# Prep data

In [ ]:
# ===============================================================
# (A) PREP BLOCK — build Temporal 5-frame sequences with 138 features
#     Using: mean, std, delta, range, mean_abs_vel, vel_std
#     => 6 * 23 = 138 features (if you have 23 metrics/features)
# ===============================================================

import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)

# ---------
# Choose the temporal aggregations (6 blocks => 138 if F=23)
# ---------
TEMPORAL_AGG = ("mean", "std", "delta", "range", "mean_abs_vel", "vel_std")
SEQ_LEN = 5

def aggregate_window(window: np.ndarray, use=TEMPORAL_AGG) -> np.ndarray:
    """
    window: (T, F) array
    return: (K*F,) aggregated vector
    """
    feats = []

    if "mean" in use:
        feats.append(np.nanmean(window, axis=0))
    if "std" in use:
        feats.append(np.nanstd(window, axis=0))
    if "delta" in use:
        feats.append(window[-1] - window[0])
    if "range" in use:
        feats.append(np.nanmax(window, axis=0) - np.nanmin(window, axis=0))

    if "mean_abs_vel" in use or "vel_std" in use:
        diff = np.diff(window, axis=0)  # (T-1, F)
        if "mean_abs_vel" in use:
            feats.append(np.nanmean(np.abs(diff), axis=0))
        if "vel_std" in use:
            feats.append(np.nanstd(diff, axis=0))

    return np.concatenate(feats, axis=0)

def build_sequences_for_app(app_df: pd.DataFrame,
                           feature_cols,
                           label_col="Temporal",
                           seq_len=5,
                           agg=TEMPORAL_AGG):
    """
    Sliding window over rows within ONE app.
    Label rule: center frame label (i + seq_len//2)
    Aggregation: aggregate_window() per metric.
    """
    # Sort by EntryID so the window is temporal
    # (works for numeric IDs and timestamp-like IDs)
    app_df = app_df.sort_values("EntryID").reset_index(drop=True)

    X = app_df[feature_cols].values
    y = app_df[label_col].values.astype(int)

    if len(app_df) < seq_len:
        return np.empty((0, len(agg)*len(feature_cols))), np.empty((0,), dtype=int)

    X_seq, y_seq = [], []
    mid = seq_len // 2

    for i in range(len(app_df) - seq_len + 1):
        window = X[i:i+seq_len]  # (T,F)
        X_seq.append(aggregate_window(window, use=agg))
        y_seq.append(int(y[i + mid]))

    return np.asarray(X_seq), np.asarray(y_seq)

# ---------
# Diagnostics: expected temporal feature dimension
# ---------
print("TEMPORAL_AGG blocks:", TEMPORAL_AGG, "| blocks =", len(TEMPORAL_AGG))
print("SEQ_LEN:", SEQ_LEN)

# After you load df and compute feature_cols in your pipeline:
# print("Frame-level feature count F:", len(feature_cols))
# print("Expected seq-level feature count:", len(TEMPORAL_AGG)*len(feature_cols))

TEMPORAL_AGG blocks: ('mean', 'std', 'delta', 'range', 'mean_abs_vel', 'vel_std') | blocks = 6
SEQ_LEN: 5


In [ ]:
# ===============================================================
# PRE-CELL: Build df_all by reading the Google Sheets tabs
# Creates: df_all with a SheetName column
# ===============================================================

from google.colab import auth, drive
auth.authenticate_user()

import gspread
from google.auth import default
import pandas as pd
import numpy as np

creds, _ = default()
gc = gspread.authorize(creds)

drive.mount("/content/drive", force_remount=False)

# Your sheet list (must match your big cell)
SHEET_NAMES = [
    # PhantomLimb train
    "PhantomLimb_Data",
    "PhantomLimb-Data-Extra-person1-1fps",
    "PhantomLimb-Data-Extra-person1-3fps",
    "PhantomLimb-Data-Extra-person1-5fps",
    "PhantomLimb-Data-Extra-person2-1fps",
    "PhantomLimb-Data-Extra-person2-3fps",
    "PhantomLimb-Data-Extra-person2-5fps",
    "PhantomLimb-Data-Extra-person3-5fps",
    "PhantomLimb-Data-Extra-person4-1fps",
    "PhantomLimb-Data-Extra-person4-3fps",
    "PhantomLimb-Data-Extra-person4-5fps",
    "PhantomLimb-Data-Extra-person5-1fps",
    "PhantomLimb-Data-Extra-person5-3fps",
    "PhantomLimb-Data-Extra-person5-5fps",
    "PhantomLimb-Data-Extra-person6-1fps",
    "PhantomLimb-Data-Extra-person6-3fps",
    "PhantomLimb-Data-Extra-person6-5fps",

    # PhantomLimb holdout/test
    "PhantomLimb-Data-Test-person1-5fps-black",
    "PhantomLimb-Data-Test-person2-3fps-black",
    "PhantomLimb-Data-Test-person2-5fps-black",
    "PhantomLimb-Data-Test-person3-1fps-black",
    "PhantomLimb-Data-Test-person3-1fps-white",
    "PhantomLimb-Data-Test-person3-5fps-white",

    # Other apps
    "PianoTiles_Data_p1_5fps",
    "archery_data_5fps_p1_01",
    "archery_data_5fps_p1_02",
    "archery_data_5fps_p1_03",
    "puzzle_data_5fps_p1_02",
    "gameover_sea_data_5fps_p1_01",
    "gameover_war_data_5fps_p1_01",
]

def load_sheet_to_df(sheet_name):
    ws = gc.open(sheet_name).sheet1
    rows = ws.get_all_values()
    if len(rows) < 2:
        raise RuntimeError(f"[{sheet_name}] sheet seems empty or missing header.")
    df = pd.DataFrame(rows[1:], columns=rows[0])
    df["SheetName"] = sheet_name
    return df

dfs = []
for name in SHEET_NAMES:
    df = load_sheet_to_df(name)
    dfs.append(df)
    print(f"[OK] loaded sheet: {name:<45} rows={len(df)}")

df_all = pd.concat(dfs, ignore_index=True)

print("\n[ALL] df_all shape:", df_all.shape)
if "Spatial" in df_all.columns:
    print("[ALL] Spatial dist (raw strings):", df_all["Spatial"].value_counts().head(10).to_dict())
if "Temporal" in df_all.columns:
    print("[ALL] Temporal dist (raw strings):", df_all["Temporal"].value_counts().head(10).to_dict())

# Optional sanity checks
if "EntryID" not in df_all.columns and "entryid" in df_all.columns:
    print("[WARN] Found 'entryid' but not 'EntryID' (your later code expects EntryID).")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[OK] loaded sheet: PhantomLimb_Data                              rows=1300
[OK] loaded sheet: PhantomLimb-Data-Extra-person1-1fps           rows=180
[OK] loaded sheet: PhantomLimb-Data-Extra-person1-3fps           rows=168
[OK] loaded sheet: PhantomLimb-Data-Extra-person1-5fps           rows=150
[OK] loaded sheet: PhantomLimb-Data-Extra-person2-1fps           rows=100
[OK] loaded sheet: PhantomLimb-Data-Extra-person2-3fps           rows=123
[OK] loaded sheet: PhantomLimb-Data-Extra-person2-5fps           rows=150
[OK] loaded sheet: PhantomLimb-Data-Extra-person3-5fps           rows=175
[OK] loaded sheet: PhantomLimb-Data-Extra-person4-1fps           rows=140
[OK] loaded sheet: PhantomLimb-Data-Extra-person4-3fps           rows=229
[OK] loaded sheet: PhantomLimb-Data-Extra-person4-5fps           rows=325
[OK] loaded sheet: PhantomLimb-Data-Extra-person5-1fps  

In [ ]:
# ===============================================================
# LOAD ALL SHEETS -> df_all  (6242 rows)
# LOAD IMAGES (.npy) from the SAME Drive locations as before
# FIX alignment:
#   - per-dataset npy stack gives 4942 rows
#   - BIG file preprocessed_images_grey.npy gives 1300 rows
#   - if (df_all - per_dataset) == BIG, prepend BIG => 6242
# OUTPUT:
#   - df_all (6242, ...)
#   - images_112 (6242, 112, 112, 1)
# ===============================================================

import os
import numpy as np
import pandas as pd
import cv2 as cv

from google.colab import drive

# -----------------------------
# 0) Mount drive (same as before)
# -----------------------------
drive.mount("/content/drive", force_remount=False)

# -----------------------------
# 1) Helpers: paths + resizing
# -----------------------------
def first_existing_path(paths):
    for p in paths:
        if p and os.path.exists(p):
            return p
    return None

def candidate_paths_for_npy(filename):
    # keep this list short and "same-drive-style" as your old notebooks
    return [
        f"/content/drive/MyDrive/VR_AR_testing/Data_Logger_and_Scripts/processed_img/{filename}",
        f"/content/drive/MyDrive/VR_Archery/archery01/processed_img/{filename}",
        f"/content/drive/MyDrive/VR_Archery/archery02/processed_img/{filename}",
        f"/content/drive/MyDrive/VR_Archery/archery03/processed_img/{filename}",
        f"/content/drive/MyDrive/VR_Puzzle/puzzle02/processed_img/{filename}",
        f"/content/drive/MyDrive/VR_GameOver_Sea/gameover_sea01/processed_img/{filename}",
        f"/content/drive/MyDrive/VR_GameOver_War/gameover_war01/processed_img/{filename}",
    ]

def ensure_nhwc1(arr):
    # expect (N,H,W) or (N,H,W,1)
    if arr.ndim == 3:
        arr = arr[..., None]
    if arr.ndim != 4 or arr.shape[-1] != 1:
        raise ValueError(f"Expected grayscale NHWC with C=1, got {arr.shape}")
    return arr

def resize_to(images_nhwc1, target_hw=(112, 112)):
    # images_nhwc1: (N,H,W,1) float or uint8
    N, H, W, C = images_nhwc1.shape
    out = np.empty((N, target_hw[0], target_hw[1], 1), dtype=np.float32)

    for i in range(N):
        img = images_nhwc1[i, :, :, 0]
        # cv2 expects (W,H) order in resize target: (width,height)
        img_r = cv.resize(img, (target_hw[1], target_hw[0]), interpolation=cv.INTER_AREA)
        out[i, :, :, 0] = img_r

    # normalize if looks like 0..255
    mx = out.max()
    if mx > 1.5:
        out = out / 255.0
    return out.astype(np.float32)

# -----------------------------
# 2) Load ALL sheets -> df_all
#     (This assumes you already know the Google Sheet name/path used in your .py file)
# -----------------------------
# If your old notebook loads from a Google Sheet via gspread, keep that same method.
# Here’s a minimal "load by local CSV" fallback ONLY if you already exported one:
#
# df_all = pd.read_csv("/content/drive/MyDrive/.../your_all_data.csv")
#
# BUT since you already ran code that prints "[OK] loaded sheet: ...", you likely have:
# - a Google Sheet with multiple tabs (one per dataset)
# - code that reads each tab and concatenates
#
# So: paste your existing "load sheets" block here.
#
# For completeness, I’m including a version that loads from an Excel file with sheets
# IF that’s what your script does. If not, replace this with your exact loader.

# ---------- EDIT THIS (ONE LINE) ----------
SHEETS_XLSX_PATH = None  # e.g., "/content/drive/MyDrive/VR_AR_testing/.../all_data_sheets.xlsx"
# ----------------------------------------

def load_all_sheets_xlsx(xlsx_path, sheet_names):
    xls = pd.ExcelFile(xlsx_path)
    dfs = []
    for name in sheet_names:
        df = pd.read_excel(xls, sheet_name=name)
        df["SheetName"] = name
        dfs.append(df)
        print(f"[OK] loaded sheet: {name:<45} rows={len(df)}")
    return pd.concat(dfs, ignore_index=True)

# Your sheet list (exactly what you posted)
SHEET_NAMES = [
    # PhantomLimb train
    "PhantomLimb_Data",
    "PhantomLimb-Data-Extra-person1-1fps",
    "PhantomLimb-Data-Extra-person1-3fps",
    "PhantomLimb-Data-Extra-person1-5fps",
    "PhantomLimb-Data-Extra-person2-1fps",
    "PhantomLimb-Data-Extra-person2-3fps",
    "PhantomLimb-Data-Extra-person2-5fps",
    "PhantomLimb-Data-Extra-person3-5fps",
    "PhantomLimb-Data-Extra-person4-1fps",
    "PhantomLimb-Data-Extra-person4-3fps",
    "PhantomLimb-Data-Extra-person4-5fps",
    "PhantomLimb-Data-Extra-person5-1fps",
    "PhantomLimb-Data-Extra-person5-3fps",
    "PhantomLimb-Data-Extra-person5-5fps",
    "PhantomLimb-Data-Extra-person6-1fps",
    "PhantomLimb-Data-Extra-person6-3fps",
    "PhantomLimb-Data-Extra-person6-5fps",

    # PhantomLimb holdout/test
    "PhantomLimb-Data-Test-person1-5fps-black",
    "PhantomLimb-Data-Test-person2-3fps-black",
    "PhantomLimb-Data-Test-person2-5fps-black",
    "PhantomLimb-Data-Test-person3-1fps-black",
    "PhantomLimb-Data-Test-person3-1fps-white",
    "PhantomLimb-Data-Test-person3-5fps-white",

    # Other apps
    "PianoTiles_Data_p1_5fps",
    "archery_data_5fps_p1_01",
    "archery_data_5fps_p1_02",
    "archery_data_5fps_p1_03",
    "puzzle_data_5fps_p1_02",
    "gameover_sea_data_5fps_p1_01",
    "gameover_war_data_5fps_p1_01",
]

# If you already have df_all from your earlier cell, comment this whole block out.
if "df_all" not in globals():
    if SHEETS_XLSX_PATH is None:
        raise RuntimeError(
            "df_all not found in globals AND SHEETS_XLSX_PATH is None.\n"
            "Either:\n"
            "  (A) run your existing sheet-loading cell that creates df_all, OR\n"
            "  (B) set SHEETS_XLSX_PATH to the .xlsx that contains these sheet tabs."
        )
    df_all = load_all_sheets_xlsx(SHEETS_XLSX_PATH, SHEET_NAMES)

print("\n[ALL] df_all shape:", df_all.shape)
if "Temporal" in df_all.columns:
    print("[ALL] Temporal dist:", df_all["Temporal"].value_counts().to_dict())

# -----------------------------
# 3) Load images from Drive (same locations as before)
#    - Try BIG file first (1300)
#    - Load per-dataset files (4942)
#    - If missing == BIG rows, prepend BIG to reach 6242
# -----------------------------
BIG_NAME = "preprocessed_images_grey.npy"
BIG_PATH = first_existing_path(candidate_paths_for_npy(BIG_NAME))
if BIG_PATH:
    print("[INFO] Found BIG images file:", BIG_PATH)
else:
    print("[INFO] BIG images file not found in common locations.")

# Per-dataset npy file names (exact filenames you used in the old notebook)
PER_DATASET_FILES = [
    # PhantomLimb train (EXTRAS ONLY, since PhantomLimb_Data likely covered by BIG)
    "img_grey_PL_p1_5fps.npy",
    "img_grey_PL_p1_1fps.npy",
    "img_grey_PL_p1_3fps.npy",
    "img_grey_PL_p2_5fps.npy",
    "img_grey_PL_p2_3fps.npy",
    "img_grey_PL_p2_1fps.npy",
    "img_grey_PL_p3_5fps.npy",
    "img_grey_PL_p4_1fps.npy",
    "img_grey_PL_p4_3fps.npy",
    "img_grey_PL_p4_5fps.npy",
    "img_grey_PL_p5_1fps.npy",
    "img_grey_PL_p5_3fps.npy",
    "img_grey_PL_p5_5fps.npy",
    "img_grey_PL_p6_1fps.npy",
    "img_grey_PL_p6_3fps.npy",
    "img_grey_PL_p6_5fps.npy",

    # PhantomLimb holdout/test
    "img_grey_PL_H_p1_5fps-black.npy",
    "img_grey_PL_H_p2_5fps-black.npy",
    "img_grey_PL_H_p2_3fps-black.npy",
    "img_grey_PL_H_p3_1fps-black.npy",
    "img_grey_PL_H_p3_1fps-white.npy",
    "img_grey_PL_H_p3_5fps-white.npy",

    # Other apps
    "img_grey_PT_p1_5fps.npy",
    "img_grey_AR_p1_5fps_01.npy",
    "img_grey_AR_p1_5fps_02.npy",
    "img_grey_AR_p1_5fps_03.npy",
    "img_grey_PUZ_p1_5fps_02.npy",
    "img_grey_GO_SEA_p1_5fps_01.npy",
    "img_grey_GO_WAR_p1_5fps_01.npy",
]

images_list = []
for fn in PER_DATASET_FILES:
    p = first_existing_path(candidate_paths_for_npy(fn))
    if p is None:
        raise FileNotFoundError(f"Missing required per-dataset image file in Drive locations: {fn}")
    arr = np.load(p)
    arr = ensure_nhwc1(arr)
    images_list.append(arr)
    print(f"[OK] {os.path.basename(p):<40} rows={len(arr)}")

images_all = np.concatenate(images_list, axis=0)
print("[ALL] concatenated images (per-dataset only):", images_all.shape)

# -----------------------------
# 4) FIX alignment using BIG file (1300) if needed
# -----------------------------
missing = len(df_all) - len(images_all)
print("[INFO] df_all rows:", len(df_all), "| images_all rows:", len(images_all), "| missing:", missing)

if missing != 0:
    if BIG_PATH is None:
        raise FileNotFoundError(
            f"images_all has {len(images_all)} rows but df_all has {len(df_all)} rows, "
            "and BIG preprocessed_images_grey.npy was not found."
        )

    big = np.load(BIG_PATH)
    big = ensure_nhwc1(big)
    print("[INFO] BIG rows:", len(big), "| BIG shape:", big.shape)

    # If the gap equals BIG, prepend BIG (PhantomLimb_Data loaded first)
    if missing == len(big):
        print("[FIX] Adding BIG images to fill the missing rows (likely PhantomLimb_Data=1300).")
        images_all = np.concatenate([big, images_all], axis=0)
        print("[OK] images_all after adding BIG:", images_all.shape)
    else:
        raise AssertionError(
            f"Mismatch cannot be resolved safely.\n"
            f"  df_all rows   = {len(df_all)}\n"
            f"  images_all    = {len(images_all)}\n"
            f"  missing       = {missing}\n"
            f"  BIG rows      = {len(big)}\n"
            f"Expected missing == BIG rows (1300) for your case."
        )

# -----------------------------
# 5) Resize to 112 and final assert
# -----------------------------
images_112 = resize_to(images_all, target_hw=(112, 112))
print("[FINAL] images_112:", images_112.shape)
print("[CHECK] df_all rows:", len(df_all), "| images_112 rows:", len(images_112))
assert len(images_112) == len(df_all), "Mismatch: images rows != df_all rows"
print("[OK] images_112 aligned with df_all")

# Done: you now have:
#   df_all    (6242, ...)
#   images_112 (6242, 112, 112, 1)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

[ALL] df_all shape: (6242, 619)
[ALL] Temporal dist: {'0': 3875, '1': 2367}
[INFO] Found BIG images file: /content/drive/MyDrive/VR_AR_testing/Data_Logger_and_Scripts/processed_img/preprocessed_images_grey.npy
[OK] img_grey_PL_p1_5fps.npy                  rows=150
[OK] img_grey_PL_p1_1fps.npy                  rows=180
[OK] img_grey_PL_p1_3fps.npy                  rows=168
[OK] img_grey_PL_p2_5fps.npy                  rows=150
[OK] img_grey_PL_p2_3fps.npy                  rows=123
[OK] img_grey_PL_p2_1fps.npy                  rows=100
[OK] img_grey_PL_p3_5fps.npy                  rows=175
[OK] img_grey_PL_p4_1fps.npy                  rows=140
[OK] img_grey_PL_p4_3fps.npy                  rows=229
[OK] img_grey_PL_p4_5fps.npy                  rows=325
[OK] img_grey_PL_p5_1fps.npy                  rows=75
[OK] img_grey_PL_p5_3fps.npy                  rows=152
[

In [ ]:
import numpy as np
import pandas as pd
def make_image_sequence_dataset(df_ordered, images_112, label_col="Temporal", seq_len=5):
    assert len(df_ordered) == len(images_112), "df/images mismatch"

    y = df_ordered[label_col].astype(int).values
    center = seq_len // 2

    X_seq, y_seq = [], []

    for i in range(len(df_ordered) - seq_len + 1):
        X_seq.append(images_112[i:i+seq_len])   # (5,112,112,1)
        y_seq.append(y[i + center])              # center label

    return np.stack(X_seq, axis=0), np.array(y_seq, dtype=int)

SEQ_LEN = 5

X_img_all = []
y_all = []

for sheet_name, g in df_all.groupby("SheetName"):   # ← change ONLY if your column differs
    # order frames temporally inside each dataset
    g = g.sort_values("EntryID")

    # get the aligned image rows
    imgs_g = images_112[g.index.values]

    if len(g) < SEQ_LEN:
        continue

    Xg, yg = make_image_sequence_dataset(
        g,
        imgs_g,
        label_col="Temporal",
        seq_len=SEQ_LEN
    )

    X_img_all.append(Xg)
    y_all.append(yg)

X_img_seq = np.concatenate(X_img_all, axis=0)
y_seq     = np.concatenate(y_all, axis=0)

print("✅ Image sequences:", X_img_seq.shape)
print("✅ Labels:", y_seq.shape)
print("Label distribution:", dict(zip(*np.unique(y_seq, return_counts=True))))

✅ Image sequences: (6122, 5, 112, 112, 1)
✅ Labels: (6122,)
Label distribution: {np.int64(0): np.int64(3780), np.int64(1): np.int64(2342)}


# Raports

In [ ]:
import pandas as pd
import numpy as np

FEATURES = [
    'missing_joints_count','missing_joints_ratio','collapsed_joints_count',
    'center_of_mass_x','center_of_mass_y','center_of_mass_z',
    'distance_from_origin',
    'bbox_width','bbox_height','bbox_depth','bbox_volume',
    'max_joint_distance_from_com',
    'distance_from_floor','below_floor',
    'left_forearm_length','right_forearm_length',
    'left_shin_length','right_shin_length',
    'arm_length_symmetry','leg_length_symmetry',
    'body_forward_x','body_forward_y','body_forward_z'
]

def safe_pearson_corr(x, y, min_n=10):
    """Returns pearson r, or np.nan if undefined."""
    x = pd.to_numeric(pd.Series(x), errors="coerce")
    y = pd.to_numeric(pd.Series(y), errors="coerce")

    m = x.notna() & y.notna()
    x = x[m].astype(float).values
    y = y[m].astype(float).values

    if len(x) < min_n:
        return np.nan
    if np.std(x) == 0 or np.std(y) == 0:
        return np.nan

    # stable pearson
    r = np.corrcoef(x, y)[0, 1]
    return float(r)

def per_app_corr(df, label, min_n=10):
    rows = []
    apps = sorted(df["App"].dropna().unique().tolist())
    for app in apps:
        dfa = df[df["App"] == app]
        # ensure label numeric
        y = pd.to_numeric(dfa[label], errors="coerce")
        for f in FEATURES:
            x = pd.to_numeric(dfa[f], errors="coerce")
            r = safe_pearson_corr(x, y, min_n=min_n)
            rows.append({"App": app, "Feature": f, "Corr": r, "N_used": int((x.notna() & y.notna()).sum())})
    return pd.DataFrame(rows)

# Run
corr_spatial  = per_app_corr(df, "Spatial",  min_n=10)
corr_temporal = per_app_corr(df, "Temporal", min_n=10)

# Pivot tables for inspection
pivot_spatial  = corr_spatial.pivot(index="Feature", columns="App", values="Corr")
pivot_temporal = corr_temporal.pivot(index="Feature", columns="App", values="Corr")

print("Spatial corr table shape:", pivot_spatial.shape)
print("Temporal corr table shape:", pivot_temporal.shape)

# Optional: show strongest |corr| per app (top 8)
def top_corrs(corr_df, app, k=8):
    d = corr_df[corr_df["App"] == app].copy()
    d["abs"] = d["Corr"].abs()
    return d.sort_values("abs", ascending=False).head(k)[["Feature","Corr","N_used"]]

for app in sorted(df["App"].unique()):
    print("\n===", app, "Spatial top corrs ===")
    print(top_corrs(corr_spatial, app, k=8).to_string(index=False))


def per_app_constant_features(df, eps=1e-12):
    out = []
    for app in sorted(df["App"].unique()):
        dfa = df[df["App"] == app]
        for f in FEATURES:
            x = pd.to_numeric(dfa[f], errors="coerce").dropna().astype(float).values
            if len(x) == 0:
                out.append((app, f, "all_nan"))
            elif np.std(x) <= eps:
                out.append((app, f, "constant"))
    return pd.DataFrame(out, columns=["App","Feature","Issue"])

const_df = per_app_constant_features(df)
print(const_df.head(50))
print("\nCounts:\n", const_df.groupby(["App","Issue"]).size())

Spatial corr table shape: (23, 6)
Temporal corr table shape: (23, 6)

=== Archery Spatial top corrs ===
             Feature      Corr  N_used
    left_shin_length -0.583391     100
      body_forward_x  0.505786     100
    center_of_mass_x -0.491096     100
distance_from_origin -0.472625     100
    center_of_mass_z -0.435250     100
      body_forward_z  0.358173     100
 leg_length_symmetry -0.323245     100
right_forearm_length  0.216497     100

=== PhantomLimb Spatial top corrs ===
             Feature      Corr  N_used
          bbox_width  0.404916     325
    left_shin_length  0.289470     325
         bbox_height -0.250010     325
   right_shin_length  0.223479     325
distance_from_origin  0.215984     325
    center_of_mass_z -0.207993     325
right_forearm_length -0.205070     325
      body_forward_z  0.186565     325

=== PianoTiles Spatial top corrs ===
                    Feature      Corr  N_used
           center_of_mass_y  0.627201     440
                bbox_heig

In [ ]:
print("\n=== Per-app label balance ===")
for app in sorted(df["App"].unique()):
    sub = df[df["App"] == app]
    print(f"\nAPP={app}  N={len(sub)}")
    print("  Spatial:", sub["Spatial"].value_counts().to_dict())
    print("  Temporal:", sub["Temporal"].value_counts().to_dict())

print("\n=== Majority baseline (per app) ===")
for app in sorted(df["App"].unique()):
    sub = df[df["App"] == app]
    for lab in ["Spatial", "Temporal"]:
        vc = sub[lab].value_counts()
        maj_acc = (vc.max() / vc.sum()) if len(vc) else np.nan
        print(f"APP={app:12s} label={lab:8s} majority_acc={maj_acc:.3f} dist={vc.to_dict()}")


=== Per-app label balance ===

APP=Archery  N=100
  Spatial: {1: 86, 0: 14}
  Temporal: {1: 86, 0: 14}

APP=PhantomLimb  N=325
  Spatial: {0: 293, 1: 32}
  Temporal: {0: 190, 1: 135}

APP=PianoTiles  N=440
  Spatial: {0: 228, 1: 212}
  Temporal: {0: 386, 1: 54}

APP=Puzzle  N=299
  Spatial: {1: 213, 0: 86}
  Temporal: {1: 213, 0: 86}

APP=Sea  N=300
  Spatial: {0: 168, 1: 132}
  Temporal: {0: 164, 1: 136}

APP=War  N=280
  Spatial: {0: 176, 1: 104}
  Temporal: {0: 164, 1: 116}

=== Majority baseline (per app) ===
APP=Archery      label=Spatial  majority_acc=0.860 dist={1: 86, 0: 14}
APP=Archery      label=Temporal majority_acc=0.860 dist={1: 86, 0: 14}
APP=PhantomLimb  label=Spatial  majority_acc=0.902 dist={0: 293, 1: 32}
APP=PhantomLimb  label=Temporal majority_acc=0.585 dist={0: 190, 1: 135}
APP=PianoTiles   label=Spatial  majority_acc=0.518 dist={0: 228, 1: 212}
APP=PianoTiles   label=Temporal majority_acc=0.877 dist={0: 386, 1: 54}
APP=Puzzle       label=Spatial  majority_acc=0.7


# LR/HGB Spatial

In [ ]:
# =========================
# 0) INSTALL / IMPORTS
# =========================
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier

SEED = 42
np.random.seed(SEED)

# =========================
# 1) LOAD DATA
# =========================
CANDIDATE_PATHS = [
    "/content/extracted_metrics_all_apps.csv",
    "/content/drive/MyDrive/extracted_metrics_all_apps.csv",
    "/content/drive/MyDrive/VR_Metrics/extracted_metrics_all_apps.csv",
]

DATA_PATH = None
for p in CANDIDATE_PATHS:
    if os.path.exists(p):
        DATA_PATH = p
        break

if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find extracted_metrics_all_apps.csv in common locations.\n"
        "Put it in /content OR adjust CANDIDATE_PATHS."
    )

df = pd.read_csv(DATA_PATH)
print("Loaded:", DATA_PATH, "| shape:", df.shape)

# =========================
# 2) CLEAN / NORMALIZE TYPES
# =========================
df = df.replace("", np.nan)

required = {"App", "Spatial", "Temporal"}
missing_req = required - set(df.columns)
if missing_req:
    raise ValueError(f"Missing required columns in CSV: {missing_req}")

# Convert bool-like strings to 0/1 BEFORE pd.to_numeric()
BOOL_MAP = {"True": 1, "False": 0, "TRUE": 1, "FALSE": 0, True: 1, False: 0}
df = df.replace(BOOL_MAP)

# Force both labels present (your extraction pipeline should guarantee this)
df = df.dropna(subset=["App", "Spatial", "Temporal"]).copy()

# Normalize labels to int
for col in ["Spatial", "Temporal"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Drop any rows that became NaN after coercion (safety)
df = df.dropna(subset=["Spatial", "Temporal"]).copy()
df["Spatial"] = df["Spatial"].astype(int)
df["Temporal"] = df["Temporal"].astype(int)

apps = sorted(df["App"].unique().tolist())
print("Apps:", apps)
print("Label balance (Spatial):", df["Spatial"].value_counts(dropna=False).to_dict())
print("Label balance (Temporal):", df["Temporal"].value_counts(dropna=False).to_dict())

# =========================
# 3) FEATURE COLUMN SELECTION
# =========================
ID_COLS = [c for c in ["GlobalID", "EntryID"] if c in df.columns]
DROP_COLS = ["App", "Spatial", "Temporal"] + ID_COLS
feature_cols = [c for c in df.columns if c not in DROP_COLS]

# Coerce features to numeric (booleans already mapped)
for c in feature_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

print("Num features:", len(feature_cols))
print("Example feature cols:", feature_cols[:15])

# =========================
# 4) MODELS (two baselines)
# =========================
# A) Tree-ish model (strong baseline)
hgb_model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("clf", HistGradientBoostingClassifier(
        random_state=SEED,
        max_depth=6,
        learning_rate=0.05,
        max_iter=400,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=20
    ))
])

# B) Linear baseline (good sanity check)
lr_model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        random_state=SEED,
        max_iter=2000,
        class_weight="balanced",
        n_jobs=-1
    ))
])

MODELS = {
    "HGB": hgb_model,
    "LR_balanced": lr_model
}

# =========================
# 5) EVAL HELPERS
# =========================
def eval_binary(y_true, y_pred):
    return {
        "acc": float(accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "cm": confusion_matrix(y_true, y_pred).tolist(),
        "pred_dist": dict(zip(*np.unique(y_pred, return_counts=True))),
    }

def run_loao(label_col: str, model_name: str, model):
    results = []
    for test_app in apps:
        train_df = df[df["App"] != test_app].copy()
        test_df  = df[df["App"] == test_app].copy()

        X_train = train_df[feature_cols].values
        y_train = train_df[label_col].values

        X_test = test_df[feature_cols].values
        y_test = test_df[label_col].values

        # If training collapses to a single class, skip (can't learn a classifier)
        uniq = np.unique(y_train)
        if len(uniq) < 2:
            print(f"[{label_col} | {model_name}] HELD-OUT={test_app:12s}  "
                  f"SKIP (train has single class: {uniq})")
            continue

        # Fit on ALL training data (no wasted split)
        model.fit(X_train, y_train)

        # Evaluate on held-out app
        yhat = model.predict(X_test)
        m = eval_binary(y_test, yhat)

        row = {
            "label": label_col,
            "model": model_name,
            "held_out_app": test_app,
            "n_train": int(len(y_train)),
            "n_test": int(len(y_test)),
            **{k: v for k, v in m.items() if k not in ["cm", "pred_dist"]},
            "cm": m["cm"],
            "pred_dist": m["pred_dist"],
        }

        print(f"[{label_col} | {model_name}] HELD-OUT={test_app:12s}  "
              f"acc={row['acc']:.3f} f1={row['f1']:.3f}  "
              f"prec={row['precision']:.3f} rec={row['recall']:.3f}  "
              f"n_test={row['n_test']}")
        results.append(row)

    return pd.DataFrame(results)

# =========================
# 6) RUN LOAO FOR BOTH LABELS
# =========================
all_runs = []
for label_col in ["Spatial", "Temporal"]:
    for model_name, model in MODELS.items():
        res = run_loao(label_col, model_name, model)
        all_runs.append(res)

results_df = pd.concat(all_runs, ignore_index=True)

print("\n================ SUMMARY (mean over held-out apps) ================")
summary = results_df.groupby(["label", "model"])[["acc", "f1", "precision", "recall"]].mean().reset_index()
print(summary)

OUT_PATH = "/content/loao_metrics_results.csv"
results_df.to_csv(OUT_PATH, index=False)
print("\nSaved per-app LOAO results to:", OUT_PATH)

Loaded: /content/extracted_metrics_all_apps.csv | shape: (1744, 28)
Apps: ['Archery', 'PhantomLimb', 'PianoTiles', 'Puzzle', 'Sea', 'War']
Label balance (Spatial): {0: 965, 1: 779}
Label balance (Temporal): {0: 1004, 1: 740}
Num features: 23
Example feature cols: ['missing_joints_count', 'missing_joints_ratio', 'collapsed_joints_count', 'center_of_mass_x', 'center_of_mass_y', 'center_of_mass_z', 'distance_from_origin', 'bbox_width', 'bbox_height', 'bbox_depth', 'bbox_volume', 'max_joint_distance_from_com', 'distance_from_floor', 'below_floor', 'left_forearm_length']


/tmp/ipython-input-3144438320.py:55: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace(BOOL_MAP)


[Spatial | HGB] HELD-OUT=Archery       acc=0.860 f1=0.925  prec=0.860 rec=1.000  n_test=100
[Spatial | HGB] HELD-OUT=PhantomLimb   acc=0.480 f1=0.214  prec=0.126 rec=0.719  n_test=325
[Spatial | HGB] HELD-OUT=PianoTiles    acc=0.534 f1=0.081  prec=0.818 rec=0.042  n_test=440
[Spatial | HGB] HELD-OUT=Puzzle        acc=0.201 f1=0.000  prec=0.000 rec=0.000  n_test=299
[Spatial | HGB] HELD-OUT=Sea           acc=0.907 f1=0.886  prec=0.956 rec=0.826  n_test=300
[Spatial | HGB] HELD-OUT=War           acc=0.671 f1=0.685  prec=0.532 rec=0.962  n_test=280
[Spatial | LR_balanced] HELD-OUT=Archery       acc=0.860 f1=0.925  prec=0.860 rec=1.000  n_test=100
[Spatial | LR_balanced] HELD-OUT=PhantomLimb   acc=0.332 f1=0.069  prec=0.040 rec=0.250  n_test=325
[Spatial | LR_balanced] HELD-OUT=PianoTiles    acc=0.500 f1=0.000  prec=0.000 rec=0.000  n_test=440
[Spatial | LR_balanced] HELD-OUT=Puzzle        acc=0.288 f1=0.000  prec=0.000 rec=0.000  n_test=299
[Spatial | LR_balanced] HELD-OUT=Sea           a

In [ ]:
# ===============================================================
# NOTE (IMPORTANT): PhantomLimb dominates the dataset (~4105 rows)
# compared to other apps (~280–440 rows). To keep LOAO evaluation
# app-fair and reduce training bias, we CAP PhantomLimb to N rows
# (default: 400) via deterministic sampling (SEED).
#
# This script prints BEFORE/AFTER per-app row counts so we can
# confirm the cap is applied.
# ===============================================================

# =========================
# 0) IMPORTS / SEED
# =========================
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier

SEED = 42
np.random.seed(SEED)

# =========================
# 1) LOAD DATA
# =========================
CANDIDATE_PATHS = [
    "/content/extracted_metrics_all_apps.csv",
    "/content/drive/MyDrive/extracted_metrics_all_apps.csv",
    "/content/drive/MyDrive/VR_Metrics/extracted_metrics_all_apps.csv",
]

DATA_PATH = None
for p in CANDIDATE_PATHS:
    if os.path.exists(p):
        DATA_PATH = p
        break

if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find extracted_metrics_all_apps.csv in common locations.\n"
        "Put it in /content OR adjust CANDIDATE_PATHS."
    )

df = pd.read_csv(DATA_PATH)
print("Loaded:", DATA_PATH, "| shape:", df.shape)

# =========================
# 2) CLEAN / NORMALIZE TYPES
# =========================
df = df.replace("", np.nan)

required = {"App", "Spatial", "Temporal"}
missing_req = required - set(df.columns)
if missing_req:
    raise ValueError(f"Missing required columns in CSV: {missing_req}")

# Convert bool-like strings to 0/1 BEFORE numeric coercion
BOOL_MAP = {"True": 1, "False": 0, "TRUE": 1, "FALSE": 0, True: 1, False: 0}
df = df.replace(BOOL_MAP)

# Require both labels present
df = df.dropna(subset=["App", "Spatial", "Temporal"]).copy()

# Normalize labels to int
for col in ["Spatial", "Temporal"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
df = df.dropna(subset=["Spatial", "Temporal"]).copy()

df["Spatial"] = df["Spatial"].astype(int)
df["Temporal"] = df["Temporal"].astype(int)

# Print counts BEFORE capping
print("\n================ PER-APP COUNTS (BEFORE CAP) ================")
before_counts = df["App"].value_counts().sort_index()
print(before_counts.to_string())
print("TOTAL (before):", int(len(df)))

# =========================
# 2.5) CAP PhantomLimb (FAIRNESS CONTROL)
# =========================
PHANTOMLIMB_CAP = 400  # <-- change to 300 or 400 as desired

print("\n===============================================================")
print(f"Applying PhantomLimb cap: PHANTOMLIMB_CAP = {PHANTOMLIMB_CAP} (SEED={SEED})")
print("===============================================================\n")

df_parts = []
for app, g in df.groupby("App"):
    if app == "PhantomLimb" and len(g) > PHANTOMLIMB_CAP:
        g = g.sample(n=PHANTOMLIMB_CAP, random_state=SEED)
    df_parts.append(g)

df = pd.concat(df_parts, ignore_index=True)

# Print counts AFTER capping
print("================ PER-APP COUNTS (AFTER CAP) ================")
after_counts = df["App"].value_counts().sort_index()
print(after_counts.to_string())
print("TOTAL (after):", int(len(df)))

apps = sorted(df["App"].unique().tolist())
print("\nApps:", apps)
print("Label balance (Spatial):", df["Spatial"].value_counts(dropna=False).to_dict())
print("Label balance (Temporal):", df["Temporal"].value_counts(dropna=False).to_dict())

# =========================
# 3) FEATURE COLUMN SELECTION
# =========================
ID_COLS = [c for c in ["GlobalID", "EntryID"] if c in df.columns]
DROP_COLS = ["App", "Spatial", "Temporal"] + ID_COLS
feature_cols = [c for c in df.columns if c not in DROP_COLS]

# Coerce features to numeric
for c in feature_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

print("\nNum features:", len(feature_cols))
print("Example feature cols:", feature_cols[:15])

# =========================
# 4) MODELS (two baselines)
# =========================
hgb_model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("clf", HistGradientBoostingClassifier(
        random_state=SEED,
        max_depth=6,
        learning_rate=0.05,
        max_iter=400,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=20
    ))
])

lr_model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        random_state=SEED,
        max_iter=2000,
        class_weight="balanced",
        n_jobs=-1
    ))
])

MODELS = {
    "HGB": hgb_model,
    "LR_balanced": lr_model
}

# =========================
# 5) EVAL HELPERS
# =========================
def eval_binary(y_true, y_pred):
    return {
        "acc": float(accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "cm": confusion_matrix(y_true, y_pred).tolist(),
        "pred_dist": dict(zip(*np.unique(y_pred, return_counts=True))),
    }

def run_loao(label_col: str, model_name: str, model):
    results = []
    for test_app in apps:
        train_df = df[df["App"] != test_app].copy()
        test_df  = df[df["App"] == test_app].copy()

        X_train = train_df[feature_cols].values
        y_train = train_df[label_col].values

        X_test = test_df[feature_cols].values
        y_test = test_df[label_col].values

        # If training collapses to a single class, skip (can't learn a classifier)
        uniq = np.unique(y_train)
        if len(uniq) < 2:
            print(f"[{label_col} | {model_name}] HELD-OUT={test_app:12s}  "
                  f"SKIP (train has single class: {uniq})")
            continue

        # Fit on ALL training data
        model.fit(X_train, y_train)

        # Evaluate on held-out app
        yhat = model.predict(X_test)
        m = eval_binary(y_test, yhat)

        row = {
            "label": label_col,
            "model": model_name,
            "held_out_app": test_app,
            "n_train": int(len(y_train)),
            "n_test": int(len(y_test)),
            **{k: v for k, v in m.items() if k not in ["cm", "pred_dist"]},
            "cm": m["cm"],
            "pred_dist": m["pred_dist"],
        }

        print(f"[{label_col} | {model_name}] HELD-OUT={test_app:12s}  "
              f"acc={row['acc']:.3f} f1={row['f1']:.3f}  "
              f"prec={row['precision']:.3f} rec={row['recall']:.3f}  "
              f"n_test={row['n_test']}")
        results.append(row)

    return pd.DataFrame(results)

# =========================
# 6) RUN LOAO FOR BOTH LABELS
# =========================
all_runs = []
for label_col in ["Spatial", "Temporal"]:
    for model_name, model in MODELS.items():
        res = run_loao(label_col, model_name, model)
        all_runs.append(res)

results_df = pd.concat(all_runs, ignore_index=True)

print("\n================ SUMMARY (mean over held-out apps) ================")
summary = results_df.groupby(["label", "model"])[["acc", "f1", "precision", "recall"]].mean().reset_index()
print(summary)

OUT_PATH = "/content/loao_metrics_results.csv"
results_df.to_csv(OUT_PATH, index=False)
print("\nSaved per-app LOAO results to:", OUT_PATH)

Loaded: /content/extracted_metrics_all_apps.csv | shape: (1744, 28)

================ PER-APP COUNTS (BEFORE CAP) ================
App
Archery        100
PhantomLimb    325
PianoTiles     440
Puzzle         299
Sea            300
War            280
TOTAL (before): 1744

Applying PhantomLimb cap: PHANTOMLIMB_CAP = 400 (SEED=42)

================ PER-APP COUNTS (AFTER CAP) ================
App
Archery        100
PhantomLimb    325
PianoTiles     440
Puzzle         299
Sea            300
War            280
TOTAL (after): 1744

Apps: ['Archery', 'PhantomLimb', 'PianoTiles', 'Puzzle', 'Sea', 'War']
Label balance (Spatial): {0: 965, 1: 779}
Label balance (Temporal): {0: 1004, 1: 740}

Num features: 23
Example feature cols: ['missing_joints_count', 'missing_joints_ratio', 'collapsed_joints_count', 'center_of_mass_x', 'center_of_mass_y', 'center_of_mass_z', 'distance_from_origin', 'bbox_width', 'bbox_height', 'bbox_depth', 'bbox_volume', 'max_joint_distance_from_com', 'distance_from_floor', 'b

/tmp/ipython-input-3292830653.py:65: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace(BOOL_MAP)


[Spatial | HGB] HELD-OUT=Archery       acc=0.860 f1=0.925  prec=0.860 rec=1.000  n_test=100
[Spatial | HGB] HELD-OUT=PhantomLimb   acc=0.551 f1=0.291  prec=0.172 rec=0.938  n_test=325
[Spatial | HGB] HELD-OUT=PianoTiles    acc=0.534 f1=0.113  prec=0.684 rec=0.061  n_test=440
[Spatial | HGB] HELD-OUT=Puzzle        acc=0.288 f1=0.000  prec=0.000 rec=0.000  n_test=299
[Spatial | HGB] HELD-OUT=Sea           acc=0.867 f1=0.829  prec=0.951 rec=0.735  n_test=300
[Spatial | HGB] HELD-OUT=War           acc=0.686 f1=0.692  prec=0.544 rec=0.952  n_test=280
[Spatial | LR_balanced] HELD-OUT=Archery       acc=0.860 f1=0.925  prec=0.860 rec=1.000  n_test=100
[Spatial | LR_balanced] HELD-OUT=PhantomLimb   acc=0.332 f1=0.069  prec=0.040 rec=0.250  n_test=325
[Spatial | LR_balanced] HELD-OUT=PianoTiles    acc=0.500 f1=0.000  prec=0.000 rec=0.000  n_test=440
[Spatial | LR_balanced] HELD-OUT=Puzzle        acc=0.288 f1=0.000  prec=0.000 rec=0.000  n_test=299
[Spatial | LR_balanced] HELD-OUT=Sea           a

# LR/HGB Temoporal

In [ ]:
# ===============================================================
# (B) MODEL BLOCK — LOAO:
#     - Spatial: frame-level (same as before)
#     - Temporal: sequence-level (SEQ_LEN=5, 138 features if F=23)
# ===============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier

SEED = 42

# --- Models (same idea as before) ---
hgb_model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("clf", HistGradientBoostingClassifier(
        random_state=SEED,
        max_depth=6,
        learning_rate=0.05,
        max_iter=400,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=20
    ))
])

lr_model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        random_state=SEED,
        max_iter=2000,
        class_weight="balanced",
        n_jobs=-1
    ))
])

MODELS = {
    "HGB": hgb_model,
    "LR_balanced": lr_model
}

def eval_binary(y_true, y_pred):
    return {
        "acc": float(accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "cm": confusion_matrix(y_true, y_pred).tolist(),
        "pred_dist": dict(zip(*np.unique(y_pred, return_counts=True))),
    }

def build_sequences_from_df(df_in: pd.DataFrame, feature_cols, label_col="Temporal"):
    """
    Build sequences for multiple apps and concatenate.
    Prevents windows crossing app boundaries by doing groupby("App").
    """
    X_list, y_list = [], []
    for app, g in df_in.groupby("App"):
        X_a, y_a = build_sequences_for_app(
            g, feature_cols,
            label_col=label_col,
            seq_len=SEQ_LEN,
            agg=TEMPORAL_AGG
        )
        if len(y_a) > 0:
            X_list.append(X_a)
            y_list.append(y_a)

    if not X_list:
        return np.empty((0, len(TEMPORAL_AGG)*len(feature_cols))), np.empty((0,), dtype=int)

    return np.vstack(X_list), np.concatenate(y_list)

def run_loao(label_col: str, model_name: str, model, df: pd.DataFrame, feature_cols):
    apps = sorted(df["App"].unique().tolist())
    results = []

    for test_app in apps:
        train_df = df[df["App"] != test_app].copy()
        test_df  = df[df["App"] == test_app].copy()

        # --------------------------
        # Spatial: frame-level
        # Temporal: seq-level
        # --------------------------
        if label_col == "Temporal":
            X_train, y_train = build_sequences_from_df(train_df, feature_cols, label_col="Temporal")
            X_test,  y_test  = build_sequences_for_app(test_df, feature_cols, label_col="Temporal",
                                                       seq_len=SEQ_LEN, agg=TEMPORAL_AGG)

        else:
            X_train = train_df[feature_cols].values
            y_train = train_df[label_col].values.astype(int)
            X_test  = test_df[feature_cols].values
            y_test  = test_df[label_col].values.astype(int)

        if len(y_train) == 0 or len(y_test) == 0:
            print(f"[{label_col} | {model_name}] HELD-OUT={test_app:12s} SKIP (empty split)")
            continue

        uniq = np.unique(y_train)
        if len(uniq) < 2:
            print(f"[{label_col} | {model_name}] HELD-OUT={test_app:12s} SKIP (single-class train: {uniq})")
            continue

        model.fit(X_train, y_train)
        yhat = model.predict(X_test)
        m = eval_binary(y_test, yhat)

        row = {
            "label": label_col,
            "model": model_name,
            "held_out_app": test_app,
            "n_train": int(len(y_train)),
            "n_test": int(len(y_test)),
            "acc": m["acc"],
            "f1": m["f1"],
            "precision": m["precision"],
            "recall": m["recall"],
            "cm": m["cm"],
            "pred_dist": m["pred_dist"],
        }

        # Print with extra info about seq dimensionality for Temporal
        if label_col == "Temporal":
            print(f"[{label_col} | {model_name}] HELD-OUT={test_app:12s} "
                  f"(SEQ_LEN={SEQ_LEN}, seqF={X_train.shape[1]})  "
                  f"acc={row['acc']:.3f} f1={row['f1']:.3f} "
                  f"prec={row['precision']:.3f} rec={row['recall']:.3f} "
                  f"n_test={row['n_test']}")
        else:
            print(f"[{label_col} | {model_name}] HELD-OUT={test_app:12s}  "
                  f"acc={row['acc']:.3f} f1={row['f1']:.3f} "
                  f"prec={row['precision']:.3f} rec={row['recall']:.3f} "
                  f"n_test={row['n_test']}")

        results.append(row)

    return pd.DataFrame(results)

# =========================
# RUN BOTH LABELS
# =========================
all_runs = []
for label_col in ["Spatial", "Temporal"]:
    for model_name, model in MODELS.items():
        res = run_loao(label_col, model_name, model, df, feature_cols)
        all_runs.append(res)

results_df = pd.concat(all_runs, ignore_index=True)

print("\n================ SUMMARY (mean over held-out apps) ================")
summary = results_df.groupby(["label", "model"])[["acc", "f1", "precision", "recall"]].mean().reset_index()
print(summary)

OUT_PATH = "/content/loao_metrics_results_seq_temporal.csv"
results_df.to_csv(OUT_PATH, index=False)
print("\nSaved per-app LOAO results to:", OUT_PATH)

[Spatial | HGB] HELD-OUT=Archery       acc=0.860 f1=0.925 prec=0.860 rec=1.000 n_test=100
[Spatial | HGB] HELD-OUT=PhantomLimb   acc=0.551 f1=0.291 prec=0.172 rec=0.938 n_test=325
[Spatial | HGB] HELD-OUT=PianoTiles    acc=0.534 f1=0.113 prec=0.684 rec=0.061 n_test=440
[Spatial | HGB] HELD-OUT=Puzzle        acc=0.288 f1=0.000 prec=0.000 rec=0.000 n_test=299
[Spatial | HGB] HELD-OUT=Sea           acc=0.867 f1=0.829 prec=0.951 rec=0.735 n_test=300
[Spatial | HGB] HELD-OUT=War           acc=0.686 f1=0.692 prec=0.544 rec=0.952 n_test=280
[Spatial | LR_balanced] HELD-OUT=Archery       acc=0.860 f1=0.925 prec=0.860 rec=1.000 n_test=100
[Spatial | LR_balanced] HELD-OUT=PhantomLimb   acc=0.332 f1=0.069 prec=0.040 rec=0.250 n_test=325
[Spatial | LR_balanced] HELD-OUT=PianoTiles    acc=0.500 f1=0.000 prec=0.000 rec=0.000 n_test=440
[Spatial | LR_balanced] HELD-OUT=Puzzle        acc=0.288 f1=0.000 prec=0.000 rec=0.000 n_test=299
[Spatial | LR_balanced] HELD-OUT=Sea           acc=0.617 f1=0.228 pr

/tmp/ipython-input-68296431.py:27: RuntimeWarning: Mean of empty slice
  feats.append(np.nanmean(window, axis=0))
/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:2035: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/ipython-input-68296431.py:33: RuntimeWarning: All-NaN slice encountered
  feats.append(np.nanmax(window, axis=0) - np.nanmin(window, axis=0))
/tmp/ipython-input-68296431.py:38: RuntimeWarning: Mean of empty slice
  feats.append(np.nanmean(np.abs(diff), axis=0))
/tmp/ipython-input-68296431.py:27: RuntimeWarning: Mean of empty slice
  feats.append(np.nanmean(window, axis=0))
/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:2035: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/ipython-input-68296431.py:33: RuntimeWarning: All-NaN slice encountered
  feats.append(np.nanmax(window, axis=0) - np.

[Temporal | HGB] HELD-OUT=Archery      (SEQ_LEN=5, seqF=138)  acc=0.896 f1=0.943 prec=0.902 rec=0.988 n_test=96


/tmp/ipython-input-68296431.py:27: RuntimeWarning: Mean of empty slice
  feats.append(np.nanmean(window, axis=0))
/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:2035: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/ipython-input-68296431.py:33: RuntimeWarning: All-NaN slice encountered
  feats.append(np.nanmax(window, axis=0) - np.nanmin(window, axis=0))
/tmp/ipython-input-68296431.py:38: RuntimeWarning: Mean of empty slice
  feats.append(np.nanmean(np.abs(diff), axis=0))
/tmp/ipython-input-68296431.py:27: RuntimeWarning: Mean of empty slice
  feats.append(np.nanmean(window, axis=0))
/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:2035: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/ipython-input-68296431.py:33: RuntimeWarning: All-NaN slice encountered
  feats.append(np.nanmax(window, axis=0) - np.

[Temporal | HGB] HELD-OUT=PhantomLimb  (SEQ_LEN=5, seqF=138)  acc=0.654 f1=0.284 prec=1.000 rec=0.165 n_test=321


...

[Temporal | LR_balanced] HELD-OUT=Puzzle       (SEQ_LEN=5, seqF=138)  acc=0.586 f1=0.679 prec=0.772 rec=0.606 n_test=295


/tmp/ipython-input-68296431.py:27: RuntimeWarning: Mean of empty slice
  feats.append(np.nanmean(window, axis=0))
/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:2035: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/ipython-input-68296431.py:33: RuntimeWarning: All-NaN slice encountered
  feats.append(np.nanmax(window, axis=0) - np.nanmin(window, axis=0))
/tmp/ipython-input-68296431.py:38: RuntimeWarning: Mean of empty slice
  feats.append(np.nanmean(np.abs(diff), axis=0))
/tmp/ipython-input-68296431.py:27: RuntimeWarning: Mean of empty slice
  feats.append(np.nanmean(window, axis=0))
/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:2035: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/ipython-input-68296431.py:33: RuntimeWarning: All-NaN slice encountered
  feats.append(np.nanmax(window, axis=0) - np.

[Temporal | LR_balanced] HELD-OUT=Sea          (SEQ_LEN=5, seqF=138)  acc=0.453 f1=0.623 prec=0.453 rec=1.000 n_test=296


/tmp/ipython-input-68296431.py:27: RuntimeWarning: Mean of empty slice
  feats.append(np.nanmean(window, axis=0))
/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:2035: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/ipython-input-68296431.py:33: RuntimeWarning: All-NaN slice encountered
  feats.append(np.nanmax(window, axis=0) - np.nanmin(window, axis=0))
/tmp/ipython-input-68296431.py:38: RuntimeWarning: Mean of empty slice
  feats.append(np.nanmean(np.abs(diff), axis=0))
/tmp/ipython-input-68296431.py:27: RuntimeWarning: Mean of empty slice
  feats.append(np.nanmean(window, axis=0))
/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:2035: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/ipython-input-68296431.py:33: RuntimeWarning: All-NaN slice encountered
  feats.append(np.nanmax(window, axis=0) - np.

[Temporal | LR_balanced] HELD-OUT=War          (SEQ_LEN=5, seqF=138)  acc=0.558 f1=0.655 prec=0.487 rec=1.000 n_test=276

================ SUMMARY (mean over held-out apps) ================
      label        model       acc        f1  precision    recall
0   Spatial          HGB  0.630811  0.474986   0.535260  0.614265
1   Spatial  LR_balanced  0.542290  0.317633   0.403300  0.396465
2  Temporal          HGB  0.700241  0.528135   0.621713  0.557131
3  Temporal  LR_balanced  0.709248  0.699356   0.651604  0.816130

Saved per-app LOAO results to: /content/loao_metrics_results_seq_temporal.csv


In [ ]:
# ===============================================================
# VR APP-INDEPENDENT METRICS - LOAO BASELINES
# Patch A: Per-app label sanity + majority baseline
# Patch B: NaN-safe temporal window aggregation (removes warnings)
#
# NOTE:
# - Spatial is evaluated frame-level (single row -> label)
# - Temporal is evaluated sequence-level with sliding window (SEQ_LEN)
# - Optional: Downsample PhantomLimb to match other apps (toggle below)
# ===============================================================

import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier

SEED = 42
np.random.seed(SEED)

# =========================
# 1) LOAD DATA
# =========================
CANDIDATE_PATHS = [
    "/content/extracted_metrics_all_apps.csv",
    "/content/drive/MyDrive/extracted_metrics_all_apps.csv",
    "/content/drive/MyDrive/VR_Metrics/extracted_metrics_all_apps.csv",
]

DATA_PATH = None
for p in CANDIDATE_PATHS:
    if os.path.exists(p):
        DATA_PATH = p
        break

if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find extracted_metrics_all_apps.csv in common locations.\n"
        "Put it in /content OR adjust CANDIDATE_PATHS."
    )

df = pd.read_csv(DATA_PATH)
print("\n================= LOADED DATA =================")
print("Loaded:", DATA_PATH, "| shape:", df.shape)

# =========================
# 2) CLEAN / NORMALIZE TYPES
# =========================
df = df.replace("", np.nan)

required = {"App", "Spatial", "Temporal"}
missing_req = required - set(df.columns)
if missing_req:
    raise ValueError(f"Missing required columns in CSV: {missing_req}")

for col in ["Spatial", "Temporal"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=["App"]).copy()
df = df.dropna(subset=["Spatial", "Temporal"], how="all").copy()

df["Spatial"] = df["Spatial"].astype(int)
df["Temporal"] = df["Temporal"].astype(int)

apps = sorted(df["App"].unique().tolist())
print("Apps:", apps)

# =========================
# OPTIONAL: Downsample PhantomLimb
# =========================
DOWNSAMPLE_PHANTOMLIMB = True
TARGET_N_PL = 400

if DOWNSAMPLE_PHANTOMLIMB and ("PhantomLimb" in df["App"].unique()):
    pl = df[df["App"] == "PhantomLimb"]
    others = df[df["App"] != "PhantomLimb"]
    if len(pl) > TARGET_N_PL:
        pl = pl.sample(TARGET_N_PL, random_state=SEED)
    df = pd.concat([others, pl], ignore_index=True)
    apps = sorted(df["App"].unique().tolist())
    print("\n[INFO] Downsample PhantomLimb =", DOWNSAMPLE_PHANTOMLIMB,
          "| TARGET_N_PL =", TARGET_N_PL)
    print("[INFO] PhantomLimb N =", len(df[df["App"] == "PhantomLimb"]),
          "| Total N =", len(df))

# =========================
# Patch A: Label sanity + baselines
# =========================
print("\n=== Per-app label balance ===")
for app in apps:
    sub = df[df["App"] == app]
    print(f"\nAPP={app}  N={len(sub)}")
    print("  Spatial:", sub["Spatial"].value_counts().to_dict())
    print("  Temporal:", sub["Temporal"].value_counts().to_dict())

print("\n=== Majority baseline (per app) ===")
for app in apps:
    sub = df[df["App"] == app]
    for lab in ["Spatial", "Temporal"]:
        vc = sub[lab].value_counts()
        maj_acc = (vc.max() / vc.sum()) if len(vc) else np.nan
        print(f"APP={app:12s} label={lab:8s} majority_acc={maj_acc:.3f} dist={vc.to_dict()}")

# =========================
# 3) FEATURE COLUMN SELECTION
# =========================
ID_COLS = [c for c in ["GlobalID", "EntryID"] if c in df.columns]
DROP_COLS = ["App", "Spatial", "Temporal"] + ID_COLS
feature_cols = [c for c in df.columns if c not in DROP_COLS]

# Coerce features to numeric
for c in feature_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

print("\n================= FEATURES =================")
print("Num features:", len(feature_cols))
print("First 25 feature cols:", feature_cols[:25])

# =========================
# 4) MODELS
# =========================
hgb_model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("clf", HistGradientBoostingClassifier(
        random_state=SEED,
        max_depth=6,
        learning_rate=0.05,
        max_iter=400
    ))
])

lr_model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        random_state=SEED,
        max_iter=3000,
        class_weight="balanced",
        n_jobs=-1
    ))
])

MODELS = {
    "HGB": hgb_model,
    "LR_balanced": lr_model
}

# =========================
# 5) EVAL HELPERS
# =========================
def eval_binary(y_true, y_pred):
    return {
        "acc": float(accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "cm": confusion_matrix(y_true, y_pred).tolist(),
        "pred_dist": dict(zip(*np.unique(y_pred, return_counts=True))),
    }

# ===============================================================
# Patch B: NaN-safe temporal aggregation for SEQ features
# ===============================================================
def _nanmean_safe(x, axis=0):
    with np.errstate(all="ignore"):
        return np.nanmean(x, axis=axis)

def _nanstd_safe(x, axis=0):
    with np.errstate(all="ignore"):
        return np.nanstd(x, axis=axis)

def _nanrange_safe(x, axis=0):
    with np.errstate(all="ignore"):
        mx = np.nanmax(x, axis=axis)
        mn = np.nanmin(x, axis=axis)
    return mx - mn

def window_is_valid(window, min_non_nan_ratio=0.4):
    total = window.size
    non_nan = np.isfinite(window).sum()
    return (non_nan / total) >= min_non_nan_ratio

def aggregate_window(window: np.ndarray,
                     use=("mean","std","delta","range","mean_abs_vel","vel_std")) -> np.ndarray:
    feats = []

    if "mean" in use:
        feats.append(_nanmean_safe(window, axis=0))
    if "std" in use:
        feats.append(_nanstd_safe(window, axis=0))
    if "delta" in use:
        feats.append(window[-1] - window[0])
    if "range" in use:
        feats.append(_nanrange_safe(window, axis=0))

    if "mean_abs_vel" in use or "vel_std" in use:
        diff = np.diff(window, axis=0)  # (T-1,F)
        if "mean_abs_vel" in use:
            feats.append(_nanmean_safe(np.abs(diff), axis=0))
        if "vel_std" in use:
            feats.append(_nanstd_safe(diff, axis=0))

    return np.concatenate(feats, axis=0)

def make_sequence_dataset(app_df, label_col, seq_len=5,
                          agg_use=("mean","std","delta","range","mean_abs_vel","vel_std"),
                          min_non_nan_ratio=0.4):
    """
    Converts frame-level features into sequence-level features by sliding window.
    Label is taken from center frame (like your earlier temporal labeling).
    """
    X = app_df[feature_cols].values.astype(float)
    y = app_df[label_col].values.astype(int)

    if len(app_df) < seq_len:
        return np.empty((0, 1)), np.empty((0,), dtype=int)

    X_seq = []
    y_seq = []
    center = seq_len // 2

    for i in range(len(app_df) - seq_len + 1):
        window = X[i:i+seq_len]

        # drop too-empty windows to avoid all-NaN aggregates
        if not window_is_valid(window, min_non_nan_ratio=min_non_nan_ratio):
            continue

        feat = aggregate_window(window, use=agg_use)
        X_seq.append(feat)
        y_seq.append(y[i + center])

    if len(X_seq) == 0:
        return np.empty((0, 1)), np.empty((0,), dtype=int)

    return np.vstack(X_seq), np.array(y_seq, dtype=int)

# =========================
# 6) LOAO: Spatial (frame-level)
# =========================
def run_loao_frame(label_col: str, model_name: str, model):
    results = []
    for test_app in apps:
        train_df = df[df["App"] != test_app].copy()
        test_df  = df[df["App"] == test_app].copy()

        X_train_full = train_df[feature_cols].values
        y_train_full = train_df[label_col].values

        X_test = test_df[feature_cols].values
        y_test = test_df[label_col].values

        # Inner val split for debugging (optional)
        X_tr, X_val, y_tr, y_val = train_test_split(
            X_train_full, y_train_full,
            test_size=0.15,
            random_state=SEED,
            stratify=y_train_full
        )

        model.fit(X_tr, y_tr)
        yhat = model.predict(X_test)
        m = eval_binary(y_test, yhat)

        row = {
            "label": label_col,
            "task": "frame",
            "model": model_name,
            "held_out_app": test_app,
            "n_train": int(len(y_tr)),
            "n_val": int(len(y_val)),
            "n_test": int(len(y_test)),
            **{k: v for k, v in m.items() if k not in ["cm", "pred_dist"]},
            "cm": m["cm"],
            "pred_dist": m["pred_dist"],
        }

        print(f"[{label_col} | {model_name}] HELD-OUT={test_app:12s}  "
              f"acc={row['acc']:.3f} f1={row['f1']:.3f}  "
              f"prec={row['precision']:.3f} rec={row['recall']:.3f}  "
              f"n_test={row['n_test']}")
        results.append(row)

    return pd.DataFrame(results)

# =========================
# 7) LOAO: Temporal (sequence-level)
# =========================
def run_loao_sequence(label_col: str, model_name: str, model,
                      seq_len=5,
                      agg_use=("mean","std","delta","range","mean_abs_vel","vel_std"),
                      min_non_nan_ratio=0.4):
    results = []

    # Determine final seq feature length for printing
    seq_feature_dim = len(agg_use) * len(feature_cols)

    for test_app in apps:
        train_df = df[df["App"] != test_app].copy()
        test_df  = df[df["App"] == test_app].copy()

        # Build sequence datasets PER split (prevents any leakage)
        X_train_full, y_train_full = [], []
        for a in apps:
            if a == test_app:
                continue
            Xa, ya = make_sequence_dataset(
                df[df["App"] == a],
                label_col=label_col,
                seq_len=seq_len,
                agg_use=agg_use,
                min_non_nan_ratio=min_non_nan_ratio
            )
            if len(ya) > 0:
                X_train_full.append(Xa)
                y_train_full.append(ya)

        if len(X_train_full) == 0:
            print(f"[{label_col} | {model_name}] HELD-OUT={test_app}  -> No train sequences.")
            continue

        X_train_full = np.vstack(X_train_full)
        y_train_full = np.concatenate(y_train_full)

        X_test, y_test = make_sequence_dataset(
            test_df,
            label_col=label_col,
            seq_len=seq_len,
            agg_use=agg_use,
            min_non_nan_ratio=min_non_nan_ratio
        )

        if len(y_test) == 0:
            print(f"[{label_col} | {model_name}] HELD-OUT={test_app}  -> No test sequences.")
            continue

        # Inner val split
        X_tr, X_val, y_tr, y_val = train_test_split(
            X_train_full, y_train_full,
            test_size=0.15,
            random_state=SEED,
            stratify=y_train_full
        )

        model.fit(X_tr, y_tr)
        yhat = model.predict(X_test)
        m = eval_binary(y_test, yhat)

        row = {
            "label": label_col,
            "task": f"sequence_len_{seq_len}",
            "model": model_name,
            "held_out_app": test_app,
            "seq_len": int(seq_len),
            "seq_feature_dim": int(seq_feature_dim),
            "n_train": int(len(y_tr)),
            "n_val": int(len(y_val)),
            "n_test": int(len(y_test)),
            **{k: v for k, v in m.items() if k not in ["cm", "pred_dist"]},
            "cm": m["cm"],
            "pred_dist": m["pred_dist"],
        }

        print(f"[{label_col} | {model_name}] HELD-OUT={test_app:12s} "
              f"(SEQ_LEN={seq_len}, seqF={seq_feature_dim})  "
              f"acc={row['acc']:.3f} f1={row['f1']:.3f} "
              f"prec={row['precision']:.3f} rec={row['recall']:.3f} "
              f"n_test={row['n_test']}")
        results.append(row)

    return pd.DataFrame(results)

# =========================
# 8) RUN EVERYTHING
# =========================
SEQ_LEN = 5
AGG_USE = ("mean","std","delta","range","mean_abs_vel","vel_std")  # 6 * F -> e.g., 6*23=138
MIN_NON_NAN_RATIO = 0.4

all_runs = []

print("\n================= LOAO: SPATIAL (FRAME) =================")
for model_name, model in MODELS.items():
    res = run_loao_frame("Spatial", model_name, model)
    all_runs.append(res)

print("\n================= LOAO: TEMPORAL (SEQUENCE) =================")
for model_name, model in MODELS.items():
    res = run_loao_sequence(
        "Temporal", model_name, model,
        seq_len=SEQ_LEN,
        agg_use=AGG_USE,
        min_non_nan_ratio=MIN_NON_NAN_RATIO
    )
    all_runs.append(res)

results_df = pd.concat(all_runs, ignore_index=True)

print("\n================ SUMMARY (mean over held-out apps) ================")
summary = results_df.groupby(["label", "task", "model"])[["acc", "f1", "precision", "recall"]].mean().reset_index()
print(summary)

OUT_PATH = "/content/loao_metrics_results_PATCHB.csv"
results_df.to_csv(OUT_PATH, index=False)
print("\nSaved results to:", OUT_PATH)


================= LOADED DATA =================
Loaded: /content/extracted_metrics_all_apps.csv | shape: (1744, 28)
Apps: ['Archery', 'PhantomLimb', 'PianoTiles', 'Puzzle', 'Sea', 'War']

[INFO] Downsample PhantomLimb = True | TARGET_N_PL = 400
[INFO] PhantomLimb N = 325 | Total N = 1744

=== Per-app label balance ===

APP=Archery  N=100
  Spatial: {1: 86, 0: 14}
  Temporal: {1: 86, 0: 14}

APP=PhantomLimb  N=325
  Spatial: {0: 293, 1: 32}
  Temporal: {0: 190, 1: 135}

APP=PianoTiles  N=440
  Spatial: {0: 228, 1: 212}
  Temporal: {0: 386, 1: 54}

APP=Puzzle  N=299
  Spatial: {1: 213, 0: 86}
  Temporal: {1: 213, 0: 86}

APP=Sea  N=300
  Spatial: {0: 168, 1: 132}
  Temporal: {0: 164, 1: 136}

APP=War  N=280
  Spatial: {0: 176, 1: 104}
  Temporal: {0: 164, 1: 116}

=== Majority baseline (per app) ===
APP=Archery      label=Spatial  majority_acc=0.860 dist={1: 86, 0: 14}
APP=Archery      label=Temporal majority_acc=0.860 dist={1: 86, 0: 14}
APP=PhantomLimb  label=Spatial  majority_acc=0.9

/tmp/ipython-input-2067879188.py:172: RuntimeWarning: Mean of empty slice
  return np.nanmean(x, axis=axis)
/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:2035: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/ipython-input-2067879188.py:180: RuntimeWarning: All-NaN slice encountered
  mx = np.nanmax(x, axis=axis)
/tmp/ipython-input-2067879188.py:181: RuntimeWarning: All-NaN slice encountered
  mn = np.nanmin(x, axis=axis)
/tmp/ipython-input-2067879188.py:172: RuntimeWarning: Mean of empty slice
  return np.nanmean(x, axis=axis)
/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:2035: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/ipython-input-2067879188.py:180: RuntimeWarning: All-NaN slice encountered
  mx = np.nanmax(x, axis=axis)
/tmp/ipython-input-2067879188.py:181: RuntimeWarning: All-NaN slice e

[Temporal | HGB] HELD-OUT=Archery      (SEQ_LEN=5, seqF=138)  acc=0.885 f1=0.939 prec=0.884 rec=1.000 n_test=96


/tmp/ipython-input-2067879188.py:172: RuntimeWarning: Mean of empty slice
  return np.nanmean(x, axis=axis)
/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:2035: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/ipython-input-2067879188.py:180: RuntimeWarning: All-NaN slice encountered
  mx = np.nanmax(x, axis=axis)
/tmp/ipython-input-2067879188.py:181: RuntimeWarning: All-NaN slice encountered
  mn = np.nanmin(x, axis=axis)
/tmp/ipython-input-2067879188.py:172: RuntimeWarning: Mean of empty slice
  return np.nanmean(x, axis=axis)
/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:2035: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/ipython-input-2067879188.py:180: RuntimeWarning: All-NaN slice encountered
  mx = np.nanmax(x, axis=axis)
/tmp/ipython-input-2067879188.py:181: RuntimeWarning: All-NaN slice e

[Temporal | HGB] HELD-OUT=PhantomLimb  (SEQ_LEN=5, seqF=138)  acc=0.632 f1=0.253 prec=0.800 rec=0.150 n_test=321


...

[Temporal | LR_balanced] HELD-OUT=Puzzle       (SEQ_LEN=5, seqF=138)  acc=0.549 f1=0.670 prec=0.773 rec=0.592 n_test=275


/tmp/ipython-input-2067879188.py:172: RuntimeWarning: Mean of empty slice
  return np.nanmean(x, axis=axis)
/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:2035: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/ipython-input-2067879188.py:180: RuntimeWarning: All-NaN slice encountered
  mx = np.nanmax(x, axis=axis)
/tmp/ipython-input-2067879188.py:181: RuntimeWarning: All-NaN slice encountered
  mn = np.nanmin(x, axis=axis)
/tmp/ipython-input-2067879188.py:172: RuntimeWarning: Mean of empty slice
  return np.nanmean(x, axis=axis)
/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:2035: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/ipython-input-2067879188.py:180: RuntimeWarning: All-NaN slice encountered
  mx = np.nanmax(x, axis=axis)
/tmp/ipython-input-2067879188.py:181: RuntimeWarning: All-NaN slice e

[Temporal | LR_balanced] HELD-OUT=Sea          (SEQ_LEN=5, seqF=138)  acc=0.456 f1=0.625 prec=0.454 rec=1.000 n_test=296


/tmp/ipython-input-2067879188.py:172: RuntimeWarning: Mean of empty slice
  return np.nanmean(x, axis=axis)
/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:2035: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/ipython-input-2067879188.py:180: RuntimeWarning: All-NaN slice encountered
  mx = np.nanmax(x, axis=axis)
/tmp/ipython-input-2067879188.py:181: RuntimeWarning: All-NaN slice encountered
  mn = np.nanmin(x, axis=axis)
/tmp/ipython-input-2067879188.py:172: RuntimeWarning: Mean of empty slice
  return np.nanmean(x, axis=axis)
/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:2035: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/ipython-input-2067879188.py:180: RuntimeWarning: All-NaN slice encountered
  mx = np.nanmax(x, axis=axis)
/tmp/ipython-input-2067879188.py:181: RuntimeWarning: All-NaN slice e

[Temporal | LR_balanced] HELD-OUT=War          (SEQ_LEN=5, seqF=138)  acc=0.533 f1=0.641 prec=0.473 rec=0.991 n_test=276

================ SUMMARY (mean over held-out apps) ================
      label            task        model       acc        f1  precision  \
0   Spatial           frame          HGB  0.607209  0.414001   0.456190   
1   Spatial           frame  LR_balanced  0.547156  0.338287   0.405481   
2  Temporal  sequence_len_5          HGB  0.670631  0.490174   0.549573   
3  Temporal  sequence_len_5  LR_balanced  0.704574  0.706668   0.669461   

     recall  
0  0.489744  
1  0.423453  
2  0.522079  
3  0.815432  

Saved results to: /content/loao_metrics_results_PATCHB.csv


In [ ]:
# ===============================================================
# VR APP-INDEPENDENT METRICS - LOAO BASELINES
# Patch A: Per-app label sanity + majority baseline
# Patch B: NaN-safe temporal window aggregation (removes warnings)
#
# NOTE:
# - Spatial is evaluated frame-level (single row -> label)
# - Temporal is evaluated sequence-level with sliding window (SEQ_LEN)
# - Optional: Downsample PhantomLimb to match other apps (toggle below)
# ===============================================================

import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier

SEED = 42
np.random.seed(SEED)

# =========================
# 1) LOAD DATA
# =========================
CANDIDATE_PATHS = [
    "/content/extracted_metrics_all_apps.csv",
    "/content/drive/MyDrive/extracted_metrics_all_apps.csv",
    "/content/drive/MyDrive/VR_Metrics/extracted_metrics_all_apps.csv",
]

DATA_PATH = None
for p in CANDIDATE_PATHS:
    if os.path.exists(p):
        DATA_PATH = p
        break

if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find extracted_metrics_all_apps.csv in common locations.\n"
        "Put it in /content OR adjust CANDIDATE_PATHS."
    )

df = pd.read_csv(DATA_PATH)
print("\n================= LOADED DATA =================")
print("Loaded:", DATA_PATH, "| shape:", df.shape)

# =========================
# 2) CLEAN / NORMALIZE TYPES
# =========================
df = df.replace("", np.nan)

required = {"App", "Spatial", "Temporal"}
missing_req = required - set(df.columns)
if missing_req:
    raise ValueError(f"Missing required columns in CSV: {missing_req}")

for col in ["Spatial", "Temporal"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=["App"]).copy()
df = df.dropna(subset=["Spatial", "Temporal"], how="all").copy()

df["Spatial"] = df["Spatial"].astype(int)
df["Temporal"] = df["Temporal"].astype(int)

apps = sorted(df["App"].unique().tolist())
print("Apps:", apps)

# =========================
# OPTIONAL: Downsample PhantomLimb
# =========================
DOWNSAMPLE_PHANTOMLIMB = True
TARGET_N_PL = 400

if DOWNSAMPLE_PHANTOMLIMB and ("PhantomLimb" in df["App"].unique()):
    pl = df[df["App"] == "PhantomLimb"]
    others = df[df["App"] != "PhantomLimb"]
    if len(pl) > TARGET_N_PL:
        pl = pl.sample(TARGET_N_PL, random_state=SEED)
    df = pd.concat([others, pl], ignore_index=True)
    apps = sorted(df["App"].unique().tolist())
    print("\n[INFO] Downsample PhantomLimb =", DOWNSAMPLE_PHANTOMLIMB,
          "| TARGET_N_PL =", TARGET_N_PL)
    print("[INFO] PhantomLimb N =", len(df[df["App"] == "PhantomLimb"]),
          "| Total N =", len(df))

# =========================
# Patch A: Label sanity + baselines
# =========================
print("\n=== Per-app label balance ===")
for app in apps:
    sub = df[df["App"] == app]
    print(f"\nAPP={app}  N={len(sub)}")
    print("  Spatial:", sub["Spatial"].value_counts().to_dict())
    print("  Temporal:", sub["Temporal"].value_counts().to_dict())

print("\n=== Majority baseline (per app) ===")
for app in apps:
    sub = df[df["App"] == app]
    for lab in ["Spatial", "Temporal"]:
        vc = sub[lab].value_counts()
        maj_acc = (vc.max() / vc.sum()) if len(vc) else np.nan
        print(f"APP={app:12s} label={lab:8s} majority_acc={maj_acc:.3f} dist={vc.to_dict()}")

# =========================
# 3) FEATURE COLUMN SELECTION
# =========================
ID_COLS = [c for c in ["GlobalID", "EntryID"] if c in df.columns]
DROP_COLS = ["App", "Spatial", "Temporal"] + ID_COLS
feature_cols = [c for c in df.columns if c not in DROP_COLS]

# Coerce features to numeric
for c in feature_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

print("\n================= FEATURES =================")
print("Num features:", len(feature_cols))
print("First 25 feature cols:", feature_cols[:25])

# =========================
# 4) MODELS
# =========================
hgb_model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("clf", HistGradientBoostingClassifier(
        random_state=SEED,
        max_depth=6,
        learning_rate=0.05,
        max_iter=400
    ))
])

lr_model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        random_state=SEED,
        max_iter=3000,
        class_weight="balanced",
        n_jobs=-1
    ))
])

MODELS = {
    "HGB": hgb_model,
    "LR_balanced": lr_model
}

# =========================
# 5) EVAL HELPERS
# =========================
def eval_binary(y_true, y_pred):
    return {
        "acc": float(accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "cm": confusion_matrix(y_true, y_pred).tolist(),
        "pred_dist": dict(zip(*np.unique(y_pred, return_counts=True))),
    }

# =========================
# Patch B FIX: warning-free NaN reducers
# =========================
import numpy as np

def nanmean_no_warn(x, axis=0):
    x = np.asarray(x, dtype=float)
    valid = np.isfinite(x)
    denom = valid.sum(axis=axis)
    # sum with NaNs turned to 0
    num = np.where(valid, x, 0.0).sum(axis=axis)
    out = np.divide(num, denom, out=np.full_like(num, np.nan, dtype=float), where=(denom > 0))
    return out

def nanstd_no_warn(x, axis=0):
    x = np.asarray(x, dtype=float)
    mu = nanmean_no_warn(x, axis=axis)
    # broadcast mu to x shape
    mu_b = np.expand_dims(mu, axis=axis)
    diff2 = (x - mu_b) ** 2
    valid = np.isfinite(diff2)
    denom = valid.sum(axis=axis)
    num = np.where(valid, diff2, 0.0).sum(axis=axis)
    var = np.divide(num, denom, out=np.full_like(num, np.nan, dtype=float), where=(denom > 0))
    return np.sqrt(var)

def nanrange_no_warn(x, axis=0):
    x = np.asarray(x, dtype=float)
    valid = np.isfinite(x)
    # Replace NaNs with +/- inf so min/max ignore them
    x_for_max = np.where(valid, x, -np.inf)
    x_for_min = np.where(valid, x, +np.inf)

    mx = np.max(x_for_max, axis=axis)
    mn = np.min(x_for_min, axis=axis)

    # If a column is all-NaN, mx will be -inf and mn will be +inf -> set to NaN
    all_nan = ~valid.any(axis=axis)
    out = mx - mn
    out = np.where(all_nan, np.nan, out)
    return out

def window_is_valid(window, min_non_nan_ratio=0.4):
    total = window.size
    non_nan = np.isfinite(window).sum()
    return (non_nan / total) >= min_non_nan_ratio

def aggregate_window(window, use=("mean","std","delta","range","mean_abs_vel","vel_std")):
    window = np.asarray(window, dtype=float)
    feats = []

    if "mean" in use:
        feats.append(nanmean_no_warn(window, axis=0))
    if "std" in use:
        feats.append(nanstd_no_warn(window, axis=0))
    if "delta" in use:
        feats.append(window[-1] - window[0])
    if "range" in use:
        feats.append(nanrange_no_warn(window, axis=0))

    if "mean_abs_vel" in use or "vel_std" in use:
        diff = np.diff(window, axis=0)  # (T-1,F)
        if "mean_abs_vel" in use:
            feats.append(nanmean_no_warn(np.abs(diff), axis=0))
        if "vel_std" in use:
            feats.append(nanstd_no_warn(diff, axis=0))

    return np.concatenate(feats, axis=0)

def make_sequence_dataset(app_df, label_col, seq_len=5,
                          agg_use=("mean","std","delta","range","mean_abs_vel","vel_std"),
                          min_non_nan_ratio=0.4):
    """
    Converts frame-level features into sequence-level features by sliding window.
    Label is taken from center frame (like your earlier temporal labeling).
    """
    X = app_df[feature_cols].values.astype(float)
    y = app_df[label_col].values.astype(int)

    if len(app_df) < seq_len:
        return np.empty((0, 1)), np.empty((0,), dtype=int)

    X_seq = []
    y_seq = []
    center = seq_len // 2

    for i in range(len(app_df) - seq_len + 1):
        window = X[i:i+seq_len]

        # drop too-empty windows to avoid all-NaN aggregates
        if not window_is_valid(window, min_non_nan_ratio=min_non_nan_ratio):
            continue

        feat = aggregate_window(window, use=agg_use)
        X_seq.append(feat)
        y_seq.append(y[i + center])

    if len(X_seq) == 0:
        return np.empty((0, 1)), np.empty((0,), dtype=int)

    return np.vstack(X_seq), np.array(y_seq, dtype=int)

# =========================
# 6) LOAO: Spatial (frame-level)
# =========================
def run_loao_frame(label_col: str, model_name: str, model):
    results = []
    for test_app in apps:
        train_df = df[df["App"] != test_app].copy()
        test_df  = df[df["App"] == test_app].copy()

        X_train_full = train_df[feature_cols].values
        y_train_full = train_df[label_col].values

        X_test = test_df[feature_cols].values
        y_test = test_df[label_col].values

        # Inner val split for debugging (optional)
        X_tr, X_val, y_tr, y_val = train_test_split(
            X_train_full, y_train_full,
            test_size=0.15,
            random_state=SEED,
            stratify=y_train_full
        )

        model.fit(X_tr, y_tr)
        yhat = model.predict(X_test)
        m = eval_binary(y_test, yhat)

        row = {
            "label": label_col,
            "task": "frame",
            "model": model_name,
            "held_out_app": test_app,
            "n_train": int(len(y_tr)),
            "n_val": int(len(y_val)),
            "n_test": int(len(y_test)),
            **{k: v for k, v in m.items() if k not in ["cm", "pred_dist"]},
            "cm": m["cm"],
            "pred_dist": m["pred_dist"],
        }

        print(f"[{label_col} | {model_name}] HELD-OUT={test_app:12s}  "
              f"acc={row['acc']:.3f} f1={row['f1']:.3f}  "
              f"prec={row['precision']:.3f} rec={row['recall']:.3f}  "
              f"n_test={row['n_test']}")
        results.append(row)

    return pd.DataFrame(results)

# =========================
# 7) LOAO: Temporal (sequence-level)
# =========================
def run_loao_sequence(label_col: str, model_name: str, model,
                      seq_len=5,
                      agg_use=("mean","std","delta","range","mean_abs_vel","vel_std"),
                      min_non_nan_ratio=0.4):
    results = []

    # Determine final seq feature length for printing
    seq_feature_dim = len(agg_use) * len(feature_cols)

    for test_app in apps:
        train_df = df[df["App"] != test_app].copy()
        test_df  = df[df["App"] == test_app].copy()

        # Build sequence datasets PER split (prevents any leakage)
        X_train_full, y_train_full = [], []
        for a in apps:
            if a == test_app:
                continue
            Xa, ya = make_sequence_dataset(
                df[df["App"] == a],
                label_col=label_col,
                seq_len=seq_len,
                agg_use=agg_use,
                min_non_nan_ratio=min_non_nan_ratio
            )
            if len(ya) > 0:
                X_train_full.append(Xa)
                y_train_full.append(ya)

        if len(X_train_full) == 0:
            print(f"[{label_col} | {model_name}] HELD-OUT={test_app}  -> No train sequences.")
            continue

        X_train_full = np.vstack(X_train_full)
        y_train_full = np.concatenate(y_train_full)

        X_test, y_test = make_sequence_dataset(
            test_df,
            label_col=label_col,
            seq_len=seq_len,
            agg_use=agg_use,
            min_non_nan_ratio=min_non_nan_ratio
        )

        if len(y_test) == 0:
            print(f"[{label_col} | {model_name}] HELD-OUT={test_app}  -> No test sequences.")
            continue

        # Inner val split
        X_tr, X_val, y_tr, y_val = train_test_split(
            X_train_full, y_train_full,
            test_size=0.15,
            random_state=SEED,
            stratify=y_train_full
        )

        model.fit(X_tr, y_tr)
        yhat = model.predict(X_test)
        m = eval_binary(y_test, yhat)

        row = {
            "label": label_col,
            "task": f"sequence_len_{seq_len}",
            "model": model_name,
            "held_out_app": test_app,
            "seq_len": int(seq_len),
            "seq_feature_dim": int(seq_feature_dim),
            "n_train": int(len(y_tr)),
            "n_val": int(len(y_val)),
            "n_test": int(len(y_test)),
            **{k: v for k, v in m.items() if k not in ["cm", "pred_dist"]},
            "cm": m["cm"],
            "pred_dist": m["pred_dist"],
        }

        print(f"[{label_col} | {model_name}] HELD-OUT={test_app:12s} "
              f"(SEQ_LEN={seq_len}, seqF={seq_feature_dim})  "
              f"acc={row['acc']:.3f} f1={row['f1']:.3f} "
              f"prec={row['precision']:.3f} rec={row['recall']:.3f} "
              f"n_test={row['n_test']}")
        results.append(row)

    return pd.DataFrame(results)

# =========================
# 8) RUN EVERYTHING
# =========================
SEQ_LEN = 5
AGG_USE = ("mean","std","delta","range","mean_abs_vel","vel_std")  # 6 * F -> e.g., 6*23=138
MIN_NON_NAN_RATIO = 0.4

all_runs = []

print("\n================= LOAO: SPATIAL (FRAME) =================")
for model_name, model in MODELS.items():
    res = run_loao_frame("Spatial", model_name, model)
    all_runs.append(res)

print("\n================= LOAO: TEMPORAL (SEQUENCE) =================")
for model_name, model in MODELS.items():
    res = run_loao_sequence(
        "Temporal", model_name, model,
        seq_len=SEQ_LEN,
        agg_use=AGG_USE,
        min_non_nan_ratio=MIN_NON_NAN_RATIO
    )
    all_runs.append(res)

results_df = pd.concat(all_runs, ignore_index=True)

print("\n================ SUMMARY (mean over held-out apps) ================")
summary = results_df.groupby(["label", "task", "model"])[["acc", "f1", "precision", "recall"]].mean().reset_index()
print(summary)

OUT_PATH = "/content/loao_metrics_results_PATCHB.csv"
results_df.to_csv(OUT_PATH, index=False)
print("\nSaved results to:", OUT_PATH)


================= LOADED DATA =================
Loaded: /content/extracted_metrics_all_apps.csv | shape: (1744, 28)
Apps: ['Archery', 'PhantomLimb', 'PianoTiles', 'Puzzle', 'Sea', 'War']

[INFO] Downsample PhantomLimb = True | TARGET_N_PL = 400
[INFO] PhantomLimb N = 325 | Total N = 1744

=== Per-app label balance ===

APP=Archery  N=100
  Spatial: {1: 86, 0: 14}
  Temporal: {1: 86, 0: 14}

APP=PhantomLimb  N=325
  Spatial: {0: 293, 1: 32}
  Temporal: {0: 190, 1: 135}

APP=PianoTiles  N=440
  Spatial: {0: 228, 1: 212}
  Temporal: {0: 386, 1: 54}

APP=Puzzle  N=299
  Spatial: {1: 213, 0: 86}
  Temporal: {1: 213, 0: 86}

APP=Sea  N=300
  Spatial: {0: 168, 1: 132}
  Temporal: {0: 164, 1: 136}

APP=War  N=280
  Spatial: {0: 176, 1: 104}
  Temporal: {0: 164, 1: 116}

=== Majority baseline (per app) ===
APP=Archery      label=Spatial  majority_acc=0.860 dist={1: 86, 0: 14}
APP=Archery      label=Temporal majority_acc=0.860 dist={1: 86, 0: 14}
APP=PhantomLimb  label=Spatial  majority_acc=0.9

In [ ]:
!pip -q install catboost

# MLP Spatial

In [ ]:
# ============================================================
# PATCH B - Spatial (FRAME) Neural Baseline (MLP) + LOAO
# Notes:
#  - Uses ALL numeric metrics columns (excludes App/labels/IDs)
#  - LOAO: train on 5 apps, test on held-out app
#  - Includes per-app label balance + majority baselines
#  - Uses class weights + early stopping
# ============================================================

import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# =========================
# 1) LOAD DATA
# =========================
CANDIDATE_PATHS = [
    "/content/extracted_metrics_all_apps.csv",
    "/content/drive/MyDrive/extracted_metrics_all_apps.csv",
    "/content/drive/MyDrive/VR_Metrics/extracted_metrics_all_apps.csv",
]

DATA_PATH = None
for p in CANDIDATE_PATHS:
    if os.path.exists(p):
        DATA_PATH = p
        break
if DATA_PATH is None:
    raise FileNotFoundError("Could not find extracted_metrics_all_apps.csv. Put it in /content or adjust paths.")

df = pd.read_csv(DATA_PATH)
print("\n================= LOADED DATA =================")
print("Loaded:", DATA_PATH, "| shape:", df.shape)

# =========================
# 2) BASIC CLEANING
# =========================
df = df.replace("", np.nan)

required = {"App", "Spatial", "Temporal"}
missing_req = required - set(df.columns)
if missing_req:
    raise ValueError(f"Missing required columns: {missing_req}")

df["App"] = df["App"].astype(str)

# For Spatial training we need Spatial present
df["Spatial"] = pd.to_numeric(df["Spatial"], errors="coerce")
df = df.dropna(subset=["App", "Spatial"]).copy()
df["Spatial"] = df["Spatial"].astype(int)

apps = sorted(df["App"].unique().tolist())
print("Apps:", apps)

# =========================
# 3) OPTIONAL: Downsample PhantomLimb (only if larger than others)
# =========================
DOWNSAMPLE_PHANTOMLIMB = True
TARGET_N_PL = 400  # you asked 300-400; but current PL is 325 so no change

print(f"\n[INFO] Downsample PhantomLimb = {DOWNSAMPLE_PHANTOMLIMB} | TARGET_N_PL = {TARGET_N_PL}")
if DOWNSAMPLE_PHANTOMLIMB and "PhantomLimb" in apps:
    df_pl = df[df["App"] == "PhantomLimb"]
    if len(df_pl) > TARGET_N_PL:
        df_pl = df_pl.sample(n=TARGET_N_PL, random_state=SEED)
        df_other = df[df["App"] != "PhantomLimb"]
        df = pd.concat([df_other, df_pl], ignore_index=True)
        print("[INFO] PhantomLimb was downsampled.")
print(f"[INFO] PhantomLimb N = {len(df[df['App']=='PhantomLimb']) if 'PhantomLimb' in apps else 0} | Total N = {len(df)}")

# =========================
# 4) FEATURE COLUMN SELECTION (USE ALL METRICS)
# =========================
ID_COLS = [c for c in ["GlobalID", "EntryID"] if c in df.columns]
DROP_COLS = ["App", "Spatial", "Temporal"] + ID_COLS
feature_cols = [c for c in df.columns if c not in DROP_COLS]

# Coerce to numeric
for c in feature_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

print("\n================= FEATURES =================")
print("Num features:", len(feature_cols))
print("First 25 feature cols:", feature_cols[:25])

# =========================
# 5) PRINT PER-APP LABEL BALANCE + MAJORITY BASELINE
# =========================
print("\n=== Per-app label balance ===")
for a in apps:
    sub = df[df["App"] == a]
    dist = sub["Spatial"].value_counts().to_dict()
    print(f"\nAPP={a:12s}  N={len(sub)}")
    print("  Spatial:", dist)

print("\n=== Majority baseline (per app) ===")
for a in apps:
    sub = df[df["App"] == a]
    dist = sub["Spatial"].value_counts().to_dict()
    maj = max(dist, key=dist.get)
    maj_acc = dist[maj] / len(sub)
    print(f"APP={a:12s} label=Spatial majority_acc={maj_acc:.3f} dist={dist}")

# =========================
# 6) MODEL: simple MLP
# =========================
def build_mlp(input_dim: int) -> keras.Model:
    inp = keras.Input(shape=(input_dim,))
    x = layers.Dense(64, activation="relu")(inp)
    x = layers.Dropout(0.25)(x)
    x = layers.Dense(32, activation="relu")(x)
    x = layers.Dropout(0.20)(x)
    x = layers.Dense(16, activation="relu")(x)
    out = layers.Dense(1, activation="sigmoid")(x)
    model = keras.Model(inp, out)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=[keras.metrics.BinaryAccuracy(name="acc")]
    )
    return model

def eval_binary(y_true, y_pred):
    return {
        "acc": float(accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "cm": confusion_matrix(y_true, y_pred).tolist(),
        "pred_dist": dict(zip(*np.unique(y_pred, return_counts=True))),
    }

def compute_class_weight(y):
    # returns dict {0: w0, 1: w1} (balanced)
    y = np.asarray(y).astype(int)
    n0 = np.sum(y == 0)
    n1 = np.sum(y == 1)
    if n0 == 0 or n1 == 0:
        return None
    w0 = (n0 + n1) / (2.0 * n0)
    w1 = (n0 + n1) / (2.0 * n1)
    return {0: float(w0), 1: float(w1)}

# =========================
# 7) LOAO EVAL
# =========================
results = []

print("\n================= LOAO: SPATIAL (FRAME) - MLP =================")
for test_app in apps:
    train_df = df[df["App"] != test_app].copy()
    test_df  = df[df["App"] == test_app].copy()

    X_train_full = train_df[feature_cols].values
    y_train_full = train_df["Spatial"].values.astype(int)

    X_test = test_df[feature_cols].values
    y_test = test_df["Spatial"].values.astype(int)

    # train/val split inside training apps
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train_full, y_train_full,
        test_size=0.15,
        random_state=SEED,
        stratify=y_train_full
    )

    # impute + scale using TRAIN ONLY
    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()

    X_tr = imputer.fit_transform(X_tr)
    X_val = imputer.transform(X_val)
    X_test_i = imputer.transform(X_test)

    X_tr = scaler.fit_transform(X_tr)
    X_val = scaler.transform(X_val)
    X_test_s = scaler.transform(X_test_i)

    # class weights computed on training split
    cw = compute_class_weight(y_tr)

    model = build_mlp(input_dim=X_tr.shape[1])

    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=15,
            restore_best_weights=True
        )
    ]

    model.fit(
        X_tr, y_tr,
        validation_data=(X_val, y_val),
        epochs=200,
        batch_size=32,
        verbose=0,
        class_weight=cw
    )

    yhat_prob = model.predict(X_test_s, verbose=0).reshape(-1)
    yhat = (yhat_prob >= 0.5).astype(int)

    m = eval_binary(y_test, yhat)

    print(f"[Spatial | MLP] HELD-OUT={test_app:12s}  "
          f"acc={m['acc']:.3f} f1={m['f1']:.3f} prec={m['precision']:.3f} rec={m['recall']:.3f}  n_test={len(y_test)}")

    results.append({
        "label": "Spatial",
        "task": "frame",
        "model": "MLP",
        "held_out_app": test_app,
        "n_train": int(len(y_tr)),
        "n_val": int(len(y_val)),
        "n_test": int(len(y_test)),
        **{k: v for k, v in m.items() if k not in ["cm", "pred_dist"]},
        "cm": m["cm"],
        "pred_dist": m["pred_dist"],
    })

results_df = pd.DataFrame(results)

print("\n================ SUMMARY (mean over held-out apps) ================")
print(results_df[["acc","f1","precision","recall"]].mean())

OUT_PATH = "/content/loao_results_spatial_mlp.csv"
results_df.to_csv(OUT_PATH, index=False)
print("\nSaved:", OUT_PATH)


================= LOADED DATA =================
Loaded: /content/extracted_metrics_all_apps.csv | shape: (1744, 28)
Apps: ['Archery', 'PhantomLimb', 'PianoTiles', 'Puzzle', 'Sea', 'War']

[INFO] Downsample PhantomLimb = True | TARGET_N_PL = 400
[INFO] PhantomLimb N = 325 | Total N = 1744

================= FEATURES =================
Num features: 23
First 25 feature cols: ['missing_joints_count', 'missing_joints_ratio', 'collapsed_joints_count', 'center_of_mass_x', 'center_of_mass_y', 'center_of_mass_z', 'distance_from_origin', 'bbox_width', 'bbox_height', 'bbox_depth', 'bbox_volume', 'max_joint_distance_from_com', 'distance_from_floor', 'below_floor', 'left_forearm_length', 'right_forearm_length', 'left_shin_length', 'right_shin_length', 'arm_length_symmetry', 'leg_length_symmetry', 'body_forward_x', 'body_forward_y', 'body_forward_z']

=== Per-app label balance ===

APP=Archery       N=100
  Spatial: {1: 86, 0: 14}

APP=PhantomLimb   N=325
  Spatial: {0: 293, 1: 32}

APP=PianoTiles 

# 1DCNN Temporal

In [ ]:
# ============================================================
# PATCH B - Temporal (SEQUENCE) Neural Baseline (GRU) + LOAO
# Notes:
#  - Builds sliding windows per app: X shape = (N-4, 5, F)
#  - Label for a window = center frame label (i+2)
#  - Uses imputer+scaler fitted on TRAIN ONLY (frame-level), then sequences built
#  - GRU is small and regularized (safe for small data)
# ============================================================

import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

SEQ_LEN = 5
CENTER = SEQ_LEN // 2  # 2 for seq_len=5

# =========================
# 1) LOAD DATA
# =========================
CANDIDATE_PATHS = [
    "/content/extracted_metrics_all_apps.csv",
    "/content/drive/MyDrive/extracted_metrics_all_apps.csv",
    "/content/drive/MyDrive/VR_Metrics/extracted_metrics_all_apps.csv",
]

DATA_PATH = None
for p in CANDIDATE_PATHS:
    if os.path.exists(p):
        DATA_PATH = p
        break
if DATA_PATH is None:
    raise FileNotFoundError("Could not find extracted_metrics_all_apps.csv. Put it in /content or adjust paths.")

df = pd.read_csv(DATA_PATH)
print("\n================= LOADED DATA =================")
print("Loaded:", DATA_PATH, "| shape:", df.shape)

df = df.replace("", np.nan)
df["App"] = df["App"].astype(str)

# Need Temporal present for temporal training
df["Temporal"] = pd.to_numeric(df["Temporal"], errors="coerce")
df = df.dropna(subset=["App", "Temporal"]).copy()
df["Temporal"] = df["Temporal"].astype(int)

apps = sorted(df["App"].unique().tolist())
print("Apps:", apps)

# =========================
# 2) FEATURE COLUMN SELECTION (USE ALL METRICS)
# =========================
ID_COLS = [c for c in ["GlobalID", "EntryID"] if c in df.columns]
DROP_COLS = ["App", "Spatial", "Temporal"] + ID_COLS
feature_cols = [c for c in df.columns if c not in DROP_COLS]

for c in feature_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

print("\n================= FEATURES =================")
print("Num features:", len(feature_cols))
print("First 25 feature cols:", feature_cols[:25])

# =========================
# 3) SEQUENCE BUILDER (per app)
# =========================
def make_sequences(X_frame, y_frame, seq_len=5):
    n = len(X_frame)
    if n < seq_len:
        return np.empty((0, seq_len, X_frame.shape[1])), np.empty((0,), dtype=int)
    Xs, ys = [], []
    for i in range(n - seq_len + 1):
        Xs.append(X_frame[i:i+seq_len])
        ys.append(int(y_frame[i + (seq_len//2)]))
    return np.asarray(Xs), np.asarray(ys)

def eval_binary(y_true, y_pred):
    return {
        "acc": float(accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "cm": confusion_matrix(y_true, y_pred).tolist(),
        "pred_dist": dict(zip(*np.unique(y_pred, return_counts=True))),
    }

def compute_class_weight(y):
    y = np.asarray(y).astype(int)
    n0 = np.sum(y == 0)
    n1 = np.sum(y == 1)
    if n0 == 0 or n1 == 0:
        return None
    w0 = (n0 + n1) / (2.0 * n0)
    w1 = (n0 + n1) / (2.0 * n1)
    return {0: float(w0), 1: float(w1)}

# =========================
# 4) MODEL: small GRU
# =========================
def build_gru(seq_len: int, feat_dim: int) -> keras.Model:
    inp = keras.Input(shape=(seq_len, feat_dim))
    x = layers.GRU(32, dropout=0.25, recurrent_dropout=0.0, return_sequences=False)(inp)
    x = layers.Dense(32, activation="relu")(x)
    x = layers.Dropout(0.25)(x)
    out = layers.Dense(1, activation="sigmoid")(x)
    model = keras.Model(inp, out)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=[keras.metrics.BinaryAccuracy(name="acc")]
    )
    return model

# =========================
# 5) LOAO EVAL (Temporal sequence)
# =========================
results = []
print(f"\n================= LOAO: TEMPORAL (SEQUENCE) - GRU =================")
print(f"[INFO] SEQ_LEN={SEQ_LEN} | label = center frame (i+{CENTER})")

for test_app in apps:
    train_df = df[df["App"] != test_app].copy()
    test_df  = df[df["App"] == test_app].copy()

    # ---- Fit preprocessing on TRAIN FRAMES ONLY (good practice)
    X_train_full = train_df[feature_cols].values
    y_train_full = train_df["Temporal"].values.astype(int)

    # small validation split at frame-level before sequences
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train_full, y_train_full,
        test_size=0.15,
        random_state=SEED,
        stratify=y_train_full
    )

    imputer = SimpleImputer(strategy="median")
    scaler  = StandardScaler()

    X_tr = imputer.fit_transform(X_tr)
    X_val = imputer.transform(X_val)
    X_test_frames = imputer.transform(test_df[feature_cols].values)

    X_tr = scaler.fit_transform(X_tr)
    X_val = scaler.transform(X_val)
    X_test_frames = scaler.transform(X_test_frames)

    # ---- Now we must rebuild per-app sequences consistently.
    # For training sequences: we need scaled+imputed frames in original train_df order.
    X_train_frames_all = scaler.transform(imputer.transform(train_df[feature_cols].values))
    y_train_frames_all = train_df["Temporal"].values.astype(int)

    X_val_frames_all = X_val
    y_val_frames_all = y_val

    # Create sequences:
    # Train sequences from ALL training frames (not only X_tr) to maximize data.
    # (You can tighten this later to avoid slight leakage into validation; for now we keep it simple.)
    X_train_seq, y_train_seq = make_sequences(X_train_frames_all, y_train_frames_all, seq_len=SEQ_LEN)
    X_test_seq, y_test_seq   = make_sequences(X_test_frames, test_df["Temporal"].values.astype(int), seq_len=SEQ_LEN)

    # Validation sequences: built from X_val split (works as quick early stopping signal)
    X_val_seq, y_val_seq = make_sequences(X_val_frames_all, y_val_frames_all, seq_len=SEQ_LEN)

    # If val has too few sequences, fall back to using a small slice of train_seq as val
    if len(X_val_seq) < 20:
        k = min(200, len(X_train_seq)//5)
        X_val_seq, y_val_seq = X_train_seq[:k], y_train_seq[:k]
        X_train_seq, y_train_seq = X_train_seq[k:], y_train_seq[k:]

    cw = compute_class_weight(y_train_seq)

    model = build_gru(seq_len=SEQ_LEN, feat_dim=X_train_seq.shape[2])

    callbacks = [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=12, restore_best_weights=True)
    ]

    model.fit(
        X_train_seq, y_train_seq,
        validation_data=(X_val_seq, y_val_seq),
        epochs=120,
        batch_size=32,
        verbose=0,
        class_weight=cw
    )

    yhat_prob = model.predict(X_test_seq, verbose=0).reshape(-1)
    yhat = (yhat_prob >= 0.5).astype(int)

    m = eval_binary(y_test_seq, yhat)

    print(f"[Temporal | GRU] HELD-OUT={test_app:12s}  "
          f"(SEQ_LEN={SEQ_LEN}) acc={m['acc']:.3f} f1={m['f1']:.3f} "
          f"prec={m['precision']:.3f} rec={m['recall']:.3f}  n_test={len(y_test_seq)}")

    results.append({
        "label": "Temporal",
        "task": f"sequence_len_{SEQ_LEN}",
        "model": "GRU",
        "held_out_app": test_app,
        "n_train_seq": int(len(y_train_seq)),
        "n_val_seq": int(len(y_val_seq)),
        "n_test_seq": int(len(y_test_seq)),
        **{k: v for k, v in m.items() if k not in ["cm", "pred_dist"]},
        "cm": m["cm"],
        "pred_dist": m["pred_dist"],
    })

results_df = pd.DataFrame(results)

print("\n================ SUMMARY (mean over held-out apps) ================")
print(results_df[["acc","f1","precision","recall"]].mean())

OUT_PATH = "/content/loao_results_temporal_gru.csv"
results_df.to_csv(OUT_PATH, index=False)
print("\nSaved:", OUT_PATH)


================= LOADED DATA =================
Loaded: /content/extracted_metrics_all_apps.csv | shape: (1744, 28)
Apps: ['Archery', 'PhantomLimb', 'PianoTiles', 'Puzzle', 'Sea', 'War']

================= FEATURES =================
Num features: 23
First 25 feature cols: ['missing_joints_count', 'missing_joints_ratio', 'collapsed_joints_count', 'center_of_mass_x', 'center_of_mass_y', 'center_of_mass_z', 'distance_from_origin', 'bbox_width', 'bbox_height', 'bbox_depth', 'bbox_volume', 'max_joint_distance_from_com', 'distance_from_floor', 'below_floor', 'left_forearm_length', 'right_forearm_length', 'left_shin_length', 'right_shin_length', 'arm_length_symmetry', 'leg_length_symmetry', 'body_forward_x', 'body_forward_y', 'body_forward_z']

================= LOAO: TEMPORAL (SEQUENCE) - GRU =================
[INFO] SEQ_LEN=5 | label = center frame (i+2)
[Temporal | GRU] HELD-OUT=Archery       (SEQ_LEN=5) acc=0.875 f1=0.933 prec=0.875 rec=1.000  n_test=96
[Temporal | GRU] HELD-OUT=PhantomL

# RF Spatial

In [ ]:
# ============================================================
# PATCH: Random Forest baselines (NO SMOTE / NO OVERSAMPLING)
#  - Spatial = frame-level classification using 23 metrics
#  - Temporal will be separate block (sequence RF)
#
# Notes:
#  - Uses ALL feature columns found in CSV (excluding App/labels/IDs)
#  - Optional downsample PhantomLimb is included (prints status)
#  - LOAO = Leave-One-App-Out evaluation
# ============================================================

import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix

from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

SEED = 42
np.random.seed(SEED)

# =========================
# 1) LOAD DATA
# =========================
CANDIDATE_PATHS = [
    "/content/extracted_metrics_all_apps.csv",
    "/content/drive/MyDrive/extracted_metrics_all_apps.csv",
    "/content/drive/MyDrive/VR_Metrics/extracted_metrics_all_apps.csv",
]

DATA_PATH = None
for p in CANDIDATE_PATHS:
    if os.path.exists(p):
        DATA_PATH = p
        break

if DATA_PATH is None:
    raise FileNotFoundError("Could not find extracted_metrics_all_apps.csv. Update CANDIDATE_PATHS.")

df = pd.read_csv(DATA_PATH)
print("\n================= LOADED DATA =================")
print("Loaded:", DATA_PATH, "| shape:", df.shape)

# =========================
# 2) CLEAN / NORMALIZE TYPES
# =========================
df = df.replace("", np.nan)

required = {"App", "Spatial", "Temporal"}
missing_req = required - set(df.columns)
if missing_req:
    raise ValueError(f"Missing required columns in CSV: {missing_req}")

df["App"] = df["App"].astype(str)

# labels
for col in ["Spatial", "Temporal"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# keep rows that have Spatial label (since this is spatial block)
df = df.dropna(subset=["App", "Spatial"]).copy()
df["Spatial"] = df["Spatial"].astype(int)

apps = sorted(df["App"].unique().tolist())
print("Apps:", apps)

# =========================
# 3) OPTIONAL: Downsample PhantomLimb to be comparable
# =========================
DOWNSAMPLE_PHANTOMLIMB = True
TARGET_N_PL = 400

print(f"\n[INFO] Downsample PhantomLimb = {DOWNSAMPLE_PHANTOMLIMB} | TARGET_N_PL = {TARGET_N_PL}")

if DOWNSAMPLE_PHANTOMLIMB and "PhantomLimb" in apps:
    df_pl = df[df["App"] == "PhantomLimb"]
    n_pl = len(df_pl)
    if n_pl > TARGET_N_PL:
        # stratified downsample by Spatial label
        df_pl_down, _ = train_test_split(
            df_pl,
            train_size=TARGET_N_PL,
            random_state=SEED,
            stratify=df_pl["Spatial"]
        )
        df_other = df[df["App"] != "PhantomLimb"]
        df = pd.concat([df_other, df_pl_down], ignore_index=True)
        print(f"[INFO] PhantomLimb downsampled: {n_pl} -> {len(df_pl_down)}")
    else:
        print(f"[INFO] PhantomLimb N = {n_pl} (<= target), no downsample performed.")
else:
    print("[INFO] PhantomLimb downsample skipped.")

print(f"[INFO] Total N after downsample step = {len(df)}")

# =========================
# 4) FEATURE COLUMN SELECTION
# =========================
ID_COLS = [c for c in ["GlobalID", "EntryID"] if c in df.columns]
DROP_COLS = ["App", "Spatial", "Temporal"] + ID_COLS
feature_cols = [c for c in df.columns if c not in DROP_COLS]

# coerce features numeric
for c in feature_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

print("\n================= FEATURES =================")
print("Num features:", len(feature_cols))
print("First 25 feature cols:", feature_cols[:25])

# =========================
# 5) REPORT LABEL BALANCE + MAJORITY BASELINE
# =========================
print("\n=== Per-app label balance ===")
for app in apps:
    sub = df[df["App"] == app]
    dist = sub["Spatial"].value_counts().to_dict()
    print(f"\nAPP={app:12s} N={len(sub)}")
    print("  Spatial:", dist)

print("\n=== Majority baseline (per app) ===")
for app in apps:
    sub = df[df["App"] == app]
    dist = sub["Spatial"].value_counts().to_dict()
    maj = max(dist, key=dist.get)
    maj_acc = dist[maj] / len(sub)
    print(f"APP={app:12s} label=Spatial  majority_acc={maj_acc:.3f} dist={dist}")

# =========================
# 6) MODEL: Random Forest
# =========================
rf_model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("clf", RandomForestClassifier(
        n_estimators=400,
        max_depth=None,
        min_samples_leaf=5,
        max_features="sqrt",
        class_weight="balanced",
        random_state=SEED,
        n_jobs=-1
    ))
])

# =========================
# 7) EVAL HELPERS
# =========================
def eval_binary(y_true, y_pred):
    return {
        "acc": float(accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "cm": confusion_matrix(y_true, y_pred).tolist(),
        "pred_dist": dict(zip(*np.unique(y_pred, return_counts=True))),
    }

def run_loao_spatial_rf():
    results = []
    for test_app in apps:
        train_df = df[df["App"] != test_app].copy()
        test_df  = df[df["App"] == test_app].copy()

        X_train_full = train_df[feature_cols].values
        y_train_full = train_df["Spatial"].values
        X_test = test_df[feature_cols].values
        y_test = test_df["Spatial"].values

        # internal val split (optional, but consistent w/ your pipeline)
        X_tr, X_val, y_tr, y_val = train_test_split(
            X_train_full, y_train_full,
            test_size=0.15,
            random_state=SEED,
            stratify=y_train_full
        )

        rf_model.fit(X_tr, y_tr)
        yhat = rf_model.predict(X_test)
        m = eval_binary(y_test, yhat)

        row = {
            "label": "Spatial",
            "task": "frame",
            "model": "RF",
            "held_out_app": test_app,
            "n_train": int(len(y_tr)),
            "n_val": int(len(y_val)),
            "n_test": int(len(y_test)),
            **{k: v for k, v in m.items() if k not in ["cm", "pred_dist"]},
            "cm": m["cm"],
            "pred_dist": m["pred_dist"],
        }

        print(f"[Spatial | RF] HELD-OUT={test_app:12s} "
              f"acc={row['acc']:.3f} f1={row['f1']:.3f} "
              f"prec={row['precision']:.3f} rec={row['recall']:.3f} "
              f"n_test={row['n_test']} pred_dist={row['pred_dist']}")
        results.append(row)

    return pd.DataFrame(results)

# =========================
# 8) RUN
# =========================
print("\n================= LOAO: SPATIAL (FRAME) - RF =================")
results_df = run_loao_spatial_rf()

print("\n================ SUMMARY (mean over held-out apps) ================")
summary = results_df[["acc", "f1", "precision", "recall"]].mean()
print(summary)

OUT_PATH = "/content/loao_results_spatial_rf.csv"
results_df.to_csv(OUT_PATH, index=False)
print("\nSaved:", OUT_PATH)


================= LOADED DATA =================
Loaded: /content/extracted_metrics_all_apps.csv | shape: (1744, 28)
Apps: ['Archery', 'PhantomLimb', 'PianoTiles', 'Puzzle', 'Sea', 'War']

[INFO] Downsample PhantomLimb = True | TARGET_N_PL = 400
[INFO] PhantomLimb N = 325 (<= target), no downsample performed.
[INFO] Total N after downsample step = 1744

================= FEATURES =================
Num features: 23
First 25 feature cols: ['missing_joints_count', 'missing_joints_ratio', 'collapsed_joints_count', 'center_of_mass_x', 'center_of_mass_y', 'center_of_mass_z', 'distance_from_origin', 'bbox_width', 'bbox_height', 'bbox_depth', 'bbox_volume', 'max_joint_distance_from_com', 'distance_from_floor', 'below_floor', 'left_forearm_length', 'right_forearm_length', 'left_shin_length', 'right_shin_length', 'arm_length_symmetry', 'leg_length_symmetry', 'body_forward_x', 'body_forward_y', 'body_forward_z']

=== Per-app label balance ===

APP=Archery      N=100
  Spatial: {1: 86, 0: 14}

APP

# RF Temporal

In [ ]:
# ============================================================
# PATCH: Temporal Random Forest on SEQUENCES (NO SMOTE)
#  - Build seq_len=5 windows per app
#  - Flatten frames -> RF input (seq_len * num_features)
#  - Label = center frame (i + seq_len//2)
#  - LOAO evaluation
# ============================================================

import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix

from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

SEED = 42
np.random.seed(SEED)

SEQ_LEN = 5
CENTER = SEQ_LEN // 2

# =========================
# 1) LOAD DATA
# =========================
CANDIDATE_PATHS = [
    "/content/extracted_metrics_all_apps.csv",
    "/content/drive/MyDrive/extracted_metrics_all_apps.csv",
    "/content/drive/MyDrive/VR_Metrics/extracted_metrics_all_apps.csv",
]

DATA_PATH = None
for p in CANDIDATE_PATHS:
    if os.path.exists(p):
        DATA_PATH = p
        break

if DATA_PATH is None:
    raise FileNotFoundError("Could not find extracted_metrics_all_apps.csv. Update CANDIDATE_PATHS.")

df = pd.read_csv(DATA_PATH)
print("\n================= LOADED DATA =================")
print("Loaded:", DATA_PATH, "| shape:", df.shape)

# =========================
# 2) CLEAN / NORMALIZE TYPES
# =========================
df = df.replace("", np.nan)

required = {"App", "Temporal"}
missing_req = required - set(df.columns)
if missing_req:
    raise ValueError(f"Missing required columns in CSV: {missing_req}")

df["App"] = df["App"].astype(str)
df["Temporal"] = pd.to_numeric(df["Temporal"], errors="coerce")

# keep rows that have Temporal label
df = df.dropna(subset=["App", "Temporal"]).copy()
df["Temporal"] = df["Temporal"].astype(int)

apps = sorted(df["App"].unique().tolist())
print("Apps:", apps)

# =========================
# 3) OPTIONAL: Downsample PhantomLimb (frame-level BEFORE seq build)
# =========================
DOWNSAMPLE_PHANTOMLIMB = True
TARGET_N_PL = 400

print(f"\n[INFO] Downsample PhantomLimb = {DOWNSAMPLE_PHANTOMLIMB} | TARGET_N_PL = {TARGET_N_PL}")

if DOWNSAMPLE_PHANTOMLIMB and "PhantomLimb" in apps:
    df_pl = df[df["App"] == "PhantomLimb"]
    n_pl = len(df_pl)
    if n_pl > TARGET_N_PL:
        df_pl_down, _ = train_test_split(
            df_pl,
            train_size=TARGET_N_PL,
            random_state=SEED,
            stratify=df_pl["Temporal"]
        )
        df_other = df[df["App"] != "PhantomLimb"]
        df = pd.concat([df_other, df_pl_down], ignore_index=True)
        print(f"[INFO] PhantomLimb downsampled: {n_pl} -> {len(df_pl_down)}")
    else:
        print(f"[INFO] PhantomLimb N = {n_pl} (<= target), no downsample performed.")
else:
    print("[INFO] PhantomLimb downsample skipped.")

print(f"[INFO] Total N after downsample step = {len(df)}")

# =========================
# 4) FEATURE COLUMN SELECTION
# =========================
ID_COLS = [c for c in ["GlobalID", "EntryID"] if c in df.columns]
DROP_COLS = ["App", "Spatial", "Temporal"] + ID_COLS  # Spatial may or may not exist; harmless
DROP_COLS = [c for c in DROP_COLS if c in df.columns]
feature_cols = [c for c in df.columns if c not in DROP_COLS]

for c in feature_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

print("\n================= FEATURES =================")
print("Num features:", len(feature_cols))
print("First 25 feature cols:", feature_cols[:25])
print(f"[INFO] SEQ_LEN={SEQ_LEN} | label = center frame (i+{CENTER})")
print(f"[INFO] Temporal seq RF will use flattened features: {len(feature_cols)} * {SEQ_LEN} = {len(feature_cols)*SEQ_LEN}")

# =========================
# 5) Build sequences per app
# =========================
def make_sequences_from_app(df_app: pd.DataFrame, feature_cols, label_col="Temporal", seq_len=5):
    X = df_app[feature_cols].values  # (N, F)
    y = df_app[label_col].values    # (N,)
    N, F = X.shape
    if N < seq_len:
        return np.empty((0, seq_len*F)), np.empty((0,), dtype=int)

    seqX, seqY = [], []
    center = seq_len // 2
    for i in range(N - seq_len + 1):
        window = X[i:i+seq_len]          # (seq_len, F)
        seqX.append(window.reshape(-1))  # (seq_len*F,)
        seqY.append(int(y[i + center]))  # center label

    return np.asarray(seqX), np.asarray(seqY)

# =========================
# 6) MODEL: Random Forest
# =========================
rf_seq_model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("clf", RandomForestClassifier(
        n_estimators=500,
        max_depth=None,
        min_samples_leaf=5,
        max_features="sqrt",
        class_weight="balanced",
        random_state=SEED,
        n_jobs=-1
    ))
])

# =========================
# 7) EVAL HELPERS
# =========================
def eval_binary(y_true, y_pred):
    return {
        "acc": float(accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "cm": confusion_matrix(y_true, y_pred).tolist(),
        "pred_dist": dict(zip(*np.unique(y_pred, return_counts=True))),
    }

# =========================
# 8) LOAO: Temporal sequence RF
# =========================
def run_loao_temporal_seq_rf():
    results = []

    for test_app in apps:
        # build train sequences from all other apps
        train_apps = [a for a in apps if a != test_app]

        X_train_list, y_train_list = [], []
        for a in train_apps:
            df_a = df[df["App"] == a].copy()
            X_a, y_a = make_sequences_from_app(df_a, feature_cols, label_col="Temporal", seq_len=SEQ_LEN)
            if len(y_a) > 0:
                X_train_list.append(X_a)
                y_train_list.append(y_a)

        X_test, y_test = make_sequences_from_app(
            df[df["App"] == test_app].copy(),
            feature_cols,
            label_col="Temporal",
            seq_len=SEQ_LEN
        )

        if len(y_test) == 0 or len(y_train_list) == 0:
            print(f"[Temporal | RF_SEQ] HELD-OUT={test_app:12s}  SKIP (not enough sequences)")
            continue

        X_train_full = np.vstack(X_train_list)
        y_train_full = np.concatenate(y_train_list)

        # internal val split
        X_tr, X_val, y_tr, y_val = train_test_split(
            X_train_full, y_train_full,
            test_size=0.15,
            random_state=SEED,
            stratify=y_train_full
        )

        rf_seq_model.fit(X_tr, y_tr)
        yhat = rf_seq_model.predict(X_test)
        m = eval_binary(y_test, yhat)

        row = {
            "label": "Temporal",
            "task": f"sequence_len_{SEQ_LEN}_flattened",
            "model": "RF",
            "held_out_app": test_app,
            "n_train": int(len(y_tr)),
            "n_val": int(len(y_val)),
            "n_test": int(len(y_test)),
            "seq_len": int(SEQ_LEN),
            "seq_features": int(X_train_full.shape[1]),
            **{k: v for k, v in m.items() if k not in ["cm", "pred_dist"]},
            "cm": m["cm"],
            "pred_dist": m["pred_dist"],
        }

        print(f"[Temporal | RF_SEQ] HELD-OUT={test_app:12s} "
              f"(SEQ_LEN={SEQ_LEN}, seqF={row['seq_features']}) "
              f"acc={row['acc']:.3f} f1={row['f1']:.3f} "
              f"prec={row['precision']:.3f} rec={row['recall']:.3f} "
              f"n_test={row['n_test']} pred_dist={row['pred_dist']}")
        results.append(row)

    return pd.DataFrame(results)

# =========================
# 9) RUN
# =========================
print("\n================= LOAO: TEMPORAL (SEQUENCE) - RF =================")
results_df = run_loao_temporal_seq_rf()

print("\n================ SUMMARY (mean over held-out apps) ================")
if len(results_df) > 0:
    summary = results_df[["acc", "f1", "precision", "recall"]].mean()
    print(summary)
else:
    print("No results (all apps skipped).")

OUT_PATH = f"/content/loao_results_temporal_seq_rf_len{SEQ_LEN}.csv"
results_df.to_csv(OUT_PATH, index=False)
print("\nSaved:", OUT_PATH)


================= LOADED DATA =================
Loaded: /content/extracted_metrics_all_apps.csv | shape: (1744, 28)
Apps: ['Archery', 'PhantomLimb', 'PianoTiles', 'Puzzle', 'Sea', 'War']

[INFO] Downsample PhantomLimb = True | TARGET_N_PL = 400
[INFO] PhantomLimb N = 325 (<= target), no downsample performed.
[INFO] Total N after downsample step = 1744

================= FEATURES =================
Num features: 23
First 25 feature cols: ['missing_joints_count', 'missing_joints_ratio', 'collapsed_joints_count', 'center_of_mass_x', 'center_of_mass_y', 'center_of_mass_z', 'distance_from_origin', 'bbox_width', 'bbox_height', 'bbox_depth', 'bbox_volume', 'max_joint_distance_from_com', 'distance_from_floor', 'below_floor', 'left_forearm_length', 'right_forearm_length', 'left_shin_length', 'right_shin_length', 'arm_length_symmetry', 'leg_length_symmetry', 'body_forward_x', 'body_forward_y', 'body_forward_z']
[INFO] SEQ_LEN=5 | label = center frame (i+2)
[INFO] Temporal seq RF will use flatten

In [ ]:
# ============================================================
# Temporal Random Forest on SEQUENCE AGGREGATES (NO SMOTE)
#  - seq_len=5 sliding windows per app
#  - label = center frame (i+2)
#  - features: per-metric aggregates over the 5 frames:
#      mean, std, range, mean_abs_diff, (last-first)
#    => 23 * 5 = 115 aggregate dims
#    + optional: global scalars (motion energy etc.) to reach ~138-ish if desired
#
# IMPORTANT:
#  - This avoids the "flattened 115" issue where RF struggles.
#  - It’s the closest RF baseline to your successful 138-feature approach.
# ============================================================

import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

SEED = 42
np.random.seed(SEED)

SEQ_LEN = 5
CENTER = SEQ_LEN // 2

# -------------------------
# 1) LOAD DATA
# -------------------------
CANDIDATE_PATHS = [
    "/content/extracted_metrics_all_apps.csv",
    "/content/drive/MyDrive/extracted_metrics_all_apps.csv",
    "/content/drive/MyDrive/VR_Metrics/extracted_metrics_all_apps.csv",
]
DATA_PATH = next((p for p in CANDIDATE_PATHS if os.path.exists(p)), None)
if DATA_PATH is None:
    raise FileNotFoundError("Could not find extracted_metrics_all_apps.csv. Update CANDIDATE_PATHS.")

df = pd.read_csv(DATA_PATH).replace("", np.nan)
print("\n================= LOADED DATA =================")
print("Loaded:", DATA_PATH, "| shape:", df.shape)

required = {"App", "Temporal"}
missing_req = required - set(df.columns)
if missing_req:
    raise ValueError(f"Missing required columns in CSV: {missing_req}")

df["App"] = df["App"].astype(str)
df["Temporal"] = pd.to_numeric(df["Temporal"], errors="coerce")
df = df.dropna(subset=["App", "Temporal"]).copy()
df["Temporal"] = df["Temporal"].astype(int)

apps = sorted(df["App"].unique().tolist())
print("Apps:", apps)

# -------------------------
# 2) OPTIONAL DOWN-SAMPLE PL
# -------------------------
DOWNSAMPLE_PHANTOMLIMB = True
TARGET_N_PL = 400
print(f"\n[INFO] Downsample PhantomLimb = {DOWNSAMPLE_PHANTOMLIMB} | TARGET_N_PL = {TARGET_N_PL}")

if DOWNSAMPLE_PHANTOMLIMB and "PhantomLimb" in apps:
    df_pl = df[df["App"] == "PhantomLimb"]
    if len(df_pl) > TARGET_N_PL:
        df_pl_down, _ = train_test_split(
            df_pl,
            train_size=TARGET_N_PL,
            random_state=SEED,
            stratify=df_pl["Temporal"]
        )
        df = pd.concat([df[df["App"] != "PhantomLimb"], df_pl_down], ignore_index=True)
        print(f"[INFO] PhantomLimb downsampled -> {len(df_pl_down)}")
    else:
        print(f"[INFO] PhantomLimb N = {len(df_pl)} (<= target), no downsample performed.")

print(f"[INFO] Total N after downsample step = {len(df)}")

# -------------------------
# 3) FEATURE COLS
# -------------------------
ID_COLS = [c for c in ["GlobalID", "EntryID"] if c in df.columns]
DROP_COLS = [c for c in ["App", "Spatial", "Temporal"] + ID_COLS if c in df.columns]
feature_cols = [c for c in df.columns if c not in DROP_COLS]

for c in feature_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

print("\n================= FEATURES =================")
print("Num features:", len(feature_cols))
print("First 25 feature cols:", feature_cols[:25])
print(f"[INFO] SEQ_LEN={SEQ_LEN} | label = center frame (i+{CENTER})")

# -------------------------
# 4) Sequence aggregate features
# -------------------------
def nanmean(x, axis=0): return np.nanmean(x, axis=axis)
def nanstd(x, axis=0):  return np.nanstd(x, axis=axis)
def nanrange(x, axis=0):
    mx = np.nanmax(x, axis=axis)
    mn = np.nanmin(x, axis=axis)
    return mx - mn

def seq_aggregate_features(window_2d: np.ndarray):
    """
    window_2d: (SEQ_LEN, F)
    returns: (F*5 + extra,)
    """
    feats = []
    feats.append(nanmean(window_2d, axis=0))                 # mean
    feats.append(nanstd(window_2d, axis=0))                  # std
    feats.append(nanrange(window_2d, axis=0))                # range
    diff = np.diff(window_2d, axis=0)                        # (SEQ_LEN-1, F)
    feats.append(nanmean(np.abs(diff), axis=0))              # mean abs step (smoothness proxy)
    feats.append(window_2d[-1] - window_2d[0])               # last-first (net change)

    base = np.concatenate(feats, axis=0)                     # (F*5,)

    # Optional extra global motion features (few scalars)
    # These help RF sometimes, and push dims toward your "138-ish" idea.
    # (they do NOT require timestamps)
    motion_energy = np.nanmean(diff**2) if diff.size else np.nan
    motion_l1 = np.nanmean(np.abs(diff)) if diff.size else np.nan
    net_move_l2 = np.nanmean((window_2d[-1] - window_2d[0])**2)
    extra = np.array([motion_energy, motion_l1, net_move_l2], dtype=float)

    return np.concatenate([base, extra], axis=0)

def make_seq_dataset(df_app: pd.DataFrame, label_col="Temporal"):
    X = df_app[feature_cols].values  # (N, F)
    y = df_app[label_col].values     # (N,)
    N, F = X.shape
    if N < SEQ_LEN:
        return np.empty((0, 0)), np.empty((0,), dtype=int)

    X_seq, y_seq = [], []
    for i in range(N - SEQ_LEN + 1):
        window = X[i:i+SEQ_LEN]  # (SEQ_LEN, F)
        X_seq.append(seq_aggregate_features(window))
        y_seq.append(int(y[i + CENTER]))
    return np.asarray(X_seq), np.asarray(y_seq)

# -------------------------
# 5) MODEL: RF
# -------------------------
rf = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("clf", RandomForestClassifier(
        n_estimators=600,
        max_depth=None,
        min_samples_leaf=5,
        max_features="sqrt",
        class_weight="balanced",
        random_state=SEED,
        n_jobs=-1
    ))
])

def eval_binary(y_true, y_pred):
    return {
        "acc": float(accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "cm": confusion_matrix(y_true, y_pred).tolist(),
        "pred_dist": dict(zip(*np.unique(y_pred, return_counts=True))),
    }

# -------------------------
# 6) LOAO
# -------------------------
print("\n================= LOAO: TEMPORAL (SEQUENCE AGG) - RF =================")
results = []

for test_app in apps:
    train_apps = [a for a in apps if a != test_app]

    X_train_list, y_train_list = [], []
    for a in train_apps:
        Xa, ya = make_seq_dataset(df[df["App"] == a].copy(), label_col="Temporal")
        if len(ya) > 0:
            X_train_list.append(Xa)
            y_train_list.append(ya)

    X_test, y_test = make_seq_dataset(df[df["App"] == test_app].copy(), label_col="Temporal")
    if len(y_test) == 0 or len(y_train_list) == 0:
        print(f"[Temporal | RF_AGG] HELD-OUT={test_app:12s}  SKIP (not enough sequences)")
        continue

    X_train_full = np.vstack(X_train_list)
    y_train_full = np.concatenate(y_train_list)

    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train_full, y_train_full,
        test_size=0.15,
        random_state=SEED,
        stratify=y_train_full
    )

    rf.fit(X_tr, y_tr)
    yhat = rf.predict(X_test)
    m = eval_binary(y_test, yhat)

    row = {
        "label": "Temporal",
        "task": f"sequence_len_{SEQ_LEN}_agg",
        "model": "RF",
        "held_out_app": test_app,
        "n_train": int(len(y_tr)),
        "n_val": int(len(y_val)),
        "n_test": int(len(y_test)),
        "seq_len": int(SEQ_LEN),
        "seq_features": int(X_train_full.shape[1]),
        **{k: v for k, v in m.items() if k not in ["cm", "pred_dist"]},
        "cm": m["cm"],
        "pred_dist": m["pred_dist"],
    }

    print(f"[Temporal | RF_AGG] HELD-OUT={test_app:12s} "
          f"(SEQ_LEN={SEQ_LEN}, seqF={row['seq_features']}) "
          f"acc={row['acc']:.3f} f1={row['f1']:.3f} "
          f"prec={row['precision']:.3f} rec={row['recall']:.3f} "
          f"n_test={row['n_test']} pred_dist={row['pred_dist']}")

    results.append(row)

results_df = pd.DataFrame(results)

print("\n================ SUMMARY (mean over held-out apps) ================")
if len(results_df) > 0:
    print(results_df[["acc", "f1", "precision", "recall"]].mean())
else:
    print("No results.")

OUT_PATH = f"/content/loao_results_temporal_rf_agg_len{SEQ_LEN}.csv"
results_df.to_csv(OUT_PATH, index=False)
print("\nSaved:", OUT_PATH)


================= LOADED DATA =================
Loaded: /content/extracted_metrics_all_apps.csv | shape: (1744, 28)
Apps: ['Archery', 'PhantomLimb', 'PianoTiles', 'Puzzle', 'Sea', 'War']

[INFO] Downsample PhantomLimb = True | TARGET_N_PL = 400
[INFO] PhantomLimb N = 325 (<= target), no downsample performed.
[INFO] Total N after downsample step = 1744

================= FEATURES =================
Num features: 23
First 25 feature cols: ['missing_joints_count', 'missing_joints_ratio', 'collapsed_joints_count', 'center_of_mass_x', 'center_of_mass_y', 'center_of_mass_z', 'distance_from_origin', 'bbox_width', 'bbox_height', 'bbox_depth', 'bbox_volume', 'max_joint_distance_from_com', 'distance_from_floor', 'below_floor', 'left_forearm_length', 'right_forearm_length', 'left_shin_length', 'right_shin_length', 'arm_length_symmetry', 'leg_length_symmetry', 'body_forward_x', 'body_forward_y', 'body_forward_z']
[INFO] SEQ_LEN=5 | label = center frame (i+2)

================= LOAO: TEMPORAL (SEQU

/tmp/ipython-input-2291642898.py:101: RuntimeWarning: Mean of empty slice
  def nanmean(x, axis=0): return np.nanmean(x, axis=axis)
/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:2035: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/ipython-input-2291642898.py:104: RuntimeWarning: All-NaN slice encountered
  mx = np.nanmax(x, axis=axis)
/tmp/ipython-input-2291642898.py:105: RuntimeWarning: All-NaN slice encountered
  mn = np.nanmin(x, axis=axis)
/tmp/ipython-input-2291642898.py:101: RuntimeWarning: Mean of empty slice
  def nanmean(x, axis=0): return np.nanmean(x, axis=axis)
/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:2035: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/ipython-input-2291642898.py:104: RuntimeWarning: All-NaN slice encountered
  mx = np.nanmax(x, axis=axis)
/tmp/ipython-input-22

[Temporal | RF_AGG] HELD-OUT=Archery      (SEQ_LEN=5, seqF=118) acc=0.365 f1=0.440 prec=0.960 rec=0.286 n_test=96 pred_dist={np.int64(0): np.int64(71), np.int64(1): np.int64(25)}


/tmp/ipython-input-2291642898.py:101: RuntimeWarning: Mean of empty slice
  def nanmean(x, axis=0): return np.nanmean(x, axis=axis)
/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:2035: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/ipython-input-2291642898.py:104: RuntimeWarning: All-NaN slice encountered
  mx = np.nanmax(x, axis=axis)
/tmp/ipython-input-2291642898.py:105: RuntimeWarning: All-NaN slice encountered
  mn = np.nanmin(x, axis=axis)
/tmp/ipython-input-2291642898.py:101: RuntimeWarning: Mean of empty slice
  def nanmean(x, axis=0): return np.nanmean(x, axis=axis)
/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:2035: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/ipython-input-2291642898.py:104: RuntimeWarning: All-NaN slice encountered
  mx = np.nanmax(x, axis=axis)
/tmp/ipython-input-22

[Temporal | RF_AGG] HELD-OUT=PhantomLimb  (SEQ_LEN=5, seqF=118) acc=0.586 f1=0.000 prec=0.000 rec=0.000 n_test=321 pred_dist={np.int64(0): np.int64(321)}


...

[Temporal | RF_AGG] HELD-OUT=Puzzle       (SEQ_LEN=5, seqF=118) acc=0.278 f1=0.000 prec=0.000 rec=0.000 n_test=295 pred_dist={np.int64(0): np.int64(295)}


/tmp/ipython-input-2291642898.py:101: RuntimeWarning: Mean of empty slice
  def nanmean(x, axis=0): return np.nanmean(x, axis=axis)
/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:2035: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/ipython-input-2291642898.py:104: RuntimeWarning: All-NaN slice encountered
  mx = np.nanmax(x, axis=axis)
/tmp/ipython-input-2291642898.py:105: RuntimeWarning: All-NaN slice encountered
  mn = np.nanmin(x, axis=axis)
/tmp/ipython-input-2291642898.py:101: RuntimeWarning: Mean of empty slice
  def nanmean(x, axis=0): return np.nanmean(x, axis=axis)
/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:2035: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/ipython-input-2291642898.py:104: RuntimeWarning: All-NaN slice encountered
  mx = np.nanmax(x, axis=axis)
/tmp/ipython-input-22

[Temporal | RF_AGG] HELD-OUT=Sea          (SEQ_LEN=5, seqF=118) acc=0.848 f1=0.813 prec=0.916 rec=0.731 n_test=296 pred_dist={np.int64(0): np.int64(189), np.int64(1): np.int64(107)}


/tmp/ipython-input-2291642898.py:101: RuntimeWarning: Mean of empty slice
  def nanmean(x, axis=0): return np.nanmean(x, axis=axis)
/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:2035: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/ipython-input-2291642898.py:104: RuntimeWarning: All-NaN slice encountered
  mx = np.nanmax(x, axis=axis)
/tmp/ipython-input-2291642898.py:105: RuntimeWarning: All-NaN slice encountered
  mn = np.nanmin(x, axis=axis)
/tmp/ipython-input-2291642898.py:101: RuntimeWarning: Mean of empty slice
  def nanmean(x, axis=0): return np.nanmean(x, axis=axis)
/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:2035: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/ipython-input-2291642898.py:104: RuntimeWarning: All-NaN slice encountered
  mx = np.nanmax(x, axis=axis)
/tmp/ipython-input-22

[Temporal | RF_AGG] HELD-OUT=War          (SEQ_LEN=5, seqF=118) acc=0.710 f1=0.744 prec=0.592 rec=1.000 n_test=276 pred_dist={np.int64(0): np.int64(80), np.int64(1): np.int64(196)}

================ SUMMARY (mean over held-out apps) ================
acc          0.588625
f1           0.393825
precision    0.455365
recall       0.434942
dtype: float64

Saved: /content/loao_results_temporal_rf_agg_len5.csv


# Summary

# [XG/HG/Cat]-Boost/LR S+T

In [ ]:
# ===============================================================
# VR APP-INDEPENDENT METRICS - LOAO BASELINES + CatBoost + XGBoost
# Starting from your Patch B (NaN-safe temporal aggregation)
#
# Adds:
#   - CatBoostClassifier (handles nonlinearity well on tabular)
#   - XGBoost (if installed; otherwise auto-skip with a message)
#
# Notes:
# - No SMOTE / oversampling here (as requested)
# - Keeps your Spatial frame-level and Temporal sequence-level setup
# - Uses the same Pipeline structure (imputer; scaler where appropriate)
# ===============================================================

import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier

SEED = 42
np.random.seed(SEED)

# =========================
# 1) LOAD DATA
# =========================
CANDIDATE_PATHS = [
    "/content/extracted_metrics_all_apps.csv",
    "/content/drive/MyDrive/extracted_metrics_all_apps.csv",
    "/content/drive/MyDrive/VR_Metrics/extracted_metrics_all_apps.csv",
]

DATA_PATH = None
for p in CANDIDATE_PATHS:
    if os.path.exists(p):
        DATA_PATH = p
        break

if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find extracted_metrics_all_apps.csv in common locations.\n"
        "Put it in /content OR adjust CANDIDATE_PATHS."
    )

df = pd.read_csv(DATA_PATH)
print("\n================= LOADED DATA =================")
print("Loaded:", DATA_PATH, "| shape:", df.shape)

# =========================
# 2) CLEAN / NORMALIZE TYPES
# =========================
df = df.replace("", np.nan)

required = {"App", "Spatial", "Temporal"}
missing_req = required - set(df.columns)
if missing_req:
    raise ValueError(f"Missing required columns in CSV: {missing_req}")

for col in ["Spatial", "Temporal"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=["App"]).copy()
df = df.dropna(subset=["Spatial", "Temporal"], how="all").copy()

df["Spatial"] = df["Spatial"].astype(int)
df["Temporal"] = df["Temporal"].astype(int)

apps = sorted(df["App"].unique().tolist())
print("Apps:", apps)

# =========================
# OPTIONAL: Downsample PhantomLimb
# =========================
DOWNSAMPLE_PHANTOMLIMB = True
TARGET_N_PL = 400

if DOWNSAMPLE_PHANTOMLIMB and ("PhantomLimb" in df["App"].unique()):
    pl = df[df["App"] == "PhantomLimb"]
    others = df[df["App"] != "PhantomLimb"]
    if len(pl) > TARGET_N_PL:
        pl = pl.sample(TARGET_N_PL, random_state=SEED)
    df = pd.concat([others, pl], ignore_index=True)
    apps = sorted(df["App"].unique().tolist())
    print("\n[INFO] Downsample PhantomLimb =", DOWNSAMPLE_PHANTOMLIMB,
          "| TARGET_N_PL =", TARGET_N_PL)
    print("[INFO] PhantomLimb N =", len(df[df["App"] == "PhantomLimb"]),
          "| Total N =", len(df))

# =========================
# Patch A: Label sanity + baselines
# =========================
print("\n=== Per-app label balance ===")
for app in apps:
    sub = df[df["App"] == app]
    print(f"\nAPP={app}  N={len(sub)}")
    print("  Spatial:", sub["Spatial"].value_counts().to_dict())
    print("  Temporal:", sub["Temporal"].value_counts().to_dict())

print("\n=== Majority baseline (per app) ===")
for app in apps:
    sub = df[df["App"] == app]
    for lab in ["Spatial", "Temporal"]:
        vc = sub[lab].value_counts()
        maj_acc = (vc.max() / vc.sum()) if len(vc) else np.nan
        print(f"APP={app:12s} label={lab:8s} majority_acc={maj_acc:.3f} dist={vc.to_dict()}")

# =========================
# 3) FEATURE COLUMN SELECTION
# =========================
ID_COLS = [c for c in ["GlobalID", "EntryID"] if c in df.columns]
DROP_COLS = ["App", "Spatial", "Temporal"] + ID_COLS
feature_cols = [c for c in df.columns if c not in DROP_COLS]

for c in feature_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

print("\n================= FEATURES =================")
print("Num features:", len(feature_cols))
print("First 25 feature cols:", feature_cols[:25])

# =========================
# 4) MODELS
# =========================
# ---- Baselines you already had ----
hgb_model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("clf", HistGradientBoostingClassifier(
        random_state=SEED,
        max_depth=6,
        learning_rate=0.05,
        max_iter=400
    ))
])

lr_model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        random_state=SEED,
        max_iter=3000,
        class_weight="balanced",
        n_jobs=-1
    ))
])

# ---- CatBoost (try to import; if missing, skip cleanly) ----
cat_model = None
try:
    from catboost import CatBoostClassifier
    cat_model = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", CatBoostClassifier(
            random_seed=SEED,
            iterations=2000,
            learning_rate=0.03,
            depth=6,
            l2_leaf_reg=6.0,
            loss_function="Logloss",
            eval_metric="F1",
            auto_class_weights="Balanced",  # CatBoost's built-in balancing
            verbose=False
        ))
    ])
    print("\n[INFO] CatBoost imported OK.")
except Exception as e:
    print("\n[WARN] CatBoost not available. Install with: !pip -q install catboost")
    print("       Error:", repr(e))

# ---- XGBoost (try to import; if missing, skip cleanly) ----
xgb_model = None
try:
    import xgboost as xgb
    xgb_model = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", xgb.XGBClassifier(
            random_state=SEED,
            n_estimators=1200,
            learning_rate=0.03,
            max_depth=6,
            subsample=0.9,
            colsample_bytree=0.9,
            reg_lambda=1.0,
            reg_alpha=0.0,
            min_child_weight=1.0,
            gamma=0.0,
            tree_method="hist",         # fast on CPU in Colab
            objective="binary:logistic",
            eval_metric="logloss",
            n_jobs=-1
        ))
    ])
    print("[INFO] XGBoost imported OK.")
except Exception as e:
    print("\n[WARN] XGBoost not available. Install with: !pip -q install xgboost")
    print("       Error:", repr(e))

MODELS = {
    "HGB": hgb_model,
    "LR_balanced": lr_model,
}
if cat_model is not None:
    MODELS["CatBoost"] = cat_model
if xgb_model is not None:
    MODELS["XGBoost"] = xgb_model

print("\n[INFO] Models to run:", list(MODELS.keys()))

# =========================
# 5) EVAL HELPERS
# =========================
def eval_binary(y_true, y_pred):
    return {
        "acc": float(accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "cm": confusion_matrix(y_true, y_pred).tolist(),
        "pred_dist": dict(zip(*np.unique(y_pred, return_counts=True))),
    }

# =========================
# Patch B FIX: warning-free NaN reducers
# =========================
def nanmean_no_warn(x, axis=0):
    x = np.asarray(x, dtype=float)
    valid = np.isfinite(x)
    denom = valid.sum(axis=axis)
    num = np.where(valid, x, 0.0).sum(axis=axis)
    out = np.divide(num, denom, out=np.full_like(num, np.nan, dtype=float), where=(denom > 0))
    return out

def nanstd_no_warn(x, axis=0):
    x = np.asarray(x, dtype=float)
    mu = nanmean_no_warn(x, axis=axis)
    mu_b = np.expand_dims(mu, axis=axis)
    diff2 = (x - mu_b) ** 2
    valid = np.isfinite(diff2)
    denom = valid.sum(axis=axis)
    num = np.where(valid, diff2, 0.0).sum(axis=axis)
    var = np.divide(num, denom, out=np.full_like(num, np.nan, dtype=float), where=(denom > 0))
    return np.sqrt(var)

def nanrange_no_warn(x, axis=0):
    x = np.asarray(x, dtype=float)
    valid = np.isfinite(x)
    x_for_max = np.where(valid, x, -np.inf)
    x_for_min = np.where(valid, x, +np.inf)

    mx = np.max(x_for_max, axis=axis)
    mn = np.min(x_for_min, axis=axis)

    all_nan = ~valid.any(axis=axis)
    out = mx - mn
    out = np.where(all_nan, np.nan, out)
    return out

def window_is_valid(window, min_non_nan_ratio=0.4):
    total = window.size
    non_nan = np.isfinite(window).sum()
    return (non_nan / total) >= min_non_nan_ratio

def aggregate_window(window, use=("mean","std","delta","range","mean_abs_vel","vel_std")):
    window = np.asarray(window, dtype=float)
    feats = []

    if "mean" in use:
        feats.append(nanmean_no_warn(window, axis=0))
    if "std" in use:
        feats.append(nanstd_no_warn(window, axis=0))
    if "delta" in use:
        feats.append(window[-1] - window[0])
    if "range" in use:
        feats.append(nanrange_no_warn(window, axis=0))

    if "mean_abs_vel" in use or "vel_std" in use:
        diff = np.diff(window, axis=0)
        if "mean_abs_vel" in use:
            feats.append(nanmean_no_warn(np.abs(diff), axis=0))
        if "vel_std" in use:
            feats.append(nanstd_no_warn(diff, axis=0))

    return np.concatenate(feats, axis=0)

def make_sequence_dataset(app_df, label_col, seq_len=5,
                          agg_use=("mean","std","delta","range","mean_abs_vel","vel_std"),
                          min_non_nan_ratio=0.4):
    X = app_df[feature_cols].values.astype(float)
    y = app_df[label_col].values.astype(int)

    if len(app_df) < seq_len:
        return np.empty((0, 1)), np.empty((0,), dtype=int)

    X_seq, y_seq = [], []
    center = seq_len // 2

    for i in range(len(app_df) - seq_len + 1):
        window = X[i:i+seq_len]
        if not window_is_valid(window, min_non_nan_ratio=min_non_nan_ratio):
            continue
        feat = aggregate_window(window, use=agg_use)
        X_seq.append(feat)
        y_seq.append(y[i + center])

    if len(X_seq) == 0:
        return np.empty((0, 1)), np.empty((0,), dtype=int)

    return np.vstack(X_seq), np.array(y_seq, dtype=int)

# =========================
# 6) LOAO: Spatial (frame-level)
# =========================
def run_loao_frame(label_col: str, model_name: str, model):
    results = []
    for test_app in apps:
        train_df = df[df["App"] != test_app].copy()
        test_df  = df[df["App"] == test_app].copy()

        X_train_full = train_df[feature_cols].values
        y_train_full = train_df[label_col].values

        X_test = test_df[feature_cols].values
        y_test = test_df[label_col].values

        # Inner val split (kept same as your patch)
        X_tr, X_val, y_tr, y_val = train_test_split(
            X_train_full, y_train_full,
            test_size=0.15,
            random_state=SEED,
            stratify=y_train_full
        )

        model.fit(X_tr, y_tr)
        yhat = model.predict(X_test)
        m = eval_binary(y_test, yhat)

        row = {
            "label": label_col,
            "task": "frame",
            "model": model_name,
            "held_out_app": test_app,
            "n_train": int(len(y_tr)),
            "n_val": int(len(y_val)),
            "n_test": int(len(y_test)),
            **{k: v for k, v in m.items() if k not in ["cm", "pred_dist"]},
            "cm": m["cm"],
            "pred_dist": m["pred_dist"],
        }

        print(f"[{label_col} | {model_name}] HELD-OUT={test_app:12s}  "
              f"acc={row['acc']:.3f} f1={row['f1']:.3f}  "
              f"prec={row['precision']:.3f} rec={row['recall']:.3f}  "
              f"n_test={row['n_test']}")
        results.append(row)

    return pd.DataFrame(results)

# =========================
# 7) LOAO: Temporal (sequence-level)
# =========================
def run_loao_sequence(label_col: str, model_name: str, model,
                      seq_len=5,
                      agg_use=("mean","std","delta","range","mean_abs_vel","vel_std"),
                      min_non_nan_ratio=0.4):
    results = []
    seq_feature_dim = len(agg_use) * len(feature_cols)

    for test_app in apps:
        train_df = df[df["App"] != test_app].copy()
        test_df  = df[df["App"] == test_app].copy()

        # Build sequence datasets per-app (prevents leakage)
        X_train_full, y_train_full = [], []
        for a in apps:
            if a == test_app:
                continue
            Xa, ya = make_sequence_dataset(
                df[df["App"] == a],
                label_col=label_col,
                seq_len=seq_len,
                agg_use=agg_use,
                min_non_nan_ratio=min_non_nan_ratio
            )
            if len(ya) > 0:
                X_train_full.append(Xa)
                y_train_full.append(ya)

        if len(X_train_full) == 0:
            print(f"[{label_col} | {model_name}] HELD-OUT={test_app}  -> No train sequences.")
            continue

        X_train_full = np.vstack(X_train_full)
        y_train_full = np.concatenate(y_train_full)

        X_test, y_test = make_sequence_dataset(
            test_df,
            label_col=label_col,
            seq_len=seq_len,
            agg_use=agg_use,
            min_non_nan_ratio=min_non_nan_ratio
        )

        if len(y_test) == 0:
            print(f"[{label_col} | {model_name}] HELD-OUT={test_app}  -> No test sequences.")
            continue

        X_tr, X_val, y_tr, y_val = train_test_split(
            X_train_full, y_train_full,
            test_size=0.15,
            random_state=SEED,
            stratify=y_train_full
        )

        model.fit(X_tr, y_tr)
        yhat = model.predict(X_test)
        m = eval_binary(y_test, yhat)

        row = {
            "label": label_col,
            "task": f"sequence_len_{seq_len}",
            "model": model_name,
            "held_out_app": test_app,
            "seq_len": int(seq_len),
            "seq_feature_dim": int(seq_feature_dim),
            "n_train": int(len(y_tr)),
            "n_val": int(len(y_val)),
            "n_test": int(len(y_test)),
            **{k: v for k, v in m.items() if k not in ["cm", "pred_dist"]},
            "cm": m["cm"],
            "pred_dist": m["pred_dist"],
        }

        print(f"[{label_col} | {model_name}] HELD-OUT={test_app:12s} "
              f"(SEQ_LEN={seq_len}, seqF={seq_feature_dim})  "
              f"acc={row['acc']:.3f} f1={row['f1']:.3f} "
              f"prec={row['precision']:.3f} rec={row['recall']:.3f} "
              f"n_test={row['n_test']}")
        results.append(row)

    return pd.DataFrame(results)

# =========================
# 8) RUN EVERYTHING
# =========================
SEQ_LEN = 5
AGG_USE = ("mean","std","delta","range","mean_abs_vel","vel_std")  # 6 * F
MIN_NON_NAN_RATIO = 0.4

all_runs = []

print("\n================= LOAO: SPATIAL (FRAME) =================")
for model_name, model in MODELS.items():
    res = run_loao_frame("Spatial", model_name, model)
    all_runs.append(res)

print("\n================= LOAO: TEMPORAL (SEQUENCE) =================")
for model_name, model in MODELS.items():
    res = run_loao_sequence(
        "Temporal", model_name, model,
        seq_len=SEQ_LEN,
        agg_use=AGG_USE,
        min_non_nan_ratio=MIN_NON_NAN_RATIO
    )
    all_runs.append(res)

results_df = pd.concat(all_runs, ignore_index=True)

print("\n================ SUMMARY (mean over held-out apps) ================")
summary = results_df.groupby(["label", "task", "model"])[["acc", "f1", "precision", "recall"]].mean().reset_index()
print(summary)

OUT_PATH = "/content/loao_metrics_results_PATCHB_with_cat_xgb.csv"
results_df.to_csv(OUT_PATH, index=False)
print("\nSaved results to:", OUT_PATH)


================= LOADED DATA =================
Loaded: /content/extracted_metrics_all_apps.csv | shape: (1744, 28)
Apps: ['Archery', 'PhantomLimb', 'PianoTiles', 'Puzzle', 'Sea', 'War']

[INFO] Downsample PhantomLimb = True | TARGET_N_PL = 400
[INFO] PhantomLimb N = 325 | Total N = 1744

=== Per-app label balance ===

APP=Archery  N=100
  Spatial: {1: 86, 0: 14}
  Temporal: {1: 86, 0: 14}

APP=PhantomLimb  N=325
  Spatial: {0: 293, 1: 32}
  Temporal: {0: 190, 1: 135}

APP=PianoTiles  N=440
  Spatial: {0: 228, 1: 212}
  Temporal: {0: 386, 1: 54}

APP=Puzzle  N=299
  Spatial: {1: 213, 0: 86}
  Temporal: {1: 213, 0: 86}

APP=Sea  N=300
  Spatial: {0: 168, 1: 132}
  Temporal: {0: 164, 1: 136}

APP=War  N=280
  Spatial: {0: 176, 1: 104}
  Temporal: {0: 164, 1: 116}

=== Majority baseline (per app) ===
APP=Archery      label=Spatial  majority_acc=0.860 dist={1: 86, 0: 14}
APP=Archery      label=Temporal majority_acc=0.860 dist={1: 86, 0: 14}
APP=PhantomLimb  label=Spatial  majority_acc=0.9

In [ ]:
# ===============================================================
# SAVE ONLY THE BEST TEMPORAL MODEL (from your summary)
# Best (mean over held-out apps) for Temporal is: LR_balanced
#
# This trains LR_balanced on ALL temporal sequences (all apps),
# then saves:
#   - model.joblib  (Pipeline: imputer+scaler+LR)
#   - meta.json     (feature order + temporal aggregation config)
#
# Output folder:
#   /content/saved_best_temporal/
# ===============================================================

import os, json
import numpy as np
import joblib

SAVE_DIR = "/content/saved_best_temporal"
os.makedirs(SAVE_DIR, exist_ok=True)

BEST_MODEL_NAME = "LR_balanced"
BEST_MODEL = MODELS[BEST_MODEL_NAME]  # uses your already-defined Pipeline

# ---- Build ONE big temporal dataset (aggregated 138-dim features) over ALL apps ----
X_all, y_all = [], []
for app in apps:
    Xa, ya = make_sequence_dataset(
        df[df["App"] == app],
        label_col="Temporal",
        seq_len=SEQ_LEN,
        agg_use=AGG_USE,
        min_non_nan_ratio=MIN_NON_NAN_RATIO
    )
    if len(ya) > 0:
        X_all.append(Xa)
        y_all.append(ya)

if len(X_all) == 0:
    raise RuntimeError("No temporal sequences were created. Check seq_len / min_non_nan_ratio / NaNs.")

X_all = np.vstack(X_all)
y_all = np.concatenate(y_all).astype(int)

print("[INFO] Training BEST temporal model on ALL apps' sequences...")
print("[INFO] X_all shape:", X_all.shape, "| y_all:", dict(zip(*np.unique(y_all, return_counts=True))))

BEST_MODEL.fit(X_all, y_all)

# ---- Save the pipeline ----
model_path = os.path.join(SAVE_DIR, "model.joblib")
joblib.dump(BEST_MODEL, model_path)

# ---- Save metadata needed by the server to recreate the SAME input -> 138D vector ----
meta = {
    "best_model_name": BEST_MODEL_NAME,
    "task": f"sequence_len_{int(SEQ_LEN)}",
    "label": "Temporal",
    "seq_len": int(SEQ_LEN),
    "agg_use": list(AGG_USE),                 # must match server-side aggregation
    "min_non_nan_ratio": float(MIN_NON_NAN_RATIO),
    "base_feature_cols_order": feature_cols,  # must match server-side feature ordering from JSON/input
    "seq_feature_dim": int(len(AGG_USE) * len(feature_cols)),  # should be 138
    "apps_seen_in_training": apps,
    "training_counts": {
        "n_sequences": int(len(y_all)),
        "class_dist": {str(k): int(v) for k, v in zip(*np.unique(y_all, return_counts=True))}
    },
    "notes": "Server must compute the SAME aggregated temporal vector using base_feature_cols_order + agg_use over a seq_len window."
}

meta_path = os.path.join(SAVE_DIR, "meta.json")
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print("\n[SAVED] Best temporal model bundle:")
print(" -", model_path)
print(" -", meta_path)

# OPTIONAL sanity check: load and predict on a small batch
_loaded = joblib.load(model_path)
p = _loaded.predict(X_all[:10])
print("[SANITY] loaded.predict(X[:10]) =", p.tolist())

[INFO] Training BEST temporal model on ALL apps' sequences...
[INFO] X_all shape: (1700, 138) | y_all: {np.int64(0): np.int64(966), np.int64(1): np.int64(734)}

[SAVED] Best temporal model bundle:
 - /content/saved_best_temporal/model.joblib
 - /content/saved_best_temporal/meta.json
[SANITY] loaded.predict(X[:10]) = [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [ ]:
import numpy as np
import joblib

_loaded = joblib.load("/content/saved_best_temporal/model.joblib")

proba = _loaded.predict_proba(X_all)[:, 1]
pred  = = (proba >= 0.5).astype(int)

print("[SANITY] pred distribution:", dict(zip(*np.unique(pred, return_counts=True))))
print("[SANITY] proba stats: min/mean/max/std:",
      float(np.min(proba)), float(np.mean(proba)), float(np.max(proba)), float(np.std(proba)))

# Compare against training label distribution (just a sanity baseline)
print("[SANITY] true distribution:", dict(zip(*np.unique(y_all, return_counts=True))))

In [ ]:
from sklearn.calibration import CalibratedClassifierCV

def best_threshold_f1(y_true, probs, grid=None):
    if grid is None:
        grid = np.linspace(0.05, 0.95, 181)
    best_t, best_f1 = 0.5, -1.0
    for t in grid:
        y_pred = (probs >= t).astype(int)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, float(t)
    return best_t, float(best_f1)

def run_loao_sequence_lr_threshold(label_col="Temporal",
                                  seq_len=5,
                                  agg_use=("mean","std","delta","range","mean_abs_vel","vel_std"),
                                  min_non_nan_ratio=0.4,
                                  do_calibrate=False,
                                  calibrate_method="sigmoid"):
    results = []
    seq_feature_dim = len(agg_use) * len(feature_cols)

    # base LR model (same as yours)
    base_lr = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            random_state=SEED,
            max_iter=5000,
            class_weight="balanced",
            n_jobs=-1
        ))
    ])

    for test_app in apps:
        # Build train sequences from all other apps
        X_train_full, y_train_full = [], []
        for a in apps:
            if a == test_app:
                continue
            Xa, ya = make_sequence_dataset(
                df[df["App"] == a],
                label_col=label_col,
                seq_len=seq_len,
                agg_use=agg_use,
                min_non_nan_ratio=min_non_nan_ratio
            )
            if len(ya) > 0:
                X_train_full.append(Xa)
                y_train_full.append(ya)

        if len(X_train_full) == 0:
            print(f"[{label_col} | LR_thr] HELD-OUT={test_app} -> No train sequences.")
            continue

        X_train_full = np.vstack(X_train_full)
        y_train_full = np.concatenate(y_train_full)

        # Build test sequences
        X_test, y_test = make_sequence_dataset(
            df[df["App"] == test_app],
            label_col=label_col,
            seq_len=seq_len,
            agg_use=agg_use,
            min_non_nan_ratio=min_non_nan_ratio
        )
        if len(y_test) == 0:
            print(f"[{label_col} | LR_thr] HELD-OUT={test_app} -> No test sequences.")
            continue

        # Inner val split
        X_tr, X_val, y_tr, y_val = train_test_split(
            X_train_full, y_train_full,
            test_size=0.15,
            random_state=SEED,
            stratify=y_train_full
        )

        model = base_lr

        # Optional calibration (fit only on training split; calibrator uses CV inside)
        # Note: calibration makes probs more meaningful; threshold tuning then is more stable.
        if do_calibrate:
            # Need an estimator that supports predict_proba at the end; pipeline is OK.
            model = CalibratedClassifierCV(estimator=base_lr, method=calibrate_method, cv=3)

        model.fit(X_tr, y_tr)

        # probs on val -> pick threshold by F1
        val_probs = model.predict_proba(X_val)[:, 1]
        best_t, best_val_f1 = best_threshold_f1(y_val, val_probs)

        # apply to test
        test_probs = model.predict_proba(X_test)[:, 1]
        yhat = (test_probs >= best_t).astype(int)

        m = eval_binary(y_test, yhat)

        row = {
            "label": label_col,
            "task": f"sequence_len_{seq_len}",
            "model": f"LR_thr_calib={do_calibrate}",
            "held_out_app": test_app,
            "seq_len": int(seq_len),
            "seq_feature_dim": int(seq_feature_dim),
            "n_train": int(len(y_tr)),
            "n_val": int(len(y_val)),
            "n_test": int(len(y_test)),
            "best_thr": float(best_t),
            "best_val_f1": float(best_val_f1),
            **{k: v for k, v in m.items() if k not in ["cm", "pred_dist"]},
            "cm": m["cm"],
            "pred_dist": m["pred_dist"],
        }

        print(f"[Temporal | LR_thr{'_cal' if do_calibrate else ''}] HELD-OUT={test_app:12s} "
              f"(SEQ_LEN={seq_len}, seqF={seq_feature_dim}) "
              f"thr={best_t:.2f} val_f1={best_val_f1:.3f}  "
              f"acc={row['acc']:.3f} f1={row['f1']:.3f} "
              f"prec={row['precision']:.3f} rec={row['recall']:.3f} "
              f"n_test={row['n_test']}")
        results.append(row)

    return pd.DataFrame(results)

# ---- RUN IT ----
res_thr = run_loao_sequence_lr_threshold(
    label_col="Temporal",
    seq_len=SEQ_LEN,
    agg_use=AGG_USE,
    min_non_nan_ratio=MIN_NON_NAN_RATIO,
    do_calibrate=False
)

print("\n=== SUMMARY (LR threshold tuned) ===")
print(res_thr[["acc","f1","precision","recall"]].mean())

OUT_THR = "/content/loao_temporal_lr_threshold_tuned.csv"
res_thr.to_csv(OUT_THR, index=False)
print("Saved:", OUT_THR)

# Optional calibrated run (often helps when domains differ)
res_thr_cal = run_loao_sequence_lr_threshold(
    label_col="Temporal",
    seq_len=SEQ_LEN,
    agg_use=AGG_USE,
    min_non_nan_ratio=MIN_NON_NAN_RATIO,
    do_calibrate=True,
    calibrate_method="sigmoid"
)

print("\n=== SUMMARY (LR calibrated + threshold tuned) ===")
print(res_thr_cal[["acc","f1","precision","recall"]].mean())

OUT_THR_CAL = "/content/loao_temporal_lr_calibrated_threshold_tuned.csv"
res_thr_cal.to_csv(OUT_THR_CAL, index=False)
print("Saved:", OUT_THR_CAL)

[Temporal | LR_thr] HELD-OUT=Archery      (SEQ_LEN=5, seqF=138) thr=0.48 val_f1=0.888  acc=0.875 f1=0.933 prec=0.875 rec=1.000 n_test=96
[Temporal | LR_thr] HELD-OUT=PhantomLimb  (SEQ_LEN=5, seqF=138) thr=0.38 val_f1=0.930  acc=0.938 f1=0.923 prec=0.945 rec=0.902 n_test=321
[Temporal | LR_thr] HELD-OUT=PianoTiles   (SEQ_LEN=5, seqF=138) thr=0.61 val_f1=0.945  acc=0.897 f1=0.458 prec=0.655 rec=0.352 n_test=436
[Temporal | LR_thr] HELD-OUT=Puzzle       (SEQ_LEN=5, seqF=138) thr=0.59 val_f1=0.867  acc=0.549 f1=0.670 prec=0.773 rec=0.592 n_test=275
[Temporal | LR_thr] HELD-OUT=Sea          (SEQ_LEN=5, seqF=138) thr=0.65 val_f1=0.901  acc=0.463 f1=0.628 prec=0.457 rec=1.000 n_test=296
[Temporal | LR_thr] HELD-OUT=War          (SEQ_LEN=5, seqF=138) thr=0.44 val_f1=0.925  acc=0.518 f1=0.634 prec=0.466 rec=0.991 n_test=276

=== SUMMARY (LR threshold tuned) ===
acc          0.706588
f1           0.707616
precision    0.695164
recall       0.806173
dtype: float64
Saved: /content/loao_temporal_lr

In [ ]:
from sklearn.calibration import CalibratedClassifierCV

def run_loao_sequence_lr_threshold(label_col="Temporal",
                                  seq_len=5,
                                  agg_use=("mean","std","delta","range","mean_abs_vel","vel_std"),
                                  min_non_nan_ratio=0.4,
                                  do_calibrate=False,
                                  calibrate_method="sigmoid"):
    results = []
    seq_feature_dim = len(agg_use) * len(feature_cols)

    base_lr = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            random_state=SEED,
            max_iter=5000,
            class_weight="balanced",
            n_jobs=-1
        ))
    ])

    for test_app in apps:
        X_train_full, y_train_full = [], []
        for a in apps:
            if a == test_app:
                continue
            Xa, ya = make_sequence_dataset(
                df[df["App"] == a],
                label_col=label_col,
                seq_len=seq_len,
                agg_use=agg_use,
                min_non_nan_ratio=min_non_nan_ratio
            )
            if len(ya) > 0:
                X_train_full.append(Xa)
                y_train_full.append(ya)

        if len(X_train_full) == 0:
            print(f"[{label_col} | LR_thr] HELD-OUT={test_app} -> No train sequences.")
            continue

        X_train_full = np.vstack(X_train_full)
        y_train_full = np.concatenate(y_train_full)

        X_test, y_test = make_sequence_dataset(
            df[df["App"] == test_app],
            label_col=label_col,
            seq_len=seq_len,
            agg_use=agg_use,
            min_non_nan_ratio=min_non_nan_ratio
        )
        if len(y_test) == 0:
            print(f"[{label_col} | LR_thr] HELD-OUT={test_app} -> No test sequences.")
            continue

        X_tr, X_val, y_tr, y_val = train_test_split(
            X_train_full, y_train_full,
            test_size=0.15,
            random_state=SEED,
            stratify=y_train_full
        )

        model = base_lr
        if do_calibrate:
            model = CalibratedClassifierCV(
                estimator=base_lr,   # <-- FIXED NAME
                method=calibrate_method,
                cv=3
            )

        model.fit(X_tr, y_tr)

        val_probs = model.predict_proba(X_val)[:, 1]
        best_t, best_val_f1 = best_threshold_f1(y_val, val_probs)

        test_probs = model.predict_proba(X_test)[:, 1]
        yhat = (test_probs >= best_t).astype(int)

        m = eval_binary(y_test, yhat)

        row = {
            "label": label_col,
            "task": f"sequence_len_{seq_len}",
            "model": f"LR_thr_calib={do_calibrate}_{calibrate_method}",
            "held_out_app": test_app,
            "seq_len": int(seq_len),
            "seq_feature_dim": int(seq_feature_dim),
            "n_train": int(len(y_tr)),
            "n_val": int(len(y_val)),
            "n_test": int(len(y_test)),
            "best_thr": float(best_t),
            "best_val_f1": float(best_val_f1),
            **{k: v for k, v in m.items() if k not in ["cm", "pred_dist"]},
            "cm": m["cm"],
            "pred_dist": m["pred_dist"],
        }

        print(f"[Temporal | LR_thr{'_cal' if do_calibrate else ''}] HELD-OUT={test_app:12s} "
              f"thr={best_t:.2f} val_f1={best_val_f1:.3f}  "
              f"acc={row['acc']:.3f} f1={row['f1']:.3f} "
              f"prec={row['precision']:.3f} rec={row['recall']:.3f} "
              f"n_test={row['n_test']}")
        results.append(row)

    return pd.DataFrame(results)

# Run calibrated
res_thr_cal = run_loao_sequence_lr_threshold(
    label_col="Temporal",
    seq_len=SEQ_LEN,
    agg_use=AGG_USE,
    min_non_nan_ratio=MIN_NON_NAN_RATIO,
    do_calibrate=True,
    calibrate_method="sigmoid"  # try "isotonic" after if you want
)

print("\n=== SUMMARY (LR calibrated + threshold tuned) ===")
print(res_thr_cal[["acc","f1","precision","recall"]].mean())

OUT_THR_CAL = "/content/loao_temporal_lr_calibrated_threshold_tuned.csv"
res_thr_cal.to_csv(OUT_THR_CAL, index=False)
print("Saved:", OUT_THR_CAL)

[Temporal | LR_thr_cal] HELD-OUT=Archery      thr=0.39 val_f1=0.889  acc=0.875 f1=0.933 prec=0.875 rec=1.000 n_test=96
[Temporal | LR_thr_cal] HELD-OUT=PhantomLimb  thr=0.46 val_f1=0.921  acc=0.938 f1=0.923 prec=0.945 rec=0.902 n_test=321
[Temporal | LR_thr_cal] HELD-OUT=PianoTiles   thr=0.39 val_f1=0.943  acc=0.773 f1=0.327 prec=0.258 rec=0.444 n_test=436
[Temporal | LR_thr_cal] HELD-OUT=Puzzle       thr=0.46 val_f1=0.863  acc=0.567 f1=0.681 prec=0.794 rec=0.596 n_test=275
[Temporal | LR_thr_cal] HELD-OUT=Sea          thr=0.42 val_f1=0.906  acc=0.470 f1=0.631 prec=0.460 rec=1.000 n_test=296
[Temporal | LR_thr_cal] HELD-OUT=War          thr=0.41 val_f1=0.925  acc=0.540 f1=0.646 prec=0.477 rec=1.000 n_test=276

=== SUMMARY (LR calibrated + threshold tuned) ===
acc          0.693725
f1           0.690122
precision    0.634924
recall       0.823824
dtype: float64
Saved: /content/loao_temporal_lr_calibrated_threshold_tuned.csv


# Improvements

In [ ]:
# ===============================================================
# TEMPORAL LOAO - LR ablations:
#   A) Baseline LR_balanced (your best)
#   B) + RandomOverSampler (train only)
#   C) + Per-app normalization (train apps fit, applied per app)
#   D) + App-aware one-hot (train app id appended, test = zeros)
#
# Saves:
#   /content/loao_temporal_lr_baseline.csv
#   /content/loao_temporal_lr_ros.csv
#   /content/loao_temporal_lr_perapp_norm.csv
#   /content/loao_temporal_lr_appaware.csv
# ===============================================================

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix

# Optional: imblearn for RandomOverSampler
try:
    from imblearn.over_sampling import RandomOverSampler
    IMBLEARN_OK = True
    print("[INFO] imblearn RandomOverSampler imported OK.")
except Exception as e:
    IMBLEARN_OK = False
    print("[WARN] imblearn not available. RandomOverSampler experiment will be skipped.", e)

SEED = 42
np.random.seed(SEED)

# -------------------------
# Metrics helper (same style as yours)
# -------------------------
def eval_binary(y_true, y_pred):
    return {
        "acc": float(accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "cm": confusion_matrix(y_true, y_pred).tolist(),
        "pred_dist": dict(zip(*np.unique(y_pred, return_counts=True))),
    }

# -------------------------
# Your best LR model (balanced)
# -------------------------
def make_lr_balanced():
    return Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            random_state=SEED,
            max_iter=4000,
            class_weight="balanced",
            n_jobs=-1
        ))
    ])

# -------------------------
# Sequence dataset builder wrapper: return also "source app" label per sequence
# -------------------------
def make_sequence_dataset_with_source(app_df, label_col, source_app_name,
                                      seq_len=5,
                                      agg_use=("mean","std","delta","range","mean_abs_vel","vel_std"),
                                      min_non_nan_ratio=0.4):
    Xs, ys = make_sequence_dataset(
        app_df=app_df,
        label_col=label_col,
        seq_len=seq_len,
        agg_use=agg_use,
        min_non_nan_ratio=min_non_nan_ratio
    )
    if len(ys) == 0:
        return Xs, ys, np.array([], dtype=object)
    src = np.array([source_app_name] * len(ys), dtype=object)
    return Xs, ys, src

# -------------------------
# Per-app normalization:
# Fit scaler on each app's TRAIN sequences only, apply to that app's sequences.
# For held-out app: fit scaler on its own (test-only) sequences? -> NO (would leak test stats).
# Instead: apply "global train scaler" to held-out sequences OR leave them raw.
#
# I recommend: GLOBAL train scaler for held-out app (no leakage), per-app for training apps.
# -------------------------
def per_app_normalize_train_then_global_for_test(X_train, src_train, X_test, src_test):
    """
    X_train: (N_train, D), src_train: (N_train,)
    X_test:  (N_test, D),  src_test: (N_test,)  -> typically all held-out app
    Returns normalized X_train, X_test
    """
    X_train = X_train.astype(float)
    X_test  = X_test.astype(float)

    # Global scaler fit on all training sequences (safe)
    global_scaler = StandardScaler()
    global_scaler.fit(X_train)
    X_test_norm = global_scaler.transform(X_test)

    # Per-app scalers for training apps
    X_train_norm = np.empty_like(X_train, dtype=float)

    for a in np.unique(src_train):
        idx = np.where(src_train == a)[0]
        scaler = StandardScaler()
        scaler.fit(X_train[idx])
        X_train_norm[idx] = scaler.transform(X_train[idx])

    return X_train_norm, X_test_norm

# -------------------------
# App-aware one-hot
# Train: append one-hot for SOURCE app (among training apps).
# Test (held-out): append all zeros.
# -------------------------
def append_app_onehot(X_train, src_train, X_test, held_out_app):
    train_apps = sorted([a for a in np.unique(src_train) if a != held_out_app])
    app2i = {a:i for i,a in enumerate(train_apps)}
    K = len(train_apps)

    def onehot(src_arr):
        oh = np.zeros((len(src_arr), K), dtype=float)
        for i, a in enumerate(src_arr):
            if a in app2i:
                oh[i, app2i[a]] = 1.0
        return oh

    oh_train = onehot(src_train)
    # held-out app => all zeros
    oh_test = np.zeros((len(X_test), K), dtype=float)

    X_train2 = np.hstack([X_train, oh_train])
    X_test2  = np.hstack([X_test, oh_test])
    return X_train2, X_test2, K

# -------------------------
# Core LOAO runner (temporal, seq)
# Options:
#   use_ros: RandomOverSampler on training set
#   use_per_app_norm: per-app normalization on train apps, global scaler on test
#   use_app_aware: append one-hot app id
# -------------------------
def run_loao_temporal_lr(
    label_col="Temporal",
    seq_len=5,
    agg_use=("mean","std","delta","range","mean_abs_vel","vel_std"),
    min_non_nan_ratio=0.4,
    use_ros=False,
    use_per_app_norm=False,
    use_app_aware=False,
    tag="baseline"
):
    results = []
    seq_feature_dim = len(agg_use) * len(feature_cols)

    for test_app in apps:
        # Build train sequences PER APP to avoid leakage & to keep src labels
        X_train_list, y_train_list, src_train_list = [], [], []
        for a in apps:
            if a == test_app:
                continue
            app_df = df[df["App"] == a]
            Xa, ya, srca = make_sequence_dataset_with_source(
                app_df, label_col, source_app_name=a,
                seq_len=seq_len, agg_use=agg_use, min_non_nan_ratio=min_non_nan_ratio
            )
            if len(ya) > 0:
                X_train_list.append(Xa)
                y_train_list.append(ya)
                src_train_list.append(srca)

        if len(X_train_list) == 0:
            print(f"[Temporal | LR_{tag}] HELD-OUT={test_app:12s} -> No train sequences.")
            continue

        X_train_full = np.vstack(X_train_list)
        y_train_full = np.concatenate(y_train_list)
        src_train_full = np.concatenate(src_train_list)

        # Test sequences for held-out app
        test_df = df[df["App"] == test_app].copy()
        X_test, y_test, src_test = make_sequence_dataset_with_source(
            test_df, label_col, source_app_name=test_app,
            seq_len=seq_len, agg_use=agg_use, min_non_nan_ratio=min_non_nan_ratio
        )
        if len(y_test) == 0:
            print(f"[Temporal | LR_{tag}] HELD-OUT={test_app:12s} -> No test sequences.")
            continue

        # Inner split (keep your style)
        X_tr, X_val, y_tr, y_val, src_tr, src_val = train_test_split(
            X_train_full, y_train_full, src_train_full,
            test_size=0.15, random_state=SEED, stratify=y_train_full
        )

        # (C) Per-app normalization
        if use_per_app_norm:
            X_tr, X_test_norm = per_app_normalize_train_then_global_for_test(
                X_tr, src_tr, X_test, src_test
            )
            # IMPORTANT: Also normalize val using GLOBAL scaler fit on X_tr
            # (keep consistent with "test uses global train scaler")
            global_scaler = StandardScaler().fit(X_tr)
            X_val = global_scaler.transform(X_val)
            X_test = X_test_norm

        # (D) App-aware features
        appaware_K = 0
        if use_app_aware:
            X_tr, X_test, appaware_K = append_app_onehot(X_tr, src_tr, X_test, held_out_app=test_app)
            # For val: append one-hot for val source app too
            X_val, _, _ = append_app_onehot(X_val, src_val, X_val[:1], held_out_app=test_app)
            # note: second return unused; we just needed same onehot mapping length

        # (B) RandomOverSampler (train only)
        if use_ros:
            if not IMBLEARN_OK:
                raise RuntimeError("imblearn not installed but use_ros=True.")
            ros = RandomOverSampler(random_state=SEED)
            X_tr, y_tr = ros.fit_resample(X_tr, y_tr)

        # Fit/predict
        model = make_lr_balanced()
        model.fit(X_tr, y_tr)
        yhat = model.predict(X_test)

        m = eval_binary(y_test, yhat)
        row = {
            "label": label_col,
            "task": f"sequence_len_{seq_len}",
            "model": f"LR_{tag}",
            "held_out_app": test_app,
            "seq_len": int(seq_len),
            "seq_feature_dim": int(seq_feature_dim),
            "appaware_K": int(appaware_K),
            "use_ros": bool(use_ros),
            "use_per_app_norm": bool(use_per_app_norm),
            "use_app_aware": bool(use_app_aware),
            "n_train": int(len(y_tr)),
            "n_val": int(len(y_val)),
            "n_test": int(len(y_test)),
            **{k: v for k, v in m.items() if k not in ["cm", "pred_dist"]},
            "cm": m["cm"],
            "pred_dist": m["pred_dist"],
        }

        print(f"[Temporal | LR_{tag}] HELD-OUT={test_app:12s} "
              f"(SEQ_LEN={seq_len}, seqF={seq_feature_dim}) "
              f"acc={row['acc']:.3f} f1={row['f1']:.3f} "
              f"prec={row['precision']:.3f} rec={row['recall']:.3f} "
              f"n_test={row['n_test']}")

        results.append(row)

    return pd.DataFrame(results)

# -------------------------
# Run the three requested experiments (+ baseline for comparison)
# -------------------------
SEQ_LEN = 5
AGG_USE = ("mean","std","delta","range","mean_abs_vel","vel_std")
MIN_NON_NAN_RATIO = 0.4

print("\n================= TEMPORAL LOAO: BASELINE LR =================")
res_base = run_loao_temporal_lr(
    label_col="Temporal",
    seq_len=SEQ_LEN,
    agg_use=AGG_USE,
    min_non_nan_ratio=MIN_NON_NAN_RATIO,
    use_ros=False,
    use_per_app_norm=False,
    use_app_aware=False,
    tag="baseline"
)
base_path = "/content/loao_temporal_lr_baseline.csv"
res_base.to_csv(base_path, index=False)
print("Saved:", base_path)

print("\n================= TEMPORAL LOAO: + RandomOverSampler =================")
if IMBLEARN_OK:
    res_ros = run_loao_temporal_lr(
        label_col="Temporal",
        seq_len=SEQ_LEN,
        agg_use=AGG_USE,
        min_non_nan_ratio=MIN_NON_NAN_RATIO,
        use_ros=True,
        use_per_app_norm=False,
        use_app_aware=False,
        tag="ROS"
    )
    ros_path = "/content/loao_temporal_lr_ros.csv"
    res_ros.to_csv(ros_path, index=False)
    print("Saved:", ros_path)
else:
    res_ros = None

print("\n================= TEMPORAL LOAO: + Per-app normalization =================")
res_norm = run_loao_temporal_lr(
    label_col="Temporal",
    seq_len=SEQ_LEN,
    agg_use=AGG_USE,
    min_non_nan_ratio=MIN_NON_NAN_RATIO,
    use_ros=False,
    use_per_app_norm=True,
    use_app_aware=False,
    tag="PerAppNorm"
)
norm_path = "/content/loao_temporal_lr_perapp_norm.csv"
res_norm.to_csv(norm_path, index=False)
print("Saved:", norm_path)

print("\n================= TEMPORAL LOAO: + App-aware one-hot =================")
res_appaware = run_loao_temporal_lr(
    label_col="Temporal",
    seq_len=SEQ_LEN,
    agg_use=AGG_USE,
    min_non_nan_ratio=MIN_NON_NAN_RATIO,
    use_ros=False,
    use_per_app_norm=False,
    use_app_aware=True,
    tag="AppAware"
)
aa_path = "/content/loao_temporal_lr_appaware.csv"
res_appaware.to_csv(aa_path, index=False)
print("Saved:", aa_path)

# -------------------------
# Summary table
# -------------------------
all_res = [res_base, res_norm, res_appaware]
if res_ros is not None:
    all_res.append(res_ros)

all_res_df = pd.concat(all_res, ignore_index=True)
summary = all_res_df.groupby(["model"])[["acc","f1","precision","recall"]].mean().sort_values("f1", ascending=False)
print("\n================ SUMMARY (mean over held-out apps) ================")
print(summary)

[INFO] imblearn RandomOverSampler imported OK.

================= TEMPORAL LOAO: BASELINE LR =================
[Temporal | LR_baseline] HELD-OUT=Archery      (SEQ_LEN=5, seqF=138) acc=0.875 f1=0.933 prec=0.875 rec=1.000 n_test=96
[Temporal | LR_baseline] HELD-OUT=PhantomLimb  (SEQ_LEN=5, seqF=138) acc=0.941 f1=0.927 prec=0.952 rec=0.902 n_test=321
[Temporal | LR_baseline] HELD-OUT=PianoTiles   (SEQ_LEN=5, seqF=138) acc=0.874 f1=0.444 prec=0.489 rec=0.407 n_test=436
[Temporal | LR_baseline] HELD-OUT=Puzzle       (SEQ_LEN=5, seqF=138) acc=0.549 f1=0.670 prec=0.773 rec=0.592 n_test=275
[Temporal | LR_baseline] HELD-OUT=Sea          (SEQ_LEN=5, seqF=138) acc=0.456 f1=0.625 prec=0.454 rec=1.000 n_test=296
[Temporal | LR_baseline] HELD-OUT=War          (SEQ_LEN=5, seqF=138) acc=0.533 f1=0.641 prec=0.473 rec=0.991 n_test=276
Saved: /content/loao_temporal_lr_baseline.csv

================= TEMPORAL LOAO: + RandomOverSampler =================
[Temporal | LR_ROS] HELD-OUT=Archery      (SEQ_LEN=5

/usr/local/lib/python3.12/dist-packages/sklearn/utils/extmath.py:1101: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/usr/local/lib/python3.12/dist-packages/sklearn/utils/extmath.py:1106: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
/usr/local/lib/python3.12/dist-packages/sklearn/utils/extmath.py:1126: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count
/usr/local/lib/python3.12/dist-packages/sklearn/utils/extmath.py:1101: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/usr/local/lib/python3.12/dist-packages/sklearn/utils/extmath.py:1106: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
/usr/local/lib/python3.12/dist-packages/sklearn/utils/extmath.py:1126: RuntimeWarning: invalid value encountered in divide
  new_unno

[Temporal | LR_PerAppNorm] HELD-OUT=Archery      (SEQ_LEN=5, seqF=138) acc=0.875 f1=0.933 prec=0.875 rec=1.000 n_test=96


/usr/local/lib/python3.12/dist-packages/sklearn/utils/extmath.py:1101: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/usr/local/lib/python3.12/dist-packages/sklearn/utils/extmath.py:1106: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
/usr/local/lib/python3.12/dist-packages/sklearn/utils/extmath.py:1126: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count
/usr/local/lib/python3.12/dist-packages/sklearn/utils/extmath.py:1101: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/usr/local/lib/python3.12/dist-packages/sklearn/utils/extmath.py:1106: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
/usr/local/lib/python3.12/dist-packages/sklearn/utils/extmath.py:1126: RuntimeWarning: invalid value encountered in divide
  new_unno

[Temporal | LR_PerAppNorm] HELD-OUT=PhantomLimb  (SEQ_LEN=5, seqF=138) acc=0.738 f1=0.750 prec=0.621 rec=0.947 n_test=321


...

[Temporal | LR_PerAppNorm] HELD-OUT=Puzzle       (SEQ_LEN=5, seqF=138) acc=0.738 f1=0.849 prec=0.768 rec=0.948 n_test=275


/usr/local/lib/python3.12/dist-packages/sklearn/utils/extmath.py:1101: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/usr/local/lib/python3.12/dist-packages/sklearn/utils/extmath.py:1106: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
/usr/local/lib/python3.12/dist-packages/sklearn/utils/extmath.py:1126: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count
/usr/local/lib/python3.12/dist-packages/sklearn/utils/extmath.py:1101: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/usr/local/lib/python3.12/dist-packages/sklearn/utils/extmath.py:1106: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
/usr/local/lib/python3.12/dist-packages/sklearn/utils/extmath.py:1126: RuntimeWarning: invalid value encountered in divide
  new_unno

[Temporal | LR_PerAppNorm] HELD-OUT=Sea          (SEQ_LEN=5, seqF=138) acc=0.453 f1=0.623 prec=0.453 rec=1.000 n_test=296


/usr/local/lib/python3.12/dist-packages/sklearn/utils/extmath.py:1101: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/usr/local/lib/python3.12/dist-packages/sklearn/utils/extmath.py:1106: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
/usr/local/lib/python3.12/dist-packages/sklearn/utils/extmath.py:1126: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count
/usr/local/lib/python3.12/dist-packages/sklearn/utils/extmath.py:1101: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/usr/local/lib/python3.12/dist-packages/sklearn/utils/extmath.py:1106: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
/usr/local/lib/python3.12/dist-packages/sklearn/utils/extmath.py:1126: RuntimeWarning: invalid value encountered in divide
  new_unno

[Temporal | LR_PerAppNorm] HELD-OUT=War          (SEQ_LEN=5, seqF=138) acc=0.743 f1=0.742 prec=0.642 rec=0.879 n_test=276
Saved: /content/loao_temporal_lr_perapp_norm.csv

================= TEMPORAL LOAO: + App-aware one-hot =================
[Temporal | LR_AppAware] HELD-OUT=Archery      (SEQ_LEN=5, seqF=138) acc=0.875 f1=0.933 prec=0.875 rec=1.000 n_test=96
[Temporal | LR_AppAware] HELD-OUT=PhantomLimb  (SEQ_LEN=5, seqF=138) acc=0.769 f1=0.773 prec=0.653 rec=0.947 n_test=321
[Temporal | LR_AppAware] HELD-OUT=PianoTiles   (SEQ_LEN=5, seqF=138) acc=0.704 f1=0.416 prec=0.275 rec=0.852 n_test=436
[Temporal | LR_AppAware] HELD-OUT=Puzzle       (SEQ_LEN=5, seqF=138) acc=0.444 f1=0.514 prec=0.794 rec=0.380 n_test=275
[Temporal | LR_AppAware] HELD-OUT=Sea          (SEQ_LEN=5, seqF=138) acc=0.449 f1=0.615 prec=0.450 rec=0.970 n_test=296
[Temporal | LR_AppAware] HELD-OUT=War          (SEQ_LEN=5, seqF=138) acc=0.692 f1=0.728 prec=0.579 rec=0.983 n_test=276
Saved: /content/loao_temporal_lr_appaw

# Transformers


In [ ]:
# ===============================================================
# LOAO Transformer (tabular-only) for YOUR VR metrics
# - Spatial: frame-level (T=1)  -> input (N, 1, F)
# - Temporal: sequence-level (T=SEQ_LEN) with your sliding-window + NaN-safe agg (optional)
#   Option A (recommended): raw sequence tokens (N, SEQ_LEN, F)  [Transformer sees time]
#   Label: center frame (i + SEQ_LEN//2), same as your setup
#
# This code:
# 1) Loads extracted_metrics_all_apps.csv (same as before)
# 2) Builds LOAO splits by App
# 3) Creates sequences for Temporal (RAW sequence, no agg)
# 4) Trains a small Transformer with focal loss + early stopping
# 5) Reports per-app acc/prec/rec/f1 + mean summary
# ===============================================================

import os
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.utils import shuffle

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# -------------------------
# 1) LOAD DATA
# -------------------------
CANDIDATE_PATHS = [
    "/content/extracted_metrics_all_apps.csv",
    "/content/drive/MyDrive/extracted_metrics_all_apps.csv",
    "/content/drive/MyDrive/VR_Metrics/extracted_metrics_all_apps.csv",
]
DATA_PATH = None
for p in CANDIDATE_PATHS:
    if os.path.exists(p):
        DATA_PATH = p
        break
if DATA_PATH is None:
    raise FileNotFoundError("Could not find extracted_metrics_all_apps.csv")

df = pd.read_csv(DATA_PATH)
df = df.replace("", np.nan)

required = {"App", "Spatial", "Temporal"}
missing_req = required - set(df.columns)
if missing_req:
    raise ValueError(f"Missing required columns: {missing_req}")

for col in ["Spatial", "Temporal"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=["App"]).copy()
df = df.dropna(subset=["Spatial", "Temporal"], how="all").copy()
df["Spatial"] = df["Spatial"].astype(int)
df["Temporal"] = df["Temporal"].astype(int)

apps = sorted(df["App"].unique().tolist())
print("\n================= LOADED DATA =================")
print("Loaded:", DATA_PATH, "| shape:", df.shape)
print("Apps:", apps)

# Optional: downsample PhantomLimb
DOWNSAMPLE_PHANTOMLIMB = True
TARGET_N_PL = 400
if DOWNSAMPLE_PHANTOMLIMB and ("PhantomLimb" in df["App"].unique()):
    pl = df[df["App"] == "PhantomLimb"]
    others = df[df["App"] != "PhantomLimb"]
    if len(pl) > TARGET_N_PL:
        pl = pl.sample(TARGET_N_PL, random_state=SEED)
    df = pd.concat([others, pl], ignore_index=True)
    apps = sorted(df["App"].unique().tolist())
    print("\n[INFO] Downsample PhantomLimb =", DOWNSAMPLE_PHANTOMLIMB, "| TARGET_N_PL =", TARGET_N_PL)
    print("[INFO] PhantomLimb N =", len(df[df["App"] == "PhantomLimb"]), "| Total N =", len(df))

# -------------------------
# 2) FEATURES
# -------------------------
ID_COLS = [c for c in ["GlobalID", "EntryID"] if c in df.columns]
DROP_COLS = ["App", "Spatial", "Temporal"] + ID_COLS
feature_cols = [c for c in df.columns if c not in DROP_COLS]

for c in feature_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

F = len(feature_cols)
print("\n================= FEATURES =================")
print("Num features:", F)
print("First 25 feature cols:", feature_cols[:25])

# -------------------------
# 3) METRICS
# -------------------------
def compute_metrics(y_true, y_pred):
    return (
        float(accuracy_score(y_true, y_pred)),
        float(precision_score(y_true, y_pred, zero_division=0)),
        float(recall_score(y_true, y_pred, zero_division=0)),
        float(f1_score(y_true, y_pred, zero_division=0)),
    )

def print_eval(prefix, y_true, y_pred):
    acc, prec, rec, f1 = compute_metrics(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred).tolist()
    return acc, prec, rec, f1, cm

# -------------------------
# 4) FOCAL LOSS (same spirit as yours)
# -------------------------
import tensorflow.keras.backend as K
def focal_loss(alpha=0.25, gamma=4.0):
    def loss(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        bce = K.binary_crossentropy(y_true, y_pred)
        p_t = y_true * y_pred + (1 - y_true) * (1 - y_pred)
        loss = alpha * K.pow((1 - p_t), gamma) * bce
        return loss
    return loss

# -------------------------
# 5) TRANSFORMER ENCODER + MODEL
# Small + stable (don’t overdo capacity on 1.7k rows)
# -------------------------
from tensorflow.keras.layers import Input, Dense, Dropout, LayerNormalization, MultiHeadAttention, GlobalAveragePooling1D
from tensorflow.keras.models import Model

def transformer_encoder(inputs, num_heads=2, key_dim=32, ff_dim=128, dropout_rate=0.1):
    attn = MultiHeadAttention(num_heads=num_heads, key_dim=key_dim)(inputs, inputs)
    attn = Dropout(dropout_rate)(attn)
    x = LayerNormalization(epsilon=1e-6)(inputs + attn)

    ffn = Dense(ff_dim, activation="relu")(x)
    ffn = Dense(inputs.shape[-1])(ffn)
    ffn = Dropout(dropout_rate)(ffn)
    out = LayerNormalization(epsilon=1e-6)(x + ffn)
    return out

def create_tabular_transformer(tabular_input_shape, embed_dim=64,
                              num_heads=2, key_dim=32, ff_dim=128,
                              depth=2, dropout_rate=0.1,
                              lr=1e-4):
    inp = Input(shape=tabular_input_shape, name="tabular_input")  # (T, F)

    # Token embedding per timestep
    x = Dense(embed_dim, activation="relu")(inp)

    for _ in range(depth):
        x = transformer_encoder(x, num_heads=num_heads, key_dim=key_dim, ff_dim=ff_dim, dropout_rate=dropout_rate)

    x = GlobalAveragePooling1D()(x)
    x = Dense(64, activation="relu")(x)
    x = Dropout(dropout_rate)(x)
    out = Dense(1, activation="sigmoid")(x)

    model = Model(inp, out)
    opt = tf.keras.optimizers.Adam(learning_rate=lr)
    model.compile(optimizer=opt, loss=focal_loss(), metrics=["accuracy"])
    return model

# -------------------------
# 6) DATA BUILDERS
# -------------------------
def impute_per_split(X_train, X_test):
    """Median impute using ONLY training split statistics."""
    med = np.nanmedian(X_train, axis=0)
    med = np.where(np.isfinite(med), med, 0.0)

    X_train_f = np.where(np.isfinite(X_train), X_train, med)
    X_test_f  = np.where(np.isfinite(X_test),  med, X_test)  # careful broadcast shape
    # X_test may be 3D; handle properly
    if X_test.ndim == 3:
        X_test_f = np.where(np.isfinite(X_test), X_test, med[None, None, :])
    else:
        X_test_f = np.where(np.isfinite(X_test), X_test, med)

    return X_train_f, X_test_f, med

def standardize_per_split(X_train, X_test, eps=1e-8):
    """Standardize using ONLY training split statistics."""
    mu = X_train.mean(axis=0)
    sd = X_train.std(axis=0)
    sd = np.where(sd < eps, 1.0, sd)

    if X_train.ndim == 3:
        X_train_z = (X_train - mu[None, None, :]) / sd[None, None, :]
        X_test_z  = (X_test  - mu[None, None, :]) / sd[None, None, :]
    else:
        X_train_z = (X_train - mu) / sd
        X_test_z  = (X_test  - mu) / sd

    return X_train_z, X_test_z

def make_raw_sequences(app_df, label_col, seq_len=5):
    """
    RAW sequence tokens:
      X_seq: (N_seq, seq_len, F)
      y_seq: (N_seq,) label from center frame
    """
    X = app_df[feature_cols].values.astype(float)
    y = app_df[label_col].values.astype(int)

    if len(app_df) < seq_len:
        return np.empty((0, seq_len, F), dtype=float), np.empty((0,), dtype=int)

    center = seq_len // 2
    Xs, ys = [], []
    for i in range(len(app_df) - seq_len + 1):
        window = X[i:i+seq_len]
        Xs.append(window)
        ys.append(y[i + center])

    return np.stack(Xs, axis=0), np.array(ys, dtype=int)

# -------------------------
# 7) LOAO RUNNERS
# -------------------------
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

def train_and_eval_transformer_loao_frame(label_col="Spatial",
                                         epochs=25, batch_size=32):
    """
    Frame-level: treat each sample as a 1-token sequence (T=1).
    Input shape: (1, F)
    """
    rows = []
    for test_app in apps:
        train_df = df[df["App"] != test_app].copy()
        test_df  = df[df["App"] == test_app].copy()

        X_train_full = train_df[feature_cols].values.astype(float)
        y_train_full = train_df[label_col].values.astype(int)
        X_test = test_df[feature_cols].values.astype(float)
        y_test = test_df[label_col].values.astype(int)

        # inner val split
        X_tr, X_val, y_tr, y_val = train_test_split(
            X_train_full, y_train_full,
            test_size=0.15, random_state=SEED, stratify=y_train_full
        )

        # impute + standardize using training only
        X_tr, X_val, _ = impute_per_split(X_tr, X_val)
        X_tr, X_val = standardize_per_split(X_tr, X_val)

        # apply same stats to test using training med/mu/sd:
        # easiest: re-run impute & standardize with train stats by concatenation trick
        X_tr2, X_test2, _ = impute_per_split(X_tr, X_test)  # X_tr already imputed but ok
        X_tr2, X_test2 = standardize_per_split(X_tr2, X_test2)

        # reshape to (N, 1, F)
        X_tr3  = X_tr2[:, None, :]
        X_val3 = X_val[:, None, :]
        X_te3  = X_test2[:, None, :]

        model = create_tabular_transformer(tabular_input_shape=(1, F),
                                           embed_dim=64, num_heads=2, key_dim=32,
                                           ff_dim=128, depth=2, dropout_rate=0.1, lr=1e-4)

        callbacks = [
            EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True, verbose=0),
            ReduceLROnPlateau(monitor="val_accuracy", factor=0.2, patience=2, min_lr=1e-7, verbose=0),
        ]

        model.fit(X_tr3, y_tr, validation_data=(X_val3, y_val),
                  epochs=epochs, batch_size=batch_size, verbose=0, callbacks=callbacks)

        y_prob = model.predict(X_te3, verbose=0).reshape(-1)
        y_pred = (y_prob >= 0.5).astype(int)

        acc, prec, rec, f1, cm = print_eval("", y_test, y_pred)
        print(f"[{label_col} | T-Frame] HELD-OUT={test_app:12s} acc={acc:.3f} f1={f1:.3f} prec={prec:.3f} rec={rec:.3f} n_test={len(y_test)}")

        rows.append({
            "label": label_col, "task":"frame", "model":"Transformer_frame",
            "held_out_app": test_app, "n_test": int(len(y_test)),
            "acc": acc, "precision": prec, "recall": rec, "f1": f1, "cm": cm
        })

        tf.keras.backend.clear_session()

    return pd.DataFrame(rows)

def train_and_eval_transformer_loao_temporal_raw(label_col="Temporal",
                                                 seq_len=5,
                                                 epochs=25, batch_size=32):
    """
    Temporal: RAW sequences (N, seq_len, F) -> Transformer
    Label from center frame like your pipeline.
    """
    rows = []
    for test_app in apps:
        # build train sequences from all other apps
        X_train_list, y_train_list = [], []
        for a in apps:
            if a == test_app:
                continue
            Xa, ya = make_raw_sequences(df[df["App"] == a], label_col=label_col, seq_len=seq_len)
            if len(ya) > 0:
                X_train_list.append(Xa)
                y_train_list.append(ya)

        if len(X_train_list) == 0:
            print(f"[{label_col} | T-SeqRaw] HELD-OUT={test_app} -> no train seqs")
            continue

        X_train_full = np.concatenate(X_train_list, axis=0).astype(float)  # (N, T, F)
        y_train_full = np.concatenate(y_train_list, axis=0).astype(int)

        # test sequences
        X_test, y_test = make_raw_sequences(df[df["App"] == test_app], label_col=label_col, seq_len=seq_len)
        if len(y_test) == 0:
            print(f"[{label_col} | T-SeqRaw] HELD-OUT={test_app} -> no test seqs")
            continue

        # inner val split (stratify on y)
        idx = np.arange(len(y_train_full))
        tr_idx, val_idx = train_test_split(idx, test_size=0.15, random_state=SEED, stratify=y_train_full)

        X_tr = X_train_full[tr_idx]
        y_tr = y_train_full[tr_idx]
        X_val = X_train_full[val_idx]
        y_val = y_train_full[val_idx]

        # impute + standardize using ONLY X_tr stats, feature-wise (across all timesteps)
        X_tr_flat = X_tr.reshape(-1, F)  # (N*T, F)
        X_val_flat = X_val.reshape(-1, F)
        X_te_flat = X_test.reshape(-1, F)

        X_tr_flat, X_val_flat, med = impute_per_split(X_tr_flat, X_val_flat)
        # impute test using same med
        X_te_flat = np.where(np.isfinite(X_te_flat), X_te_flat, med)

        # standardize using X_tr_flat stats
        mu = X_tr_flat.mean(axis=0)
        sd = X_tr_flat.std(axis=0)
        sd = np.where(sd < 1e-8, 1.0, sd)

        X_tr_flat = (X_tr_flat - mu) / sd
        X_val_flat = (X_val_flat - mu) / sd
        X_te_flat = (X_te_flat - mu) / sd

        # reshape back
        X_tr2 = X_tr_flat.reshape(-1, seq_len, F)
        X_val2 = X_val_flat.reshape(-1, seq_len, F)
        X_te2 = X_te_flat.reshape(-1, seq_len, F)

        model = create_tabular_transformer(tabular_input_shape=(seq_len, F),
                                           embed_dim=64, num_heads=2, key_dim=32,
                                           ff_dim=128, depth=2, dropout_rate=0.1, lr=1e-4)

        callbacks = [
            EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True, verbose=0),
            ReduceLROnPlateau(monitor="val_accuracy", factor=0.2, patience=2, min_lr=1e-7, verbose=0),
        ]

        model.fit(X_tr2, y_tr, validation_data=(X_val2, y_val),
                  epochs=epochs, batch_size=batch_size, verbose=0, callbacks=callbacks)

        y_prob = model.predict(X_te2, verbose=0).reshape(-1)
        y_pred = (y_prob >= 0.5).astype(int)

        acc, prec, rec, f1, cm = print_eval("", y_test, y_pred)
        print(f"[{label_col} | T-SeqRaw] HELD-OUT={test_app:12s} (SEQ_LEN={seq_len}) "
              f"acc={acc:.3f} f1={f1:.3f} prec={prec:.3f} rec={rec:.3f} n_test={len(y_test)}")

        rows.append({
            "label": label_col, "task": f"sequence_raw_len_{seq_len}", "model":"Transformer_seqraw",
            "held_out_app": test_app, "seq_len": int(seq_len), "n_test": int(len(y_test)),
            "acc": acc, "precision": prec, "recall": rec, "f1": f1, "cm": cm
        })

        tf.keras.backend.clear_session()

    return pd.DataFrame(rows)

# -------------------------
# 8) RUN BOTH (Spatial + Temporal)
# -------------------------
SEQ_LEN = 5
EPOCHS = 25
BATCH = 32

print("\n================= LOAO: SPATIAL (FRAME) - TRANSFORMER =================")
res_sp = train_and_eval_transformer_loao_frame(label_col="Spatial", epochs=EPOCHS, batch_size=BATCH)
out_sp = "/content/loao_spatial_transformer_frame.csv"
res_sp.to_csv(out_sp, index=False)
print("Saved:", out_sp)

print("\n================= LOAO: TEMPORAL (RAW SEQ) - TRANSFORMER =================")
res_tmp = train_and_eval_transformer_loao_temporal_raw(label_col="Temporal", seq_len=SEQ_LEN, epochs=EPOCHS, batch_size=BATCH)
out_tmp = f"/content/loao_temporal_transformer_seqraw_len{SEQ_LEN}.csv"
res_tmp.to_csv(out_tmp, index=False)
print("Saved:", out_tmp)

print("\n================ SUMMARY =================")
all_df = pd.concat([res_sp, res_tmp], ignore_index=True)
summary = all_df.groupby(["label","task","model"])[["acc","precision","recall","f1"]].mean().reset_index()
print(summary)

out_all = "/content/loao_transformer_spatial_temporal.csv"
all_df.to_csv(out_all, index=False)
print("Saved:", out_all)


================= LOADED DATA =================
Loaded: /content/extracted_metrics_all_apps.csv | shape: (1744, 28)
Apps: ['Archery', 'PhantomLimb', 'PianoTiles', 'Puzzle', 'Sea', 'War']

[INFO] Downsample PhantomLimb = True | TARGET_N_PL = 400
[INFO] PhantomLimb N = 325 | Total N = 1744

================= FEATURES =================
Num features: 23
First 25 feature cols: ['missing_joints_count', 'missing_joints_ratio', 'collapsed_joints_count', 'center_of_mass_x', 'center_of_mass_y', 'center_of_mass_z', 'distance_from_origin', 'bbox_width', 'bbox_height', 'bbox_depth', 'bbox_volume', 'max_joint_distance_from_com', 'distance_from_floor', 'below_floor', 'left_forearm_length', 'right_forearm_length', 'left_shin_length', 'right_shin_length', 'arm_length_symmetry', 'leg_length_symmetry', 'body_forward_x', 'body_forward_y', 'body_forward_z']

================= LOAO: SPATIAL (FRAME) - TRANSFORMER =================
[Spatial | T-Frame] HELD-OUT=Archery      acc=0.140 f1=0.000 prec=0.000 rec=0

In [ ]:
# ===============================================================
# VR APP-INDEPENDENT METRICS - LOAO (Spatial + Temporal Transformer)
# FULL SCRIPT (single block)
#
# - Loads extracted_metrics_all_apps.csv
# - Optional: downsample PhantomLimb
# - Spatial: "Transformer" (implemented as token-per-feature; otherwise T=1 is meaningless)
# - Temporal: raw sequences (SEQ_LEN) -> Transformer with per-sequence normalization
#
# Notes:
# - Temporal sequence dataset uses RAW windows (no aggregation) and center-frame label (i+SEQ_LEN//2)
# - No SMOTE/ROS here (you can add later)
# ===============================================================

import os
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# =========================
# 1) LOAD DATA
# =========================
CANDIDATE_PATHS = [
    "/content/extracted_metrics_all_apps.csv",
    "/content/drive/MyDrive/extracted_metrics_all_apps.csv",
    "/content/drive/MyDrive/VR_Metrics/extracted_metrics_all_apps.csv",
]

DATA_PATH = None
for p in CANDIDATE_PATHS:
    if os.path.exists(p):
        DATA_PATH = p
        break

if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find extracted_metrics_all_apps.csv in common locations.\n"
        "Put it in /content OR adjust CANDIDATE_PATHS."
    )

df = pd.read_csv(DATA_PATH)
print("\n================= LOADED DATA =================")
print("Loaded:", DATA_PATH, "| shape:", df.shape)

# =========================
# 2) CLEAN / NORMALIZE TYPES
# =========================
df = df.replace("", np.nan)

required = {"App", "Spatial", "Temporal"}
missing_req = required - set(df.columns)
if missing_req:
    raise ValueError(f"Missing required columns in CSV: {missing_req}")

for col in ["Spatial", "Temporal"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=["App"]).copy()
df = df.dropna(subset=["Spatial", "Temporal"], how="all").copy()

df["Spatial"] = df["Spatial"].astype(int)
df["Temporal"] = df["Temporal"].astype(int)

apps = sorted(df["App"].unique().tolist())
print("Apps:", apps)

# =========================
# OPTIONAL: Downsample PhantomLimb
# =========================
DOWNSAMPLE_PHANTOMLIMB = True
TARGET_N_PL = 400

if DOWNSAMPLE_PHANTOMLIMB and ("PhantomLimb" in df["App"].unique()):
    pl = df[df["App"] == "PhantomLimb"]
    others = df[df["App"] != "PhantomLimb"]
    if len(pl) > TARGET_N_PL:
        pl = pl.sample(TARGET_N_PL, random_state=SEED)
    df = pd.concat([others, pl], ignore_index=True)
    apps = sorted(df["App"].unique().tolist())
    print("\n[INFO] Downsample PhantomLimb =", DOWNSAMPLE_PHANTOMLIMB,
          "| TARGET_N_PL =", TARGET_N_PL)
    print("[INFO] PhantomLimb N =", len(df[df["App"] == "PhantomLimb"]),
          "| Total N =", len(df))

# =========================
# 3) FEATURE COLUMN SELECTION
# =========================
ID_COLS = [c for c in ["GlobalID", "EntryID"] if c in df.columns]
DROP_COLS = ["App", "Spatial", "Temporal"] + ID_COLS
feature_cols = [c for c in df.columns if c not in DROP_COLS]

for c in feature_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

print("\n================= FEATURES =================")
print("Num features:", len(feature_cols))
print("First 25 feature cols:", feature_cols[:25])

# =========================
# 4) METRICS
# =========================
def eval_binary(y_true, y_pred):
    return {
        "acc": float(accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "cm": confusion_matrix(y_true, y_pred).tolist(),
        "pred_dist": dict(zip(*np.unique(y_pred, return_counts=True))),
    }

# ===============================================================
# 5) TRANSFORMER BUILDERS
# ===============================================================

from tensorflow.keras.layers import (
    Input, Dense, Dropout, LayerNormalization, MultiHeadAttention,
    GlobalAveragePooling1D, Lambda, Reshape
)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

def transformer_encoder(inputs, num_heads=2, key_dim=32, ff_dim=128, dropout_rate=0.1):
    attn = MultiHeadAttention(num_heads=num_heads, key_dim=key_dim)(inputs, inputs)
    attn = Dropout(dropout_rate)(attn)
    x = LayerNormalization(epsilon=1e-6)(inputs + attn)

    ffn = Dense(ff_dim, activation="relu")(x)
    ffn = Dense(inputs.shape[-1])(ffn)
    ffn = Dropout(dropout_rate)(ffn)
    return LayerNormalization(epsilon=1e-6)(x + ffn)

# ----- Spatial transformer (token-per-feature) -----
# This avoids the "T=1 token" issue. We treat each feature as a token.
def create_spatial_transformer_featuretokens(F,
                                             embed_dim=32, depth=2,
                                             num_heads=2, key_dim=16, ff_dim=64,
                                             dropout=0.1, lr=1e-3):
    inp = Input(shape=(F,), name="x_frame")  # (F,)

    # Replace NaNs safely inside TF
    x = Lambda(lambda t: tf.where(tf.math.is_finite(t), t, tf.zeros_like(t)))(inp)

    # Make tokens: (F, 1)
    x = Reshape((F, 1))(x)

    # Token embedding: (F, embed_dim)
    x = Dense(embed_dim, activation="relu")(x)

    # Transformer over feature tokens
    for _ in range(depth):
        x = transformer_encoder(x, num_heads=num_heads, key_dim=key_dim, ff_dim=ff_dim, dropout_rate=dropout)

    x = GlobalAveragePooling1D()(x)
    x = Dense(64, activation="relu")(x)
    x = Dropout(dropout)(x)
    out = Dense(1, activation="sigmoid")(x)

    model = Model(inp, out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(lr),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model

# ----- Temporal transformer (raw seq) with per-sequence normalization -----
def create_temporal_transformer_seqnorm(seq_len, F,
                                       embed_dim=64, depth=2,
                                       num_heads=2, key_dim=32, ff_dim=128,
                                       dropout=0.1, lr=1e-4):
    inp = Input(shape=(seq_len, F), name="x_seq")  # (T,F)

    # Replace NaNs with 0 first
    x = Lambda(lambda t: tf.where(tf.math.is_finite(t), t, tf.zeros_like(t)))(inp)

    # Per-sequence normalization across time (no test leakage: uses only the sequence)
    def seq_norm(z):
        mu = tf.reduce_mean(z, axis=1, keepdims=True)
        sd = tf.math.reduce_std(z, axis=1, keepdims=True)
        sd = tf.where(sd < 1e-6, tf.ones_like(sd), sd)
        return (z - mu) / sd

    x = Lambda(seq_norm, name="seq_norm")(x)

    # Token embedding over time tokens
    x = Dense(embed_dim, activation="relu")(x)

    for _ in range(depth):
        x = transformer_encoder(x, num_heads=num_heads, key_dim=key_dim, ff_dim=ff_dim, dropout_rate=dropout)

    x = GlobalAveragePooling1D()(x)
    x = Dense(64, activation="relu")(x)
    x = Dropout(dropout)(x)
    out = Dense(1, activation="sigmoid")(x)

    model = Model(inp, out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(lr),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model

# ===============================================================
# 6) DATASET BUILDERS
# ===============================================================

def make_raw_sequence_dataset(app_df, label_col, seq_len=5):
    """
    Raw sliding windows: X_seq shape (Nseq, seq_len, F)
    Label = center frame (i + seq_len//2)
    """
    X = app_df[feature_cols].values.astype(np.float32)
    y = app_df[label_col].values.astype(int)

    if len(app_df) < seq_len:
        return np.empty((0, seq_len, len(feature_cols)), dtype=np.float32), np.empty((0,), dtype=int)

    X_seq, y_seq = [], []
    center = seq_len // 2

    for i in range(len(app_df) - seq_len + 1):
        window = X[i:i+seq_len]
        X_seq.append(window)
        y_seq.append(y[i + center])

    return np.stack(X_seq, axis=0), np.array(y_seq, dtype=int)

# ===============================================================
# 7) LOAO RUNNERS
# ===============================================================

def run_loao_spatial_transformer():
    results = []
    F = len(feature_cols)

    for test_app in apps:
        train_df = df[df["App"] != test_app].copy()
        test_df  = df[df["App"] == test_app].copy()

        X_train_full = train_df[feature_cols].values.astype(np.float32)
        y_train_full = train_df["Spatial"].values.astype(int)

        X_test = test_df[feature_cols].values.astype(np.float32)
        y_test = test_df["Spatial"].values.astype(int)

        # Split train/val
        X_tr, X_val, y_tr, y_val = train_test_split(
            X_train_full, y_train_full,
            test_size=0.15,
            random_state=SEED,
            stratify=y_train_full
        )

        # Impute + scale using TRAIN only (outside TF for stability)
        imp = SimpleImputer(strategy="median")
        sca = StandardScaler()

        X_tr = sca.fit_transform(imp.fit_transform(X_tr)).astype(np.float32)
        X_val = sca.transform(imp.transform(X_val)).astype(np.float32)
        X_test2 = sca.transform(imp.transform(X_test)).astype(np.float32)

        # Class weights from TRAIN only
        cw = compute_class_weight(class_weight="balanced", classes=np.array([0, 1]), y=y_tr)
        class_weight = {0: float(cw[0]), 1: float(cw[1])}

        model = create_spatial_transformer_featuretokens(F=F)

        callbacks = [
            EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True, verbose=0),
            ReduceLROnPlateau(monitor="val_accuracy", factor=0.2, patience=2, min_lr=1e-7, verbose=0),
        ]

        model.fit(
            X_tr, y_tr,
            validation_data=(X_val, y_val),
            epochs=25,
            batch_size=32,
            verbose=0,
            callbacks=callbacks,
            class_weight=class_weight
        )

        y_prob = model.predict(X_test2, verbose=0).reshape(-1)
        y_pred = (y_prob >= 0.5).astype(int)

        m = eval_binary(y_test, y_pred)

        row = {
            "label": "Spatial",
            "task": "frame",
            "model": "Transformer_frame_featuretokens",
            "held_out_app": test_app,
            "n_train": int(len(y_tr)),
            "n_val": int(len(y_val)),
            "n_test": int(len(y_test)),
            **{k: v for k, v in m.items() if k not in ["cm", "pred_dist"]},
            "cm": m["cm"],
            "pred_dist": m["pred_dist"],
        }

        print(f"[Spatial | T-Frame] HELD-OUT={test_app:12s} "
              f"acc={row['acc']:.3f} f1={row['f1']:.3f} "
              f"prec={row['precision']:.3f} rec={row['recall']:.3f} "
              f"n_test={row['n_test']}")
        results.append(row)

    out = pd.DataFrame(results)
    out_path = "/content/loao_spatial_transformer_frame.csv"
    out.to_csv(out_path, index=False)
    print("Saved:", out_path)
    return out

def run_loao_temporal_transformer(seq_len=5):
    results = []
    F = len(feature_cols)

    for test_app in apps:
        # Build train sequences from all other apps (no leakage)
        X_train_list, y_train_list = [], []
        for a in apps:
            if a == test_app:
                continue
            Xa, ya = make_raw_sequence_dataset(df[df["App"] == a], label_col="Temporal", seq_len=seq_len)
            if len(ya) > 0:
                X_train_list.append(Xa)
                y_train_list.append(ya)

        if len(X_train_list) == 0:
            print(f"[Temporal | T-SeqRaw] HELD-OUT={test_app} -> No train sequences.")
            continue

        X_train_full = np.concatenate(X_train_list, axis=0)
        y_train_full = np.concatenate(y_train_list, axis=0)

        # Test sequences
        test_df = df[df["App"] == test_app].copy()
        X_test, y_test = make_raw_sequence_dataset(test_df, label_col="Temporal", seq_len=seq_len)

        if len(y_test) == 0:
            print(f"[Temporal | T-SeqRaw] HELD-OUT={test_app} -> No test sequences.")
            continue

        # Train/val split on sequences
        X_tr, X_val, y_tr, y_val = train_test_split(
            X_train_full, y_train_full,
            test_size=0.15,
            random_state=SEED,
            stratify=y_train_full
        )

        # Impute + scale per-feature using TRAIN ONLY (flatten -> fit -> reshape)
        imp = SimpleImputer(strategy="median")
        sca = StandardScaler()

        def fit_transform_seq(X, fit=False):
            # X: (N,T,F)
            N, T, F_ = X.shape
            X2 = X.reshape(N*T, F_)
            if fit:
                X2 = imp.fit_transform(X2)
                X2 = sca.fit_transform(X2)
            else:
                X2 = imp.transform(X2)
                X2 = sca.transform(X2)
            return X2.reshape(N, T, F_).astype(np.float32)

        X_tr2 = fit_transform_seq(X_tr, fit=True)
        X_val2 = fit_transform_seq(X_val, fit=False)
        X_te2  = fit_transform_seq(X_test, fit=False)

        # Class weights from TRAIN ONLY
        cw = compute_class_weight(class_weight="balanced", classes=np.array([0, 1]), y=y_tr)
        class_weight = {0: float(cw[0]), 1: float(cw[1])}

        model = create_temporal_transformer_seqnorm(seq_len=seq_len, F=F)

        callbacks = [
            EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True, verbose=0),
            ReduceLROnPlateau(monitor="val_accuracy", factor=0.2, patience=2, min_lr=1e-7, verbose=0),
        ]

        model.fit(
            X_tr2, y_tr,
            validation_data=(X_val2, y_val),
            epochs=25,
            batch_size=32,
            verbose=0,
            callbacks=callbacks,
            class_weight=class_weight
        )

        y_prob = model.predict(X_te2, verbose=0).reshape(-1)
        y_pred = (y_prob >= 0.5).astype(int)

        m = eval_binary(y_test, y_pred)

        row = {
            "label": "Temporal",
            "task": f"sequence_raw_len_{seq_len}",
            "model": "Transformer_seqraw_seqnorm",
            "held_out_app": test_app,
            "seq_len": int(seq_len),
            "n_train": int(len(y_tr)),
            "n_val": int(len(y_val)),
            "n_test": int(len(y_test)),
            **{k: v for k, v in m.items() if k not in ["cm", "pred_dist"]},
            "cm": m["cm"],
            "pred_dist": m["pred_dist"],
        }

        print(f"[Temporal | T-SeqRaw] HELD-OUT={test_app:12s} "
              f"(SEQ_LEN={seq_len}) acc={row['acc']:.3f} f1={row['f1']:.3f} "
              f"prec={row['precision']:.3f} rec={row['recall']:.3f} "
              f"n_test={row['n_test']}")
        results.append(row)

    out = pd.DataFrame(results)
    out_path = f"/content/loao_temporal_transformer_seqraw_len{seq_len}.csv"
    out.to_csv(out_path, index=False)
    print("Saved:", out_path)
    return out

# ===============================================================
# 8) RUN
# ===============================================================
SEQ_LEN = 5

print("\n================= LOAO: SPATIAL (FRAME) - TRANSFORMER =================")
spatial_df = run_loao_spatial_transformer()

print("\n================= LOAO: TEMPORAL (RAW SEQ) - TRANSFORMER =================")
temporal_df = run_loao_temporal_transformer(seq_len=SEQ_LEN)

results_df = pd.concat([spatial_df, temporal_df], ignore_index=True)

print("\n================ SUMMARY =================")
summary = results_df.groupby(["label", "task", "model"])[["acc", "precision", "recall", "f1"]].mean().reset_index()
print(summary)

OUT_PATH = "/content/loao_transformer_spatial_temporal.csv"
results_df.to_csv(OUT_PATH, index=False)
print("Saved:", OUT_PATH)


================= LOADED DATA =================
Loaded: /content/extracted_metrics_all_apps.csv | shape: (1744, 28)
Apps: ['Archery', 'PhantomLimb', 'PianoTiles', 'Puzzle', 'Sea', 'War']

[INFO] Downsample PhantomLimb = True | TARGET_N_PL = 400
[INFO] PhantomLimb N = 325 | Total N = 1744

================= FEATURES =================
Num features: 23
First 25 feature cols: ['missing_joints_count', 'missing_joints_ratio', 'collapsed_joints_count', 'center_of_mass_x', 'center_of_mass_y', 'center_of_mass_z', 'distance_from_origin', 'bbox_width', 'bbox_height', 'bbox_depth', 'bbox_volume', 'max_joint_distance_from_com', 'distance_from_floor', 'below_floor', 'left_forearm_length', 'right_forearm_length', 'left_shin_length', 'right_shin_length', 'arm_length_symmetry', 'leg_length_symmetry', 'body_forward_x', 'body_forward_y', 'body_forward_z']

================= LOAO: SPATIAL (FRAME) - TRANSFORMER =================
[Spatial | T-Frame] HELD-OUT=Archery      acc=0.140 f1=0.000 prec=0.000 rec=0

[Spatial | T-Frame] HELD-OUT=PhantomLimb  acc=0.578 f1=0.055 prec=0.035 rec=0.125 n_test=325
[Spatial | T-Frame] HELD-OUT=PianoTiles   acc=0.518 f1=0.000 prec=0.000 rec=0.000 n_test=440
[Spatial | T-Frame] HELD-OUT=Puzzle       acc=0.261 f1=0.000 prec=0.000 rec=0.000 n_test=299
[Spatial | T-Frame] HELD-OUT=Sea          acc=0.560 f1=0.000 prec=0.000 rec=0.000 n_test=300
[Spatial | T-Frame] HELD-OUT=War          acc=0.421 f1=0.557 prec=0.389 rec=0.981 n_test=280
Saved: /content/loao_spatial_transformer_frame.csv

================= LOAO: TEMPORAL (RAW SEQ) - TRANSFORMER =================
[Temporal | T-SeqRaw] HELD-OUT=Archery      (SEQ_LEN=5) acc=0.271 f1=0.352 prec=0.792 rec=0.226 n_test=96
[Temporal | T-SeqRaw] HELD-OUT=PhantomLimb  (SEQ_LEN=5) acc=0.536 f1=0.259 prec=0.382 rec=0.195 n_test=321
[Temporal | T-SeqRaw] HELD-OUT=PianoTiles   (SEQ_LEN=5) acc=0.569 f1=0.324 prec=0.201 rec=0.833 n_test=436
[Temporal | T-SeqRaw] HELD-OUT=Puzzle       (SEQ_LEN=5) acc=0.739 f1=0.816 prec=0.830 re

In [ ]:
# ===============================================================
# LOAO TEMPORAL (RAW SEQ) - TRANSFORMER with stability fixes
# - Uses raw sequences: (SEQ_LEN, F=23)
# - Per-fold scaler fit on TRAIN only (no leakage)
# - Class weights
# - Fixed predict batch_size to reduce TF retracing
# - clear_session() each fold
# ===============================================================

import os, random
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# -------------------------
# 1) LOAD DATA
# -------------------------
CANDIDATE_PATHS = [
    "/content/extracted_metrics_all_apps.csv",
    "/content/drive/MyDrive/extracted_metrics_all_apps.csv",
    "/content/drive/MyDrive/VR_Metrics/extracted_metrics_all_apps.csv",
]
DATA_PATH = next((p for p in CANDIDATE_PATHS if os.path.exists(p)), None)
if DATA_PATH is None:
    raise FileNotFoundError("Could not find extracted_metrics_all_apps.csv")

df = pd.read_csv(DATA_PATH).replace("", np.nan)

required = {"App", "Temporal"}
missing_req = required - set(df.columns)
if missing_req:
    raise ValueError(f"Missing required columns: {missing_req}")

df["Temporal"] = pd.to_numeric(df["Temporal"], errors="coerce")
df = df.dropna(subset=["App", "Temporal"]).copy()
df["Temporal"] = df["Temporal"].astype(int)

# optional downsample PhantomLimb
DOWNSAMPLE_PHANTOMLIMB = True
TARGET_N_PL = 400
if DOWNSAMPLE_PHANTOMLIMB and ("PhantomLimb" in df["App"].unique()):
    pl = df[df["App"] == "PhantomLimb"]
    others = df[df["App"] != "PhantomLimb"]
    if len(pl) > TARGET_N_PL:
        pl = pl.sample(TARGET_N_PL, random_state=SEED)
    df = pd.concat([others, pl], ignore_index=True)

apps = sorted(df["App"].unique().tolist())
print("\n================= LOADED DATA =================")
print("Loaded:", DATA_PATH, "| shape:", df.shape)
print("Apps:", apps)

# -------------------------
# 2) FEATURE COLS
# -------------------------
ID_COLS = [c for c in ["GlobalID", "EntryID"] if c in df.columns]
DROP_COLS = ["App", "Spatial", "Temporal"] + ID_COLS
feature_cols = [c for c in df.columns if c not in DROP_COLS]

for c in feature_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

print("\n================= FEATURES =================")
print("Num features:", len(feature_cols))
print("First 25 feature cols:", feature_cols[:25])

# -------------------------
# 3) METRICS
# -------------------------
def eval_binary(y_true, y_pred):
    return {
        "acc": float(accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "cm": confusion_matrix(y_true, y_pred).tolist(),
        "pred_dist": dict(zip(*np.unique(y_pred, return_counts=True))),
    }

# -------------------------
# 4) SEQUENCE DATA (RAW)
# -------------------------
def make_raw_sequence_dataset(app_df, label_col="Temporal", seq_len=5, min_non_nan_ratio=0.4):
    X = app_df[feature_cols].values.astype(float)
    y = app_df[label_col].values.astype(int)

    if len(app_df) < seq_len:
        return np.empty((0, seq_len, len(feature_cols))), np.empty((0,), dtype=int)

    X_seq, y_seq = [], []
    center = seq_len // 2

    for i in range(len(app_df) - seq_len + 1):
        window = X[i:i+seq_len]
        valid = np.isfinite(window).sum()
        total = window.size
        if (valid / total) < min_non_nan_ratio:
            continue
        X_seq.append(window)
        y_seq.append(y[i + center])

    if len(X_seq) == 0:
        return np.empty((0, seq_len, len(feature_cols))), np.empty((0,), dtype=int)

    return np.stack(X_seq, axis=0), np.array(y_seq, dtype=int)

# -------------------------
# 5) TRANSFORMER MODEL (SMALL + STABLE)
# -------------------------
from tensorflow.keras.layers import Input, Dense, Dropout, LayerNormalization, MultiHeadAttention, GlobalAveragePooling1D
from tensorflow.keras.models import Model

def transformer_encoder(x, num_heads=2, key_dim=16, ff_dim=64, dropout=0.15):
    attn = MultiHeadAttention(num_heads=num_heads, key_dim=key_dim)(x, x)
    attn = Dropout(dropout)(attn)
    x = LayerNormalization(epsilon=1e-6)(x + attn)

    ffn = Dense(ff_dim, activation="relu")(x)
    ffn = Dense(x.shape[-1])(ffn)
    ffn = Dropout(dropout)(ffn)
    return LayerNormalization(epsilon=1e-6)(x + ffn)

def build_temporal_transformer(seq_len, feat_dim,
                              embed_dim=32,
                              depth=2,
                              num_heads=2,
                              key_dim=16,
                              ff_dim=64,
                              dropout=0.15,
                              lr=2e-4):
    inp = Input(shape=(seq_len, feat_dim), name="seq_input")
    x = Dense(embed_dim, activation="relu")(inp)

    for _ in range(depth):
        x = transformer_encoder(x, num_heads=num_heads, key_dim=key_dim, ff_dim=ff_dim, dropout=dropout)

    x = GlobalAveragePooling1D()(x)
    x = Dense(64, activation="relu")(x)
    x = Dropout(dropout)(x)
    out = Dense(1, activation="sigmoid")(x)

    model = Model(inp, out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model

# -------------------------
# 6) LOAO RUN
# -------------------------
def run_loao_temporal_transformer(seq_len=5, min_non_nan_ratio=0.4,
                                 epochs=30, batch_size=64, pred_batch_size=256):
    results = []

    for test_app in apps:
        tf.keras.backend.clear_session()  # IMPORTANT for memory + retracing

        # Build sequences per app (no leakage)
        X_train_list, y_train_list = [], []
        for a in apps:
            if a == test_app:
                continue
            Xa, ya = make_raw_sequence_dataset(
                df[df["App"] == a],
                label_col="Temporal",
                seq_len=seq_len,
                min_non_nan_ratio=min_non_nan_ratio
            )
            if len(ya) > 0:
                X_train_list.append(Xa)
                y_train_list.append(ya)

        if len(X_train_list) == 0:
            print(f"[Temporal | T-SeqRaw] HELD-OUT={test_app} -> no train sequences")
            continue

        X_train_full = np.concatenate(X_train_list, axis=0)
        y_train_full = np.concatenate(y_train_list, axis=0)

        X_test, y_test = make_raw_sequence_dataset(
            df[df["App"] == test_app],
            label_col="Temporal",
            seq_len=seq_len,
            min_non_nan_ratio=min_non_nan_ratio
        )
        if len(y_test) == 0:
            print(f"[Temporal | T-SeqRaw] HELD-OUT={test_app} -> no test sequences")
            continue

        # ---------
        # Impute + scale (fit on TRAIN only)
        # ---------
        # Flatten to (N*T, F) for imputer/scaler, then reshape back
        Ntr, T, F = X_train_full.shape
        Nte = X_test.shape[0]

        imputer = SimpleImputer(strategy="median")
        scaler = StandardScaler()

        Xtr_flat = X_train_full.reshape(Ntr*T, F)
        Xte_flat = X_test.reshape(Nte*T, F)

        Xtr_flat = imputer.fit_transform(Xtr_flat)
        Xte_flat = imputer.transform(Xte_flat)

        Xtr_flat = scaler.fit_transform(Xtr_flat)
        Xte_flat = scaler.transform(Xte_flat)

        X_train_full = Xtr_flat.reshape(Ntr, T, F)
        X_test = Xte_flat.reshape(Nte, T, F)

        # inner split
        X_tr, X_val, y_tr, y_val = train_test_split(
            X_train_full, y_train_full,
            test_size=0.15,
            random_state=SEED,
            stratify=y_train_full
        )

        # class weights
        cw = compute_class_weight(class_weight="balanced", classes=np.array([0,1]), y=y_tr)
        class_weight = {0: float(cw[0]), 1: float(cw[1])}

        model = build_temporal_transformer(seq_len=T, feat_dim=F)

        callbacks = [
            tf.keras.callbacks.EarlyStopping(
                monitor="val_loss",
                patience=5,
                restore_best_weights=True,
                verbose=0
            ),
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor="val_loss",
                factor=0.5,
                patience=2,
                min_lr=1e-6,
                verbose=0
            )
        ]

        model.fit(
            X_tr, y_tr,
            validation_data=(X_val, y_val),
            epochs=epochs,
            batch_size=batch_size,
            verbose=0,
            class_weight=class_weight,
        )

        probs = model.predict(X_test, batch_size=pred_batch_size, verbose=0).reshape(-1)
        yhat = (probs >= 0.5).astype(int)

        m = eval_binary(y_test, yhat)

        print(f"[Temporal | T-SeqRaw] HELD-OUT={test_app:12s} (SEQ_LEN={seq_len}) "
              f"acc={m['acc']:.3f} f1={m['f1']:.3f} prec={m['precision']:.3f} rec={m['recall']:.3f} "
              f"n_test={len(y_test)}")

        results.append({
            "label": "Temporal",
            "task": f"sequence_raw_len_{seq_len}",
            "model": "Transformer_seqraw_stable",
            "held_out_app": test_app,
            "seq_len": int(seq_len),
            "n_train": int(len(y_tr)),
            "n_val": int(len(y_val)),
            "n_test": int(len(y_test)),
            "acc": m["acc"],
            "f1": m["f1"],
            "precision": m["precision"],
            "recall": m["recall"],
            "cm": m["cm"],
            "pred_dist": m["pred_dist"],
        })

    return pd.DataFrame(results)

SEQ_LEN = 5
MIN_NON_NAN_RATIO = 0.4
res_df = run_loao_temporal_transformer(seq_len=SEQ_LEN, min_non_nan_ratio=MIN_NON_NAN_RATIO)

print("\n================ SUMMARY (mean over held-out apps) ================")
print(res_df[["acc","f1","precision","recall"]].mean())

OUT = "/content/loao_temporal_transformer_seqraw_len5_STABLE.csv"
res_df.to_csv(OUT, index=False)
print("\nSaved:", OUT)


================= LOADED DATA =================
Loaded: /content/extracted_metrics_all_apps.csv | shape: (1744, 28)
Apps: ['Archery', 'PhantomLimb', 'PianoTiles', 'Puzzle', 'Sea', 'War']

================= FEATURES =================
Num features: 23
First 25 feature cols: ['missing_joints_count', 'missing_joints_ratio', 'collapsed_joints_count', 'center_of_mass_x', 'center_of_mass_y', 'center_of_mass_z', 'distance_from_origin', 'bbox_width', 'bbox_height', 'bbox_depth', 'bbox_volume', 'max_joint_distance_from_com', 'distance_from_floor', 'below_floor', 'left_forearm_length', 'right_forearm_length', 'left_shin_length', 'right_shin_length', 'arm_length_symmetry', 'leg_length_symmetry', 'body_forward_x', 'body_forward_y', 'body_forward_z']
[Temporal | T-SeqRaw] HELD-OUT=Archery      (SEQ_LEN=5) acc=0.875 f1=0.933 prec=0.875 rec=1.000 n_test=96


[Temporal | T-SeqRaw] HELD-OUT=PhantomLimb  (SEQ_LEN=5) acc=0.586 f1=0.000 prec=0.000 rec=0.000 n_test=321
[Temporal | T-SeqRaw] HELD-OUT=PianoTiles   (SEQ_LEN=5) acc=0.851 f1=0.425 prec=0.407 rec=0.444 n_test=436
[Temporal | T-SeqRaw] HELD-OUT=Puzzle       (SEQ_LEN=5) acc=0.778 f1=0.875 prec=0.777 rec=1.000 n_test=275
[Temporal | T-SeqRaw] HELD-OUT=Sea          (SEQ_LEN=5) acc=0.571 f1=0.099 prec=1.000 rec=0.052 n_test=296
[Temporal | T-SeqRaw] HELD-OUT=War          (SEQ_LEN=5) acc=0.757 f1=0.660 prec=0.802 rec=0.560 n_test=276

================ SUMMARY (mean over held-out apps) ================
acc          0.736327
f1           0.498674
precision    0.643604
recall       0.509505
dtype: float64

Saved: /content/loao_temporal_transformer_seqraw_len5_STABLE.csv


In [ ]:
# ===============================================================
# VR APP-INDEPENDENT METRICS - LOAO TEMPORAL (RAW SEQ) TRANSFORMER
# Improvements:
#   (1) RAW SEQ input: (SEQ_LEN, F) = (5, 23)
#   (2) NaN-safe preprocessing (median impute on train only)
#   (3) StandardScaler fitted on TRAIN only (flattened tokens)
#   (4) Class weights (per-fold) to reduce majority collapse
#   (5) Threshold tuning on VAL (maximize F1), then evaluate on TEST
#   (6) Reduce TF retracing: tf.function(reduce_retracing=True) predict
#
# Outputs:
#   - Per held-out app: acc/f1/prec/rec + best threshold
#   - Mean metrics across held-out apps
#   - CSV saved to /content/loao_temporal_transformer_seqraw_len5_IMPROVED.csv
# ===============================================================

import os
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.layers import (
    Input, Dense, Dropout, LayerNormalization,
    MultiHeadAttention, GlobalAveragePooling1D
)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# -------------------------
# 1) LOAD DATA
# -------------------------
CANDIDATE_PATHS = [
    "/content/extracted_metrics_all_apps.csv",
    "/content/drive/MyDrive/extracted_metrics_all_apps.csv",
    "/content/drive/MyDrive/VR_Metrics/extracted_metrics_all_apps.csv",
]
DATA_PATH = None
for p in CANDIDATE_PATHS:
    if os.path.exists(p):
        DATA_PATH = p
        break
if DATA_PATH is None:
    raise FileNotFoundError("Could not find extracted_metrics_all_apps.csv in common locations.")

df = pd.read_csv(DATA_PATH)
print("\n================= LOADED DATA =================")
print("Loaded:", DATA_PATH, "| shape:", df.shape)

# -------------------------
# 2) CLEAN / TYPES
# -------------------------
df = df.replace("", np.nan)

required = {"App", "Temporal"}
missing_req = required - set(df.columns)
if missing_req:
    raise ValueError(f"Missing required columns: {missing_req}")

df["Temporal"] = pd.to_numeric(df["Temporal"], errors="coerce")
df = df.dropna(subset=["App", "Temporal"]).copy()
df["Temporal"] = df["Temporal"].astype(int)

apps = sorted(df["App"].unique().tolist())
print("Apps:", apps)

# OPTIONAL: Downsample PhantomLimb
DOWNSAMPLE_PHANTOMLIMB = True
TARGET_N_PL = 400
if DOWNSAMPLE_PHANTOMLIMB and ("PhantomLimb" in df["App"].unique()):
    pl = df[df["App"] == "PhantomLimb"]
    others = df[df["App"] != "PhantomLimb"]
    if len(pl) > TARGET_N_PL:
        pl = pl.sample(TARGET_N_PL, random_state=SEED)
    df = pd.concat([others, pl], ignore_index=True)
    apps = sorted(df["App"].unique().tolist())
    print("\n[INFO] Downsample PhantomLimb =", DOWNSAMPLE_PHANTOMLIMB, "| TARGET_N_PL =", TARGET_N_PL)
    print("[INFO] PhantomLimb N =", len(df[df["App"] == "PhantomLimb"]), "| Total N =", len(df))

# -------------------------
# 3) FEATURE COLS (23)
# -------------------------
ID_COLS = [c for c in ["GlobalID", "EntryID"] if c in df.columns]
DROP_COLS = ["App", "Spatial", "Temporal"] + ID_COLS
feature_cols = [c for c in df.columns if c not in DROP_COLS]

for c in feature_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

print("\n================= FEATURES =================")
print("Num features:", len(feature_cols))
print("First 25 feature cols:", feature_cols[:25])

# -------------------------
# 4) HELPERS
# -------------------------
def eval_binary(y_true, y_pred):
    return {
        "acc": float(accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "cm": confusion_matrix(y_true, y_pred).tolist(),
        "pred_dist": dict(zip(*np.unique(y_pred, return_counts=True))),
    }

def make_raw_sequence_dataset(app_df, label_col="Temporal", seq_len=5):
    """
    Raw seq: returns X of shape (N_seq, seq_len, F) using consecutive rows.
    Label is center frame: i + seq_len//2
    """
    X = app_df[feature_cols].values.astype(float)
    y = app_df[label_col].values.astype(int)

    if len(app_df) < seq_len:
        return np.empty((0, seq_len, len(feature_cols))), np.empty((0,), dtype=int)

    X_seq, y_seq = [], []
    center = seq_len // 2
    for i in range(len(app_df) - seq_len + 1):
        X_seq.append(X[i:i+seq_len])
        y_seq.append(y[i + center])

    return np.stack(X_seq, axis=0), np.array(y_seq, dtype=int)

def build_transformer_seq_model(seq_len, n_feat,
                                d_model=64, num_heads=2, key_dim=32,
                                ff_dim=128, depth=2, dropout=0.15,
                                lr=1e-4):
    """
    Input: (seq_len, n_feat)
    We'll project features -> d_model, then transformer blocks, then GAP + head
    """
    inp = Input(shape=(seq_len, n_feat), name="seq_in")
    x = Dense(d_model, activation="relu")(inp)

    for _ in range(depth):
        attn = MultiHeadAttention(num_heads=num_heads, key_dim=key_dim)(x, x)
        attn = Dropout(dropout)(attn)
        x = LayerNormalization(epsilon=1e-6)(x + attn)

        ffn = Dense(ff_dim, activation="relu")(x)
        ffn = Dense(d_model)(ffn)
        ffn = Dropout(dropout)(ffn)
        x = LayerNormalization(epsilon=1e-6)(x + ffn)

    x = GlobalAveragePooling1D()(x)
    x = Dense(64, activation="relu")(x)
    x = Dropout(dropout)(x)
    out = Dense(1, activation="sigmoid", name="out")(x)

    model = Model(inp, out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model

def tune_threshold_on_val(y_val, p_val, grid=None):
    if grid is None:
        grid = np.linspace(0.05, 0.95, 19)  # coarse but stable
    best_thr, best_f1 = 0.5, -1.0
    for t in grid:
        yhat = (p_val >= t).astype(int)
        f1 = f1_score(y_val, yhat, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_thr = float(t)
    return best_thr, float(best_f1)

# -------------------------
# 5) TF retracing reduction: fast predict
# -------------------------
tf.config.run_functions_eagerly(False)

@tf.function(reduce_retracing=True)
def fast_predict(model, x):
    return model(x, training=False)

# -------------------------
# 6) LOAO run: RAW SEQ Transformer + val threshold tuning
# -------------------------
def run_loao_temporal_transformer_seqraw(
    label_col="Temporal",
    seq_len=5,
    d_model=64,
    num_heads=2,
    key_dim=32,
    ff_dim=128,
    depth=2,
    dropout=0.15,
    lr=1e-4,
    epochs=20,
    batch_size=64,
    val_size=0.15
):
    results = []

    print("\n================= LOAO: TEMPORAL (RAW SEQ) - TRANSFORMER (IMPROVED) =================")
    print(f"[INFO] SEQ_LEN={seq_len} | raw tokens = (T={seq_len}, F={len(feature_cols)})")

    for test_app in apps:
        # ---- build train/test (no leakage)
        train_apps = [a for a in apps if a != test_app]

        X_train_list, y_train_list = [], []
        for a in train_apps:
            Xa, ya = make_raw_sequence_dataset(df[df["App"] == a], label_col=label_col, seq_len=seq_len)
            if len(ya) > 0:
                X_train_list.append(Xa)
                y_train_list.append(ya)

        if len(X_train_list) == 0:
            print(f"[{label_col} | T-SeqRaw] HELD-OUT={test_app} -> No train sequences.")
            continue

        X_train_full = np.concatenate(X_train_list, axis=0)
        y_train_full = np.concatenate(y_train_list, axis=0)

        X_test, y_test = make_raw_sequence_dataset(df[df["App"] == test_app], label_col=label_col, seq_len=seq_len)
        if len(y_test) == 0:
            print(f"[{label_col} | T-SeqRaw] HELD-OUT={test_app} -> No test sequences.")
            continue

        # ---- train/val split (stratified)
        X_tr, X_val, y_tr, y_val = train_test_split(
            X_train_full, y_train_full,
            test_size=val_size,
            random_state=SEED,
            stratify=y_train_full
        )

        # ---- Impute + scale (fit ONLY on train)
        # We fit imputer/scaler over flattened tokens to treat each feature consistently.
        n_feat = X_tr.shape[-1]

        imp = SimpleImputer(strategy="median")
        sca = StandardScaler()

        X_tr_flat = X_tr.reshape(-1, n_feat)
        X_val_flat = X_val.reshape(-1, n_feat)
        X_te_flat  = X_test.reshape(-1, n_feat)

        X_tr_flat = imp.fit_transform(X_tr_flat)
        X_val_flat = imp.transform(X_val_flat)
        X_te_flat  = imp.transform(X_te_flat)

        X_tr_flat = sca.fit_transform(X_tr_flat)
        X_val_flat = sca.transform(X_val_flat)
        X_te_flat  = sca.transform(X_te_flat)

        X_tr = X_tr_flat.reshape(-1, seq_len, n_feat).astype(np.float32)
        X_val = X_val_flat.reshape(-1, seq_len, n_feat).astype(np.float32)
        X_te  = X_te_flat.reshape(-1, seq_len, n_feat).astype(np.float32)

        # ---- class weights (per-fold)
        classes = np.array([0, 1])
        cw = compute_class_weight(class_weight="balanced", classes=classes, y=y_tr)
        class_weight = {0: float(cw[0]), 1: float(cw[1])}

        # ---- model
        model = build_transformer_seq_model(
            seq_len=seq_len, n_feat=n_feat,
            d_model=d_model, num_heads=num_heads, key_dim=key_dim,
            ff_dim=ff_dim, depth=depth, dropout=dropout, lr=lr
        )

        early = EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True, verbose=0)
        rlrop = ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, min_lr=1e-6, verbose=0)

        model.fit(
            X_tr, y_tr,
            validation_data=(X_val, y_val),
            epochs=epochs,
            batch_size=batch_size,
            verbose=0,
            class_weight=class_weight,
            callbacks=[early, rlrop]
        )

        # ---- predict probs via tf.function (reduces retracing spam)
        p_val = fast_predict(model, tf.convert_to_tensor(X_val)).numpy().reshape(-1)
        p_te  = fast_predict(model, tf.convert_to_tensor(X_te)).numpy().reshape(-1)

        thr, val_f1 = tune_threshold_on_val(y_val, p_val)
        yhat = (p_te >= thr).astype(int)

        m = eval_binary(y_test, yhat)
        row = {
            "label": label_col,
            "task": f"sequence_raw_len_{seq_len}",
            "model": "Transformer_seqraw_improved",
            "held_out_app": test_app,
            "seq_len": int(seq_len),
            "F": int(n_feat),
            "n_train": int(len(y_tr)),
            "n_val": int(len(y_val)),
            "n_test": int(len(y_test)),
            "thr": float(thr),
            "val_f1_best": float(val_f1),
            **{k: v for k, v in m.items() if k not in ["cm", "pred_dist"]},
            "cm": m["cm"],
            "pred_dist": m["pred_dist"],
        }

        print(f"[Temporal | T-SeqRaw+thr] HELD-OUT={test_app:12s} (SEQ_LEN={seq_len}) "
              f"thr={thr:.2f} val_f1={val_f1:.3f}  "
              f"acc={row['acc']:.3f} f1={row['f1']:.3f} "
              f"prec={row['precision']:.3f} rec={row['recall']:.3f}  "
              f"n_test={row['n_test']}")
        results.append(row)

        # cleanup between folds
        tf.keras.backend.clear_session()

    return pd.DataFrame(results)

# -------------------------
# 7) RUN
# -------------------------
SEQ_LEN = 5

res = run_loao_temporal_transformer_seqraw(
    label_col="Temporal",
    seq_len=SEQ_LEN,
    d_model=64,
    num_heads=2,
    key_dim=32,
    ff_dim=128,
    depth=2,
    dropout=0.15,
    lr=1e-4,
    epochs=20,
    batch_size=64,
    val_size=0.15
)

print("\n================ SUMMARY (mean over held-out apps) ================")
if len(res) > 0:
    print(res[["acc","f1","precision","recall"]].mean())
else:
    print("No results (no sequences found).")

OUT = "/content/loao_temporal_transformer_seqraw_len5_IMPROVED.csv"
res.to_csv(OUT, index=False)
print("\nSaved:", OUT)


================= LOADED DATA =================
Loaded: /content/extracted_metrics_all_apps.csv | shape: (1744, 28)
Apps: ['Archery', 'PhantomLimb', 'PianoTiles', 'Puzzle', 'Sea', 'War']

[INFO] Downsample PhantomLimb = True | TARGET_N_PL = 400
[INFO] PhantomLimb N = 325 | Total N = 1744

================= FEATURES =================
Num features: 23
First 25 feature cols: ['missing_joints_count', 'missing_joints_ratio', 'collapsed_joints_count', 'center_of_mass_x', 'center_of_mass_y', 'center_of_mass_z', 'distance_from_origin', 'bbox_width', 'bbox_height', 'bbox_depth', 'bbox_volume', 'max_joint_distance_from_com', 'distance_from_floor', 'below_floor', 'left_forearm_length', 'right_forearm_length', 'left_shin_length', 'right_shin_length', 'arm_length_symmetry', 'leg_length_symmetry', 'body_forward_x', 'body_forward_y', 'body_forward_z']

================= LOAO: TEMPORAL (RAW SEQ) - TRANSFORMER (IMPROVED) =================
[INFO] SEQ_LEN=5 | raw tokens = (T=5, F=23)
[Temporal | T-SeqRa

[Temporal | T-SeqRaw+thr] HELD-OUT=PianoTiles   (SEQ_LEN=5) thr=0.35 val_f1=0.951  acc=0.725 f1=0.464 prec=0.306 rec=0.963  n_test=436
[Temporal | T-SeqRaw+thr] HELD-OUT=Puzzle       (SEQ_LEN=5) thr=0.85 val_f1=0.869  acc=0.278 f1=0.000 prec=0.000 rec=0.000  n_test=295
[Temporal | T-SeqRaw+thr] HELD-OUT=Sea          (SEQ_LEN=5) thr=0.65 val_f1=0.919  acc=0.547 f1=0.000 prec=0.000 rec=0.000  n_test=296
[Temporal | T-SeqRaw+thr] HELD-OUT=War          (SEQ_LEN=5) thr=0.50 val_f1=0.940  acc=0.667 f1=0.716 prec=0.558 rec=1.000  n_test=276

================ SUMMARY (mean over held-out apps) ================
acc          0.623799
f1           0.397733
precision    0.456429
recall       0.520143
dtype: float64

Saved: /content/loao_temporal_transformer_seqraw_len5_IMPROVED.csv


In [ ]:
# ===============================================================
# VR APP-INDEPENDENT METRICS - LOAO TEMPORAL (RAW SEQ) TRANSFORMER
# Stable version:
#   - RAW SEQ input: (SEQ_LEN, F) = (5, 23)
#   - Train-only median impute + StandardScaler
#   - tf.data pipelines w/ fixed batch size + drop_remainder=True
#   - Class weights (per fold)
#   - Threshold tuned on VAL (maximize F1), eval on TEST
#   - NO tf.function wrapper => avoids retracing spam
# ===============================================================

import os
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.layers import (
    Input, Dense, Dropout, LayerNormalization,
    MultiHeadAttention, GlobalAveragePooling1D
)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# -------------------------
# 1) LOAD DATA
# -------------------------
CANDIDATE_PATHS = [
    "/content/extracted_metrics_all_apps.csv",
    "/content/drive/MyDrive/extracted_metrics_all_apps.csv",
    "/content/drive/MyDrive/VR_Metrics/extracted_metrics_all_apps.csv",
]
DATA_PATH = None
for p in CANDIDATE_PATHS:
    if os.path.exists(p):
        DATA_PATH = p
        break
if DATA_PATH is None:
    raise FileNotFoundError("Could not find extracted_metrics_all_apps.csv in common locations.")

df = pd.read_csv(DATA_PATH)
print("\n================= LOADED DATA =================")
print("Loaded:", DATA_PATH, "| shape:", df.shape)

# -------------------------
# 2) CLEAN / TYPES
# -------------------------
df = df.replace("", np.nan)

required = {"App", "Temporal"}
missing_req = required - set(df.columns)
if missing_req:
    raise ValueError(f"Missing required columns: {missing_req}")

df["Temporal"] = pd.to_numeric(df["Temporal"], errors="coerce")
df = df.dropna(subset=["App", "Temporal"]).copy()
df["Temporal"] = df["Temporal"].astype(int)

apps = sorted(df["App"].unique().tolist())
print("Apps:", apps)

# OPTIONAL: Downsample PhantomLimb
DOWNSAMPLE_PHANTOMLIMB = True
TARGET_N_PL = 400
if DOWNSAMPLE_PHANTOMLIMB and ("PhantomLimb" in df["App"].unique()):
    pl = df[df["App"] == "PhantomLimb"]
    others = df[df["App"] != "PhantomLimb"]
    if len(pl) > TARGET_N_PL:
        pl = pl.sample(TARGET_N_PL, random_state=SEED)
    df = pd.concat([others, pl], ignore_index=True)
    apps = sorted(df["App"].unique().tolist())
    print("\n[INFO] Downsample PhantomLimb =", DOWNSAMPLE_PHANTOMLIMB, "| TARGET_N_PL =", TARGET_N_PL)
    print("[INFO] PhantomLimb N =", len(df[df["App"] == "PhantomLimb"]), "| Total N =", len(df))

# -------------------------
# 3) FEATURE COLS (23)
# -------------------------
ID_COLS = [c for c in ["GlobalID", "EntryID"] if c in df.columns]
DROP_COLS = ["App", "Spatial", "Temporal"] + ID_COLS
feature_cols = [c for c in df.columns if c not in DROP_COLS]

for c in feature_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

print("\n================= FEATURES =================")
print("Num features:", len(feature_cols))
print("First 25 feature cols:", feature_cols[:25])

# -------------------------
# 4) HELPERS
# -------------------------
def eval_binary(y_true, y_pred):
    return {
        "acc": float(accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "cm": confusion_matrix(y_true, y_pred).tolist(),
        "pred_dist": dict(zip(*np.unique(y_pred, return_counts=True))),
    }

def make_raw_sequence_dataset(app_df, label_col="Temporal", seq_len=5):
    """
    Raw seq: returns X of shape (N_seq, seq_len, F) using consecutive rows.
    Label is center frame: i + seq_len//2
    """
    X = app_df[feature_cols].values.astype(float)
    y = app_df[label_col].values.astype(int)

    if len(app_df) < seq_len:
        return np.empty((0, seq_len, len(feature_cols))), np.empty((0,), dtype=int)

    X_seq, y_seq = [], []
    center = seq_len // 2
    for i in range(len(app_df) - seq_len + 1):
        X_seq.append(X[i:i+seq_len])
        y_seq.append(y[i + center])

    return np.stack(X_seq, axis=0), np.array(y_seq, dtype=int)

def tune_threshold_on_val(y_val, p_val, grid=None):
    if grid is None:
        grid = np.linspace(0.05, 0.95, 19)
    best_thr, best_f1 = 0.5, -1.0
    for t in grid:
        yhat = (p_val >= t).astype(int)
        f1 = f1_score(y_val, yhat, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_thr = float(t)
    return best_thr, float(best_f1)

def build_transformer_seq_model(seq_len, n_feat,
                                d_model=64, num_heads=2, key_dim=32,
                                ff_dim=128, depth=2, dropout=0.15,
                                lr=1e-4):
    inp = Input(shape=(seq_len, n_feat), name="seq_in")
    x = Dense(d_model, activation="relu")(inp)

    for _ in range(depth):
        attn = MultiHeadAttention(num_heads=num_heads, key_dim=key_dim)(x, x)
        attn = Dropout(dropout)(attn)
        x = LayerNormalization(epsilon=1e-6)(x + attn)

        ffn = Dense(ff_dim, activation="relu")(x)
        ffn = Dense(d_model)(ffn)
        ffn = Dropout(dropout)(ffn)
        x = LayerNormalization(epsilon=1e-6)(x + ffn)

    x = GlobalAveragePooling1D()(x)
    x = Dense(64, activation="relu")(x)
    x = Dropout(dropout)(x)
    out = Dense(1, activation="sigmoid", name="out")(x)

    model = Model(inp, out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model

def make_tf_dataset(X, y, batch_size, shuffle=True):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if shuffle:
        ds = ds.shuffle(buffer_size=min(len(y), 5000), seed=SEED, reshuffle_each_iteration=True)
    # drop_remainder=True => consistent batch shapes, helps stability
    ds = ds.batch(batch_size, drop_remainder=True).prefetch(tf.data.AUTOTUNE)
    return ds

# -------------------------
# 5) LOAO: Temporal raw-seq Transformer (stable)
# -------------------------
def run_loao_temporal_transformer_seqraw(
    label_col="Temporal",
    seq_len=5,
    d_model=64,
    num_heads=2,
    key_dim=32,
    ff_dim=128,
    depth=2,
    dropout=0.15,
    lr=1e-4,
    epochs=25,
    batch_size=64,
    val_size=0.15,
):
    results = []

    print("\n================= LOAO: TEMPORAL (RAW SEQ) - TRANSFORMER (STABLE) =================")
    print(f"[INFO] SEQ_LEN={seq_len} | raw tokens = (T={seq_len}, F={len(feature_cols)})")

    for test_app in apps:
        train_apps = [a for a in apps if a != test_app]

        # Build train sequences (no leakage)
        X_train_list, y_train_list = [], []
        for a in train_apps:
            Xa, ya = make_raw_sequence_dataset(df[df["App"] == a], label_col=label_col, seq_len=seq_len)
            if len(ya) > 0:
                X_train_list.append(Xa)
                y_train_list.append(ya)

        if len(X_train_list) == 0:
            print(f"[{label_col} | T-SeqRaw] HELD-OUT={test_app} -> No train sequences.")
            continue

        X_train_full = np.concatenate(X_train_list, axis=0)
        y_train_full = np.concatenate(y_train_list, axis=0)

        # Build test sequences
        X_test, y_test = make_raw_sequence_dataset(df[df["App"] == test_app], label_col=label_col, seq_len=seq_len)
        if len(y_test) == 0:
            print(f"[{label_col} | T-SeqRaw] HELD-OUT={test_app} -> No test sequences.")
            continue

        # Train/val split
        X_tr, X_val, y_tr, y_val = train_test_split(
            X_train_full, y_train_full,
            test_size=val_size,
            random_state=SEED,
            stratify=y_train_full
        )

        n_feat = X_tr.shape[-1]

        # Train-only impute + scale (flatten tokens)
        imp = SimpleImputer(strategy="median")
        sca = StandardScaler()

        X_tr_flat = X_tr.reshape(-1, n_feat)
        X_val_flat = X_val.reshape(-1, n_feat)
        X_te_flat  = X_test.reshape(-1, n_feat)

        X_tr_flat = imp.fit_transform(X_tr_flat)
        X_val_flat = imp.transform(X_val_flat)
        X_te_flat  = imp.transform(X_te_flat)

        X_tr_flat = sca.fit_transform(X_tr_flat)
        X_val_flat = sca.transform(X_val_flat)
        X_te_flat  = sca.transform(X_te_flat)

        X_tr = X_tr_flat.reshape(-1, seq_len, n_feat).astype(np.float32)
        X_val = X_val_flat.reshape(-1, seq_len, n_feat).astype(np.float32)
        X_te  = X_te_flat.reshape(-1, seq_len, n_feat).astype(np.float32)

        # Class weights
        classes = np.array([0, 1])
        cw = compute_class_weight(class_weight="balanced", classes=classes, y=y_tr)
        class_weight = {0: float(cw[0]), 1: float(cw[1])}

        # Build model
        model = build_transformer_seq_model(
            seq_len=seq_len, n_feat=n_feat,
            d_model=d_model, num_heads=num_heads, key_dim=key_dim,
            ff_dim=ff_dim, depth=depth, dropout=dropout, lr=lr
        )

        early = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=0)
        rlrop = ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, min_lr=1e-6, verbose=0)

        # tf.data datasets (fixed batch size, drop_remainder=True)
        train_ds = make_tf_dataset(X_tr, y_tr, batch_size=batch_size, shuffle=True)
        val_ds   = make_tf_dataset(X_val, y_val, batch_size=batch_size, shuffle=False)

        model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=epochs,
            verbose=0,
            class_weight=class_weight,
            callbacks=[early, rlrop]
        )

        # Predict probabilities (no tf.function wrapper)
        p_val = model.predict(X_val, batch_size=batch_size, verbose=0).reshape(-1)
        p_te  = model.predict(X_te,  batch_size=batch_size, verbose=0).reshape(-1)

        thr, val_f1 = tune_threshold_on_val(y_val, p_val)
        yhat = (p_te >= thr).astype(int)

        m = eval_binary(y_test, yhat)
        row = {
            "label": label_col,
            "task": f"sequence_raw_len_{seq_len}",
            "model": "Transformer_seqraw_STABLE",
            "held_out_app": test_app,
            "seq_len": int(seq_len),
            "F": int(n_feat),
            "n_train": int(len(y_tr)),
            "n_val": int(len(y_val)),
            "n_test": int(len(y_test)),
            "thr": float(thr),
            "val_f1_best": float(val_f1),
            **{k: v for k, v in m.items() if k not in ["cm", "pred_dist"]},
            "cm": m["cm"],
            "pred_dist": m["pred_dist"],
        }

        print(f"[Temporal | T-SeqRaw+thr] HELD-OUT={test_app:12s} (SEQ_LEN={seq_len}) "
              f"thr={thr:.2f} val_f1={val_f1:.3f}  "
              f"acc={row['acc']:.3f} f1={row['f1']:.3f} "
              f"prec={row['precision']:.3f} rec={row['recall']:.3f}  "
              f"n_test={row['n_test']}")
        results.append(row)

        tf.keras.backend.clear_session()

    return pd.DataFrame(results)

# -------------------------
# 6) RUN
# -------------------------
SEQ_LEN = 5
res = run_loao_temporal_transformer_seqraw(
    label_col="Temporal",
    seq_len=SEQ_LEN,
    d_model=64,
    num_heads=2,
    key_dim=32,
    ff_dim=128,
    depth=2,
    dropout=0.15,
    lr=1e-4,
    epochs=25,
    batch_size=64,
    val_size=0.15,
)

print("\n================ SUMMARY (mean over held-out apps) ================")
if len(res) > 0:
    print(res[["acc","f1","precision","recall"]].mean())
else:
    print("No results (no sequences found).")

OUT = "/content/loao_temporal_transformer_seqraw_len5_STABLE.csv"
res.to_csv(OUT, index=False)
print("\nSaved:", OUT)


================= LOADED DATA =================
Loaded: /content/extracted_metrics_all_apps.csv | shape: (1744, 28)
Apps: ['Archery', 'PhantomLimb', 'PianoTiles', 'Puzzle', 'Sea', 'War']

[INFO] Downsample PhantomLimb = True | TARGET_N_PL = 400
[INFO] PhantomLimb N = 325 | Total N = 1744

================= FEATURES =================
Num features: 23
First 25 feature cols: ['missing_joints_count', 'missing_joints_ratio', 'collapsed_joints_count', 'center_of_mass_x', 'center_of_mass_y', 'center_of_mass_z', 'distance_from_origin', 'bbox_width', 'bbox_height', 'bbox_depth', 'bbox_volume', 'max_joint_distance_from_com', 'distance_from_floor', 'below_floor', 'left_forearm_length', 'right_forearm_length', 'left_shin_length', 'right_shin_length', 'arm_length_symmetry', 'leg_length_symmetry', 'body_forward_x', 'body_forward_y', 'body_forward_z']

================= LOAO: TEMPORAL (RAW SEQ) - TRANSFORMER (STABLE) =================
[INFO] SEQ_LEN=5 | raw tokens = (T=5, F=23)
[Temporal | T-SeqRaw+

In [ ]:
# ===============================================================
# TEMPORAL LOAO: Transformer on RAW SEQ (T=5, F=23)
# Fixes:
#  - NO drop_remainder for val/test (critical for threshold tuning)
#  - AdamW weight decay
#  - optional label smoothing
#  - deterministic-ish seeds
# ===============================================================

import os
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.layers import Input, Dense, Dropout, LayerNormalization, MultiHeadAttention, GlobalAveragePooling1D
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# (Optional) helps reproducibility
os.environ["TF_DETERMINISTIC_OPS"] = "1"

# -------------------------
# 1) LOAD DATA
# -------------------------
CANDIDATE_PATHS = [
    "/content/extracted_metrics_all_apps.csv",
    "/content/drive/MyDrive/extracted_metrics_all_apps.csv",
    "/content/drive/MyDrive/VR_Metrics/extracted_metrics_all_apps.csv",
]
DATA_PATH = None
for p in CANDIDATE_PATHS:
    if os.path.exists(p):
        DATA_PATH = p
        break
if DATA_PATH is None:
    raise FileNotFoundError("Could not find extracted_metrics_all_apps.csv in common locations.")

df = pd.read_csv(DATA_PATH)
print("\n================= LOADED DATA =================")
print("Loaded:", DATA_PATH, "| shape:", df.shape)

# -------------------------
# 2) CLEAN / TYPES
# -------------------------
df = df.replace("", np.nan)
df = df.dropna(subset=["App"]).copy()

df["Temporal"] = pd.to_numeric(df["Temporal"], errors="coerce")
df = df.dropna(subset=["Temporal"]).copy()
df["Temporal"] = df["Temporal"].astype(int)

apps = sorted(df["App"].unique().tolist())
print("Apps:", apps)

# OPTIONAL: Downsample PhantomLimb
DOWNSAMPLE_PHANTOMLIMB = True
TARGET_N_PL = 400
if DOWNSAMPLE_PHANTOMLIMB and ("PhantomLimb" in df["App"].unique()):
    pl = df[df["App"] == "PhantomLimb"]
    others = df[df["App"] != "PhantomLimb"]
    if len(pl) > TARGET_N_PL:
        pl = pl.sample(TARGET_N_PL, random_state=SEED)
    df = pd.concat([others, pl], ignore_index=True)
    apps = sorted(df["App"].unique().tolist())
    print("\n[INFO] Downsample PhantomLimb =", DOWNSAMPLE_PHANTOMLIMB, "| TARGET_N_PL =", TARGET_N_PL)
    print("[INFO] PhantomLimb N =", len(df[df["App"] == "PhantomLimb"]), "| Total N =", len(df))

# -------------------------
# 3) FEATURES (23)
# -------------------------
ID_COLS = [c for c in ["GlobalID", "EntryID"] if c in df.columns]
DROP_COLS = ["App", "Spatial", "Temporal"] + ID_COLS
feature_cols = [c for c in df.columns if c not in DROP_COLS]

for c in feature_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

F = len(feature_cols)
print("\n================= FEATURES =================")
print("Num features:", F)
print("First 25 feature cols:", feature_cols[:25])

# -------------------------
# 4) Sliding windows: RAW seq dataset (N, T, F)
# -------------------------
def window_is_valid(window, min_non_nan_ratio=0.4):
    total = window.size
    non_nan = np.isfinite(window).sum()
    return (non_nan / total) >= min_non_nan_ratio

def make_raw_sequence_dataset(app_df, label_col="Temporal", seq_len=5, min_non_nan_ratio=0.4):
    X = app_df[feature_cols].values.astype(float)
    y = app_df[label_col].values.astype(int)

    if len(app_df) < seq_len:
        return np.empty((0, seq_len, F)), np.empty((0,), dtype=int)

    X_seq, y_seq = [], []
    center = seq_len // 2

    for i in range(len(app_df) - seq_len + 1):
        window = X[i:i+seq_len]
        if not window_is_valid(window, min_non_nan_ratio=min_non_nan_ratio):
            continue
        X_seq.append(window)
        y_seq.append(y[i + center])

    if len(X_seq) == 0:
        return np.empty((0, seq_len, F)), np.empty((0,), dtype=int)

    return np.stack(X_seq, axis=0), np.array(y_seq, dtype=int)

# -------------------------
# 5) Metrics + threshold tuning
# -------------------------
def eval_binary(y_true, y_pred):
    return {
        "acc": float(accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "cm": confusion_matrix(y_true, y_pred).tolist(),
    }

def tune_threshold_on_val(y_val, p_val, grid=None):
    if grid is None:
        grid = np.linspace(0.05, 0.95, 19)
    best_thr, best_f1 = 0.5, -1.0
    for t in grid:
        yhat = (p_val >= t).astype(int)
        f1 = f1_score(y_val, yhat, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_thr = float(t)
    return best_thr, float(best_f1)

# -------------------------
# 6) Transformer model (RAW seq)
# -------------------------
def transformer_encoder(inputs, num_heads, key_dim, ff_dim, dropout=0.15):
    attn = MultiHeadAttention(num_heads=num_heads, key_dim=key_dim)(inputs, inputs)
    attn = Dropout(dropout)(attn)
    x = LayerNormalization(epsilon=1e-6)(inputs + attn)

    ffn = Dense(ff_dim, activation="relu")(x)
    ffn = Dense(x.shape[-1])(ffn)
    ffn = Dropout(dropout)(ffn)
    out = LayerNormalization(epsilon=1e-6)(x + ffn)
    return out

def build_rawseq_transformer(seq_len=5, feat_dim=23,
                            d_model=64, num_heads=2, key_dim=32,
                            ff_dim=128, depth=2, dropout=0.15,
                            lr=7e-5, weight_decay=1e-4,
                            label_smoothing=0.0):
    inp = Input(shape=(seq_len, feat_dim), name="raw_seq_in")

    # Project features per timestep
    x = Dense(d_model, activation="relu")(inp)

    for _ in range(depth):
        x = transformer_encoder(x, num_heads=num_heads, key_dim=key_dim, ff_dim=ff_dim, dropout=dropout)

    x = GlobalAveragePooling1D()(x)
    x = Dense(64, activation="relu")(x)
    x = Dropout(dropout)(x)
    out = Dense(1, activation="sigmoid")(x)

    model = Model(inp, out)

    # AdamW for stability under domain shift
    opt = tf.keras.optimizers.AdamW(learning_rate=lr, weight_decay=weight_decay)

    loss = tf.keras.losses.BinaryCrossentropy(label_smoothing=label_smoothing)
    model.compile(optimizer=opt, loss=loss, metrics=["accuracy"])
    return model

# -------------------------
# 7) tf.data (IMPORTANT: drop_remainder ONLY for train)
# -------------------------
def make_tf_dataset(X, y, batch_size, shuffle=True, drop_remainder=False):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if shuffle:
        ds = ds.shuffle(buffer_size=min(len(y), 5000), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(batch_size, drop_remainder=drop_remainder).prefetch(tf.data.AUTOTUNE)
    return ds

# -------------------------
# 8) LOAO runner
# -------------------------
def run_loao_temporal_transformer_rawseq(
    seq_len=5,
    min_non_nan_ratio=0.4,
    epochs=25,
    batch_size=64,
    val_size=0.15,
):
    results = []
    print("\n================= LOAO: TEMPORAL (RAW SEQ) - TRANSFORMER (FIXED) =================")
    print(f"[INFO] SEQ_LEN={seq_len} | raw tokens = (T={seq_len}, F={F})")

    for test_app in apps:
        # Build train sequences across all other apps
        X_train_list, y_train_list = [], []
        for a in apps:
            if a == test_app:
                continue
            Xa, ya = make_raw_sequence_dataset(
                df[df["App"] == a],
                label_col="Temporal",
                seq_len=seq_len,
                min_non_nan_ratio=min_non_nan_ratio
            )
            if len(ya) > 0:
                X_train_list.append(Xa)
                y_train_list.append(ya)

        if len(X_train_list) == 0:
            print(f"[Temporal | T-SeqRaw] HELD-OUT={test_app} -> No train sequences.")
            continue

        X_train_full = np.concatenate(X_train_list, axis=0)  # (N,T,F)
        y_train_full = np.concatenate(y_train_list, axis=0)

        # Test sequences
        X_test, y_test = make_raw_sequence_dataset(
            df[df["App"] == test_app],
            label_col="Temporal",
            seq_len=seq_len,
            min_non_nan_ratio=min_non_nan_ratio
        )
        if len(y_test) == 0:
            print(f"[Temporal | T-SeqRaw] HELD-OUT={test_app} -> No test sequences.")
            continue

        # Train/val split
        X_tr, X_val, y_tr, y_val = train_test_split(
            X_train_full, y_train_full,
            test_size=val_size,
            random_state=SEED,
            stratify=y_train_full
        )

        # Impute + scale TRAIN ONLY (flatten time)
        imp = SimpleImputer(strategy="median")
        sca = StandardScaler()

        Xtr_flat = X_tr.reshape(-1, F)
        Xval_flat = X_val.reshape(-1, F)
        Xte_flat = X_test.reshape(-1, F)

        Xtr_flat = imp.fit_transform(Xtr_flat)
        Xval_flat = imp.transform(Xval_flat)
        Xte_flat = imp.transform(Xte_flat)

        Xtr_flat = sca.fit_transform(Xtr_flat)
        Xval_flat = sca.transform(Xval_flat)
        Xte_flat = sca.transform(Xte_flat)

        X_tr = Xtr_flat.reshape(-1, seq_len, F).astype(np.float32)
        X_val = Xval_flat.reshape(-1, seq_len, F).astype(np.float32)
        X_te  = Xte_flat.reshape(-1, seq_len, F).astype(np.float32)

        # Class weights
        classes = np.array([0, 1])
        cw = compute_class_weight(class_weight="balanced", classes=classes, y=y_tr)
        class_weight = {0: float(cw[0]), 1: float(cw[1])}

        model = build_rawseq_transformer(
            seq_len=seq_len, feat_dim=F,
            d_model=64, num_heads=2, key_dim=32,
            ff_dim=128, depth=2, dropout=0.15,
            lr=7e-5, weight_decay=1e-4,
            label_smoothing=0.0  # try 0.05 if still collapses
        )

        early = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=0)
        rlrop = ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, min_lr=1e-6, verbose=0)

        train_ds = make_tf_dataset(X_tr, y_tr, batch_size=batch_size, shuffle=True,  drop_remainder=True)
        val_ds   = make_tf_dataset(X_val, y_val, batch_size=batch_size, shuffle=False, drop_remainder=False)

        model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=epochs,
            verbose=0,
            class_weight=class_weight,
            callbacks=[early, rlrop]
        )

        # IMPORTANT: predict on FULL val/test arrays (no dropped samples)
        p_val = model.predict(X_val, batch_size=batch_size, verbose=0).reshape(-1)
        p_te  = model.predict(X_te,  batch_size=batch_size, verbose=0).reshape(-1)

        thr, val_f1 = tune_threshold_on_val(y_val, p_val)
        yhat = (p_te >= thr).astype(int)

        m = eval_binary(y_test, yhat)

        print(f"[Temporal | T-SeqRaw+thr] HELD-OUT={test_app:12s} (SEQ_LEN={seq_len}) "
              f"thr={thr:.2f} val_f1={val_f1:.3f}  "
              f"acc={m['acc']:.3f} f1={m['f1']:.3f} prec={m['precision']:.3f} rec={m['recall']:.3f}  "
              f"n_test={len(y_test)}")

        results.append({
            "held_out_app": test_app,
            "acc": m["acc"], "f1": m["f1"], "precision": m["precision"], "recall": m["recall"],
            "thr": thr, "val_f1": val_f1, "cm": m["cm"],
            "n_test": int(len(y_test)),
        })

        tf.keras.backend.clear_session()

    return pd.DataFrame(results)

# -------------------------
# 9) RUN
# -------------------------
SEQ_LEN = 5
MIN_NON_NAN_RATIO = 0.4

res = run_loao_temporal_transformer_rawseq(
    seq_len=SEQ_LEN,
    min_non_nan_ratio=MIN_NON_NAN_RATIO,
    epochs=25,
    batch_size=64,
    val_size=0.15
)

print("\n================ SUMMARY (mean over held-out apps) ================")
if len(res) > 0:
    print(res[["acc","f1","precision","recall"]].mean())
else:
    print("No results.")

OUT = "/content/loao_temporal_transformer_seqraw_len5_FIXED.csv"
res.to_csv(OUT, index=False)
print("\nSaved:", OUT)


================= LOADED DATA =================
Loaded: /content/extracted_metrics_all_apps.csv | shape: (1744, 28)
Apps: ['Archery', 'PhantomLimb', 'PianoTiles', 'Puzzle', 'Sea', 'War']

[INFO] Downsample PhantomLimb = True | TARGET_N_PL = 400
[INFO] PhantomLimb N = 325 | Total N = 1744

================= FEATURES =================
Num features: 23
First 25 feature cols: ['missing_joints_count', 'missing_joints_ratio', 'collapsed_joints_count', 'center_of_mass_x', 'center_of_mass_y', 'center_of_mass_z', 'distance_from_origin', 'bbox_width', 'bbox_height', 'bbox_depth', 'bbox_volume', 'max_joint_distance_from_com', 'distance_from_floor', 'below_floor', 'left_forearm_length', 'right_forearm_length', 'left_shin_length', 'right_shin_length', 'arm_length_symmetry', 'leg_length_symmetry', 'body_forward_x', 'body_forward_y', 'body_forward_z']

================= LOAO: TEMPORAL (RAW SEQ) - TRANSFORMER (FIXED) =================
[INFO] SEQ_LEN=5 | raw tokens = (T=5, F=23)
[Temporal | T-SeqRaw+t

In [ ]:
# ===============================================================
# VR APP-INDEPENDENT METRICS - LOAO TEMPORAL - TRANSFORMER on AGG TOKENS
#
# Goal:
#   Use your *aggregated* temporal features (the same idea behind the 138-dim RF/LR),
#   but represent them as tokens so a Transformer can attend over (mean/std/delta/range/absvel/velstd).
#
# Input construction:
#   - Start from raw window (SEQ_LEN=5, F=23)
#   - Compute agg blocks: ("mean","std","delta","range","mean_abs_vel","vel_std")
#   - Instead of concatenating to (6*F=138) vector, reshape to tokens:
#         X_tokens shape = (N_seq, 6, F)  => (N_seq, 6, 23)
#
# Train:
#   - LOAO (leave-one-app-out)
#   - Train-only impute+scale (flatten across tokens)
#   - Class weights
#   - Threshold tuned on VAL F1 (like your LR_thr)
#   - tf.data (fixed batch, drop_remainder=True) for stability
# ===============================================================

import os
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.layers import (
    Input, Dense, Dropout, LayerNormalization,
    MultiHeadAttention, GlobalAveragePooling1D
)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# -------------------------
# 1) LOAD DATA
# -------------------------
CANDIDATE_PATHS = [
    "/content/extracted_metrics_all_apps.csv",
    "/content/drive/MyDrive/extracted_metrics_all_apps.csv",
    "/content/drive/MyDrive/VR_Metrics/extracted_metrics_all_apps.csv",
]
DATA_PATH = None
for p in CANDIDATE_PATHS:
    if os.path.exists(p):
        DATA_PATH = p
        break
if DATA_PATH is None:
    raise FileNotFoundError("Could not find extracted_metrics_all_apps.csv in common locations.")

df = pd.read_csv(DATA_PATH)
print("\n================= LOADED DATA =================")
print("Loaded:", DATA_PATH, "| shape:", df.shape)

# -------------------------
# 2) CLEAN / TYPES
# -------------------------
df = df.replace("", np.nan)

required = {"App", "Temporal"}
missing_req = required - set(df.columns)
if missing_req:
    raise ValueError(f"Missing required columns: {missing_req}")

df["Temporal"] = pd.to_numeric(df["Temporal"], errors="coerce")
df = df.dropna(subset=["App", "Temporal"]).copy()
df["Temporal"] = df["Temporal"].astype(int)

apps = sorted(df["App"].unique().tolist())
print("Apps:", apps)

# OPTIONAL: Downsample PhantomLimb
DOWNSAMPLE_PHANTOMLIMB = True
TARGET_N_PL = 400
if DOWNSAMPLE_PHANTOMLIMB and ("PhantomLimb" in df["App"].unique()):
    pl = df[df["App"] == "PhantomLimb"]
    others = df[df["App"] != "PhantomLimb"]
    if len(pl) > TARGET_N_PL:
        pl = pl.sample(TARGET_N_PL, random_state=SEED)
    df = pd.concat([others, pl], ignore_index=True)
    apps = sorted(df["App"].unique().tolist())
    print("\n[INFO] Downsample PhantomLimb =", DOWNSAMPLE_PHANTOMLIMB, "| TARGET_N_PL =", TARGET_N_PL)
    print("[INFO] PhantomLimb N =", len(df[df["App"] == "PhantomLimb"]), "| Total N =", len(df))

# -------------------------
# 3) FEATURE COLS (23)
# -------------------------
ID_COLS = [c for c in ["GlobalID", "EntryID"] if c in df.columns]
DROP_COLS = ["App", "Spatial", "Temporal"] + ID_COLS
feature_cols = [c for c in df.columns if c not in DROP_COLS]

for c in feature_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

F = len(feature_cols)
print("\n================= FEATURES =================")
print("Num features:", F)
print("First 25 feature cols:", feature_cols[:25])

# -------------------------
# 4) NaN-safe reducers (Patch B style)
# -------------------------
def nanmean_no_warn(x, axis=0):
    x = np.asarray(x, dtype=float)
    valid = np.isfinite(x)
    denom = valid.sum(axis=axis)
    num = np.where(valid, x, 0.0).sum(axis=axis)
    out = np.divide(num, denom, out=np.full_like(num, np.nan, dtype=float), where=(denom > 0))
    return out

def nanstd_no_warn(x, axis=0):
    x = np.asarray(x, dtype=float)
    mu = nanmean_no_warn(x, axis=axis)
    mu_b = np.expand_dims(mu, axis=axis)
    diff2 = (x - mu_b) ** 2
    valid = np.isfinite(diff2)
    denom = valid.sum(axis=axis)
    num = np.where(valid, diff2, 0.0).sum(axis=axis)
    var = np.divide(num, denom, out=np.full_like(num, np.nan, dtype=float), where=(denom > 0))
    return np.sqrt(var)

def nanrange_no_warn(x, axis=0):
    x = np.asarray(x, dtype=float)
    valid = np.isfinite(x)
    x_for_max = np.where(valid, x, -np.inf)
    x_for_min = np.where(valid, x, +np.inf)
    mx = np.max(x_for_max, axis=axis)
    mn = np.min(x_for_min, axis=axis)
    all_nan = ~valid.any(axis=axis)
    out = mx - mn
    out = np.where(all_nan, np.nan, out)
    return out

def window_is_valid(window, min_non_nan_ratio=0.4):
    total = window.size
    non_nan = np.isfinite(window).sum()
    return (non_nan / total) >= min_non_nan_ratio

# -------------------------
# 5) Build AGG TOKENS dataset: (N, 6, F)
# -------------------------
AGG_USE = ("mean","std","delta","range","mean_abs_vel","vel_std")
N_TOKENS = len(AGG_USE)

def aggregate_window_to_tokens(window, use=AGG_USE):
    """
    window: (T, F)
    returns tokens: (K, F) where K=len(use)
    """
    window = np.asarray(window, dtype=float)
    tokens = []

    if "mean" in use:
        tokens.append(nanmean_no_warn(window, axis=0))
    if "std" in use:
        tokens.append(nanstd_no_warn(window, axis=0))
    if "delta" in use:
        tokens.append(window[-1] - window[0])
    if "range" in use:
        tokens.append(nanrange_no_warn(window, axis=0))

    if "mean_abs_vel" in use or "vel_std" in use:
        diff = np.diff(window, axis=0)  # (T-1,F)
        if "mean_abs_vel" in use:
            tokens.append(nanmean_no_warn(np.abs(diff), axis=0))
        if "vel_std" in use:
            tokens.append(nanstd_no_warn(diff, axis=0))

    toks = np.stack(tokens, axis=0)  # (K,F)
    assert toks.shape[0] == len(use) and toks.shape[1] == window.shape[1]
    return toks

def make_agg_token_sequence_dataset(app_df, label_col="Temporal", seq_len=5,
                                   use=AGG_USE, min_non_nan_ratio=0.4):
    """
    Sliding window -> token matrix (K,F) per window.
    Returns:
      X: (N_seq, K, F)
      y: (N_seq,)
    """
    X = app_df[feature_cols].values.astype(float)
    y = app_df[label_col].values.astype(int)

    if len(app_df) < seq_len:
        return np.empty((0, len(use), len(feature_cols))), np.empty((0,), dtype=int)

    X_seq, y_seq = [], []
    center = seq_len // 2

    for i in range(len(app_df) - seq_len + 1):
        window = X[i:i+seq_len]
        if not window_is_valid(window, min_non_nan_ratio=min_non_nan_ratio):
            continue
        toks = aggregate_window_to_tokens(window, use=use)  # (K,F)
        X_seq.append(toks)
        y_seq.append(y[i + center])

    if len(X_seq) == 0:
        return np.empty((0, len(use), len(feature_cols))), np.empty((0,), dtype=int)

    return np.stack(X_seq, axis=0), np.array(y_seq, dtype=int)

# -------------------------
# 6) Metrics + threshold tuning
# -------------------------
def eval_binary(y_true, y_pred):
    return {
        "acc": float(accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "cm": confusion_matrix(y_true, y_pred).tolist(),
        "pred_dist": dict(zip(*np.unique(y_pred, return_counts=True))),
    }

def tune_threshold_on_val(y_val, p_val, grid=None):
    if grid is None:
        grid = np.linspace(0.05, 0.95, 19)
    best_thr, best_f1 = 0.5, -1.0
    for t in grid:
        yhat = (p_val >= t).astype(int)
        f1 = f1_score(y_val, yhat, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_thr = float(t)
    return best_thr, float(best_f1)

# -------------------------
# 7) Transformer on AGG TOKENS
# -------------------------
def build_transformer_token_model(n_tokens, n_feat,
                                  d_model=64, num_heads=2, key_dim=32,
                                  ff_dim=128, depth=2, dropout=0.15,
                                  lr=1e-4):
    """
    Input: (K, F) tokens.
    Dense projects feature-dim -> d_model per token, then attention across tokens.
    """
    inp = Input(shape=(n_tokens, n_feat), name="agg_tokens_in")
    x = Dense(d_model, activation="relu")(inp)

    for _ in range(depth):
        attn = MultiHeadAttention(num_heads=num_heads, key_dim=key_dim)(x, x)
        attn = Dropout(dropout)(attn)
        x = LayerNormalization(epsilon=1e-6)(x + attn)

        ffn = Dense(ff_dim, activation="relu")(x)
        ffn = Dense(d_model)(ffn)
        ffn = Dropout(dropout)(ffn)
        x = LayerNormalization(epsilon=1e-6)(x + ffn)

    x = GlobalAveragePooling1D()(x)
    x = Dense(64, activation="relu")(x)
    x = Dropout(dropout)(x)
    out = Dense(1, activation="sigmoid")(x)

    model = Model(inp, out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model

def make_tf_dataset(X, y, batch_size, shuffle=True):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if shuffle:
        ds = ds.shuffle(buffer_size=min(len(y), 5000), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(batch_size, drop_remainder=True).prefetch(tf.data.AUTOTUNE)
    return ds

# -------------------------
# 8) LOAO runner
# -------------------------
def run_loao_temporal_transformer_aggtokens(
    label_col="Temporal",
    seq_len=5,
    use=AGG_USE,
    min_non_nan_ratio=0.4,
    d_model=64,
    num_heads=2,
    key_dim=32,
    ff_dim=128,
    depth=2,
    dropout=0.15,
    lr=1e-4,
    epochs=25,
    batch_size=64,
    val_size=0.15,
):
    results = []

    print("\n================= LOAO: TEMPORAL (AGG TOKENS) - TRANSFORMER =================")
    print(f"[INFO] SEQ_LEN={seq_len} | tokens K={len(use)} | token dim F={len(feature_cols)}")
    print(f"[INFO] This corresponds to 6*F = {len(use)*len(feature_cols)} aggregated dims (i.e., ~138).")

    for test_app in apps:
        train_apps = [a for a in apps if a != test_app]

        # Build train token-seqs
        X_train_list, y_train_list = [], []
        for a in train_apps:
            Xa, ya = make_agg_token_sequence_dataset(
                df[df["App"] == a],
                label_col=label_col,
                seq_len=seq_len,
                use=use,
                min_non_nan_ratio=min_non_nan_ratio
            )
            if len(ya) > 0:
                X_train_list.append(Xa)
                y_train_list.append(ya)

        if len(X_train_list) == 0:
            print(f"[{label_col} | T-AggTokens] HELD-OUT={test_app} -> No train sequences.")
            continue

        X_train_full = np.concatenate(X_train_list, axis=0)  # (N, K, F)
        y_train_full = np.concatenate(y_train_list, axis=0)

        # Build test token-seqs
        X_test, y_test = make_agg_token_sequence_dataset(
            df[df["App"] == test_app],
            label_col=label_col,
            seq_len=seq_len,
            use=use,
            min_non_nan_ratio=min_non_nan_ratio
        )
        if len(y_test) == 0:
            print(f"[{label_col} | T-AggTokens] HELD-OUT={test_app} -> No test sequences.")
            continue

        # Split train/val
        X_tr, X_val, y_tr, y_val = train_test_split(
            X_train_full, y_train_full,
            test_size=val_size,
            random_state=SEED,
            stratify=y_train_full
        )

        K = X_tr.shape[1]
        n_feat = X_tr.shape[2]

        # Train-only impute+scale across tokens/features
        # Flatten (N*K, F)
        imp = SimpleImputer(strategy="median")
        sca = StandardScaler()

        X_tr_flat = X_tr.reshape(-1, n_feat)
        X_val_flat = X_val.reshape(-1, n_feat)
        X_te_flat  = X_test.reshape(-1, n_feat)

        X_tr_flat = imp.fit_transform(X_tr_flat)
        X_val_flat = imp.transform(X_val_flat)
        X_te_flat  = imp.transform(X_te_flat)

        X_tr_flat = sca.fit_transform(X_tr_flat)
        X_val_flat = sca.transform(X_val_flat)
        X_te_flat  = sca.transform(X_te_flat)

        X_tr = X_tr_flat.reshape(-1, K, n_feat).astype(np.float32)
        X_val = X_val_flat.reshape(-1, K, n_feat).astype(np.float32)
        X_te  = X_te_flat.reshape(-1, K, n_feat).astype(np.float32)

        # Class weights
        classes = np.array([0, 1])
        cw = compute_class_weight(class_weight="balanced", classes=classes, y=y_tr)
        class_weight = {0: float(cw[0]), 1: float(cw[1])}

        # Build model
        model = build_transformer_token_model(
            n_tokens=K, n_feat=n_feat,
            d_model=d_model, num_heads=num_heads, key_dim=key_dim,
            ff_dim=ff_dim, depth=depth, dropout=dropout, lr=lr
        )

        early = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=0)
        rlrop = ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, min_lr=1e-6, verbose=0)

        train_ds = make_tf_dataset(X_tr, y_tr, batch_size=batch_size, shuffle=True)
        val_ds   = make_tf_dataset(X_val, y_val, batch_size=batch_size, shuffle=False)

        model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=epochs,
            verbose=0,
            class_weight=class_weight,
            callbacks=[early, rlrop]
        )

        p_val = model.predict(X_val, batch_size=batch_size, verbose=0).reshape(-1)
        p_te  = model.predict(X_te,  batch_size=batch_size, verbose=0).reshape(-1)

        thr, val_f1 = tune_threshold_on_val(y_val, p_val)
        yhat = (p_te >= thr).astype(int)

        m = eval_binary(y_test, yhat)

        row = {
            "label": label_col,
            "task": f"sequence_aggtokens_len_{seq_len}",
            "model": "Transformer_aggtokens",
            "held_out_app": test_app,
            "seq_len": int(seq_len),
            "K_tokens": int(K),
            "F_token": int(n_feat),
            "agg_dims_equiv": int(K * n_feat),
            "n_train": int(len(y_tr)),
            "n_val": int(len(y_val)),
            "n_test": int(len(y_test)),
            "thr": float(thr),
            "val_f1_best": float(val_f1),
            **{k: v for k, v in m.items() if k not in ["cm", "pred_dist"]},
            "cm": m["cm"],
            "pred_dist": m["pred_dist"],
        }

        print(f"[Temporal | T-AggTokens+thr] HELD-OUT={test_app:12s} "
              f"(SEQ_LEN={seq_len}, K={K}, F={n_feat}, equiv={K*n_feat}) "
              f"thr={thr:.2f} val_f1={val_f1:.3f}  "
              f"acc={row['acc']:.3f} f1={row['f1']:.3f} "
              f"prec={row['precision']:.3f} rec={row['recall']:.3f} "
              f"n_test={row['n_test']}")

        results.append(row)
        tf.keras.backend.clear_session()

    return pd.DataFrame(results)

# -------------------------
# 9) RUN
# -------------------------
SEQ_LEN = 5
MIN_NON_NAN_RATIO = 0.4

res = run_loao_temporal_transformer_aggtokens(
    label_col="Temporal",
    seq_len=SEQ_LEN,
    use=AGG_USE,
    min_non_nan_ratio=MIN_NON_NAN_RATIO,
    d_model=64,
    num_heads=2,
    key_dim=32,
    ff_dim=128,
    depth=2,
    dropout=0.15,
    lr=1e-4,
    epochs=25,
    batch_size=64,
    val_size=0.15,
)

print("\n================ SUMMARY (mean over held-out apps) ================")
if len(res) > 0:
    print(res[["acc", "f1", "precision", "recall"]].mean())
else:
    print("No results (no sequences found).")

OUT = "/content/loao_temporal_transformer_aggtokens_len5.csv"
res.to_csv(OUT, index=False)
print("\nSaved:", OUT)


================= LOADED DATA =================
Loaded: /content/extracted_metrics_all_apps.csv | shape: (1744, 28)
Apps: ['Archery', 'PhantomLimb', 'PianoTiles', 'Puzzle', 'Sea', 'War']

[INFO] Downsample PhantomLimb = True | TARGET_N_PL = 400
[INFO] PhantomLimb N = 325 | Total N = 1744

================= FEATURES =================
Num features: 23
First 25 feature cols: ['missing_joints_count', 'missing_joints_ratio', 'collapsed_joints_count', 'center_of_mass_x', 'center_of_mass_y', 'center_of_mass_z', 'distance_from_origin', 'bbox_width', 'bbox_height', 'bbox_depth', 'bbox_volume', 'max_joint_distance_from_com', 'distance_from_floor', 'below_floor', 'left_forearm_length', 'right_forearm_length', 'left_shin_length', 'right_shin_length', 'arm_length_symmetry', 'leg_length_symmetry', 'body_forward_x', 'body_forward_y', 'body_forward_z']

================= LOAO: TEMPORAL (AGG TOKENS) - TRANSFORMER =================
[INFO] SEQ_LEN=5 | tokens K=6 | token dim F=23
[INFO] This corresponds 

# Per-app evaluation

# -Temporal

## LR - invariant metrics

In [ ]:
# ============================================================
# PHANTOMLIMB (TRAIN) -> PHANTOMLIMB HOLDOUT (EVAL)
# Temporal LR_balanced on INVARIANT27 + Patch-B aggregation (SEQ_LEN=5)
#
# Key points:
#  - Train ONE final model on ALL PhantomLimb TRAIN sequences
#  - Evaluate ONCE on PhantomLimb HOLDOUT sequences
#  - Uses ONLY invariant27 metrics (27) aggregated over a 5-frame window
#  - Patch-B aggregation: mean, std, delta, range, mean_abs_vel, vel_std => 162-dim vector
#  - Pipeline (imputer+scaler+LR) is fit ONLY on TRAIN
# ============================================================

import os, json
import numpy as np
import pandas as pd
import joblib

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

SEED = 42
np.random.seed(SEED)

SEQ_LEN = 5
MID = SEQ_LEN // 2
AGG_USE = ("mean", "std", "delta", "range", "mean_abs_vel", "vel_std")
MIN_NON_NAN_RATIO = 0.4
LABEL_COL = "Temporal"

TRAIN_CSV   = "/content/metrics_PhantomLimb.csv"
HOLDOUT_CSV = "/content/metrics_PhantomLimb_Holdout.csv"

OUT_DIR = "/content/phantomlimb_temporal_lr_balanced_holdout"
os.makedirs(OUT_DIR, exist_ok=True)

INVARIANT27 = [
    "missing_joints_count","missing_joints_ratio","collapsed_joints_count",
    "center_of_mass_x","center_of_mass_y","center_of_mass_z","distance_from_origin",
    "bbox_width","bbox_height","bbox_depth","bbox_volume","max_joint_distance_from_com",
    "distance_from_floor","min_foot_height_above_floor","below_floor",
    "left_forearm_length","right_forearm_length","left_shin_length","right_shin_length",
    "arm_length_symmetry","leg_length_symmetry",
    "body_upright_x","body_upright_y","body_upright_z",
    "body_forward_x","body_forward_y","body_forward_z"
]
TAB_DIM = len(INVARIANT27) * len(AGG_USE)  # 27*6=162

# --------------------------
# Patch-B NaN-safe aggregation
# --------------------------
def nanmean_no_warn(x, axis=0):
    x = np.asarray(x, float)
    valid = np.isfinite(x)
    denom = valid.sum(axis=axis)
    num = np.where(valid, x, 0.0).sum(axis=axis)
    return np.divide(num, denom, out=np.full_like(num, np.nan), where=(denom > 0))

def nanstd_no_warn(x, axis=0):
    mu = nanmean_no_warn(x, axis=axis)
    diff2 = (x - np.expand_dims(mu, axis=axis)) ** 2
    valid = np.isfinite(diff2)
    denom = valid.sum(axis=axis)
    num = np.where(valid, diff2, 0.0).sum(axis=axis)
    return np.sqrt(np.divide(num, denom, out=np.full_like(num, np.nan), where=(denom > 0)))

def nanrange_no_warn(x, axis=0):
    x = np.asarray(x, float)
    valid = np.isfinite(x)
    mx = np.max(np.where(valid, x, -np.inf), axis=axis)
    mn = np.min(np.where(valid, x, +np.inf), axis=axis)
    return np.where(valid.any(axis=axis), mx - mn, np.nan)

def window_is_valid(window):
    window = np.asarray(window, float)
    return (np.isfinite(window).sum() / window.size) >= MIN_NON_NAN_RATIO

def aggregate_window_patchB(window_5x27):
    # window_5x27 shape: (SEQ_LEN, 27)
    window = np.asarray(window_5x27, float)
    feats = [
        nanmean_no_warn(window, 0),           # mean
        nanstd_no_warn(window, 0),            # std
        window[-1] - window[0],               # delta
        nanrange_no_warn(window, 0),          # range
    ]
    diff = np.diff(window, axis=0)           # (4,27)
    feats += [
        nanmean_no_warn(np.abs(diff), 0),     # mean_abs_vel
        nanstd_no_warn(diff, 0),              # vel_std
    ]
    out = np.concatenate(feats, axis=0)      # (162,)
    if out.shape[0] != TAB_DIM:
        raise RuntimeError(f"Patch-B dim mismatch: got {out.shape[0]} expected {TAB_DIM}")
    return out

def clean_and_order(df):
    df = df.copy().replace("", np.nan)
    df[LABEL_COL] = pd.to_numeric(df[LABEL_COL], errors="coerce")
    df = df.dropna(subset=[LABEL_COL]).copy()
    df[LABEL_COL] = df[LABEL_COL].astype(int)

    order_col = None
    for c in ["EntryID", "entryid", "Timestamp", "timestamp", "Frame", "frame"]:
        if c in df.columns:
            order_col = c
            break

    if order_col is not None:
        tmp = pd.to_numeric(df[order_col], errors="coerce")
        if tmp.notna().any():
            df = df.assign(_ord=tmp).sort_values("_ord").drop(columns=["_ord"])
        else:
            df = df.sort_values(order_col)

    return df.reset_index(drop=True), order_col

def make_sequence_dataset_from_df(df_raw):
    df, order_col = clean_and_order(df_raw)

    missing = [c for c in INVARIANT27 if c not in df.columns]
    if missing:
        raise RuntimeError(f"Missing invariant cols: {missing[:10]}")

    X_frame = df[INVARIANT27].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=float)  # (N,27)
    y_frame = df[LABEL_COL].to_numpy(dtype=int)

    X_seq, y_seq = [], []
    for i in range(len(X_frame) - SEQ_LEN + 1):
        w = X_frame[i:i+SEQ_LEN]
        if not window_is_valid(w):
            continue
        X_seq.append(aggregate_window_patchB(w))
        y_seq.append(int(y_frame[i + MID]))

    X_seq = np.asarray(X_seq, dtype=float)
    y_seq = np.asarray(y_seq, dtype=int)

    # keep-dims / NaN handling for LR input:
    # - columns that are all-NaN in this split -> set to 0.0 (then imputer/scaler still fit on train)
    if X_seq.size > 0:
        col_all_nan = np.all(~np.isfinite(X_seq), axis=0)
        if np.any(col_all_nan):
            X_seq[:, col_all_nan] = 0.0

    return X_seq, y_seq, order_col

def make_lr_balanced():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            class_weight="balanced",
            max_iter=3000,
            random_state=SEED,
            n_jobs=-1
        ))
    ])

def eval_metrics(y_true, y_pred):
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "cm": confusion_matrix(y_true, y_pred, labels=[0,1]).tolist(),
    }

# --------------------------
# LOAD + BUILD SEQUENCES
# --------------------------
df_train_raw = pd.read_csv(TRAIN_CSV)
df_hold_raw  = pd.read_csv(HOLDOUT_CSV)

X_tr, y_tr, order_tr = make_sequence_dataset_from_df(df_train_raw)
X_h,  y_h,  order_h  = make_sequence_dataset_from_df(df_hold_raw)

print("\n================= PHANTOMLIMB TEMPORAL LR (PATCH-B, INVARIANT27) =================")
print(f"[TRAIN]   rows={len(df_train_raw)} order_col={order_tr} sequences={X_tr.shape} ydist={dict(zip(*np.unique(y_tr, return_counts=True)))}")
print(f"[HOLDOUT] rows={len(df_hold_raw)}  order_col={order_h}  sequences={X_h.shape}  ydist={dict(zip(*np.unique(y_h,  return_counts=True)))}")
print(f"[INFO] tab_dim={TAB_DIM} (27*6)")

if len(y_tr) < 50:
    raise RuntimeError("Too few TRAIN sequences after filtering (check ordering / NaNs / MIN_NON_NAN_RATIO).")
if len(y_h) == 0:
    raise RuntimeError("HOLDOUT produced 0 sequences (check ordering / NaNs / SEQ_LEN).")

# --------------------------
# TRAIN FINAL MODEL ON ALL TRAIN SEQUENCES
# --------------------------
model = make_lr_balanced()
model.fit(X_tr, y_tr)

# --------------------------
# EVALUATE ON HOLDOUT (ONCE)
# --------------------------
yhat_h = model.predict(X_h)
m_h = eval_metrics(y_h, yhat_h)

print("\n================= TRAIN -> HOLDOUT (LR_balanced) =================")
print(f"[HOLDOUT] acc={m_h['accuracy']:.3f} f1={m_h['f1']:.3f} prec={m_h['precision']:.3f} rec={m_h['recall']:.3f} | cm={m_h['cm']}")

# --------------------------
# SAVE MODEL + META
# --------------------------
model_path = os.path.join(OUT_DIR, "model.joblib")
meta_path  = os.path.join(OUT_DIR, "meta.json")

joblib.dump(model, model_path)

meta = {
    "app": "PhantomLimb",
    "task": "Temporal_sequence",
    "model": "LR_balanced",
    "seed": SEED,
    "label_col": LABEL_COL,
    "seq_len": SEQ_LEN,
    "mid_label_index": MID,
    "min_non_nan_ratio": float(MIN_NON_NAN_RATIO),
    "frame_feature_cols": INVARIANT27,
    "agg_use": list(AGG_USE),
    "seq_feature_dim": int(TAB_DIM),
    "order_col_train": order_tr,
    "order_col_holdout": order_h,
    "n_train_sequences": int(len(y_tr)),
    "n_holdout_sequences": int(len(y_h)),
    "trained_on": TRAIN_CSV,
    "evaluated_on_holdout": HOLDOUT_CSV,
    "holdout_metrics": m_h,
    "notes": "Trained on PhantomLimb train sequences only; evaluated once on PhantomLimb holdout sequences. Preprocessing fit on train only."
}
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print("\nSaved:")
print(" -", model_path)
print(" -", meta_path)


================= PHANTOMLIMB TEMPORAL LR (PATCH-B, INVARIANT27) =================
[TRAIN]   rows=4105 order_col=EntryID sequences=(4101, 162) ydist={np.int64(0): np.int64(2660), np.int64(1): np.int64(1441)}
[HOLDOUT] rows=391  order_col=EntryID  sequences=(387, 162)  ydist={np.int64(0): np.int64(286), np.int64(1): np.int64(101)}
[INFO] tab_dim=162 (27*6)

================= TRAIN -> HOLDOUT (LR_balanced) =================
[HOLDOUT] acc=0.757 f1=0.669 prec=0.519 rec=0.941 | cm=[[198, 88], [6, 95]]

Saved:
 - /content/phantomlimb_temporal_lr_balanced_holdout/model.joblib
 - /content/phantomlimb_temporal_lr_balanced_holdout/meta.json


In [ ]:
# ============================================================
# Temporal per-app LR_balanced (Invariant27, Patch-B agg)
# + PhantomLimb Holdout evaluation
# ============================================================

import os, json
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

SEED = 42
np.random.seed(SEED)

SEQ_LEN = 5
AGG_USE = ("mean", "std", "delta", "range", "mean_abs_vel", "vel_std")  # Patch-B
MIN_NON_NAN_RATIO = 0.4
LABEL_COL = "Temporal"

INVARIANT27 = [
    "missing_joints_count","missing_joints_ratio","collapsed_joints_count",
    "center_of_mass_x","center_of_mass_y","center_of_mass_z","distance_from_origin",
    "bbox_width","bbox_height","bbox_depth","bbox_volume","max_joint_distance_from_com",
    "distance_from_floor","min_foot_height_above_floor","below_floor",
    "left_forearm_length","right_forearm_length","left_shin_length","right_shin_length",
    "arm_length_symmetry","leg_length_symmetry",
    "body_upright_x","body_upright_y","body_upright_z",
    "body_forward_x","body_forward_y","body_forward_z"
]

# --------------------------
# Patch-B NaN-safe aggregation
# --------------------------
def nanmean_no_warn(x, axis=0):
    x = np.asarray(x, float)
    valid = np.isfinite(x)
    denom = valid.sum(axis=axis)
    num = np.where(valid, x, 0.0).sum(axis=axis)
    return np.divide(num, denom, out=np.full_like(num, np.nan), where=(denom > 0))

def nanstd_no_warn(x, axis=0):
    mu = nanmean_no_warn(x, axis=axis)
    diff2 = (x - np.expand_dims(mu, axis=axis)) ** 2
    valid = np.isfinite(diff2)
    denom = valid.sum(axis=axis)
    num = np.where(valid, diff2, 0.0).sum(axis=axis)
    return np.sqrt(np.divide(num, denom, out=np.full_like(num, np.nan), where=(denom > 0)))

def nanrange_no_warn(x, axis=0):
    x = np.asarray(x, float)
    valid = np.isfinite(x)
    mx = np.max(np.where(valid, x, -np.inf), axis=axis)
    mn = np.min(np.where(valid, x, +np.inf), axis=axis)
    return np.where(valid.any(axis=axis), mx - mn, np.nan)

def window_is_valid(window):
    window = np.asarray(window, float)
    return (np.isfinite(window).sum() / window.size) >= MIN_NON_NAN_RATIO

def aggregate_window(window):
    window = np.asarray(window, float)
    feats = [
        nanmean_no_warn(window, 0),
        nanstd_no_warn(window, 0),
        window[-1] - window[0],
        nanrange_no_warn(window, 0),
    ]
    diff = np.diff(window, axis=0)
    feats += [
        nanmean_no_warn(np.abs(diff), 0),
        nanstd_no_warn(diff, 0),
    ]
    return np.concatenate(feats)

def make_sequence_dataset(df_app, feature_cols):
    df = df_app.copy().replace("", np.nan)

    # label
    df[LABEL_COL] = pd.to_numeric(df[LABEL_COL], errors="coerce")
    df = df.dropna(subset=[LABEL_COL]).copy()
    df[LABEL_COL] = df[LABEL_COL].astype(int)

    # order (critical for temporal)
    ORDER_COL = None
    for c in ["EntryID", "entryid", "Timestamp", "timestamp", "Frame", "frame"]:
        if c in df.columns:
            ORDER_COL = c
            break

    if ORDER_COL is not None:
        tmp = pd.to_numeric(df[ORDER_COL], errors="coerce")
        if tmp.notna().any():
            df = df.assign(_order_tmp=tmp).sort_values("_order_tmp").drop(columns="_order_tmp")
        else:
            df = df.sort_values(ORDER_COL)

    df = df.reset_index(drop=True)

    # numeric features
    df_num = df.apply(pd.to_numeric, errors="coerce")
    X = df_num[feature_cols].to_numpy(dtype=float)
    y = df[LABEL_COL].to_numpy(dtype=int)

    X_seq, y_seq = [], []
    mid = SEQ_LEN // 2
    for i in range(len(X) - SEQ_LEN + 1):
        w = X[i:i+SEQ_LEN]
        if not window_is_valid(w):
            continue
        X_seq.append(aggregate_window(w))
        y_seq.append(int(y[i + mid]))

    return np.asarray(X_seq), np.asarray(y_seq), ORDER_COL

def make_lr_balanced():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            class_weight="balanced",
            max_iter=3000,
            random_state=SEED,
            n_jobs=-1
        ))
    ])

def eval_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=[0,1]).tolist()
    return acc, f1, prec, rec, cm

def run_app_from_metrics_csv(app_name, csv_path, save_dir, df_holdout=None):
    df_app = pd.read_csv(csv_path)

    # enforce invariant27 only
    missing = [c for c in INVARIANT27 if c not in df_app.columns]
    if missing:
        raise RuntimeError(f"[{app_name}] metrics csv missing invariant cols: {missing[:10]}")
    if LABEL_COL not in df_app.columns:
        raise RuntimeError(f"[{app_name}] metrics csv missing label '{LABEL_COL}'")

    feature_cols = INVARIANT27[:]  # exactly 27
    X_all, y_all, ORDER_COL = make_sequence_dataset(df_app, feature_cols)

    print(f"\n================= {app_name.upper()} =================")
    print("[INFO] rows:", len(df_app), "| label dist:", pd.Series(df_app[LABEL_COL]).value_counts().to_dict())
    print("[INFO] ORDER_COL:", ORDER_COL)
    print("[INFO] sequences:", X_all.shape, "| y dist:", dict(zip(*np.unique(y_all, return_counts=True))))
    print("[INFO] expected dim:", len(AGG_USE) * len(feature_cols))

    if len(y_all) < 50:
        raise RuntimeError(f"[{app_name}] too few sequences after filtering (check ordering / NaNs / MIN_NON_NAN_RATIO).")

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

    fold_rows = []
    best_key = (-1.0, -1.0)
    best_fold = None
    best_model = None

    for fold, (tr, te) in enumerate(skf.split(X_all, y_all), 1):
        m = make_lr_balanced()
        m.fit(X_all[tr], y_all[tr])
        yhat = m.predict(X_all[te])

        acc, f1, prec, rec, cm = eval_metrics(y_all[te], yhat)
        fold_rows.append({
            "app": app_name, "fold": fold,
            "accuracy": float(acc), "f1": float(f1),
            "precision": float(prec), "recall": float(rec),
            "cm": cm
        })
        print(f"[FOLD {fold}] acc={acc:.3f} f1={f1:.3f} prec={prec:.3f} rec={rec:.3f} | cm={cm}")

        # accuracy-first, f1 tie-break
        key = (float(acc), float(f1))
        if key > best_key:
            best_key = key
            best_fold = fold
            best_model = m

    holdout_report = None
    if df_holdout is not None:
        miss_h = [c for c in INVARIANT27 if c not in df_holdout.columns]
        if miss_h:
            raise RuntimeError(f"[{app_name}] holdout df missing invariant cols: {miss_h[:10]}")
        if LABEL_COL not in df_holdout.columns:
            raise RuntimeError(f"[{app_name}] holdout df missing label '{LABEL_COL}'")

        X_h, y_h, _ = make_sequence_dataset(df_holdout, feature_cols)
        if len(y_h) == 0:
            raise RuntimeError(f"[{app_name}] holdout produced 0 sequences (check ORDER_COL/NaNs/SEQ_LEN).")

        yhat_h = best_model.predict(X_h)
        acc, f1, prec, rec, cm = eval_metrics(y_h, yhat_h)
        holdout_report = {"accuracy": float(acc), "f1": float(f1), "precision": float(prec), "recall": float(rec), "cm": cm}
        print(f"[HOLDOUT(best_fold={best_fold})] acc={acc:.3f} f1={f1:.3f} prec={prec:.3f} rec={rec:.3f} | cm={cm}")

    os.makedirs(save_dir, exist_ok=True)
    joblib.dump(best_model, os.path.join(save_dir, "model.joblib"))

    meta = {
        "app": app_name,
        "task": "Temporal_sequence",
        "model": "LR_balanced",
        "seed": SEED,
        "label_col": LABEL_COL,
        "seq_len": SEQ_LEN,
        "agg_use": list(AGG_USE),
        "min_non_nan_ratio": float(MIN_NON_NAN_RATIO),
        "order_col": ORDER_COL,
        "frame_feature_cols": feature_cols,
        "frame_F": len(feature_cols),
        "seq_feature_dim": int(len(AGG_USE) * len(feature_cols)),
        "cv_n_splits": 5,
        "fold_metrics": fold_rows,
        "best_fold": int(best_fold),
        "best_key_(acc,f1)": [float(best_key[0]), float(best_key[1])],
        "holdout": holdout_report,
        "notes": "Per-app Temporal LR_balanced using invariant27 only + Patch-B aggregation."
    }
    with open(os.path.join(save_dir, "meta.json"), "w") as f:
        json.dump(meta, f, indent=2)

    return meta


# ============================================================
# RUN (includes PhantomLimb holdout)
# ============================================================

PHANTOM_HOLDOUT_PATH = "/content/metrics_PhantomLimb_Holdout.csv"
df_phantom_holdout = pd.read_csv(PHANTOM_HOLDOUT_PATH)
print("[LOAD] PhantomLimb HOLDOUT rows=", len(df_phantom_holdout), "cols=", len(df_phantom_holdout.columns))

apps = {
    "Archery":      "/content/metrics_Archery.csv",
    "PhantomLimb":  "/content/metrics_PhantomLimb.csv",
    "PianoTiles":   "/content/metrics_PianoTiles.csv",
    "Puzzle":       "/content/metrics_Puzzle.csv",
    "Sea":          "/content/metrics_Sea.csv",
    "War":          "/content/metrics_War.csv",
}

results = []
for app, path in apps.items():
    save_dir = f"/content/saved_{app.lower()}_temporal_lr_balanced_5fold_invariant27_from_metrics_csv"
    holdout_df = df_phantom_holdout if app == "PhantomLimb" else None

    meta = run_app_from_metrics_csv(app, path, save_dir, df_holdout=holdout_df)

    results.append({
        "app": app,
        "best_fold": meta["best_fold"],
        "best_acc": meta["best_key_(acc,f1)"][0],
        "best_f1":  meta["best_key_(acc,f1)"][1],
        "cv_mean_acc": float(np.mean([r["accuracy"] for r in meta["fold_metrics"]])),
        "cv_mean_f1":  float(np.mean([r["f1"] for r in meta["fold_metrics"]])),
        "holdout_acc": (meta["holdout"]["accuracy"] if meta["holdout"] else np.nan),
        "holdout_f1":  (meta["holdout"]["f1"] if meta["holdout"] else np.nan),
    })

summary = pd.DataFrame(results).sort_values("cv_mean_acc", ascending=False)
print("\n================= SUMMARY =================")
print(summary)

[LOAD] PhantomLimb HOLDOUT rows= 391 cols= 32

================= ARCHERY =================
[INFO] rows: 425 | label dist: {1: 304, 0: 121}
[INFO] ORDER_COL: EntryID
[INFO] sequences: (421, 162) | y dist: {np.int64(0): np.int64(119), np.int64(1): np.int64(302)}
[INFO] expected dim: 162
[FOLD 1] acc=0.929 f1=0.951 prec=0.951 rec=0.951 | cm=[[21, 3], [3, 58]]
[FOLD 2] acc=0.952 f1=0.966 prec=0.983 rec=0.950 | cm=[[23, 1], [3, 57]]
[FOLD 3] acc=0.905 f1=0.930 prec=0.981 rec=0.883 | cm=[[23, 1], [7, 53]]
[FOLD 4] acc=0.952 f1=0.967 prec=0.952 rec=0.983 | cm=[[21, 3], [1, 59]]
[FOLD 5] acc=0.988 f1=0.992 prec=1.000 rec=0.984 | cm=[[23, 0], [1, 60]]

================= PHANTOMLIMB =================
[INFO] rows: 4105 | label dist: {0: 2663, 1: 1442}
[INFO] ORDER_COL: EntryID
[INFO] sequences: (4101, 162) | y dist: {np.int64(0): np.int64(2660), np.int64(1): np.int64(1441)}
[INFO] expected dim: 162
[FOLD 1] acc=0.851 f1=0.795 prec=0.774 rec=0.817 | cm=[[463, 69], [53, 236]]
[FOLD 2] acc=0.859 f1=

/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [ 12  13  14  39  40  41  66  67  68  93  94  95 120 121 122 147 148 149]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [ 12  13  14  39  40  41  66  67  68  93  94  95 120 121 122 147 148 149]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [ 12  13  14  39  40  41  66  67  68  93  94  95 120 121 122 147 148 149]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping featur

[FOLD 2] acc=0.945 f1=0.965 prec=0.953 rec=0.976 | cm=[[11, 2], [1, 41]]
[FOLD 3] acc=1.000 f1=1.000 prec=1.000 rec=1.000 | cm=[[12, 0], [0, 43]]
[FOLD 4] acc=0.945 f1=0.965 prec=0.976 rec=0.953 | cm=[[11, 1], [2, 41]]
[FOLD 5] acc=0.964 f1=0.977 prec=0.977 rec=0.977 | cm=[[11, 1], [1, 42]]

================= SEA =================
[INFO] rows: 300 | label dist: {0: 164, 1: 136}
[INFO] ORDER_COL: EntryID
[INFO] sequences: (296, 162) | y dist: {np.int64(0): np.int64(162), np.int64(1): np.int64(134)}
[INFO] expected dim: 162
[FOLD 1] acc=0.983 f1=0.981 prec=1.000 rec=0.963 | cm=[[33, 0], [1, 26]]
[FOLD 2] acc=1.000 f1=1.000 prec=1.000 rec=1.000 | cm=[[33, 0], [0, 26]]
[FOLD 3] acc=0.983 f1=0.982 prec=0.964 rec=1.000 | cm=[[31, 1], [0, 27]]
[FOLD 4] acc=0.915 f1=0.915 prec=0.844 rec=1.000 | cm=[[27, 5], [0, 27]]
[FOLD 5] acc=0.949 f1=0.945 prec=0.929 rec=0.963 | cm=[[30, 2], [1, 26]]

================= WAR =================
[INFO] rows: 281 | label dist: {0: 165, 1: 116}
[INFO] ORDER_COL: 

## Transformer- invariant metrics

In [ ]:
# ===============================================================
# PER-APP: Temporal Transformer on RAW SEQ (T=5, F=all metric cols)
# - 5-fold Stratified CV on sequences PER APP
# - inner train/val split for EarlyStopping + threshold tuning
# - impute+scale FIT ON TRAIN ONLY (flatten time)
# - returns acc/prec/rec/f1/cm per fold + means
# ===============================================================

import os
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.layers import Input, Dense, Dropout, LayerNormalization, MultiHeadAttention, GlobalAveragePooling1D
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"


# -------------------------
# 1) Data prep helpers
# -------------------------
def _clean_df(df):
    df = df.copy().replace("", np.nan)
    df["Temporal"] = pd.to_numeric(df["Temporal"], errors="coerce")
    df = df.dropna(subset=["Temporal"]).copy()
    df["Temporal"] = df["Temporal"].astype(int)
    return df

def _get_feature_cols(df):
    id_cols = [c for c in ["GlobalID", "EntryID", "Sequence_ID", "Person_ID", "FPS"] if c in df.columns]
    drop_cols = ["App", "Spatial", "Temporal"] + id_cols
    feature_cols = [c for c in df.columns if c not in drop_cols]

    out = df.copy()
    for c in feature_cols:
        out[c] = pd.to_numeric(out[c], errors="coerce")
    return out, feature_cols

def _order_df(df):
    order_col = None
    for c in ["EntryID", "entryid", "Timestamp", "timestamp", "Frame", "frame"]:
        if c in df.columns:
            order_col = c
            break
    if order_col is None:
        return df.reset_index(drop=True), None

    tmp = pd.to_numeric(df[order_col], errors="coerce")
    if tmp.notna().any():
        df = df.assign(_order_tmp=tmp).sort_values("_order_tmp").drop(columns=["_order_tmp"]).reset_index(drop=True)
    else:
        df = df.sort_values(order_col).reset_index(drop=True)
    return df, order_col


# -------------------------
# 2) Sliding windows (RAW seq) => X: (N,T,F), y: (N,)
# -------------------------
def window_is_valid(window, min_non_nan_ratio=0.4):
    total = window.size
    non_nan = np.isfinite(window).sum()
    return (non_nan / total) >= min_non_nan_ratio

def make_raw_sequence_dataset(app_df, feature_cols, label_col="Temporal", seq_len=5, min_non_nan_ratio=0.4):
    X = app_df[feature_cols].values.astype(float)
    y = app_df[label_col].values.astype(int)

    if len(app_df) < seq_len:
        return np.empty((0, seq_len, len(feature_cols))), np.empty((0,), dtype=int)

    X_seq, y_seq = [], []
    center = seq_len // 2
    for i in range(len(app_df) - seq_len + 1):
        window = X[i:i+seq_len]
        if not window_is_valid(window, min_non_nan_ratio=min_non_nan_ratio):
            continue
        X_seq.append(window)
        y_seq.append(y[i + center])

    if len(X_seq) == 0:
        return np.empty((0, seq_len, len(feature_cols))), np.empty((0,), dtype=int)

    return np.stack(X_seq, axis=0), np.array(y_seq, dtype=int)


# -------------------------
# 3) Metrics + threshold tuning
# -------------------------
def eval_binary(y_true, y_pred):
    return {
        "acc": float(accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "cm": confusion_matrix(y_true, y_pred, labels=[0,1]).tolist(),
    }

def tune_threshold_on_val(y_val, p_val, grid=None):
    if grid is None:
        grid = np.linspace(0.05, 0.95, 19)
    best_thr, best_f1 = 0.5, -1.0
    for t in grid:
        yhat = (p_val >= t).astype(int)
        f1 = f1_score(y_val, yhat, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_thr = float(t)
    return best_thr, float(best_f1)


# -------------------------
# 4) Transformer model (RAW seq)
# -------------------------
def transformer_encoder(inputs, num_heads, key_dim, ff_dim, dropout=0.15):
    attn = MultiHeadAttention(num_heads=num_heads, key_dim=key_dim)(inputs, inputs)
    attn = Dropout(dropout)(attn)
    x = LayerNormalization(epsilon=1e-6)(inputs + attn)

    ffn = Dense(ff_dim, activation="relu")(x)
    ffn = Dense(x.shape[-1])(ffn)
    ffn = Dropout(dropout)(ffn)
    out = LayerNormalization(epsilon=1e-6)(x + ffn)
    return out

def build_rawseq_transformer(seq_len, feat_dim,
                            d_model=64, num_heads=2, key_dim=32,
                            ff_dim=128, depth=2, dropout=0.15,
                            lr=7e-5, weight_decay=1e-4,
                            label_smoothing=0.0):
    inp = Input(shape=(seq_len, feat_dim), name="raw_seq_in")
    x = Dense(d_model, activation="relu")(inp)

    for _ in range(depth):
        x = transformer_encoder(x, num_heads=num_heads, key_dim=key_dim, ff_dim=ff_dim, dropout=dropout)

    x = GlobalAveragePooling1D()(x)
    x = Dense(64, activation="relu")(x)
    x = Dropout(dropout)(x)
    out = Dense(1, activation="sigmoid")(x)

    model = Model(inp, out)
    opt = tf.keras.optimizers.AdamW(learning_rate=lr, weight_decay=weight_decay)
    loss = tf.keras.losses.BinaryCrossentropy(label_smoothing=label_smoothing)
    model.compile(optimizer=opt, loss=loss, metrics=["accuracy"])
    return model

def make_tf_dataset(X, y, batch_size, shuffle=True, drop_remainder=False):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if shuffle:
        ds = ds.shuffle(buffer_size=min(len(y), 5000), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(batch_size, drop_remainder=drop_remainder).prefetch(tf.data.AUTOTUNE)
    return ds


# -------------------------
# 4.5) KEEP-DIMS impute+scale (MINIMAL FIX for all-NaN columns)
# -------------------------
def fit_impute_scale_keepdims(X_train_flat):
    """
    X_train_flat: (N, F) float with NaNs
    Returns fill/mean/std (F,)
    - fill = per-col median; if all-NaN -> 0
    - mean/std computed after fill; std==0 -> 1
    """
    X = X_train_flat.astype(float, copy=True)
    fill = np.nanmedian(X, axis=0)
    fill = np.where(np.isfinite(fill), fill, 0.0)
    X = np.where(np.isfinite(X), X, fill)

    mean = X.mean(axis=0)
    std  = X.std(axis=0)
    std  = np.where(std > 0, std, 1.0)
    return fill, mean, std

def apply_impute_scale_keepdims(X_flat, fill, mean, std):
    X = X_flat.astype(float, copy=True)
    X = np.where(np.isfinite(X), X, fill)
    X = (X - mean) / std
    return X


# -------------------------
# 5) Per-app 5-fold runner
# -------------------------
def run_per_app_5fold_transformer(
    df_app_raw,
    app_name,
    seq_len=5,
    min_non_nan_ratio=0.4,
    n_splits=5,
    epochs=25,
    batch_size=64,
    val_size=0.15,
    label_smoothing=0.0,
    verbose_fit=0,
):
    df_app_raw = _clean_df(df_app_raw)
    df_app_raw, order_col = _order_df(df_app_raw)
    df_app, feature_cols = _get_feature_cols(df_app_raw)
    F = len(feature_cols)

    X_all, y_all = make_raw_sequence_dataset(df_app, feature_cols, "Temporal", seq_len, min_non_nan_ratio)
    print(f"\n================= {app_name.upper()} (Transformer) =================")
    print(f"[INFO] rows: {len(df_app)} | label dist: {df_app['Temporal'].value_counts().to_dict()}")
    print(f"[INFO] ORDER_COL: {order_col}")
    print(f"[INFO] sequences: {X_all.shape} | y dist: {dict(zip(*np.unique(y_all, return_counts=True)))}")
    print(f"[INFO] raw tokens = (T={seq_len}, F={F})")

    if len(y_all) < 50:
        raise RuntimeError(f"[{app_name}] Too few sequences built. Check ordering + missingness + min_non_nan_ratio.")

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    fold_rows = []

    for fold, (tr_idx, te_idx) in enumerate(skf.split(np.zeros(len(y_all)), y_all), 1):
        X_tr_full, y_tr_full = X_all[tr_idx], y_all[tr_idx]
        X_te, y_te = X_all[te_idx], y_all[te_idx]

        # inner train/val split (for early stop + threshold)
        X_tr, X_val, y_tr, y_val = train_test_split(
            X_tr_full, y_tr_full,
            test_size=val_size,
            random_state=SEED,
            stratify=y_tr_full
        )

        # ---- impute+scale TRAIN only (flatten time) ----
        Xtr_flat  = X_tr.reshape(-1, F)
        Xval_flat = X_val.reshape(-1, F)
        Xte_flat  = X_te.reshape(-1, F)

        # MINIMAL FIX: keep dims even if some cols are all-NaN (Puzzle case)
        fill, mean, std = fit_impute_scale_keepdims(Xtr_flat)

        Xtr_flat  = apply_impute_scale_keepdims(Xtr_flat,  fill, mean, std)
        Xval_flat = apply_impute_scale_keepdims(Xval_flat, fill, mean, std)
        Xte_flat  = apply_impute_scale_keepdims(Xte_flat,  fill, mean, std)

        X_tr = Xtr_flat.reshape(-1, seq_len, F).astype(np.float32)
        X_val = Xval_flat.reshape(-1, seq_len, F).astype(np.float32)
        X_te  = Xte_flat.reshape(-1, seq_len, F).astype(np.float32)

        # class weights on inner-train
        classes = np.array([0, 1])
        cw = compute_class_weight(class_weight="balanced", classes=classes, y=y_tr)
        class_weight = {0: float(cw[0]), 1: float(cw[1])}

        model = build_rawseq_transformer(
            seq_len=seq_len, feat_dim=F,
            d_model=64, num_heads=2, key_dim=32,
            ff_dim=128, depth=2, dropout=0.15,
            lr=7e-5, weight_decay=1e-4,
            label_smoothing=label_smoothing
        )

        early = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=0)
        rlrop = ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, min_lr=1e-6, verbose=0)

        train_ds = make_tf_dataset(X_tr, y_tr, batch_size=batch_size, shuffle=True,  drop_remainder=True)
        val_ds   = make_tf_dataset(X_val, y_val, batch_size=batch_size, shuffle=False, drop_remainder=False)

        model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=epochs,
            verbose=verbose_fit,
            class_weight=class_weight,
            callbacks=[early, rlrop]
        )

        p_val = model.predict(X_val, batch_size=batch_size, verbose=0).reshape(-1)
        p_te  = model.predict(X_te,  batch_size=batch_size, verbose=0).reshape(-1)

        thr, val_f1 = tune_threshold_on_val(y_val, p_val)
        yhat = (p_te >= thr).astype(int)
        m = eval_binary(y_te, yhat)

        print(f"[FOLD {fold}] thr={thr:.2f} val_f1={val_f1:.3f}  "
              f"acc={m['acc']:.3f} f1={m['f1']:.3f} prec={m['precision']:.3f} rec={m['recall']:.3f} | cm={m['cm']}")

        fold_rows.append({
            "app": app_name,
            "fold": fold,
            "thr": thr,
            "val_f1": val_f1,
            **m,
            "n_test": int(len(y_te)),
            "seq_len": int(seq_len),
            "F": int(F),
        })

        tf.keras.backend.clear_session()

    out = pd.DataFrame(fold_rows)
    summary = {
        "app": app_name,
        "cv_mean_acc": float(out["acc"].mean()),
        "cv_mean_f1": float(out["f1"].mean()),
        "cv_mean_precision": float(out["precision"].mean()),
        "cv_mean_recall": float(out["recall"].mean()),
        "cv_std_acc": float(out["acc"].std()),
        "cv_std_f1": float(out["f1"].std()),
    }
    return out, summary


# -------------------------
# 6) Run all apps (per app)
# -------------------------
def load_metrics_csv(path, app_name=None):
    df = pd.read_csv(path)
    if app_name is not None:
        df["App"] = app_name
    return df

# Use the ACTUAL uploaded paths in this chat:
df_archery = load_metrics_csv("/content/metrics_Archery.csv", "Archery")
df_phantom = load_metrics_csv("/content/metrics_PhantomLimb.csv", "PhantomLimb")
df_pt      = load_metrics_csv("/content/metrics_PianoTiles.csv", "PianoTiles")
df_puzzle  = load_metrics_csv("/content/metrics_Puzzle.csv", "Puzzle")
df_sea     = load_metrics_csv("/content/metrics_Sea.csv", "Sea")
df_war     = load_metrics_csv("/content/metrics_War.csv", "War")

apps = [
    ("Archery", df_archery),
    ("PhantomLimb", df_phantom),
    ("PianoTiles", df_pt),
    ("Puzzle", df_puzzle),
    ("Sea", df_sea),
    ("War", df_war),
]

summaries = []
all_folds = []

for app_name, df_app in apps:
    folds_df, summ = run_per_app_5fold_transformer(
        df_app, app_name,
        seq_len=5,
        min_non_nan_ratio=0.4,
        n_splits=5,
        epochs=25,
        batch_size=64,
        val_size=0.15,
        label_smoothing=0.0,
        verbose_fit=0
    )
    all_folds.append(folds_df)
    summaries.append(summ)

folds_all = pd.concat(all_folds, ignore_index=True)
summary_df = pd.DataFrame(summaries).sort_values("cv_mean_acc", ascending=False)

print("\n================= SUMMARY (PER-APP TRANSFORMER) =================")
print(summary_df[["app","cv_mean_acc","cv_mean_precision","cv_mean_recall","cv_mean_f1","cv_std_acc","cv_std_f1"]])

OUT_FOLDS = "/content/per_app_temporal_transformer_folds.csv"
OUT_SUMM  = "/content/per_app_temporal_transformer_summary.csv"
folds_all.to_csv(OUT_FOLDS, index=False)
summary_df.to_csv(OUT_SUMM, index=False)
print("\nSaved:")
print(" -", OUT_FOLDS)
print(" -", OUT_SUMM)


================= ARCHERY (Transformer) =================
[INFO] rows: 425 | label dist: {1: 304, 0: 121}
[INFO] ORDER_COL: EntryID
[INFO] sequences: (421, 5, 27) | y dist: {np.int64(0): np.int64(119), np.int64(1): np.int64(302)}
[INFO] raw tokens = (T=5, F=27)
[FOLD 1] thr=0.30 val_f1=0.974  acc=0.941 f1=0.960 prec=0.938 rec=0.984 | cm=[[20, 4], [1, 60]]
[FOLD 2] thr=0.25 val_f1=0.949  acc=0.952 f1=0.967 prec=0.952 rec=0.983 | cm=[[21, 3], [1, 59]]
[FOLD 3] thr=0.30 val_f1=0.974  acc=0.952 f1=0.967 prec=0.967 rec=0.967 | cm=[[22, 2], [2, 58]]
[FOLD 4] thr=0.40 val_f1=0.937  acc=0.917 f1=0.943 prec=0.921 rec=0.967 | cm=[[19, 5], [2, 58]]
[FOLD 5] thr=0.20 val_f1=0.921  acc=0.940 f1=0.961 prec=0.924 rec=1.000 | cm=[[18, 5], [0, 61]]

================= PHANTOMLIMB (Transformer) =================
[INFO] rows: 4105 | label dist: {0: 2663, 1: 1442}
[INFO] ORDER_COL: EntryID
[INFO] sequences: (4101, 5, 27) | y dist: {np.int64(0): np.int64(2660), np.int64(1): np.int64(1441)}
[INFO] raw token

/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:1233: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keepdims=keepdims,


[FOLD 1] thr=0.20 val_f1=0.981  acc=0.946 f1=0.966 prec=0.935 rec=1.000 | cm=[[10, 3], [0, 43]]


/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:1233: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keepdims=keepdims,


[FOLD 2] thr=0.20 val_f1=0.963  acc=0.927 f1=0.955 prec=0.913 rec=1.000 | cm=[[9, 4], [0, 42]]


...

[FOLD 3] thr=0.20 val_f1=0.981  acc=0.891 f1=0.935 prec=0.878 rec=1.000 | cm=[[6, 6], [0, 43]]


/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:1233: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keepdims=keepdims,


[FOLD 4] thr=0.50 val_f1=0.981  acc=0.945 f1=0.966 prec=0.955 rec=0.977 | cm=[[10, 2], [1, 42]]


/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:1233: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keepdims=keepdims,


[FOLD 5] thr=0.25 val_f1=0.981  acc=0.909 f1=0.944 prec=0.913 rec=0.977 | cm=[[8, 4], [1, 42]]

================= SEA (Transformer) =================
[INFO] rows: 300 | label dist: {0: 164, 1: 136}
[INFO] ORDER_COL: EntryID
[INFO] sequences: (296, 5, 27) | y dist: {np.int64(0): np.int64(162), np.int64(1): np.int64(134)}
[INFO] raw tokens = (T=5, F=27)
[FOLD 1] thr=0.45 val_f1=0.970  acc=0.967 f1=0.964 prec=0.931 rec=1.000 | cm=[[31, 2], [0, 27]]
[FOLD 2] thr=0.40 val_f1=0.970  acc=1.000 f1=1.000 prec=1.000 rec=1.000 | cm=[[33, 0], [0, 26]]
[FOLD 3] thr=0.55 val_f1=0.970  acc=0.932 f1=0.931 prec=0.871 rec=1.000 | cm=[[28, 4], [0, 27]]
[FOLD 4] thr=0.15 val_f1=0.970  acc=0.881 f1=0.885 prec=0.794 rec=1.000 | cm=[[25, 7], [0, 27]]
[FOLD 5] thr=0.55 val_f1=0.970  acc=0.983 f1=0.982 prec=0.964 rec=1.000 | cm=[[31, 1], [0, 27]]

================= WAR (Transformer) =================
[INFO] rows: 281 | label dist: {0: 165, 1: 116}
[INFO] ORDER_COL: EntryID
[INFO] sequences: (277, 5, 27) | y di

In [ ]:
# ===============================================================
# PHANTOMLIMB ONLY: Temporal Transformer on RAW SEQ (T=5, F=INVARIANT27)
# - Train on /content/metrics_PhantomLimb.csv
# - Tune threshold on an inner val split from TRAIN
# - Evaluate once on /content/metrics_PhantomLimb_Holdout.csv
# - Impute+scale FIT ON TRAIN ONLY (flatten time)  (keep-dims fix included)
# ===============================================================

import os
import json
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.layers import Input, Dense, Dropout, LayerNormalization, MultiHeadAttention, GlobalAveragePooling1D
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"

SEQ_LEN = 5
MIN_NON_NAN_RATIO = 0.4
LABEL_COL = "Temporal"

INVARIANT27 = [
    "missing_joints_count","missing_joints_ratio","collapsed_joints_count",
    "center_of_mass_x","center_of_mass_y","center_of_mass_z","distance_from_origin",
    "bbox_width","bbox_height","bbox_depth","bbox_volume","max_joint_distance_from_com",
    "distance_from_floor","min_foot_height_above_floor","below_floor",
    "left_forearm_length","right_forearm_length","left_shin_length","right_shin_length",
    "arm_length_symmetry","leg_length_symmetry",
    "body_upright_x","body_upright_y","body_upright_z",
    "body_forward_x","body_forward_y","body_forward_z"
]

TRAIN_CSV   = "/content/metrics_PhantomLimb.csv"
HOLDOUT_CSV = "/content/metrics_PhantomLimb_Holdout.csv"

OUT_DIR = "/content/phantomlimb_temporal_transformer_invariant27_holdout"
os.makedirs(OUT_DIR, exist_ok=True)

# -------------------------
# Helpers
# -------------------------
def _clean_df(df):
    df = df.copy().replace("", np.nan)
    df[LABEL_COL] = pd.to_numeric(df[LABEL_COL], errors="coerce")
    df = df.dropna(subset=[LABEL_COL]).copy()
    df[LABEL_COL] = df[LABEL_COL].astype(int)
    return df

def _order_df(df):
    order_col = None
    for c in ["EntryID", "entryid", "Timestamp", "timestamp", "Frame", "frame"]:
        if c in df.columns:
            order_col = c
            break
    if order_col is None:
        return df.reset_index(drop=True), None

    tmp = pd.to_numeric(df[order_col], errors="coerce")
    if tmp.notna().any():
        df = df.assign(_order_tmp=tmp).sort_values("_order_tmp").drop(columns=["_order_tmp"]).reset_index(drop=True)
    else:
        df = df.sort_values(order_col).reset_index(drop=True)
    return df, order_col

def window_is_valid(window, min_non_nan_ratio=0.4):
    total = window.size
    non_nan = np.isfinite(window).sum()
    return (non_nan / total) >= min_non_nan_ratio

def make_raw_sequence_dataset(app_df, feature_cols, label_col=LABEL_COL, seq_len=5, min_non_nan_ratio=0.4):
    X = app_df[feature_cols].values.astype(float)
    y = app_df[label_col].values.astype(int)

    if len(app_df) < seq_len:
        return np.empty((0, seq_len, len(feature_cols))), np.empty((0,), dtype=int)

    X_seq, y_seq = [], []
    center = seq_len // 2
    for i in range(len(app_df) - seq_len + 1):
        window = X[i:i+seq_len]
        if not window_is_valid(window, min_non_nan_ratio=min_non_nan_ratio):
            continue
        X_seq.append(window)
        y_seq.append(y[i + center])

    if len(X_seq) == 0:
        return np.empty((0, seq_len, len(feature_cols))), np.empty((0,), dtype=int)

    return np.stack(X_seq, axis=0), np.array(y_seq, dtype=int)

def eval_binary(y_true, y_pred):
    return {
        "acc": float(accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "cm": confusion_matrix(y_true, y_pred, labels=[0,1]).tolist(),
    }

def tune_threshold_on_val(y_val, p_val, grid=None):
    if grid is None:
        grid = np.linspace(0.05, 0.95, 19)
    best_thr, best_f1 = 0.5, -1.0
    for t in grid:
        yhat = (p_val >= t).astype(int)
        f1 = f1_score(y_val, yhat, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_thr = float(t)
    return best_thr, float(best_f1)

def transformer_encoder(inputs, num_heads, key_dim, ff_dim, dropout=0.15):
    attn = MultiHeadAttention(num_heads=num_heads, key_dim=key_dim)(inputs, inputs)
    attn = Dropout(dropout)(attn)
    x = LayerNormalization(epsilon=1e-6)(inputs + attn)

    ffn = Dense(ff_dim, activation="relu")(x)
    ffn = Dense(x.shape[-1])(ffn)
    ffn = Dropout(dropout)(ffn)
    out = LayerNormalization(epsilon=1e-6)(x + ffn)
    return out

def build_rawseq_transformer(seq_len, feat_dim,
                            d_model=64, num_heads=2, key_dim=32,
                            ff_dim=128, depth=2, dropout=0.15,
                            lr=7e-5, weight_decay=1e-4,
                            label_smoothing=0.0):
    inp = Input(shape=(seq_len, feat_dim), name="raw_seq_in")
    x = Dense(d_model, activation="relu")(inp)

    for _ in range(depth):
        x = transformer_encoder(x, num_heads=num_heads, key_dim=key_dim, ff_dim=ff_dim, dropout=dropout)

    x = GlobalAveragePooling1D()(x)
    x = Dense(64, activation="relu")(x)
    x = Dropout(dropout)(x)
    out = Dense(1, activation="sigmoid")(x)

    model = Model(inp, out)
    opt = tf.keras.optimizers.AdamW(learning_rate=lr, weight_decay=weight_decay)
    loss = tf.keras.losses.BinaryCrossentropy(label_smoothing=label_smoothing)
    model.compile(optimizer=opt, loss=loss, metrics=["accuracy"])
    return model

# ---- KEEP-DIMS impute+scale (all-NaN cols safe) ----
def fit_impute_scale_keepdims(X_train_flat):
    X = X_train_flat.astype(float, copy=True)
    fill = np.nanmedian(X, axis=0)
    fill = np.where(np.isfinite(fill), fill, 0.0)
    X = np.where(np.isfinite(X), X, fill)

    mean = X.mean(axis=0)
    std  = X.std(axis=0)
    std  = np.where(std > 0, std, 1.0)
    return fill, mean, std

def apply_impute_scale_keepdims(X_flat, fill, mean, std):
    X = X_flat.astype(float, copy=True)
    X = np.where(np.isfinite(X), X, fill)
    X = (X - mean) / std
    return X

# -------------------------
# Load + enforce invariant27 only
# -------------------------
df_train = pd.read_csv(TRAIN_CSV)
df_hold  = pd.read_csv(HOLDOUT_CSV)

df_train = _clean_df(df_train)
df_hold  = _clean_df(df_hold)

df_train, train_order_col = _order_df(df_train)
df_hold,  hold_order_col  = _order_df(df_hold)

missing_train = [c for c in INVARIANT27 if c not in df_train.columns]
missing_hold  = [c for c in INVARIANT27 if c not in df_hold.columns]
if missing_train:
    raise RuntimeError(f"TRAIN missing invariant cols: {missing_train}")
if missing_hold:
    raise RuntimeError(f"HOLDOUT missing invariant cols: {missing_hold}")

# numeric coercion
for c in INVARIANT27:
    df_train[c] = pd.to_numeric(df_train[c], errors="coerce")
    df_hold[c]  = pd.to_numeric(df_hold[c],  errors="coerce")

X_all, y_all = make_raw_sequence_dataset(df_train, INVARIANT27, LABEL_COL, SEQ_LEN, MIN_NON_NAN_RATIO)
X_h,   y_h   = make_raw_sequence_dataset(df_hold,  INVARIANT27, LABEL_COL, SEQ_LEN, MIN_NON_NAN_RATIO)

print("\n================= PHANTOMLIMB (Transformer, INVARIANT27) =================")
print(f"[TRAIN] rows={len(df_train)} order_col={train_order_col} label_dist={df_train[LABEL_COL].value_counts().to_dict()}")
print(f"[TRAIN] sequences={X_all.shape} y_dist={dict(zip(*np.unique(y_all, return_counts=True)))}")
print(f"[HOLD ] rows={len(df_hold)}  order_col={hold_order_col}  label_dist={df_hold[LABEL_COL].value_counts().to_dict()}")
print(f"[HOLD ] sequences={X_h.shape} y_dist={dict(zip(*np.unique(y_h, return_counts=True)))}")
print(f"[INFO] tokens=(T={SEQ_LEN}, F={len(INVARIANT27)})")

if len(y_all) < 50 or len(y_h) < 10:
    raise RuntimeError("Too few sequences. Check ordering + missingness + MIN_NON_NAN_RATIO.")

# -------------------------
# Inner train/val split (TRAIN only) for EarlyStopping + threshold
# -------------------------
X_tr, X_val, y_tr, y_val = train_test_split(
    X_all, y_all,
    test_size=0.15,
    random_state=SEED,
    stratify=y_all
)

T = SEQ_LEN
F = len(INVARIANT27)

# ---- impute+scale FIT ON TRAIN ONLY (flatten time) ----
Xtr_flat  = X_tr.reshape(-1, F)
Xval_flat = X_val.reshape(-1, F)
Xh_flat   = X_h.reshape(-1, F)

fill, mean, std = fit_impute_scale_keepdims(Xtr_flat)

Xtr_flat  = apply_impute_scale_keepdims(Xtr_flat,  fill, mean, std)
Xval_flat = apply_impute_scale_keepdims(Xval_flat, fill, mean, std)
Xh_flat   = apply_impute_scale_keepdims(Xh_flat,   fill, mean, std)

X_tr = Xtr_flat.reshape(-1, T, F).astype(np.float32)
X_val = Xval_flat.reshape(-1, T, F).astype(np.float32)
X_h  = Xh_flat.reshape(-1,  T, F).astype(np.float32)

# class weights on inner-train
classes = np.array([0, 1])
cw = compute_class_weight(class_weight="balanced", classes=classes, y=y_tr)
class_weight = {0: float(cw[0]), 1: float(cw[1])}

model = build_rawseq_transformer(
    seq_len=T, feat_dim=F,
    d_model=64, num_heads=2, key_dim=32,
    ff_dim=128, depth=2, dropout=0.15,
    lr=7e-5, weight_decay=1e-4,
    label_smoothing=0.0
)

early = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=0)
rlrop = ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, min_lr=1e-6, verbose=0)

# small tf datasets
def make_tf_dataset(X, y, batch_size, shuffle=True, drop_remainder=False):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if shuffle:
        ds = ds.shuffle(buffer_size=min(len(y), 5000), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(batch_size, drop_remainder=drop_remainder).prefetch(tf.data.AUTOTUNE)
    return ds

batch_size = 64
train_ds = make_tf_dataset(X_tr, y_tr, batch_size=batch_size, shuffle=True,  drop_remainder=True)
val_ds   = make_tf_dataset(X_val, y_val, batch_size=batch_size, shuffle=False, drop_remainder=False)

model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=25,
    verbose=0,
    class_weight=class_weight,
    callbacks=[early, rlrop]
)

# threshold tune on val, eval on holdout
p_val = model.predict(X_val, batch_size=batch_size, verbose=0).reshape(-1)
p_h   = model.predict(X_h,   batch_size=batch_size, verbose=0).reshape(-1)

thr, val_f1 = tune_threshold_on_val(y_val, p_val)
yhat_h = (p_h >= thr).astype(int)
m = eval_binary(y_h, yhat_h)

print(f"\n[VAL] tuned_thr={thr:.2f} val_f1={val_f1:.3f}")
print(f"[HOLDOUT] acc={m['acc']:.3f} f1={m['f1']:.3f} prec={m['precision']:.3f} rec={m['recall']:.3f} cm={m['cm']}")

# save
model_path = os.path.join(OUT_DIR, "model.keras")
model.save(model_path)

meta = {
    "app": "PhantomLimb",
    "task": "Temporal_sequence_raw_transformer",
    "trained_on": TRAIN_CSV,
    "evaluated_on_holdout": HOLDOUT_CSV,
    "seed": SEED,
    "label_col": LABEL_COL,
    "seq_len": SEQ_LEN,
    "min_non_nan_ratio": float(MIN_NON_NAN_RATIO),
    "features": INVARIANT27,
    "train_order_col": train_order_col,
    "holdout_order_col": hold_order_col,
    "impute_scale": {
        "fill": fill.tolist(),
        "mean": mean.tolist(),
        "std": std.tolist(),
    },
    "threshold": {"tuned_on_val": float(thr), "val_f1": float(val_f1)},
    "holdout_metrics": m,
    "notes": "Transformer trained on PhantomLimb TRAIN only; threshold tuned on TRAIN val split; evaluated once on PhantomLimb holdout; invariant27 only."
}
with open(os.path.join(OUT_DIR, "meta.json"), "w") as f:
    json.dump(meta, f, indent=2)

print("\nSaved:")
print(" -", model_path)
print(" -", os.path.join(OUT_DIR, "meta.json"))


================= PHANTOMLIMB (Transformer, INVARIANT27) =================
[TRAIN] rows=4105 order_col=EntryID label_dist={0: 2663, 1: 1442}
[TRAIN] sequences=(4101, 5, 27) y_dist={np.int64(0): np.int64(2660), np.int64(1): np.int64(1441)}
[HOLD ] rows=391  order_col=EntryID  label_dist={0: 290, 1: 101}
[HOLD ] sequences=(387, 5, 27) y_dist={np.int64(0): np.int64(286), np.int64(1): np.int64(101)}
[INFO] tokens=(T=5, F=27)

[VAL] tuned_thr=0.60 val_f1=0.907
[HOLDOUT] acc=0.933 f1=0.877 prec=0.838 rec=0.921 cm=[[268, 18], [8, 93]]

Saved:
 - /content/phantomlimb_temporal_transformer_invariant27_holdout/model.keras
 - /content/phantomlimb_temporal_transformer_invariant27_holdout/meta.json


In [ ]:
# ===============================================================
# PER-APP: Temporal Transformer on RAW SEQ (T=5, F=all metric cols)
# ✅ UPDATED to prioritize ACCURACY (minimal changes)
# - 5-fold Stratified CV on sequences PER APP
# - inner train/val split for EarlyStopping + threshold tuning
# - impute+scale FIT ON TRAIN ONLY (flatten time)
# - KEEP-DIMS impute/scale (all-NaN cols -> 0) so feature count is stable
#
# Changes vs your block:
#   1) Threshold tuned to MAXIMIZE VAL ACC (not val F1)
#   2) EarlyStopping + ReduceLROnPlateau monitor VAL ACC (not val loss)
#   3) class_weight optional toggle (default OFF for accuracy priority)
# ===============================================================

import os
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.layers import Input, Dense, Dropout, LayerNormalization, MultiHeadAttention, GlobalAveragePooling1D
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"


# -------------------------
# 1) Data prep helpers
# -------------------------
def _clean_df(df):
    df = df.copy().replace("", np.nan)
    df["Temporal"] = pd.to_numeric(df["Temporal"], errors="coerce")
    df = df.dropna(subset=["Temporal"]).copy()
    df["Temporal"] = df["Temporal"].astype(int)
    return df

def _get_feature_cols(df):
    id_cols = [c for c in ["GlobalID", "EntryID", "Sequence_ID", "Person_ID", "FPS"] if c in df.columns]
    drop_cols = ["App", "Spatial", "Temporal"] + id_cols
    feature_cols = [c for c in df.columns if c not in drop_cols]

    out = df.copy()
    for c in feature_cols:
        out[c] = pd.to_numeric(out[c], errors="coerce")
    return out, feature_cols

def _order_df(df):
    order_col = None
    for c in ["EntryID", "entryid", "Timestamp", "timestamp", "Frame", "frame"]:
        if c in df.columns:
            order_col = c
            break
    if order_col is None:
        return df.reset_index(drop=True), None

    tmp = pd.to_numeric(df[order_col], errors="coerce")
    if tmp.notna().any():
        df = df.assign(_order_tmp=tmp).sort_values("_order_tmp").drop(columns=["_order_tmp"]).reset_index(drop=True)
    else:
        df = df.sort_values(order_col).reset_index(drop=True)
    return df, order_col


# -------------------------
# 2) Sliding windows (RAW seq) => X: (N,T,F), y: (N,)
# -------------------------
def window_is_valid(window, min_non_nan_ratio=0.4):
    total = window.size
    non_nan = np.isfinite(window).sum()
    return (non_nan / total) >= min_non_nan_ratio

def make_raw_sequence_dataset(app_df, feature_cols, label_col="Temporal", seq_len=5, min_non_nan_ratio=0.4):
    X = app_df[feature_cols].values.astype(float)
    y = app_df[label_col].values.astype(int)

    if len(app_df) < seq_len:
        return np.empty((0, seq_len, len(feature_cols))), np.empty((0,), dtype=int)

    X_seq, y_seq = [], []
    center = seq_len // 2
    for i in range(len(app_df) - seq_len + 1):
        window = X[i:i+seq_len]
        if not window_is_valid(window, min_non_nan_ratio=min_non_nan_ratio):
            continue
        X_seq.append(window)
        y_seq.append(y[i + center])

    if len(X_seq) == 0:
        return np.empty((0, seq_len, len(feature_cols))), np.empty((0,), dtype=int)

    return np.stack(X_seq, axis=0), np.array(y_seq, dtype=int)


# -------------------------
# 3) Metrics + threshold tuning
# -------------------------
def eval_binary(y_true, y_pred):
    return {
        "acc": float(accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "cm": confusion_matrix(y_true, y_pred, labels=[0,1]).tolist(),
    }

# ✅ ACCURACY-PRIORITY threshold tuning
def tune_threshold_on_val_acc(y_val, p_val, grid=None):
    if grid is None:
        grid = np.linspace(0.05, 0.95, 19)
    best_thr, best_acc = 0.5, -1.0
    for t in grid:
        yhat = (p_val >= t).astype(int)
        acc = accuracy_score(y_val, yhat)
        if acc > best_acc:
            best_acc = acc
            best_thr = float(t)
    return best_thr, float(best_acc)


# -------------------------
# 4) Transformer model (RAW seq)
# -------------------------
def transformer_encoder(inputs, num_heads, key_dim, ff_dim, dropout=0.15):
    attn = MultiHeadAttention(num_heads=num_heads, key_dim=key_dim)(inputs, inputs)
    attn = Dropout(dropout)(attn)
    x = LayerNormalization(epsilon=1e-6)(inputs + attn)

    ffn = Dense(ff_dim, activation="relu")(x)
    ffn = Dense(x.shape[-1])(ffn)
    ffn = Dropout(dropout)(ffn)
    out = LayerNormalization(epsilon=1e-6)(x + ffn)
    return out

def build_rawseq_transformer(seq_len, feat_dim,
                            d_model=64, num_heads=2, key_dim=32,
                            ff_dim=128, depth=2, dropout=0.15,
                            lr=7e-5, weight_decay=1e-4,
                            label_smoothing=0.0):
    inp = Input(shape=(seq_len, feat_dim), name="raw_seq_in")
    x = Dense(d_model, activation="relu")(inp)

    for _ in range(depth):
        x = transformer_encoder(x, num_heads=num_heads, key_dim=key_dim, ff_dim=ff_dim, dropout=dropout)

    x = GlobalAveragePooling1D()(x)
    x = Dense(64, activation="relu")(x)
    x = Dropout(dropout)(x)
    out = Dense(1, activation="sigmoid")(x)

    model = Model(inp, out)
    opt = tf.keras.optimizers.AdamW(learning_rate=lr, weight_decay=weight_decay)
    loss = tf.keras.losses.BinaryCrossentropy(label_smoothing=label_smoothing)
    model.compile(optimizer=opt, loss=loss, metrics=["accuracy"])
    return model

def make_tf_dataset(X, y, batch_size, shuffle=True, drop_remainder=False):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if shuffle:
        ds = ds.shuffle(buffer_size=min(len(y), 5000), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(batch_size, drop_remainder=drop_remainder).prefetch(tf.data.AUTOTUNE)
    return ds


# -------------------------
# 4.5) KEEP-DIMS impute+scale (all-NaN cols -> 0.0)
# -------------------------
def fit_impute_scale_keepdims(X_train_flat):
    X = X_train_flat.astype(float, copy=True)
    fill = np.nanmedian(X, axis=0)
    fill = np.where(np.isfinite(fill), fill, 0.0)
    X = np.where(np.isfinite(X), X, fill)

    mean = X.mean(axis=0)
    std  = X.std(axis=0)
    std  = np.where(std > 0, std, 1.0)
    return fill, mean, std

def apply_impute_scale_keepdims(X_flat, fill, mean, std):
    X = X_flat.astype(float, copy=True)
    X = np.where(np.isfinite(X), X, fill)
    X = (X - mean) / std
    return X


# -------------------------
# 5) Per-app 5-fold runner (ACCURACY priority)
# -------------------------
def run_per_app_5fold_transformer(
    df_app_raw,
    app_name,
    seq_len=5,
    min_non_nan_ratio=0.4,
    n_splits=5,
    epochs=25,
    batch_size=64,
    val_size=0.15,
    label_smoothing=0.0,
    verbose_fit=0,
    # ✅ accuracy-first defaults:
    USE_CLASS_WEIGHT=False,          # balanced weights often help F1/recall but can hurt pure accuracy
    THR_GRID=None,                   # can pass np.linspace(0.1,0.9,17) etc.
):
    df_app_raw = _clean_df(df_app_raw)
    df_app_raw, order_col = _order_df(df_app_raw)
    df_app, feature_cols = _get_feature_cols(df_app_raw)
    F = len(feature_cols)

    X_all, y_all = make_raw_sequence_dataset(df_app, feature_cols, "Temporal", seq_len, min_non_nan_ratio)
    print(f"\n================= {app_name.upper()} (Transformer) =================")
    print(f"[INFO] rows: {len(df_app)} | label dist: {df_app['Temporal'].value_counts().to_dict()}")
    print(f"[INFO] ORDER_COL: {order_col}")
    print(f"[INFO] sequences: {X_all.shape} | y dist: {dict(zip(*np.unique(y_all, return_counts=True)))}")
    print(f"[INFO] raw tokens = (T={seq_len}, F={F})")

    if len(y_all) < 50:
        raise RuntimeError(f"[{app_name}] Too few sequences built. Check ordering + missingness + min_non_nan_ratio.")

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    fold_rows = []

    for fold, (tr_idx, te_idx) in enumerate(skf.split(np.zeros(len(y_all)), y_all), 1):
        X_tr_full, y_tr_full = X_all[tr_idx], y_all[tr_idx]
        X_te, y_te = X_all[te_idx], y_all[te_idx]

        # inner train/val split (for early stop + threshold)
        X_tr, X_val, y_tr, y_val = train_test_split(
            X_tr_full, y_tr_full,
            test_size=val_size,
            random_state=SEED,
            stratify=y_tr_full
        )

        # ---- impute+scale TRAIN only (flatten time) ----
        Xtr_flat  = X_tr.reshape(-1, F)
        Xval_flat = X_val.reshape(-1, F)
        Xte_flat  = X_te.reshape(-1, F)

        fill, mean, std = fit_impute_scale_keepdims(Xtr_flat)

        Xtr_flat  = apply_impute_scale_keepdims(Xtr_flat,  fill, mean, std)
        Xval_flat = apply_impute_scale_keepdims(Xval_flat, fill, mean, std)
        Xte_flat  = apply_impute_scale_keepdims(Xte_flat,  fill, mean, std)

        X_tr = Xtr_flat.reshape(-1, seq_len, F).astype(np.float32)
        X_val = Xval_flat.reshape(-1, seq_len, F).astype(np.float32)
        X_te  = Xte_flat.reshape(-1, seq_len, F).astype(np.float32)

        # ✅ optional class weights (OFF by default for accuracy)
        class_weight = None
        if USE_CLASS_WEIGHT and len(np.unique(y_tr)) >= 2:
            classes = np.array([0, 1])
            cw = compute_class_weight(class_weight="balanced", classes=classes, y=y_tr)
            class_weight = {0: float(cw[0]), 1: float(cw[1])}

        model = build_rawseq_transformer(
            seq_len=seq_len, feat_dim=F,
            d_model=64, num_heads=2, key_dim=32,
            ff_dim=128, depth=2, dropout=0.15,
            lr=7e-5, weight_decay=1e-4,
            label_smoothing=label_smoothing
        )

        # ✅ accuracy-first training controls
        early = EarlyStopping(
            monitor="val_accuracy", mode="max",
            patience=5, restore_best_weights=True, verbose=0
        )
        rlrop = ReduceLROnPlateau(
            monitor="val_accuracy", mode="max",
            factor=0.3, patience=2, min_lr=1e-6, verbose=0
        )

        train_ds = make_tf_dataset(X_tr, y_tr, batch_size=batch_size, shuffle=True,  drop_remainder=True)
        val_ds   = make_tf_dataset(X_val, y_val, batch_size=batch_size, shuffle=False, drop_remainder=False)

        model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=epochs,
            verbose=verbose_fit,
            class_weight=class_weight,
            callbacks=[early, rlrop]
        )

        p_val = model.predict(X_val, batch_size=batch_size, verbose=0).reshape(-1)
        p_te  = model.predict(X_te,  batch_size=batch_size, verbose=0).reshape(-1)

        # ✅ threshold tuned for VAL ACC (still report F1 etc on test)
        thr, val_acc = tune_threshold_on_val_acc(y_val, p_val, grid=THR_GRID)
        yhat = (p_te >= thr).astype(int)
        m = eval_binary(y_te, yhat)

        print(f"[FOLD {fold}] thr={thr:.2f} val_acc={val_acc:.3f}  "
              f"acc={m['acc']:.3f} f1={m['f1']:.3f} prec={m['precision']:.3f} rec={m['recall']:.3f} | cm={m['cm']}")

        fold_rows.append({
            "app": app_name,
            "fold": fold,
            "thr": thr,
            "val_acc": val_acc,
            **m,
            "n_test": int(len(y_te)),
            "seq_len": int(seq_len),
            "F": int(F),
            "use_class_weight": bool(USE_CLASS_WEIGHT),
        })

        tf.keras.backend.clear_session()

    out = pd.DataFrame(fold_rows)
    summary = {
        "app": app_name,
        "cv_mean_acc": float(out["acc"].mean()),
        "cv_mean_f1": float(out["f1"].mean()),
        "cv_mean_precision": float(out["precision"].mean()),
        "cv_mean_recall": float(out["recall"].mean()),
        "cv_std_acc": float(out["acc"].std()),
        "cv_std_f1": float(out["f1"].std()),
        "use_class_weight": bool(USE_CLASS_WEIGHT),
    }
    return out, summary


# -------------------------
# 6) Run all apps (per app)
# -------------------------
def load_metrics_csv(path, app_name=None):
    df = pd.read_csv(path)
    if app_name is not None:
        df["App"] = app_name
    return df

# Use the ACTUAL uploaded paths in this chat:
df_archery = load_metrics_csv("/content/metrics_Archery.csv", "Archery")
df_phantom = load_metrics_csv("/content/metrics_PhantomLimb.csv", "PhantomLimb")
df_pt      = load_metrics_csv("/content/metrics_PianoTiles.csv", "PianoTiles")
df_puzzle  = load_metrics_csv("/content/metrics_Puzzle.csv", "Puzzle")
df_sea     = load_metrics_csv("/content/metrics_Sea.csv", "Sea")
df_war     = load_metrics_csv("/content/metrics_War.csv", "War")

apps = [
    ("Archery", df_archery),
    ("PhantomLimb", df_phantom),
    ("PianoTiles", df_pt),
    ("Puzzle", df_puzzle),
    ("Sea", df_sea),
    ("War", df_war),
]

summaries = []
all_folds = []

for app_name, df_app in apps:
    folds_df, summ = run_per_app_5fold_transformer(
        df_app, app_name,
        seq_len=5,
        min_non_nan_ratio=0.4,
        n_splits=5,
        epochs=25,
        batch_size=64,
        val_size=0.15,
        label_smoothing=0.0,
        verbose_fit=0,
        USE_CLASS_WEIGHT=False,   # ✅ accuracy priority default
        THR_GRID=None             # ✅ default grid
    )
    all_folds.append(folds_df)
    summaries.append(summ)

folds_all = pd.concat(all_folds, ignore_index=True)
summary_df = pd.DataFrame(summaries).sort_values("cv_mean_acc", ascending=False)

print("\n================= SUMMARY (PER-APP TRANSFORMER, ACC PRIORITY) =================")
print(summary_df[["app","cv_mean_acc","cv_mean_precision","cv_mean_recall","cv_mean_f1","cv_std_acc","cv_std_f1","use_class_weight"]])

OUT_FOLDS = "/content/per_app_temporal_transformer_folds_acc_priority.csv"
OUT_SUMM  = "/content/per_app_temporal_transformer_summary_acc_priority.csv"
folds_all.to_csv(OUT_FOLDS, index=False)
summary_df.to_csv(OUT_SUMM, index=False)
print("\nSaved:")
print(" -", OUT_FOLDS)
print(" -", OUT_SUMM)


================= ARCHERY (Transformer) =================
[INFO] rows: 425 | label dist: {1: 304, 0: 121}
[INFO] ORDER_COL: EntryID
[INFO] sequences: (421, 5, 27) | y dist: {np.int64(0): np.int64(119), np.int64(1): np.int64(302)}
[INFO] raw tokens = (T=5, F=27)
[FOLD 1] thr=0.35 val_acc=0.824  acc=0.776 f1=0.865 prec=0.762 rec=1.000 | cm=[[5, 19], [0, 61]]
[FOLD 2] thr=0.60 val_acc=0.863  acc=0.821 f1=0.860 prec=0.979 rec=0.767 | cm=[[23, 1], [14, 46]]
[FOLD 3] thr=0.40 val_acc=0.882  acc=0.881 f1=0.923 prec=0.857 rec=1.000 | cm=[[14, 10], [0, 60]]
[FOLD 4] thr=0.35 val_acc=0.784  acc=0.774 f1=0.857 prec=0.781 rec=0.950 | cm=[[8, 16], [3, 57]]
[FOLD 5] thr=0.45 val_acc=0.843  acc=0.893 f1=0.930 prec=0.882 rec=0.984 | cm=[[15, 8], [1, 60]]

================= PHANTOMLIMB (Transformer) =================
[INFO] rows: 4105 | label dist: {0: 2663, 1: 1442}
[INFO] ORDER_COL: EntryID
[INFO] sequences: (4101, 5, 27) | y dist: {np.int64(0): np.int64(2660), np.int64(1): np.int64(1441)}
[INFO] ra

/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:1233: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keepdims=keepdims,


[FOLD 1] thr=0.50 val_acc=0.879  acc=0.911 f1=0.945 prec=0.896 rec=1.000 | cm=[[8, 5], [0, 43]]


/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:1233: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keepdims=keepdims,


[FOLD 2] thr=0.40 val_acc=0.853  acc=0.873 f1=0.923 prec=0.857 rec=1.000 | cm=[[6, 7], [0, 42]]


...

[FOLD 3] thr=0.50 val_acc=0.912  acc=0.818 f1=0.872 prec=0.971 rec=0.791 | cm=[[11, 1], [9, 34]]


/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:1233: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keepdims=keepdims,


[FOLD 4] thr=0.50 val_acc=1.000  acc=0.891 f1=0.933 prec=0.894 rec=0.977 | cm=[[7, 5], [1, 42]]


/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:1233: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keepdims=keepdims,


[FOLD 5] thr=0.60 val_acc=0.853  acc=0.709 f1=0.800 prec=0.865 rec=0.744 | cm=[[7, 5], [11, 32]]

================= SEA (Transformer) =================
[INFO] rows: 300 | label dist: {0: 164, 1: 136}
[INFO] ORDER_COL: EntryID
[INFO] sequences: (296, 5, 27) | y dist: {np.int64(0): np.int64(162), np.int64(1): np.int64(134)}
[INFO] raw tokens = (T=5, F=27)
[FOLD 1] thr=0.40 val_acc=0.972  acc=0.933 f1=0.926 prec=0.926 rec=0.926 | cm=[[31, 2], [2, 25]]
[FOLD 2] thr=0.50 val_acc=0.944  acc=1.000 f1=1.000 prec=1.000 rec=1.000 | cm=[[33, 0], [0, 26]]
[FOLD 3] thr=0.50 val_acc=0.944  acc=0.915 f1=0.912 prec=0.867 rec=0.963 | cm=[[28, 4], [1, 26]]
[FOLD 4] thr=0.35 val_acc=0.972  acc=0.898 f1=0.900 prec=0.818 rec=1.000 | cm=[[26, 6], [0, 27]]
[FOLD 5] thr=0.50 val_acc=0.972  acc=0.966 f1=0.963 prec=0.963 rec=0.963 | cm=[[31, 1], [1, 26]]

================= WAR (Transformer) =================
[INFO] rows: 281 | label dist: {0: 165, 1: 116}
[INFO] ORDER_COL: EntryID
[INFO] sequences: (277, 5, 27)

## Image + Metrics(LR)

In [ ]:
# ============================================================
# ✅ BEST / MOST-CORRECT COMPARISON ROUTE (ONE RUNNABLE BLOCK)
#   - Same Temporal setup as LR (SEQ_LEN=5, center label)
#   - Same invariant27 + Patch-B aggregation => 162-dim tabular vector
#   - Same NaN-window filtering (MIN_NON_NAN_RATIO)
#   - FIX: keep feature dim constant across apps:
#       * if a Patch-B feature column is all-NaN -> set to 0.0 (never drop dims)
#   - 5-fold Stratified CV PER APP (same protocol as your LR runs)
#   - Combined model = (5-frame image seq) + (tabular 162 token) + transformer
#
# REQUIRES ALREADY IN MEMORY:
#   df_all      (6242, ...) with columns: SheetName, EntryID
#   images_112  (6242, 112, 112, 1)
# ============================================================

import os, numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

import tensorflow as tf
from tensorflow.keras.layers import (Input, Dense, Flatten, Conv2D, TimeDistributed, MaxPooling2D,
                                     LayerNormalization, Dropout, MultiHeadAttention, GlobalAveragePooling1D,
                                     Concatenate, Lambda)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# -----------------------
# 0) Config
# -----------------------
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

SEQ_LEN = 5
MID = SEQ_LEN // 2
MIN_NON_NAN_RATIO = 0.4
LABEL_COL = "Temporal"

INVARIANT27 = [
    "missing_joints_count","missing_joints_ratio","collapsed_joints_count",
    "center_of_mass_x","center_of_mass_y","center_of_mass_z","distance_from_origin",
    "bbox_width","bbox_height","bbox_depth","bbox_volume","max_joint_distance_from_com",
    "distance_from_floor","min_foot_height_above_floor","below_floor",
    "left_forearm_length","right_forearm_length","left_shin_length","right_shin_length",
    "arm_length_symmetry","leg_length_symmetry",
    "body_upright_x","body_upright_y","body_upright_z",
    "body_forward_x","body_forward_y","body_forward_z"
]
AGG_USE = ("mean", "std", "delta", "range", "mean_abs_vel", "vel_std")
TAB_DIM = len(INVARIANT27) * len(AGG_USE)  # 27 * 6 = 162

apps = {
    "Archery":      "/content/metrics_Archery.csv",
    "PhantomLimb":  "/content/metrics_PhantomLimb.csv",
    "PianoTiles":   "/content/metrics_PianoTiles.csv",
    "Puzzle":       "/content/metrics_Puzzle.csv",
    "Sea":          "/content/metrics_Sea.csv",
    "War":          "/content/metrics_War.csv",
}

# -----------------------
# 1) Patch-B helpers (NaN-safe)
# -----------------------
def nanmean_no_warn(x, axis=0):
    x = np.asarray(x, float)
    valid = np.isfinite(x)
    denom = valid.sum(axis=axis)
    num = np.where(valid, x, 0.0).sum(axis=axis)
    return np.divide(num, denom, out=np.full_like(num, np.nan), where=(denom > 0))

def nanstd_no_warn(x, axis=0):
    mu = nanmean_no_warn(x, axis=axis)
    diff2 = (x - np.expand_dims(mu, axis=axis)) ** 2
    valid = np.isfinite(diff2)
    denom = valid.sum(axis=axis)
    num = np.where(valid, diff2, 0.0).sum(axis=axis)
    return np.sqrt(np.divide(num, denom, out=np.full_like(num, np.nan), where=(denom > 0)))

def nanrange_no_warn(x, axis=0):
    valid = np.isfinite(x)
    mx = np.max(np.where(valid, x, -np.inf), axis=axis)
    mn = np.min(np.where(valid, x, +np.inf), axis=axis)
    return np.where(valid.any(axis=axis), mx - mn, np.nan)

def window_is_valid(window, min_ratio=MIN_NON_NAN_RATIO):
    # window shape: (SEQ_LEN, F)
    return (np.isfinite(window).sum() / window.size) >= min_ratio

def aggregate_window_patchB(window):
    # window shape: (SEQ_LEN, 27)
    feats = [
        nanmean_no_warn(window, 0),
        nanstd_no_warn(window, 0),
        window[-1] - window[0],
        nanrange_no_warn(window, 0),
    ]
    diff = np.diff(window, axis=0)
    feats += [
        nanmean_no_warn(np.abs(diff), 0),
        nanstd_no_warn(diff, 0),
    ]
    out = np.concatenate(feats, axis=0)  # (162,)
    assert out.shape[0] == TAB_DIM
    return out

def fill_all_nan_columns_to_zero(X2d):
    # X2d: (N, D) float
    X2d = X2d.astype(np.float32, copy=False)
    col_all_nan = np.all(~np.isfinite(X2d), axis=0)
    if np.any(col_all_nan):
        X2d[:, col_all_nan] = 0.0
    # remaining NaNs (partial) -> 0.0 (since keras branch has no imputer)
    X2d[~np.isfinite(X2d)] = 0.0
    return X2d

# -----------------------
# 2) Build EntryID -> global image index mapping (from df_all)
# -----------------------
assert "df_all" in globals() and "images_112" in globals(), "Need df_all + images_112 already created."

def _to_int_safe(s):
    return pd.to_numeric(s, errors="coerce").astype("Int64")

df_map = df_all.copy()
assert "EntryID" in df_map.columns, "df_all must have EntryID column."
df_map["EntryID_int"] = _to_int_safe(df_map["EntryID"])
df_map = df_map.dropna(subset=["EntryID_int"]).copy()
df_map["EntryID_int"] = df_map["EntryID_int"].astype(int)

# if duplicates exist, keep first occurrence (should be rare; safest deterministic)
df_map = df_map.drop_duplicates(subset=["EntryID_int"], keep="first")
entryid_to_idx = dict(zip(df_map["EntryID_int"].values, df_map.index.values))
print(f"[OK] EntryID mapping size: {len(entryid_to_idx)}")

# -----------------------
# 3) Build paired temporal sequences from metrics CSV + global images
# -----------------------
def make_paired_sequences_from_metrics(csv_path, images_112, entryid_to_idx, app_name):
    df = pd.read_csv(csv_path).replace("", np.nan)

    # label -> int
    if LABEL_COL not in df.columns:
        raise RuntimeError(f"[{app_name}] missing label col {LABEL_COL}")
    df[LABEL_COL] = pd.to_numeric(df[LABEL_COL], errors="coerce")
    df = df.dropna(subset=[LABEL_COL]).copy()
    df[LABEL_COL] = df[LABEL_COL].astype(int)

    # EntryID required for pairing to images
    if "EntryID" not in df.columns:
        raise RuntimeError(f"[{app_name}] metrics CSV missing EntryID for pairing.")
    df["EntryID_int"] = _to_int_safe(df["EntryID"])
    df = df.dropna(subset=["EntryID_int"]).copy()
    df["EntryID_int"] = df["EntryID_int"].astype(int)

    # sort temporally
    df = df.sort_values("EntryID_int").reset_index(drop=True)

    # invariant features
    missing = [c for c in INVARIANT27 if c not in df.columns]
    if missing:
        raise RuntimeError(f"[{app_name}] missing invariant cols: {missing[:10]} (and {len(missing)-10} more)" if len(missing) > 10 else f"[{app_name}] missing invariant cols: {missing}")

    X_frame = df[INVARIANT27].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=float)  # (N,27)
    y_frame = df[LABEL_COL].to_numpy(dtype=int)
    entryids = df["EntryID_int"].to_numpy(dtype=int)

    X_img_seq, X_tab_seq, y_seq = [], [], []

    for i in range(len(df) - SEQ_LEN + 1):
        wX = X_frame[i:i+SEQ_LEN]          # (5,27)
        wIDs = entryids[i:i+SEQ_LEN]       # (5,)
        if not window_is_valid(wX, MIN_NON_NAN_RATIO):
            continue

        # image pairing: all 5 frames must exist in global map
        idxs = []
        ok = True
        for eid in wIDs:
            j = entryid_to_idx.get(int(eid), None)
            if j is None:
                ok = False
                break
            idxs.append(j)
        if not ok:
            continue

        img_seq = images_112[np.array(idxs, dtype=int)]  # (5,112,112,1)
        tab_162 = aggregate_window_patchB(wX)            # (162,)
        label = int(y_frame[i + MID])

        X_img_seq.append(img_seq)
        X_tab_seq.append(tab_162)
        y_seq.append(label)

    X_img_seq = np.asarray(X_img_seq, dtype=np.float32)
    X_tab_seq = np.asarray(X_tab_seq, dtype=np.float32)
    y_seq = np.asarray(y_seq, dtype=int)

    # critical: keep dims fixed across apps (never drop all-NaN cols)
    X_tab_seq = fill_all_nan_columns_to_zero(X_tab_seq)

    print(f"[{app_name}] paired sequences: img={X_img_seq.shape} tab={X_tab_seq.shape} y={y_seq.shape}")
    uniq, cnt = np.unique(y_seq, return_counts=True)
    print(f"[{app_name}] y dist:", dict(zip(uniq, cnt)))

    if X_tab_seq.shape[1] != TAB_DIM:
        raise RuntimeError(f"[{app_name}] tab dim mismatch: expected {TAB_DIM}, got {X_tab_seq.shape[1]}")

    return X_img_seq, X_tab_seq, y_seq

# -----------------------
# 4) Model (image seq tokens + ONE tabular token)
# -----------------------
import tensorflow.keras.backend as K
def focal_loss(alpha=0.25, gamma=2.0):
    def loss(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        bce = K.binary_crossentropy(y_true, y_pred)
        p_t = y_true * y_pred + (1.0 - y_true) * (1.0 - y_pred)
        return alpha * K.pow((1.0 - p_t), gamma) * bce
    return loss

def transformer_encoder(x, num_heads=4, key_dim=64, ff_dim=128, dropout_rate=0.1):
    attn = MultiHeadAttention(num_heads=num_heads, key_dim=key_dim)(x, x)
    attn = Dropout(dropout_rate)(attn)
    x = LayerNormalization(epsilon=1e-6)(x + attn)

    ffn = Dense(ff_dim, activation="relu")(x)
    ffn = Dense(x.shape[-1])(ffn)
    ffn = Dropout(dropout_rate)(ffn)
    return LayerNormalization(epsilon=1e-6)(x + ffn)

def build_combined_temporal_model(image_input_shape=(5,112,112,1), tab_dim=162,
                                  token_dim=64, num_heads=4, key_dim=64, ff_dim=128, depth=2):
    # Image branch -> 5 tokens
    img_in = Input(shape=image_input_shape, name="image_input")
    x = TimeDistributed(Conv2D(16, (3,3), activation="relu", padding="same"))(img_in)
    x = TimeDistributed(MaxPooling2D((2,2)))(x)
    x = TimeDistributed(Conv2D(32, (3,3), activation="relu", padding="same"))(x)
    x = TimeDistributed(MaxPooling2D((2,2)))(x)
    x = TimeDistributed(Flatten())(x)
    img_tokens = TimeDistributed(Dense(token_dim, activation="relu"))(x)  # (B,5,64)

    for _ in range(depth):
        img_tokens = transformer_encoder(img_tokens, num_heads=num_heads, key_dim=key_dim, ff_dim=ff_dim)

    # Tabular branch -> ONE token (Patch-B 162 -> 64, then expand to (B,1,64))
    tab_in = Input(shape=(tab_dim,), name="tabular_input")
    t = Dense(token_dim, activation="relu")(tab_in)
    tab_token = Lambda(lambda z: tf.expand_dims(z, axis=1), name="tab_token_expand")(t)  # (B,1,64)

    # Combine tokens: 5 (img) + 1 (tab) = 6 tokens
    tokens = Concatenate(axis=1)([img_tokens, tab_token])  # (B,6,64)

    for _ in range(depth):
        tokens = transformer_encoder(tokens, num_heads=num_heads, key_dim=key_dim, ff_dim=ff_dim)

    pooled = GlobalAveragePooling1D()(tokens)
    h = Dense(64, activation="relu")(pooled)
    out = Dense(1, activation="sigmoid", name="output")(h)

    model = Model(inputs=[img_in, tab_in], outputs=out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-4),
        loss=focal_loss(alpha=0.25, gamma=2.0),
        metrics=["accuracy"]
    )
    return model

def compute_metrics(y_true, y_pred01):
    acc = accuracy_score(y_true, y_pred01)
    prec = precision_score(y_true, y_pred01, zero_division=0)
    rec = recall_score(y_true, y_pred01, zero_division=0)
    f1 = f1_score(y_true, y_pred01, zero_division=0)
    cm = confusion_matrix(y_true, y_pred01, labels=[0,1])
    return acc, f1, prec, rec, cm

# -----------------------
# 5) 5-fold CV per app (same protocol as LR)
#     + scaler fitted on TRAIN fold only (tabular only)
# -----------------------
def run_5fold_combined(app_name, Ximg, Xtab, y, epochs=10, batch_size=16):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    rows = []

    for fold, (tr, te) in enumerate(skf.split(np.zeros(len(y)), y), 1):
        Ximg_tr, Ximg_te = Ximg[tr], Ximg[te]
        Xtab_tr, Xtab_te = Xtab[tr], Xtab[te]
        y_tr, y_te = y[tr], y[te]

        # scale tabular features (train-only fit) for stability + better convergence
        scaler = StandardScaler()
        Xtab_tr = scaler.fit_transform(Xtab_tr)
        Xtab_te = scaler.transform(Xtab_te)

        model = build_combined_temporal_model(image_input_shape=Ximg.shape[1:], tab_dim=Xtab.shape[1])

        es = EarlyStopping(monitor="val_accuracy", patience=3, restore_best_weights=True, verbose=0)
        rlrop = ReduceLROnPlateau(monitor="val_accuracy", factor=0.2, patience=1, min_lr=1e-7, verbose=0)

        model.fit(
            [Ximg_tr, Xtab_tr], y_tr,
            validation_data=([Ximg_te, Xtab_te], y_te),
            epochs=epochs,
            batch_size=batch_size,
            verbose=0,
            callbacks=[es, rlrop]
        )

        prob = model.predict([Ximg_te, Xtab_te], batch_size=batch_size, verbose=0).reshape(-1)
        pred = (prob > 0.5).astype(int)

        acc, f1, prec, rec, cm = compute_metrics(y_te, pred)
        print(f"[{app_name} FOLD {fold}] acc={acc:.3f} f1={f1:.3f} prec={prec:.3f} rec={rec:.3f} cm={cm.tolist()}")

        rows.append({
            "app": app_name, "fold": fold,
            "accuracy": float(acc), "f1": float(f1),
            "precision": float(prec), "recall": float(rec),
            "tn": int(cm[0,0]), "fp": int(cm[0,1]),
            "fn": int(cm[1,0]), "tp": int(cm[1,1]),
        })

        # free memory per fold (colab friendly)
        tf.keras.backend.clear_session()

    df_folds = pd.DataFrame(rows)
    print(f"\n[{app_name}] CV mean acc={df_folds['accuracy'].mean():.3f} mean f1={df_folds['f1'].mean():.3f}\n")
    return df_folds

# -----------------------
# 6) RUN ALL APPS
# -----------------------
all_results = []
for app, mpath in apps.items():
    Ximg, Xtab, y = make_paired_sequences_from_metrics(mpath, images_112, entryid_to_idx, app)
    folds_df = run_5fold_combined(app, Ximg, Xtab, y, epochs=10, batch_size=16)
    all_results.append(folds_df)

results_df = pd.concat(all_results, ignore_index=True)
summary_df = results_df.groupby("app")[["accuracy","f1","precision","recall"]].mean().sort_values("accuracy", ascending=False)

print("=========== PER-APP 5-FOLD SUMMARY (MEAN) ===========")
print(summary_df)
print("\n=========== ALL FOLDS RAW ===========")
print(results_df.head(20))

[OK] EntryID mapping size: 6240
[Archery] paired sequences: img=(421, 5, 112, 112, 1) tab=(421, 162) y=(421,)
[Archery] y dist: {np.int64(0): np.int64(119), np.int64(1): np.int64(302)}
[Archery FOLD 1] acc=0.929 f1=0.952 prec=0.937 rec=0.967 cm=[[20, 4], [2, 59]]
[Archery FOLD 2] acc=0.714 f1=0.833 prec=0.714 rec=1.000 cm=[[0, 24], [0, 60]]
[Archery FOLD 3] acc=0.714 f1=0.833 prec=0.714 rec=1.000 cm=[[0, 24], [0, 60]]
[Archery FOLD 4] acc=0.952 f1=0.968 prec=0.938 rec=1.000 cm=[[20, 4], [0, 60]]
[Archery FOLD 5] acc=0.845 f1=0.904 prec=0.824 rec=1.000 cm=[[10, 13], [0, 61]]

[Archery] CV mean acc=0.831 mean f1=0.898

[PhantomLimb] paired sequences: img=(4101, 5, 112, 112, 1) tab=(4101, 162) y=(4101,)
[PhantomLimb] y dist: {np.int64(0): np.int64(2660), np.int64(1): np.int64(1441)}
[PhantomLimb FOLD 1] acc=0.966 f1=0.951 prec=0.958 rec=0.945 cm=[[520, 12], [16, 273]]
[PhantomLimb FOLD 2] acc=0.957 f1=0.938 prec=0.950 rec=0.927 cm=[[518, 14], [21, 267]]
[PhantomLimb FOLD 3] acc=0.965 f1=0

In [ ]:
# ============================================================
# ✅ Combined Temporal (Images + Patch-B Tabular) — ACCURACY PRIORITY
# Minimal changes vs your block:
#   1) Replace focal loss -> BinaryCrossentropy (accuracy-oriented)
#   2) Tune threshold on VAL ACC (not fixed 0.5)
#   3) Proper inner train/val split inside each fold (so ES + threshold are not tuned on test)
#   4) Monitor val_accuracy (mode='max') everywhere (already mostly the case)
# Everything else kept the same (SEQ_LEN=5, Patch-B 162, same pairing logic, same CV per app)
#
# REQUIRES ALREADY IN MEMORY:
#   df_all      (6242, ...) with columns: SheetName, EntryID
#   images_112  (6242, 112, 112, 1)
# ============================================================

import os, numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

import tensorflow as tf
from tensorflow.keras.layers import (Input, Dense, Flatten, Conv2D, TimeDistributed, MaxPooling2D,
                                     LayerNormalization, Dropout, MultiHeadAttention, GlobalAveragePooling1D,
                                     Concatenate, Lambda)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# -----------------------
# 0) Config
# -----------------------
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

SEQ_LEN = 5
MID = SEQ_LEN // 2
MIN_NON_NAN_RATIO = 0.4
LABEL_COL = "Temporal"

INVARIANT27 = [
    "missing_joints_count","missing_joints_ratio","collapsed_joints_count",
    "center_of_mass_x","center_of_mass_y","center_of_mass_z","distance_from_origin",
    "bbox_width","bbox_height","bbox_depth","bbox_volume","max_joint_distance_from_com",
    "distance_from_floor","min_foot_height_above_floor","below_floor",
    "left_forearm_length","right_forearm_length","left_shin_length","right_shin_length",
    "arm_length_symmetry","leg_length_symmetry",
    "body_upright_x","body_upright_y","body_upright_z",
    "body_forward_x","body_forward_y","body_forward_z"
]
AGG_USE = ("mean", "std", "delta", "range", "mean_abs_vel", "vel_std")
TAB_DIM = len(INVARIANT27) * len(AGG_USE)  # 27 * 6 = 162

apps = {
    "Archery":      "/content/metrics_Archery.csv",
    "PhantomLimb":  "/content/metrics_PhantomLimb.csv",
    "PianoTiles":   "/content/metrics_PianoTiles.csv",
    "Puzzle":       "/content/metrics_Puzzle.csv",
    "Sea":          "/content/metrics_Sea.csv",
    "War":          "/content/metrics_War.csv",
}

# -----------------------
# 1) Patch-B helpers (NaN-safe)
# -----------------------
def nanmean_no_warn(x, axis=0):
    x = np.asarray(x, float)
    valid = np.isfinite(x)
    denom = valid.sum(axis=axis)
    num = np.where(valid, x, 0.0).sum(axis=axis)
    return np.divide(num, denom, out=np.full_like(num, np.nan), where=(denom > 0))

def nanstd_no_warn(x, axis=0):
    mu = nanmean_no_warn(x, axis=axis)
    diff2 = (x - np.expand_dims(mu, axis=axis)) ** 2
    valid = np.isfinite(diff2)
    denom = valid.sum(axis=axis)
    num = np.where(valid, diff2, 0.0).sum(axis=axis)
    return np.sqrt(np.divide(num, denom, out=np.full_like(num, np.nan), where=(denom > 0)))

def nanrange_no_warn(x, axis=0):
    valid = np.isfinite(x)
    mx = np.max(np.where(valid, x, -np.inf), axis=axis)
    mn = np.min(np.where(valid, x, +np.inf), axis=axis)
    return np.where(valid.any(axis=axis), mx - mn, np.nan)

def window_is_valid(window, min_ratio=MIN_NON_NAN_RATIO):
    return (np.isfinite(window).sum() / window.size) >= min_ratio

def aggregate_window_patchB(window):
    feats = [
        nanmean_no_warn(window, 0),
        nanstd_no_warn(window, 0),
        window[-1] - window[0],
        nanrange_no_warn(window, 0),
    ]
    diff = np.diff(window, axis=0)
    feats += [
        nanmean_no_warn(np.abs(diff), 0),
        nanstd_no_warn(diff, 0),
    ]
    out = np.concatenate(feats, axis=0)  # (162,)
    assert out.shape[0] == TAB_DIM
    return out

def fill_all_nan_columns_to_zero(X2d):
    X2d = X2d.astype(np.float32, copy=False)
    col_all_nan = np.all(~np.isfinite(X2d), axis=0)
    if np.any(col_all_nan):
        X2d[:, col_all_nan] = 0.0
    X2d[~np.isfinite(X2d)] = 0.0
    return X2d

# -----------------------
# 2) Build EntryID -> global image index mapping (from df_all)
# -----------------------
assert "df_all" in globals() and "images_112" in globals(), "Need df_all + images_112 already created."

def _to_int_safe(s):
    return pd.to_numeric(s, errors="coerce").astype("Int64")

df_map = df_all.copy()
assert "EntryID" in df_map.columns, "df_all must have EntryID column."
df_map["EntryID_int"] = _to_int_safe(df_map["EntryID"])
df_map = df_map.dropna(subset=["EntryID_int"]).copy()
df_map["EntryID_int"] = df_map["EntryID_int"].astype(int)

df_map = df_map.drop_duplicates(subset=["EntryID_int"], keep="first")
entryid_to_idx = dict(zip(df_map["EntryID_int"].values, df_map.index.values))
print(f"[OK] EntryID mapping size: {len(entryid_to_idx)}")

# -----------------------
# 3) Build paired temporal sequences from metrics CSV + global images
# -----------------------
def make_paired_sequences_from_metrics(csv_path, images_112, entryid_to_idx, app_name):
    df = pd.read_csv(csv_path).replace("", np.nan)

    if LABEL_COL not in df.columns:
        raise RuntimeError(f"[{app_name}] missing label col {LABEL_COL}")
    df[LABEL_COL] = pd.to_numeric(df[LABEL_COL], errors="coerce")
    df = df.dropna(subset=[LABEL_COL]).copy()
    df[LABEL_COL] = df[LABEL_COL].astype(int)

    if "EntryID" not in df.columns:
        raise RuntimeError(f"[{app_name}] metrics CSV missing EntryID for pairing.")
    df["EntryID_int"] = _to_int_safe(df["EntryID"])
    df = df.dropna(subset=["EntryID_int"]).copy()
    df["EntryID_int"] = df["EntryID_int"].astype(int)

    df = df.sort_values("EntryID_int").reset_index(drop=True)

    missing = [c for c in INVARIANT27 if c not in df.columns]
    if missing:
        raise RuntimeError(
            f"[{app_name}] missing invariant cols: {missing[:10]} (and {len(missing)-10} more)"
            if len(missing) > 10 else f"[{app_name}] missing invariant cols: {missing}"
        )

    X_frame = df[INVARIANT27].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=float)  # (N,27)
    y_frame = df[LABEL_COL].to_numpy(dtype=int)
    entryids = df["EntryID_int"].to_numpy(dtype=int)

    X_img_seq, X_tab_seq, y_seq = [], [], []

    for i in range(len(df) - SEQ_LEN + 1):
        wX = X_frame[i:i+SEQ_LEN]
        wIDs = entryids[i:i+SEQ_LEN]
        if not window_is_valid(wX, MIN_NON_NAN_RATIO):
            continue

        idxs = []
        ok = True
        for eid in wIDs:
            j = entryid_to_idx.get(int(eid), None)
            if j is None:
                ok = False
                break
            idxs.append(j)
        if not ok:
            continue

        img_seq = images_112[np.array(idxs, dtype=int)]  # (5,112,112,1)
        tab_162 = aggregate_window_patchB(wX)            # (162,)
        label = int(y_frame[i + MID])

        X_img_seq.append(img_seq)
        X_tab_seq.append(tab_162)
        y_seq.append(label)

    X_img_seq = np.asarray(X_img_seq, dtype=np.float32)
    X_tab_seq = np.asarray(X_tab_seq, dtype=np.float32)
    y_seq = np.asarray(y_seq, dtype=int)

    X_tab_seq = fill_all_nan_columns_to_zero(X_tab_seq)

    print(f"[{app_name}] paired sequences: img={X_img_seq.shape} tab={X_tab_seq.shape} y={y_seq.shape}")
    uniq, cnt = np.unique(y_seq, return_counts=True)
    print(f"[{app_name}] y dist:", dict(zip(uniq, cnt)))

    if X_tab_seq.shape[1] != TAB_DIM:
        raise RuntimeError(f"[{app_name}] tab dim mismatch: expected {TAB_DIM}, got {X_tab_seq.shape[1]}")

    return X_img_seq, X_tab_seq, y_seq

# -----------------------
# 4) Model (image seq tokens + ONE tabular token)
# -----------------------
def transformer_encoder(x, num_heads=4, key_dim=64, ff_dim=128, dropout_rate=0.1):
    attn = MultiHeadAttention(num_heads=num_heads, key_dim=key_dim)(x, x)
    attn = Dropout(dropout_rate)(attn)
    x = LayerNormalization(epsilon=1e-6)(x + attn)

    ffn = Dense(ff_dim, activation="relu")(x)
    ffn = Dense(x.shape[-1])(ffn)
    ffn = Dropout(dropout_rate)(ffn)
    return LayerNormalization(epsilon=1e-6)(x + ffn)

def build_combined_temporal_model(image_input_shape=(5,112,112,1), tab_dim=162,
                                  token_dim=64, num_heads=4, key_dim=64, ff_dim=128, depth=2,
                                  lr=1e-4, weight_decay=1e-4, label_smoothing=0.0):
    # Image branch -> 5 tokens
    img_in = Input(shape=image_input_shape, name="image_input")
    x = TimeDistributed(Conv2D(16, (3,3), activation="relu", padding="same"))(img_in)
    x = TimeDistributed(MaxPooling2D((2,2)))(x)
    x = TimeDistributed(Conv2D(32, (3,3), activation="relu", padding="same"))(x)
    x = TimeDistributed(MaxPooling2D((2,2)))(x)
    x = TimeDistributed(Flatten())(x)
    img_tokens = TimeDistributed(Dense(token_dim, activation="relu"))(x)  # (B,5,64)

    for _ in range(depth):
        img_tokens = transformer_encoder(img_tokens, num_heads=num_heads, key_dim=key_dim, ff_dim=ff_dim)

    # Tabular branch -> ONE token
    tab_in = Input(shape=(tab_dim,), name="tabular_input")
    t = Dense(token_dim, activation="relu")(tab_in)
    tab_token = Lambda(lambda z: tf.expand_dims(z, axis=1), name="tab_token_expand")(t)  # (B,1,64)

    tokens = Concatenate(axis=1)([img_tokens, tab_token])  # (B,6,64)

    for _ in range(depth):
        tokens = transformer_encoder(tokens, num_heads=num_heads, key_dim=key_dim, ff_dim=ff_dim)

    pooled = GlobalAveragePooling1D()(tokens)
    h = Dense(64, activation="relu")(pooled)
    out = Dense(1, activation="sigmoid", name="output")(h)

    model = Model(inputs=[img_in, tab_in], outputs=out)

    # ✅ ACCURACY-oriented loss (instead of focal)
    opt = tf.keras.optimizers.AdamW(learning_rate=lr, weight_decay=weight_decay)
    loss = tf.keras.losses.BinaryCrossentropy(label_smoothing=label_smoothing)
    model.compile(optimizer=opt, loss=loss, metrics=["accuracy"])
    return model

def compute_metrics(y_true, y_pred01):
    acc = accuracy_score(y_true, y_pred01)
    prec = precision_score(y_true, y_pred01, zero_division=0)
    rec = recall_score(y_true, y_pred01, zero_division=0)
    f1 = f1_score(y_true, y_pred01, zero_division=0)
    cm = confusion_matrix(y_true, y_pred01, labels=[0,1])
    return acc, f1, prec, rec, cm

# ✅ ACCURACY threshold tuning
def tune_threshold_on_val_acc(y_val, p_val, grid=None):
    if grid is None:
        grid = np.linspace(0.05, 0.95, 19)
    best_thr, best_acc = 0.5, -1.0
    for t in grid:
        yhat = (p_val >= t).astype(int)
        acc = accuracy_score(y_val, yhat)
        if acc > best_acc:
            best_acc = acc
            best_thr = float(t)
    return best_thr, float(best_acc)

# -----------------------
# 5) 5-fold CV per app (same protocol as LR)
#     ✅ Now uses inner val split (no tuning on test fold)
# -----------------------
def run_5fold_combined(app_name, Ximg, Xtab, y,
                       epochs=10, batch_size=16, val_size=0.15,
                       thr_grid=None, verbose_fit=0):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    rows = []

    for fold, (tr, te) in enumerate(skf.split(np.zeros(len(y)), y), 1):
        # outer split: train vs test
        Ximg_tr_full, Ximg_te = Ximg[tr], Ximg[te]
        Xtab_tr_full, Xtab_te = Xtab[tr], Xtab[te]
        y_tr_full, y_te = y[tr], y[te]

        # inner split: train vs val (for ES + threshold)
        Ximg_tr, Ximg_val, Xtab_tr, Xtab_val, y_tr, y_val = train_test_split(
            Ximg_tr_full, Xtab_tr_full, y_tr_full,
            test_size=val_size, random_state=SEED, stratify=y_tr_full
        )

        # scale tabular features (fit on INNER TRAIN only)
        scaler = StandardScaler()
        Xtab_tr  = scaler.fit_transform(Xtab_tr)
        Xtab_val = scaler.transform(Xtab_val)
        Xtab_te  = scaler.transform(Xtab_te)

        model = build_combined_temporal_model(
            image_input_shape=Ximg.shape[1:], tab_dim=Xtab.shape[1],
            lr=1e-4, weight_decay=1e-4, label_smoothing=0.0
        )

        es = EarlyStopping(monitor="val_accuracy", mode="max", patience=3, restore_best_weights=True, verbose=0)
        rlrop = ReduceLROnPlateau(monitor="val_accuracy", mode="max", factor=0.2, patience=1, min_lr=1e-7, verbose=0)

        model.fit(
            [Ximg_tr, Xtab_tr], y_tr,
            validation_data=([Ximg_val, Xtab_val], y_val),
            epochs=epochs,
            batch_size=batch_size,
            verbose=verbose_fit,
            callbacks=[es, rlrop]
        )

        # tune threshold on VAL for ACC
        p_val = model.predict([Ximg_val, Xtab_val], batch_size=batch_size, verbose=0).reshape(-1)
        thr, val_acc = tune_threshold_on_val_acc(y_val, p_val, grid=thr_grid)

        # evaluate on TEST with tuned threshold
        prob = model.predict([Ximg_te, Xtab_te], batch_size=batch_size, verbose=0).reshape(-1)
        pred = (prob >= thr).astype(int)

        acc, f1, prec, rec, cm = compute_metrics(y_te, pred)
        print(f"[{app_name} FOLD {fold}] thr={thr:.2f} val_acc={val_acc:.3f}  "
              f"acc={acc:.3f} f1={f1:.3f} prec={prec:.3f} rec={rec:.3f} cm={cm.tolist()}")

        rows.append({
            "app": app_name, "fold": fold,
            "thr": float(thr),
            "val_acc": float(val_acc),
            "accuracy": float(acc), "f1": float(f1),
            "precision": float(prec), "recall": float(rec),
            "tn": int(cm[0,0]), "fp": int(cm[0,1]),
            "fn": int(cm[1,0]), "tp": int(cm[1,1]),
        })

        tf.keras.backend.clear_session()

    df_folds = pd.DataFrame(rows)
    print(f"\n[{app_name}] CV mean acc={df_folds['accuracy'].mean():.3f} mean f1={df_folds['f1'].mean():.3f}\n")
    return df_folds

# -----------------------
# 6) RUN ALL APPS
# -----------------------
all_results = []
for app, mpath in apps.items():
    Ximg, Xtab, y = make_paired_sequences_from_metrics(mpath, images_112, entryid_to_idx, app)
    folds_df = run_5fold_combined(app, Ximg, Xtab, y, epochs=10, batch_size=16, val_size=0.15, thr_grid=None, verbose_fit=0)
    all_results.append(folds_df)

results_df = pd.concat(all_results, ignore_index=True)
summary_df = results_df.groupby("app")[["accuracy","f1","precision","recall"]].mean().sort_values("accuracy", ascending=False)

print("=========== PER-APP 5-FOLD SUMMARY (MEAN) ===========")
print(summary_df)
print("\n=========== ALL FOLDS RAW ===========")
print(results_df.head(20))

[OK] EntryID mapping size: 6240
[Archery] paired sequences: img=(421, 5, 112, 112, 1) tab=(421, 162) y=(421,)
[Archery] y dist: {np.int64(0): np.int64(119), np.int64(1): np.int64(302)}
[Archery FOLD 1] thr=0.75 val_acc=0.745  acc=0.788 f1=0.870 prec=0.779 rec=0.984 cm=[[7, 17], [1, 60]]
[Archery FOLD 2] thr=0.05 val_acc=0.725  acc=0.714 f1=0.833 prec=0.714 rec=1.000 cm=[[0, 24], [0, 60]]
[Archery FOLD 3] thr=0.05 val_acc=0.725  acc=0.714 f1=0.833 prec=0.714 rec=1.000 cm=[[0, 24], [0, 60]]
[Archery FOLD 4] thr=0.05 val_acc=0.725  acc=0.714 f1=0.833 prec=0.714 rec=1.000 cm=[[0, 24], [0, 60]]
[Archery FOLD 5] thr=0.60 val_acc=0.784  acc=0.833 f1=0.897 prec=0.813 rec=1.000 cm=[[9, 14], [0, 61]]

[Archery] CV mean acc=0.753 mean f1=0.853

[PhantomLimb] paired sequences: img=(4101, 5, 112, 112, 1) tab=(4101, 162) y=(4101,)
[PhantomLimb] y dist: {np.int64(0): np.int64(2660), np.int64(1): np.int64(1441)}
[PhantomLimb FOLD 1] thr=0.50 val_acc=0.959  acc=0.971 f1=0.958 prec=0.978 rec=0.938 cm=[[

In [ ]:
# ============================================================
# ✅ CONFERENCE-CORRECT:
# PHANTOMLIMB TRAIN -> (train/val split inside TRAIN) -> HOLDOUT TEST
# Combined temporal model = (5-frame image seq) + (Patch-B 162 tab vector) + transformer
#
# SAME AS YOUR SCRIPT:
#  - SEQ_LEN=5, center label
#  - invariant27 + Patch-B aggregation => 162-dim
#  - MIN_NON_NAN_RATIO window filter
#  - fill_all_nan_columns_to_zero (never drop dims)
#  - StandardScaler fit on TRAIN only (tabular only)
#  - threshold = 0.5
#
# IMPORTANT FIX:
#  - HOLDOUT is NOT used for early stopping or LR scheduling
#  - early stopping monitors an INNER VAL split from TRAIN
#
# REQUIRES ALREADY IN MEMORY:
#   df_all      with column EntryID (global mapping table)
#   images_112  with shape (N,112,112,1) aligned to df_all rows
#
# INPUT FILES:
#   /content/metrics_PhantomLimb.csv
#   /content/metrics_PhantomLimb_Holdout.csv
# ============================================================

import os, json
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

import tensorflow as tf
from tensorflow.keras.layers import (Input, Dense, Flatten, Conv2D, TimeDistributed, MaxPooling2D,
                                     LayerNormalization, Dropout, MultiHeadAttention, GlobalAveragePooling1D,
                                     Concatenate, Lambda)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import tensorflow.keras.backend as K
import joblib

# -----------------------
# 0) Config
# -----------------------
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

SEQ_LEN = 5
MID = SEQ_LEN // 2
MIN_NON_NAN_RATIO = 0.4
LABEL_COL = "Temporal"

INVARIANT27 = [
    "missing_joints_count","missing_joints_ratio","collapsed_joints_count",
    "center_of_mass_x","center_of_mass_y","center_of_mass_z","distance_from_origin",
    "bbox_width","bbox_height","bbox_depth","bbox_volume","max_joint_distance_from_com",
    "distance_from_floor","min_foot_height_above_floor","below_floor",
    "left_forearm_length","right_forearm_length","left_shin_length","right_shin_length",
    "arm_length_symmetry","leg_length_symmetry",
    "body_upright_x","body_upright_y","body_upright_z",
    "body_forward_x","body_forward_y","body_forward_z"
]
AGG_USE = ("mean", "std", "delta", "range", "mean_abs_vel", "vel_std")
TAB_DIM = len(INVARIANT27) * len(AGG_USE)  # 162

TRAIN_METRICS   = "/content/metrics_PhantomLimb.csv"
HOLDOUT_METRICS = "/content/metrics_PhantomLimb_Holdout.csv"

OUT_DIR = "/content/phantomlimb_temporal_combined_img_tab_holdout_CONFERENCE"
os.makedirs(OUT_DIR, exist_ok=True)

# -----------------------
# 1) Patch-B helpers (NaN-safe)
# -----------------------
def nanmean_no_warn(x, axis=0):
    x = np.asarray(x, float)
    valid = np.isfinite(x)
    denom = valid.sum(axis=axis)
    num = np.where(valid, x, 0.0).sum(axis=axis)
    return np.divide(num, denom, out=np.full_like(num, np.nan), where=(denom > 0))

def nanstd_no_warn(x, axis=0):
    mu = nanmean_no_warn(x, axis=axis)
    diff2 = (x - np.expand_dims(mu, axis=axis)) ** 2
    valid = np.isfinite(diff2)
    denom = valid.sum(axis=axis)
    num = np.where(valid, diff2, 0.0).sum(axis=axis)
    return np.sqrt(np.divide(num, denom, out=np.full_like(num, np.nan), where=(denom > 0)))

def nanrange_no_warn(x, axis=0):
    valid = np.isfinite(x)
    mx = np.max(np.where(valid, x, -np.inf), axis=axis)
    mn = np.min(np.where(valid, x, +np.inf), axis=axis)
    return np.where(valid.any(axis=axis), mx - mn, np.nan)

def window_is_valid(window, min_ratio=MIN_NON_NAN_RATIO):
    return (np.isfinite(window).sum() / window.size) >= min_ratio

def aggregate_window_patchB(window):
    feats = [
        nanmean_no_warn(window, 0),
        nanstd_no_warn(window, 0),
        window[-1] - window[0],
        nanrange_no_warn(window, 0),
    ]
    diff = np.diff(window, axis=0)
    feats += [
        nanmean_no_warn(np.abs(diff), 0),
        nanstd_no_warn(diff, 0),
    ]
    out = np.concatenate(feats, axis=0)  # (162,)
    if out.shape[0] != TAB_DIM:
        raise RuntimeError(f"Patch-B dim mismatch: got {out.shape[0]} expected {TAB_DIM}")
    return out

def fill_all_nan_columns_to_zero(X2d):
    X2d = X2d.astype(np.float32, copy=False)
    col_all_nan = np.all(~np.isfinite(X2d), axis=0)
    if np.any(col_all_nan):
        X2d[:, col_all_nan] = 0.0
    X2d[~np.isfinite(X2d)] = 0.0
    return X2d

# -----------------------
# 2) Build EntryID -> global image index mapping (from df_all)
# -----------------------
assert "df_all" in globals() and "images_112" in globals(), "Need df_all + images_112 already created."
assert "EntryID" in df_all.columns, "df_all must contain EntryID."

def _to_int_safe(s):
    return pd.to_numeric(s, errors="coerce").astype("Int64")

df_map = df_all.copy()
df_map["EntryID_int"] = _to_int_safe(df_map["EntryID"])
df_map = df_map.dropna(subset=["EntryID_int"]).copy()
df_map["EntryID_int"] = df_map["EntryID_int"].astype(int)
df_map = df_map.drop_duplicates(subset=["EntryID_int"], keep="first")
entryid_to_idx = dict(zip(df_map["EntryID_int"].values, df_map.index.values))
print(f"[OK] EntryID mapping size: {len(entryid_to_idx)} | images_112 shape: {np.asarray(images_112).shape}")

# -----------------------
# 3) Build paired temporal sequences (metrics csv + global images)
# -----------------------
def make_paired_sequences_from_metrics(csv_path, images_112, entryid_to_idx, tag):
    df = pd.read_csv(csv_path).replace("", np.nan)

    if LABEL_COL not in df.columns:
        raise RuntimeError(f"[{tag}] missing label col {LABEL_COL}")
    df[LABEL_COL] = pd.to_numeric(df[LABEL_COL], errors="coerce")
    df = df.dropna(subset=[LABEL_COL]).copy()
    df[LABEL_COL] = df[LABEL_COL].astype(int)

    if "EntryID" not in df.columns:
        raise RuntimeError(f"[{tag}] metrics CSV missing EntryID for pairing.")
    df["EntryID_int"] = _to_int_safe(df["EntryID"])
    df = df.dropna(subset=["EntryID_int"]).copy()
    df["EntryID_int"] = df["EntryID_int"].astype(int)

    # match original: sort by EntryID
    df = df.sort_values("EntryID_int").reset_index(drop=True)

    missing = [c for c in INVARIANT27 if c not in df.columns]
    if missing:
        raise RuntimeError(f"[{tag}] missing invariant cols: {missing[:10]}")

    X_frame = df[INVARIANT27].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=float)  # (N,27)
    y_frame = df[LABEL_COL].to_numpy(dtype=int)
    entryids = df["EntryID_int"].to_numpy(dtype=int)

    X_img_seq, X_tab_seq, y_seq = [], [], []

    for i in range(len(df) - SEQ_LEN + 1):
        wX = X_frame[i:i+SEQ_LEN]
        wIDs = entryids[i:i+SEQ_LEN]
        if not window_is_valid(wX, MIN_NON_NAN_RATIO):
            continue

        idxs = []
        ok = True
        for eid in wIDs:
            j = entryid_to_idx.get(int(eid), None)
            if j is None:
                ok = False
                break
            idxs.append(j)
        if not ok:
            continue

        img_seq = images_112[np.array(idxs, dtype=int)]
        tab_162 = aggregate_window_patchB(wX)
        label = int(y_frame[i + MID])

        X_img_seq.append(img_seq)
        X_tab_seq.append(tab_162)
        y_seq.append(label)

    X_img_seq = np.asarray(X_img_seq, dtype=np.float32)
    X_tab_seq = np.asarray(X_tab_seq, dtype=np.float32)
    y_seq = np.asarray(y_seq, dtype=int)

    X_tab_seq = fill_all_nan_columns_to_zero(X_tab_seq)

    print(f"[{tag}] paired sequences: img={X_img_seq.shape} tab={X_tab_seq.shape} y={y_seq.shape}")
    uniq, cnt = np.unique(y_seq, return_counts=True)
    print(f"[{tag}] y dist:", dict(zip(uniq, cnt)))

    if X_tab_seq.shape[1] != TAB_DIM:
        raise RuntimeError(f"[{tag}] tab dim mismatch: expected {TAB_DIM}, got {X_tab_seq.shape[1]}")
    if len(y_seq) < 50:
        raise RuntimeError(f"[{tag}] too few sequences after filtering/pairing. Check EntryID mapping + NaNs.")

    return X_img_seq, X_tab_seq, y_seq

# -----------------------
# 4) Model (same as your code)
# -----------------------
def focal_loss(alpha=0.25, gamma=2.0):
    def loss(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        bce = K.binary_crossentropy(y_true, y_pred)
        p_t = y_true * y_pred + (1.0 - y_true) * (1.0 - y_pred)
        return alpha * K.pow((1.0 - p_t), gamma) * bce
    return loss

def transformer_encoder(x, num_heads=4, key_dim=64, ff_dim=128, dropout_rate=0.1):
    attn = MultiHeadAttention(num_heads=num_heads, key_dim=key_dim)(x, x)
    attn = Dropout(dropout_rate)(attn)
    x = LayerNormalization(epsilon=1e-6)(x + attn)

    ffn = Dense(ff_dim, activation="relu")(x)
    ffn = Dense(x.shape[-1])(ffn)
    ffn = Dropout(dropout_rate)(ffn)
    return LayerNormalization(epsilon=1e-6)(x + ffn)

def build_combined_temporal_model(image_input_shape=(5,112,112,1), tab_dim=162,
                                  token_dim=64, num_heads=4, key_dim=64, ff_dim=128, depth=2):
    img_in = Input(shape=image_input_shape, name="image_input")
    x = TimeDistributed(Conv2D(16, (3,3), activation="relu", padding="same"))(img_in)
    x = TimeDistributed(MaxPooling2D((2,2)))(x)
    x = TimeDistributed(Conv2D(32, (3,3), activation="relu", padding="same"))(x)
    x = TimeDistributed(MaxPooling2D((2,2)))(x)
    x = TimeDistributed(Flatten())(x)
    img_tokens = TimeDistributed(Dense(token_dim, activation="relu"))(x)  # (B,5,64)

    for _ in range(depth):
        img_tokens = transformer_encoder(img_tokens, num_heads=num_heads, key_dim=key_dim, ff_dim=ff_dim)

    tab_in = Input(shape=(tab_dim,), name="tabular_input")
    t = Dense(token_dim, activation="relu")(tab_in)
    tab_token = Lambda(lambda z: tf.expand_dims(z, axis=1), name="tab_token_expand")(t)  # (B,1,64)

    tokens = Concatenate(axis=1)([img_tokens, tab_token])  # (B,6,64)

    for _ in range(depth):
        tokens = transformer_encoder(tokens, num_heads=num_heads, key_dim=key_dim, ff_dim=ff_dim)

    pooled = GlobalAveragePooling1D()(tokens)
    h = Dense(64, activation="relu")(pooled)
    out = Dense(1, activation="sigmoid", name="output")(h)

    model = Model(inputs=[img_in, tab_in], outputs=out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-4),
        loss=focal_loss(alpha=0.25, gamma=2.0),
        metrics=["accuracy"]
    )
    return model

def compute_metrics(y_true, y_pred01):
    acc = accuracy_score(y_true, y_pred01)
    prec = precision_score(y_true, y_pred01, zero_division=0)
    rec = recall_score(y_true, y_pred01, zero_division=0)
    f1 = f1_score(y_true, y_pred01, zero_division=0)
    cm = confusion_matrix(y_true, y_pred01, labels=[0,1])
    return acc, f1, prec, rec, cm

# -----------------------
# 5) Build TRAIN + HOLDOUT sequences
# -----------------------
Ximg_tr_full, Xtab_tr_full, y_tr_full = make_paired_sequences_from_metrics(TRAIN_METRICS, images_112, entryid_to_idx, "PhantomLimb_TRAIN")
Ximg_h,       Xtab_h,       y_h       = make_paired_sequences_from_metrics(HOLDOUT_METRICS, images_112, entryid_to_idx, "PhantomLimb_HOLDOUT")

# ---- inner split on TRAIN ONLY (conference-correct) ----
Ximg_tr, Ximg_val, Xtab_tr, Xtab_val, y_tr, y_val = train_test_split(
    Ximg_tr_full, Xtab_tr_full, y_tr_full,
    test_size=0.15,
    random_state=SEED,
    stratify=y_tr_full
)

# scale tabular features (fit on TRAIN only)
scaler = StandardScaler()
Xtab_tr_s  = scaler.fit_transform(Xtab_tr)
Xtab_val_s = scaler.transform(Xtab_val)
Xtab_h_s   = scaler.transform(Xtab_h)

model = build_combined_temporal_model(image_input_shape=Ximg_tr.shape[1:], tab_dim=Xtab_tr_s.shape[1])

es = EarlyStopping(monitor="val_accuracy", patience=3, restore_best_weights=True, verbose=0)
rlrop = ReduceLROnPlateau(monitor="val_accuracy", factor=0.2, patience=1, min_lr=1e-7, verbose=0)

model.fit(
    [Ximg_tr, Xtab_tr_s], y_tr,
    validation_data=([Ximg_val, Xtab_val_s], y_val),
    epochs=10,
    batch_size=16,
    verbose=0,
    callbacks=[es, rlrop]
)

# -----------------------
# 6) Evaluate ON HOLDOUT ONCE (no training decisions)
# -----------------------
prob = model.predict([Ximg_h, Xtab_h_s], batch_size=16, verbose=0).reshape(-1)
pred = (prob > 0.5).astype(int)

acc, f1, prec, rec, cm = compute_metrics(y_h, pred)
print("\n================= PHANTOMLIMB TRAIN -> HOLDOUT (COMBINED, CONFERENCE-CORRECT) =================")
print(f"[HOLDOUT] acc={acc:.3f} f1={f1:.3f} prec={prec:.3f} rec={rec:.3f} cm={cm.tolist()}")

# -----------------------
# 7) Save model + scaler + meta
# -----------------------
model_path  = os.path.join(OUT_DIR, "model.keras")
scaler_path = os.path.join(OUT_DIR, "tab_scaler.joblib")
meta_path   = os.path.join(OUT_DIR, "meta.json")

model.save(model_path)
joblib.dump(scaler, scaler_path)

meta = {
    "app": "PhantomLimb",
    "task": "Temporal_combined_img_tab_transformer",
    "protocol": "Train on PhantomLimb TRAIN only; early-stop on inner TRAIN val split; evaluate once on HOLDOUT.",
    "trained_on": TRAIN_METRICS,
    "evaluated_on_holdout": HOLDOUT_METRICS,
    "seed": SEED,
    "label_col": LABEL_COL,
    "seq_len": SEQ_LEN,
    "mid_label_index": MID,
    "min_non_nan_ratio": float(MIN_NON_NAN_RATIO),
    "features_frame_invariant27": INVARIANT27,
    "agg_use": list(AGG_USE),
    "tab_dim": int(TAB_DIM),
    "train_sequences_full": int(len(y_tr_full)),
    "train_sequences_used": int(len(y_tr)),
    "val_sequences_used": int(len(y_val)),
    "holdout_sequences": int(len(y_h)),
    "holdout_metrics": {
        "accuracy": float(acc),
        "f1": float(f1),
        "precision": float(prec),
        "recall": float(rec),
        "cm": cm.tolist(),
    },
    "notes": "Conference-correct: holdout not used for early stopping / LR scheduling / threshold selection."
}
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print("\nSaved:")
print(" -", model_path)
print(" -", scaler_path)
print(" -", meta_path)

[OK] EntryID mapping size: 6240 | images_112 shape: (6242, 112, 112, 1)
[PhantomLimb_TRAIN] paired sequences: img=(4101, 5, 112, 112, 1) tab=(4101, 162) y=(4101,)
[PhantomLimb_TRAIN] y dist: {np.int64(0): np.int64(2660), np.int64(1): np.int64(1441)}
[PhantomLimb_HOLDOUT] paired sequences: img=(387, 5, 112, 112, 1) tab=(387, 162) y=(387,)
[PhantomLimb_HOLDOUT] y dist: {np.int64(0): np.int64(286), np.int64(1): np.int64(101)}

================= PHANTOMLIMB TRAIN -> HOLDOUT (COMBINED, CONFERENCE-CORRECT) =================
[HOLDOUT] acc=0.695 f1=0.359 prec=0.398 rec=0.327 cm=[[236, 50], [68, 33]]

Saved:
 - /content/phantomlimb_temporal_combined_img_tab_holdout_CONFERENCE/model.keras
 - /content/phantomlimb_temporal_combined_img_tab_holdout_CONFERENCE/tab_scaler.joblib
 - /content/phantomlimb_temporal_combined_img_tab_holdout_CONFERENCE/meta.json


## Image + Metrics(T)

In [ ]:
# ===============================================================
# PER-APP: Temporal Transformer on RAW SEQ (T=5, F=all metric cols)
# - 5-fold Stratified CV on sequences PER APP
# - inner train/val split for EarlyStopping + threshold tuning
# - impute+scale FIT ON TRAIN ONLY (flatten time)
# - KEEP-DIMS impute/scale (all-NaN cols -> 0) so feature count is stable
# - returns acc/prec/rec/f1/cm per fold + means
# ===============================================================

import os
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.layers import Input, Dense, Dropout, LayerNormalization, MultiHeadAttention, GlobalAveragePooling1D
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"


# -------------------------
# 1) Data prep helpers
# -------------------------
def _clean_df(df):
    df = df.copy().replace("", np.nan)
    df["Temporal"] = pd.to_numeric(df["Temporal"], errors="coerce")
    df = df.dropna(subset=["Temporal"]).copy()
    df["Temporal"] = df["Temporal"].astype(int)
    return df

def _get_feature_cols(df):
    id_cols = [c for c in ["GlobalID", "EntryID", "Sequence_ID", "Person_ID", "FPS"] if c in df.columns]
    drop_cols = ["App", "Spatial", "Temporal"] + id_cols
    feature_cols = [c for c in df.columns if c not in drop_cols]

    out = df.copy()
    for c in feature_cols:
        out[c] = pd.to_numeric(out[c], errors="coerce")
    return out, feature_cols

def _order_df(df):
    order_col = None
    for c in ["EntryID", "entryid", "Timestamp", "timestamp", "Frame", "frame"]:
        if c in df.columns:
            order_col = c
            break
    if order_col is None:
        return df.reset_index(drop=True), None

    tmp = pd.to_numeric(df[order_col], errors="coerce")
    if tmp.notna().any():
        df = df.assign(_order_tmp=tmp).sort_values("_order_tmp").drop(columns=["_order_tmp"]).reset_index(drop=True)
    else:
        df = df.sort_values(order_col).reset_index(drop=True)
    return df, order_col


# -------------------------
# 2) Sliding windows (RAW seq) => X: (N,T,F), y: (N,)
# -------------------------
def window_is_valid(window, min_non_nan_ratio=0.4):
    total = window.size
    non_nan = np.isfinite(window).sum()
    return (non_nan / total) >= min_non_nan_ratio

def make_raw_sequence_dataset(app_df, feature_cols, label_col="Temporal", seq_len=5, min_non_nan_ratio=0.4):
    X = app_df[feature_cols].values.astype(float)
    y = app_df[label_col].values.astype(int)

    if len(app_df) < seq_len:
        return np.empty((0, seq_len, len(feature_cols))), np.empty((0,), dtype=int)

    X_seq, y_seq = [], []
    center = seq_len // 2
    for i in range(len(app_df) - seq_len + 1):
        window = X[i:i+seq_len]
        if not window_is_valid(window, min_non_nan_ratio=min_non_nan_ratio):
            continue
        X_seq.append(window)
        y_seq.append(y[i + center])

    if len(X_seq) == 0:
        return np.empty((0, seq_len, len(feature_cols))), np.empty((0,), dtype=int)

    return np.stack(X_seq, axis=0), np.array(y_seq, dtype=int)


# -------------------------
# 3) Metrics + threshold tuning
# -------------------------
def eval_binary(y_true, y_pred):
    return {
        "acc": float(accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "cm": confusion_matrix(y_true, y_pred, labels=[0,1]).tolist(),
    }

def tune_threshold_on_val(y_val, p_val, grid=None):
    if grid is None:
        grid = np.linspace(0.05, 0.95, 19)
    best_thr, best_f1 = 0.5, -1.0
    for t in grid:
        yhat = (p_val >= t).astype(int)
        f1 = f1_score(y_val, yhat, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_thr = float(t)
    return best_thr, float(best_f1)


# -------------------------
# 4) Transformer model (RAW seq)
# -------------------------
def transformer_encoder(inputs, num_heads, key_dim, ff_dim, dropout=0.15):
    attn = MultiHeadAttention(num_heads=num_heads, key_dim=key_dim)(inputs, inputs)
    attn = Dropout(dropout)(attn)
    x = LayerNormalization(epsilon=1e-6)(inputs + attn)

    ffn = Dense(ff_dim, activation="relu")(x)
    ffn = Dense(x.shape[-1])(ffn)
    ffn = Dropout(dropout)(ffn)
    out = LayerNormalization(epsilon=1e-6)(x + ffn)
    return out

def build_rawseq_transformer(seq_len, feat_dim,
                            d_model=64, num_heads=2, key_dim=32,
                            ff_dim=128, depth=2, dropout=0.15,
                            lr=7e-5, weight_decay=1e-4,
                            label_smoothing=0.0):
    inp = Input(shape=(seq_len, feat_dim), name="raw_seq_in")
    x = Dense(d_model, activation="relu")(inp)

    for _ in range(depth):
        x = transformer_encoder(x, num_heads=num_heads, key_dim=key_dim, ff_dim=ff_dim, dropout=dropout)

    x = GlobalAveragePooling1D()(x)
    x = Dense(64, activation="relu")(x)
    x = Dropout(dropout)(x)
    out = Dense(1, activation="sigmoid")(x)

    model = Model(inp, out)
    opt = tf.keras.optimizers.AdamW(learning_rate=lr, weight_decay=weight_decay)
    loss = tf.keras.losses.BinaryCrossentropy(label_smoothing=label_smoothing)
    model.compile(optimizer=opt, loss=loss, metrics=["accuracy"])
    return model

def make_tf_dataset(X, y, batch_size, shuffle=True, drop_remainder=False):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if shuffle:
        ds = ds.shuffle(buffer_size=min(len(y), 5000), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(batch_size, drop_remainder=drop_remainder).prefetch(tf.data.AUTOTUNE)
    return ds


# -------------------------
# 4.5) KEEP-DIMS impute+scale (all-NaN cols -> 0.0)
# -------------------------
def fit_impute_scale_keepdims(X_train_flat):
    X = X_train_flat.astype(float, copy=True)
    fill = np.nanmedian(X, axis=0)
    fill = np.where(np.isfinite(fill), fill, 0.0)
    X = np.where(np.isfinite(X), X, fill)

    mean = X.mean(axis=0)
    std  = X.std(axis=0)
    std  = np.where(std > 0, std, 1.0)
    return fill, mean, std

def apply_impute_scale_keepdims(X_flat, fill, mean, std):
    X = X_flat.astype(float, copy=True)
    X = np.where(np.isfinite(X), X, fill)
    X = (X - mean) / std
    return X


# -------------------------
# 5) Per-app 5-fold runner
# -------------------------
def run_per_app_5fold_transformer(
    df_app_raw,
    app_name,
    seq_len=5,
    min_non_nan_ratio=0.4,
    n_splits=5,
    epochs=25,
    batch_size=64,
    val_size=0.15,
    label_smoothing=0.0,
    verbose_fit=0,
):
    df_app_raw = _clean_df(df_app_raw)
    df_app_raw, order_col = _order_df(df_app_raw)
    df_app, feature_cols = _get_feature_cols(df_app_raw)
    F = len(feature_cols)

    X_all, y_all = make_raw_sequence_dataset(df_app, feature_cols, "Temporal", seq_len, min_non_nan_ratio)
    print(f"\n================= {app_name.upper()} (Transformer) =================")
    print(f"[INFO] rows: {len(df_app)} | label dist: {df_app['Temporal'].value_counts().to_dict()}")
    print(f"[INFO] ORDER_COL: {order_col}")
    print(f"[INFO] sequences: {X_all.shape} | y dist: {dict(zip(*np.unique(y_all, return_counts=True)))}")
    print(f"[INFO] raw tokens = (T={seq_len}, F={F})")

    if len(y_all) < 50:
        raise RuntimeError(f"[{app_name}] Too few sequences built. Check ordering + missingness + min_non_nan_ratio.")

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    fold_rows = []

    for fold, (tr_idx, te_idx) in enumerate(skf.split(np.zeros(len(y_all)), y_all), 1):
        X_tr_full, y_tr_full = X_all[tr_idx], y_all[tr_idx]
        X_te, y_te = X_all[te_idx], y_all[te_idx]

        # inner train/val split (for early stop + threshold)
        X_tr, X_val, y_tr, y_val = train_test_split(
            X_tr_full, y_tr_full,
            test_size=val_size,
            random_state=SEED,
            stratify=y_tr_full
        )

        # ---- impute+scale TRAIN only (flatten time) ----
        Xtr_flat  = X_tr.reshape(-1, F)
        Xval_flat = X_val.reshape(-1, F)
        Xte_flat  = X_te.reshape(-1, F)

        fill, mean, std = fit_impute_scale_keepdims(Xtr_flat)

        Xtr_flat  = apply_impute_scale_keepdims(Xtr_flat,  fill, mean, std)
        Xval_flat = apply_impute_scale_keepdims(Xval_flat, fill, mean, std)
        Xte_flat  = apply_impute_scale_keepdims(Xte_flat,  fill, mean, std)

        X_tr = Xtr_flat.reshape(-1, seq_len, F).astype(np.float32)
        X_val = Xval_flat.reshape(-1, seq_len, F).astype(np.float32)
        X_te  = Xte_flat.reshape(-1, seq_len, F).astype(np.float32)

        # ---- class_weight=balanced on inner-train (guard single-class) ----
        if len(np.unique(y_tr)) < 2:
            class_weight = None
        else:
            classes = np.array([0, 1])
            cw = compute_class_weight(class_weight="balanced", classes=classes, y=y_tr)
            class_weight = {0: float(cw[0]), 1: float(cw[1])}

        model = build_rawseq_transformer(
            seq_len=seq_len, feat_dim=F,
            d_model=64, num_heads=2, key_dim=32,
            ff_dim=128, depth=2, dropout=0.15,
            lr=7e-5, weight_decay=1e-4,
            label_smoothing=label_smoothing
        )

        early = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=0)
        rlrop = ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, min_lr=1e-6, verbose=0)

        train_ds = make_tf_dataset(X_tr, y_tr, batch_size=batch_size, shuffle=True,  drop_remainder=True)
        val_ds   = make_tf_dataset(X_val, y_val, batch_size=batch_size, shuffle=False, drop_remainder=False)

        model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=epochs,
            verbose=verbose_fit,
            class_weight=class_weight,
            callbacks=[early, rlrop]
        )

        p_val = model.predict(X_val, batch_size=batch_size, verbose=0).reshape(-1)
        p_te  = model.predict(X_te,  batch_size=batch_size, verbose=0).reshape(-1)

        thr, val_f1 = tune_threshold_on_val(y_val, p_val)
        yhat = (p_te >= thr).astype(int)
        m = eval_binary(y_te, yhat)

        print(f"[FOLD {fold}] thr={thr:.2f} val_f1={val_f1:.3f}  "
              f"acc={m['acc']:.3f} f1={m['f1']:.3f} prec={m['precision']:.3f} rec={m['recall']:.3f} | cm={m['cm']}")

        fold_rows.append({
            "app": app_name,
            "fold": fold,
            "thr": thr,
            "val_f1": val_f1,
            **m,
            "n_test": int(len(y_te)),
            "seq_len": int(seq_len),
            "F": int(F),
        })

        tf.keras.backend.clear_session()

    out = pd.DataFrame(fold_rows)
    summary = {
        "app": app_name,
        "cv_mean_acc": float(out["acc"].mean()),
        "cv_mean_f1": float(out["f1"].mean()),
        "cv_mean_precision": float(out["precision"].mean()),
        "cv_mean_recall": float(out["recall"].mean()),
        "cv_std_acc": float(out["acc"].std()),
        "cv_std_f1": float(out["f1"].std()),
    }
    return out, summary


# -------------------------
# 6) Run all apps (per app)
# -------------------------
def load_metrics_csv(path, app_name=None):
    df = pd.read_csv(path)
    if app_name is not None:
        df["App"] = app_name
    return df

# Use the ACTUAL paths you have in your runtime:
df_archery = load_metrics_csv("/content/metrics_Archery.csv", "Archery")
df_phantom = load_metrics_csv("/content/metrics_PhantomLimb.csv", "PhantomLimb")
df_pt      = load_metrics_csv("/content/metrics_PianoTiles.csv", "PianoTiles")
df_puzzle  = load_metrics_csv("/content/metrics_Puzzle.csv", "Puzzle")
df_sea     = load_metrics_csv("/content/metrics_Sea.csv", "Sea")
df_war     = load_metrics_csv("/content/metrics_War.csv", "War")

apps = [
    ("Archery", df_archery),
    ("PhantomLimb", df_phantom),
    ("PianoTiles", df_pt),
    ("Puzzle", df_puzzle),
    ("Sea", df_sea),
    ("War", df_war),
]

summaries = []
all_folds = []

for app_name, df_app in apps:
    folds_df, summ = run_per_app_5fold_transformer(
        df_app, app_name,
        seq_len=5,
        min_non_nan_ratio=0.4,
        n_splits=5,
        epochs=25,
        batch_size=64,
        val_size=0.15,
        label_smoothing=0.0,
        verbose_fit=0
    )
    all_folds.append(folds_df)
    summaries.append(summ)

folds_all = pd.concat(all_folds, ignore_index=True)
summary_df = pd.DataFrame(summaries).sort_values("cv_mean_acc", ascending=False)

print("\n================= SUMMARY (PER-APP TRANSFORMER) =================")
print(summary_df[["app","cv_mean_acc","cv_mean_precision","cv_mean_recall","cv_mean_f1","cv_std_acc","cv_std_f1"]])

OUT_FOLDS = "/content/per_app_temporal_transformer_folds.csv"
OUT_SUMM  = "/content/per_app_temporal_transformer_summary.csv"
folds_all.to_csv(OUT_FOLDS, index=False)
summary_df.to_csv(OUT_SUMM, index=False)
print("\nSaved:")
print(" -", OUT_FOLDS)
print(" -", OUT_SUMM)


================= ARCHERY (Transformer) =================
[INFO] rows: 425 | label dist: {1: 304, 0: 121}
[INFO] ORDER_COL: EntryID
[INFO] sequences: (421, 5, 27) | y dist: {np.int64(0): np.int64(119), np.int64(1): np.int64(302)}
[INFO] raw tokens = (T=5, F=27)
[FOLD 1] thr=0.25 val_f1=0.974  acc=0.953 f1=0.968 prec=0.938 rec=1.000 | cm=[[20, 4], [0, 61]]
[FOLD 2] thr=0.45 val_f1=0.947  acc=0.917 f1=0.940 prec=0.965 rec=0.917 | cm=[[22, 2], [5, 55]]
[FOLD 3] thr=0.20 val_f1=0.925  acc=0.917 f1=0.945 prec=0.896 rec=1.000 | cm=[[17, 7], [0, 60]]
[FOLD 4] thr=0.20 val_f1=0.937  acc=0.929 f1=0.952 prec=0.909 rec=1.000 | cm=[[18, 6], [0, 60]]
[FOLD 5] thr=0.20 val_f1=0.935  acc=0.940 f1=0.961 prec=0.924 rec=1.000 | cm=[[18, 5], [0, 61]]

================= PHANTOMLIMB (Transformer) =================
[INFO] rows: 4105 | label dist: {0: 2663, 1: 1442}
[INFO] ORDER_COL: EntryID
[INFO] sequences: (4101, 5, 27) | y dist: {np.int64(0): np.int64(2660), np.int64(1): np.int64(1441)}
[INFO] raw token

/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:1233: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keepdims=keepdims,


[FOLD 1] thr=0.45 val_f1=0.980  acc=0.929 f1=0.951 prec=1.000 rec=0.907 | cm=[[13, 0], [4, 39]]


/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:1233: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keepdims=keepdims,


[FOLD 2] thr=0.30 val_f1=0.963  acc=0.945 f1=0.966 prec=0.933 rec=1.000 | cm=[[10, 3], [0, 42]]


...

[FOLD 3] thr=0.25 val_f1=0.981  acc=0.891 f1=0.933 prec=0.894 rec=0.977 | cm=[[7, 5], [1, 42]]


/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:1233: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keepdims=keepdims,


[FOLD 4] thr=0.40 val_f1=0.981  acc=0.945 f1=0.966 prec=0.935 rec=1.000 | cm=[[9, 3], [0, 43]]


/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:1233: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keepdims=keepdims,


[FOLD 5] thr=0.20 val_f1=0.981  acc=0.909 f1=0.945 prec=0.896 rec=1.000 | cm=[[7, 5], [0, 43]]

================= SEA (Transformer) =================
[INFO] rows: 300 | label dist: {0: 164, 1: 136}
[INFO] ORDER_COL: EntryID
[INFO] sequences: (296, 5, 27) | y dist: {np.int64(0): np.int64(162), np.int64(1): np.int64(134)}
[INFO] raw tokens = (T=5, F=27)
[FOLD 1] thr=0.55 val_f1=0.970  acc=0.950 f1=0.945 prec=0.929 rec=0.963 | cm=[[31, 2], [1, 26]]
[FOLD 2] thr=0.10 val_f1=0.941  acc=0.932 f1=0.929 prec=0.867 rec=1.000 | cm=[[29, 4], [0, 26]]
[FOLD 3] thr=0.80 val_f1=0.970  acc=0.966 f1=0.964 prec=0.931 rec=1.000 | cm=[[30, 2], [0, 27]]
[FOLD 4] thr=0.15 val_f1=0.970  acc=0.898 f1=0.900 prec=0.818 rec=1.000 | cm=[[26, 6], [0, 27]]
[FOLD 5] thr=0.80 val_f1=0.938  acc=0.949 f1=0.943 prec=0.962 rec=0.926 | cm=[[31, 1], [2, 25]]

================= WAR (Transformer) =================
[INFO] rows: 281 | label dist: {0: 165, 1: 116}
[INFO] ORDER_COL: EntryID
[INFO] sequences: (277, 5, 27) | y di

In [ ]:
# ===============================================================
# ✅ UPDATED (MINIMAL) — Prioritize ACCURACY for RAW-SEQ Transformer
# What changed (kept minimal):
#   1) Threshold tuning now maximizes VAL ACC (instead of VAL F1)
#   2) EarlyStopping / LR scheduler monitor val_accuracy (mode="max")
#      (so training is also aligned with accuracy)
# Everything else stays the same: per-app CV, inner split, keep-dims impute/scale, class_weight, etc.
# ===============================================================

import os
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.layers import Input, Dense, Dropout, LayerNormalization, MultiHeadAttention, GlobalAveragePooling1D
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"


# -------------------------
# 1) Data prep helpers
# -------------------------
def _clean_df(df):
    df = df.copy().replace("", np.nan)
    df["Temporal"] = pd.to_numeric(df["Temporal"], errors="coerce")
    df = df.dropna(subset=["Temporal"]).copy()
    df["Temporal"] = df["Temporal"].astype(int)
    return df

def _get_feature_cols(df):
    id_cols = [c for c in ["GlobalID", "EntryID", "Sequence_ID", "Person_ID", "FPS"] if c in df.columns]
    drop_cols = ["App", "Spatial", "Temporal"] + id_cols
    feature_cols = [c for c in df.columns if c not in drop_cols]

    out = df.copy()
    for c in feature_cols:
        out[c] = pd.to_numeric(out[c], errors="coerce")
    return out, feature_cols

def _order_df(df):
    order_col = None
    for c in ["EntryID", "entryid", "Timestamp", "timestamp", "Frame", "frame"]:
        if c in df.columns:
            order_col = c
            break
    if order_col is None:
        return df.reset_index(drop=True), None

    tmp = pd.to_numeric(df[order_col], errors="coerce")
    if tmp.notna().any():
        df = df.assign(_order_tmp=tmp).sort_values("_order_tmp").drop(columns=["_order_tmp"]).reset_index(drop=True)
    else:
        df = df.sort_values(order_col).reset_index(drop=True)
    return df, order_col


# -------------------------
# 2) Sliding windows (RAW seq) => X: (N,T,F), y: (N,)
# -------------------------
def window_is_valid(window, min_non_nan_ratio=0.4):
    total = window.size
    non_nan = np.isfinite(window).sum()
    return (non_nan / total) >= min_non_nan_ratio

def make_raw_sequence_dataset(app_df, feature_cols, label_col="Temporal", seq_len=5, min_non_nan_ratio=0.4):
    X = app_df[feature_cols].values.astype(float)
    y = app_df[label_col].values.astype(int)

    if len(app_df) < seq_len:
        return np.empty((0, seq_len, len(feature_cols))), np.empty((0,), dtype=int)

    X_seq, y_seq = [], []
    center = seq_len // 2
    for i in range(len(app_df) - seq_len + 1):
        window = X[i:i+seq_len]
        if not window_is_valid(window, min_non_nan_ratio=min_non_nan_ratio):
            continue
        X_seq.append(window)
        y_seq.append(y[i + center])

    if len(X_seq) == 0:
        return np.empty((0, seq_len, len(feature_cols))), np.empty((0,), dtype=int)

    return np.stack(X_seq, axis=0), np.array(y_seq, dtype=int)


# -------------------------
# 3) Metrics + threshold tuning
# -------------------------
def eval_binary(y_true, y_pred):
    return {
        "acc": float(accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "cm": confusion_matrix(y_true, y_pred, labels=[0,1]).tolist(),
    }

# ✅ ACCURACY PRIORITY: tune threshold for best VAL ACC
def tune_threshold_on_val_acc(y_val, p_val, grid=None):
    if grid is None:
        grid = np.linspace(0.05, 0.95, 19)
    best_thr, best_acc = 0.5, -1.0
    for t in grid:
        yhat = (p_val >= t).astype(int)
        acc = accuracy_score(y_val, yhat)
        if acc > best_acc:
            best_acc = acc
            best_thr = float(t)
    return best_thr, float(best_acc)


# -------------------------
# 4) Transformer model (RAW seq)
# -------------------------
def transformer_encoder(inputs, num_heads, key_dim, ff_dim, dropout=0.15):
    attn = MultiHeadAttention(num_heads=num_heads, key_dim=key_dim)(inputs, inputs)
    attn = Dropout(dropout)(attn)
    x = LayerNormalization(epsilon=1e-6)(inputs + attn)

    ffn = Dense(ff_dim, activation="relu")(x)
    ffn = Dense(x.shape[-1])(ffn)
    ffn = Dropout(dropout)(ffn)
    out = LayerNormalization(epsilon=1e-6)(x + ffn)
    return out

def build_rawseq_transformer(seq_len, feat_dim,
                            d_model=64, num_heads=2, key_dim=32,
                            ff_dim=128, depth=2, dropout=0.15,
                            lr=7e-5, weight_decay=1e-4,
                            label_smoothing=0.0):
    inp = Input(shape=(seq_len, feat_dim), name="raw_seq_in")
    x = Dense(d_model, activation="relu")(inp)

    for _ in range(depth):
        x = transformer_encoder(x, num_heads=num_heads, key_dim=key_dim, ff_dim=ff_dim, dropout=dropout)

    x = GlobalAveragePooling1D()(x)
    x = Dense(64, activation="relu")(x)
    x = Dropout(dropout)(x)
    out = Dense(1, activation="sigmoid")(x)

    model = Model(inp, out)
    opt = tf.keras.optimizers.AdamW(learning_rate=lr, weight_decay=weight_decay)
    loss = tf.keras.losses.BinaryCrossentropy(label_smoothing=label_smoothing)
    model.compile(optimizer=opt, loss=loss, metrics=["accuracy"])
    return model

def make_tf_dataset(X, y, batch_size, shuffle=True, drop_remainder=False):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if shuffle:
        ds = ds.shuffle(buffer_size=min(len(y), 5000), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(batch_size, drop_remainder=drop_remainder).prefetch(tf.data.AUTOTUNE)
    return ds


# -------------------------
# 4.5) KEEP-DIMS impute+scale (all-NaN cols -> 0.0)
# -------------------------
def fit_impute_scale_keepdims(X_train_flat):
    X = X_train_flat.astype(float, copy=True)
    fill = np.nanmedian(X, axis=0)
    fill = np.where(np.isfinite(fill), fill, 0.0)
    X = np.where(np.isfinite(X), X, fill)

    mean = X.mean(axis=0)
    std  = X.std(axis=0)
    std  = np.where(std > 0, std, 1.0)
    return fill, mean, std

def apply_impute_scale_keepdims(X_flat, fill, mean, std):
    X = X_flat.astype(float, copy=True)
    X = np.where(np.isfinite(X), X, fill)
    X = (X - mean) / std
    return X


# -------------------------
# 5) Per-app 5-fold runner
# -------------------------
def run_per_app_5fold_transformer(
    df_app_raw,
    app_name,
    seq_len=5,
    min_non_nan_ratio=0.4,
    n_splits=5,
    epochs=25,
    batch_size=64,
    val_size=0.15,
    label_smoothing=0.0,
    verbose_fit=0,
):
    df_app_raw = _clean_df(df_app_raw)
    df_app_raw, order_col = _order_df(df_app_raw)
    df_app, feature_cols = _get_feature_cols(df_app_raw)
    F = len(feature_cols)

    X_all, y_all = make_raw_sequence_dataset(df_app, feature_cols, "Temporal", seq_len, min_non_nan_ratio)
    print(f"\n================= {app_name.upper()} (Transformer) =================")
    print(f"[INFO] rows: {len(df_app)} | label dist: {df_app['Temporal'].value_counts().to_dict()}")
    print(f"[INFO] ORDER_COL: {order_col}")
    print(f"[INFO] sequences: {X_all.shape} | y dist: {dict(zip(*np.unique(y_all, return_counts=True)))}")
    print(f"[INFO] raw tokens = (T={seq_len}, F={F})")

    if len(y_all) < 50:
        raise RuntimeError(f"[{app_name}] Too few sequences built. Check ordering + missingness + min_non_nan_ratio.")

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    fold_rows = []

    for fold, (tr_idx, te_idx) in enumerate(skf.split(np.zeros(len(y_all)), y_all), 1):
        X_tr_full, y_tr_full = X_all[tr_idx], y_all[tr_idx]
        X_te, y_te = X_all[te_idx], y_all[te_idx]

        # inner train/val split (for early stop + threshold)
        X_tr, X_val, y_tr, y_val = train_test_split(
            X_tr_full, y_tr_full,
            test_size=val_size,
            random_state=SEED,
            stratify=y_tr_full
        )

        # ---- impute+scale TRAIN only (flatten time) ----
        Xtr_flat  = X_tr.reshape(-1, F)
        Xval_flat = X_val.reshape(-1, F)
        Xte_flat  = X_te.reshape(-1, F)

        fill, mean, std = fit_impute_scale_keepdims(Xtr_flat)

        Xtr_flat  = apply_impute_scale_keepdims(Xtr_flat,  fill, mean, std)
        Xval_flat = apply_impute_scale_keepdims(Xval_flat, fill, mean, std)
        Xte_flat  = apply_impute_scale_keepdims(Xte_flat,  fill, mean, std)

        X_tr = Xtr_flat.reshape(-1, seq_len, F).astype(np.float32)
        X_val = Xval_flat.reshape(-1, seq_len, F).astype(np.float32)
        X_te  = Xte_flat.reshape(-1, seq_len, F).astype(np.float32)

        # class weights on inner-train (guard single-class)
        if len(np.unique(y_tr)) < 2:
            class_weight = None
        else:
            classes = np.array([0, 1])
            cw = compute_class_weight(class_weight="balanced", classes=classes, y=y_tr)
            class_weight = {0: float(cw[0]), 1: float(cw[1])}

        model = build_rawseq_transformer(
            seq_len=seq_len, feat_dim=F,
            d_model=64, num_heads=2, key_dim=32,
            ff_dim=128, depth=2, dropout=0.15,
            lr=7e-5, weight_decay=1e-4,
            label_smoothing=label_smoothing
        )

        # ✅ ACCURACY-aligned callbacks
        early = EarlyStopping(monitor="val_accuracy", mode="max", patience=5, restore_best_weights=True, verbose=0)
        rlrop = ReduceLROnPlateau(monitor="val_accuracy", mode="max", factor=0.3, patience=2, min_lr=1e-6, verbose=0)

        train_ds = make_tf_dataset(X_tr, y_tr, batch_size=batch_size, shuffle=True,  drop_remainder=True)
        val_ds   = make_tf_dataset(X_val, y_val, batch_size=batch_size, shuffle=False, drop_remainder=False)

        model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=epochs,
            verbose=verbose_fit,
            class_weight=class_weight,
            callbacks=[early, rlrop]
        )

        p_val = model.predict(X_val, batch_size=batch_size, verbose=0).reshape(-1)
        p_te  = model.predict(X_te,  batch_size=batch_size, verbose=0).reshape(-1)

        # ✅ tune threshold for ACC (not F1)
        thr, val_acc = tune_threshold_on_val_acc(y_val, p_val)
        yhat = (p_te >= thr).astype(int)
        m = eval_binary(y_te, yhat)

        print(f"[FOLD {fold}] thr={thr:.2f} val_acc={val_acc:.3f}  "
              f"acc={m['acc']:.3f} f1={m['f1']:.3f} prec={m['precision']:.3f} rec={m['recall']:.3f} | cm={m['cm']}")

        fold_rows.append({
            "app": app_name,
            "fold": fold,
            "thr": thr,
            "val_acc": val_acc,
            **m,
            "n_test": int(len(y_te)),
            "seq_len": int(seq_len),
            "F": int(F),
        })

        tf.keras.backend.clear_session()

    out = pd.DataFrame(fold_rows)
    summary = {
        "app": app_name,
        "cv_mean_acc": float(out["acc"].mean()),
        "cv_mean_f1": float(out["f1"].mean()),
        "cv_mean_precision": float(out["precision"].mean()),
        "cv_mean_recall": float(out["recall"].mean()),
        "cv_std_acc": float(out["acc"].std()),
        "cv_std_f1": float(out["f1"].std()),
    }
    return out, summary


# -------------------------
# 6) Run all apps (per app)
# -------------------------
def load_metrics_csv(path, app_name=None):
    df = pd.read_csv(path)
    if app_name is not None:
        df["App"] = app_name
    return df

df_archery = load_metrics_csv("/content/metrics_Archery.csv", "Archery")
df_phantom = load_metrics_csv("/content/metrics_PhantomLimb.csv", "PhantomLimb")
df_pt      = load_metrics_csv("/content/metrics_PianoTiles.csv", "PianoTiles")
df_puzzle  = load_metrics_csv("/content/metrics_Puzzle.csv", "Puzzle")
df_sea     = load_metrics_csv("/content/metrics_Sea.csv", "Sea")
df_war     = load_metrics_csv("/content/metrics_War.csv", "War")

apps = [
    ("Archery", df_archery),
    ("PhantomLimb", df_phantom),
    ("PianoTiles", df_pt),
    ("Puzzle", df_puzzle),
    ("Sea", df_sea),
    ("War", df_war),
]

summaries = []
all_folds = []

for app_name, df_app in apps:
    folds_df, summ = run_per_app_5fold_transformer(
        df_app, app_name,
        seq_len=5,
        min_non_nan_ratio=0.4,
        n_splits=5,
        epochs=25,
        batch_size=64,
        val_size=0.15,
        label_smoothing=0.0,
        verbose_fit=0
    )
    all_folds.append(folds_df)
    summaries.append(summ)

folds_all = pd.concat(all_folds, ignore_index=True)
summary_df = pd.DataFrame(summaries).sort_values("cv_mean_acc", ascending=False)

print("\n================= SUMMARY (PER-APP TRANSFORMER) =================")
print(summary_df[["app","cv_mean_acc","cv_mean_precision","cv_mean_recall","cv_mean_f1","cv_std_acc","cv_std_f1"]])

OUT_FOLDS = "/content/per_app_temporal_transformer_folds_acc_priority.csv"
OUT_SUMM  = "/content/per_app_temporal_transformer_summary_acc_priority.csv"
folds_all.to_csv(OUT_FOLDS, index=False)
summary_df.to_csv(OUT_SUMM, index=False)
print("\nSaved:")
print(" -", OUT_FOLDS)
print(" -", OUT_SUMM)


================= ARCHERY (Transformer) =================
[INFO] rows: 425 | label dist: {1: 304, 0: 121}
[INFO] ORDER_COL: EntryID
[INFO] sequences: (421, 5, 27) | y dist: {np.int64(0): np.int64(119), np.int64(1): np.int64(302)}
[INFO] raw tokens = (T=5, F=27)
[FOLD 1] thr=0.50 val_acc=0.863  acc=0.847 f1=0.898 prec=0.864 rec=0.934 | cm=[[15, 9], [4, 57]]
[FOLD 2] thr=0.50 val_acc=0.882  acc=0.881 f1=0.918 prec=0.903 rec=0.933 | cm=[[18, 6], [4, 56]]
[FOLD 3] thr=0.30 val_acc=0.843  acc=0.893 f1=0.929 prec=0.881 rec=0.983 | cm=[[16, 8], [1, 59]]
[FOLD 4] thr=0.30 val_acc=0.902  acc=0.929 f1=0.952 prec=0.922 rec=0.983 | cm=[[19, 5], [1, 59]]
[FOLD 5] thr=0.30 val_acc=0.804  acc=0.881 f1=0.922 prec=0.881 rec=0.967 | cm=[[15, 8], [2, 59]]

================= PHANTOMLIMB (Transformer) =================
[INFO] rows: 4105 | label dist: {0: 2663, 1: 1442}
[INFO] ORDER_COL: EntryID
[INFO] sequences: (4101, 5, 27) | y dist: {np.int64(0): np.int64(2660), np.int64(1): np.int64(1441)}
[INFO] raw 

/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:1233: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keepdims=keepdims,


[FOLD 1] thr=0.45 val_acc=0.939  acc=0.839 f1=0.892 prec=0.925 rec=0.860 | cm=[[10, 3], [6, 37]]


/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:1233: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keepdims=keepdims,


[FOLD 2] thr=0.50 val_acc=0.912  acc=0.800 f1=0.853 prec=0.970 rec=0.762 | cm=[[12, 1], [10, 32]]


...

[FOLD 3] thr=0.25 val_acc=0.971  acc=0.964 f1=0.977 prec=0.977 rec=0.977 | cm=[[11, 1], [1, 42]]


/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:1233: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keepdims=keepdims,


[FOLD 4] thr=0.45 val_acc=0.971  acc=0.964 f1=0.977 prec=0.956 rec=1.000 | cm=[[10, 2], [0, 43]]


/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:1233: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keepdims=keepdims,


[FOLD 5] thr=0.50 val_acc=0.971  acc=0.836 f1=0.892 prec=0.925 rec=0.860 | cm=[[9, 3], [6, 37]]

================= SEA (Transformer) =================
[INFO] rows: 300 | label dist: {0: 164, 1: 136}
[INFO] ORDER_COL: EntryID
[INFO] sequences: (296, 5, 27) | y dist: {np.int64(0): np.int64(162), np.int64(1): np.int64(134)}
[INFO] raw tokens = (T=5, F=27)
[FOLD 1] thr=0.45 val_acc=0.944  acc=0.867 f1=0.867 prec=0.788 rec=0.963 | cm=[[26, 7], [1, 26]]
[FOLD 2] thr=0.35 val_acc=0.944  acc=0.898 f1=0.897 prec=0.812 rec=1.000 | cm=[[27, 6], [0, 26]]
[FOLD 3] thr=0.35 val_acc=0.944  acc=0.932 f1=0.931 prec=0.871 rec=1.000 | cm=[[28, 4], [0, 27]]
[FOLD 4] thr=0.40 val_acc=0.972  acc=0.847 f1=0.857 prec=0.750 rec=1.000 | cm=[[23, 9], [0, 27]]
[FOLD 5] thr=0.50 val_acc=0.917  acc=0.898 f1=0.889 prec=0.889 rec=0.889 | cm=[[29, 3], [3, 24]]

================= WAR (Transformer) =================
[INFO] rows: 281 | label dist: {0: 165, 1: 116}
[INFO] ORDER_COL: EntryID
[INFO] sequences: (277, 5, 27) 

In [ ]:
# ===============================================================
# PHANTOMLIMB ONLY (TRAIN) -> PHANTOMLIMB HOLDOUT (EVAL)
# Temporal Transformer on RAW SEQ (T=5, F=ALL METRIC COLS)
#
# EXACT SAME LOGIC AS YOUR PER-APP CV SCRIPT, BUT:
#   - NO CV
#   - Train on PhantomLimb train CSV only
#   - Inner train/val split ONLY from train (for EarlyStopping + threshold tuning)
#   - Apply KEEP-DIMS impute/scale FIT ON TRAIN ONLY (flatten time)
#   - Evaluate ON HOLDOUT ONCE using tuned threshold from train/val
#
# Files:
#   /content/metrics_PhantomLimb.csv
#   /content/metrics_PhantomLimb_Holdout.csv
# Outputs:
#   /content/phantomlimb_rawseq_transformer_holdout/*
# ===============================================================

import os
import json
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.layers import Input, Dense, Dropout, LayerNormalization, MultiHeadAttention, GlobalAveragePooling1D
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"

SEQ_LEN = 5
MIN_NON_NAN_RATIO = 0.4
LABEL_COL = "Temporal"

TRAIN_CSV   = "/content/metrics_PhantomLimb.csv"
HOLDOUT_CSV = "/content/metrics_PhantomLimb_Holdout.csv"

OUT_DIR = "/content/phantomlimb_rawseq_transformer_holdout"
os.makedirs(OUT_DIR, exist_ok=True)

# -------------------------
# 1) Data prep helpers (same as your code)
# -------------------------
def _clean_df(df):
    df = df.copy().replace("", np.nan)
    df[LABEL_COL] = pd.to_numeric(df[LABEL_COL], errors="coerce")
    df = df.dropna(subset=[LABEL_COL]).copy()
    df[LABEL_COL] = df[LABEL_COL].astype(int)
    return df

def _get_feature_cols(df):
    id_cols = [c for c in ["GlobalID", "EntryID", "Sequence_ID", "Person_ID", "FPS"] if c in df.columns]
    drop_cols = ["App", "Spatial", LABEL_COL] + id_cols
    feature_cols = [c for c in df.columns if c not in drop_cols]

    out = df.copy()
    for c in feature_cols:
        out[c] = pd.to_numeric(out[c], errors="coerce")
    return out, feature_cols

def _order_df(df):
    order_col = None
    for c in ["EntryID", "entryid", "Timestamp", "timestamp", "Frame", "frame"]:
        if c in df.columns:
            order_col = c
            break
    if order_col is None:
        return df.reset_index(drop=True), None

    tmp = pd.to_numeric(df[order_col], errors="coerce")
    if tmp.notna().any():
        df = df.assign(_order_tmp=tmp).sort_values("_order_tmp").drop(columns=["_order_tmp"]).reset_index(drop=True)
    else:
        df = df.sort_values(order_col).reset_index(drop=True)
    return df, order_col

# -------------------------
# 2) Sliding windows (RAW seq)
# -------------------------
def window_is_valid(window, min_non_nan_ratio=0.4):
    total = window.size
    non_nan = np.isfinite(window).sum()
    return (non_nan / total) >= min_non_nan_ratio

def make_raw_sequence_dataset(app_df, feature_cols, label_col=LABEL_COL, seq_len=5, min_non_nan_ratio=0.4):
    X = app_df[feature_cols].values.astype(float)
    y = app_df[label_col].values.astype(int)

    if len(app_df) < seq_len:
        return np.empty((0, seq_len, len(feature_cols))), np.empty((0,), dtype=int)

    X_seq, y_seq = [], []
    center = seq_len // 2
    for i in range(len(app_df) - seq_len + 1):
        window = X[i:i+seq_len]
        if not window_is_valid(window, min_non_nan_ratio=min_non_nan_ratio):
            continue
        X_seq.append(window)
        y_seq.append(y[i + center])

    if len(X_seq) == 0:
        return np.empty((0, seq_len, len(feature_cols))), np.empty((0,), dtype=int)

    return np.stack(X_seq, axis=0), np.array(y_seq, dtype=int)

# -------------------------
# 3) Metrics + threshold tuning
# -------------------------
def eval_binary(y_true, y_pred):
    return {
        "acc": float(accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "cm": confusion_matrix(y_true, y_pred, labels=[0,1]).tolist(),
    }

def tune_threshold_on_val(y_val, p_val, grid=None):
    if grid is None:
        grid = np.linspace(0.05, 0.95, 19)
    best_thr, best_f1 = 0.5, -1.0
    for t in grid:
        yhat = (p_val >= t).astype(int)
        f1 = f1_score(y_val, yhat, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_thr = float(t)
    return best_thr, float(best_f1)

# -------------------------
# 4) Transformer model (same as your code)
# -------------------------
def transformer_encoder(inputs, num_heads, key_dim, ff_dim, dropout=0.15):
    attn = MultiHeadAttention(num_heads=num_heads, key_dim=key_dim)(inputs, inputs)
    attn = Dropout(dropout)(attn)
    x = LayerNormalization(epsilon=1e-6)(inputs + attn)

    ffn = Dense(ff_dim, activation="relu")(x)
    ffn = Dense(x.shape[-1])(ffn)
    ffn = Dropout(dropout)(ffn)
    out = LayerNormalization(epsilon=1e-6)(x + ffn)
    return out

def build_rawseq_transformer(seq_len, feat_dim,
                            d_model=64, num_heads=2, key_dim=32,
                            ff_dim=128, depth=2, dropout=0.15,
                            lr=7e-5, weight_decay=1e-4,
                            label_smoothing=0.0):
    inp = Input(shape=(seq_len, feat_dim), name="raw_seq_in")
    x = Dense(d_model, activation="relu")(inp)

    for _ in range(depth):
        x = transformer_encoder(x, num_heads=num_heads, key_dim=key_dim, ff_dim=ff_dim, dropout=dropout)

    x = GlobalAveragePooling1D()(x)
    x = Dense(64, activation="relu")(x)
    x = Dropout(dropout)(x)
    out = Dense(1, activation="sigmoid")(x)

    model = Model(inp, out)
    opt = tf.keras.optimizers.AdamW(learning_rate=lr, weight_decay=weight_decay)
    loss = tf.keras.losses.BinaryCrossentropy(label_smoothing=label_smoothing)
    model.compile(optimizer=opt, loss=loss, metrics=["accuracy"])
    return model

def make_tf_dataset(X, y, batch_size, shuffle=True, drop_remainder=False):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if shuffle:
        ds = ds.shuffle(buffer_size=min(len(y), 5000), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(batch_size, drop_remainder=drop_remainder).prefetch(tf.data.AUTOTUNE)
    return ds

# -------------------------
# 4.5) KEEP-DIMS impute+scale (same as your code)
# -------------------------
def fit_impute_scale_keepdims(X_train_flat):
    X = X_train_flat.astype(float, copy=True)
    fill = np.nanmedian(X, axis=0)
    fill = np.where(np.isfinite(fill), fill, 0.0)
    X = np.where(np.isfinite(X), X, fill)

    mean = X.mean(axis=0)
    std  = X.std(axis=0)
    std  = np.where(std > 0, std, 1.0)
    return fill, mean, std

def apply_impute_scale_keepdims(X_flat, fill, mean, std):
    X = X_flat.astype(float, copy=True)
    X = np.where(np.isfinite(X), X, fill)
    X = (X - mean) / std
    return X

# -------------------------
# 5) LOAD TRAIN + HOLDOUT, build sequences, TRAIN, THRESHOLD-TUNE, EVAL
# -------------------------
df_tr_raw = pd.read_csv(TRAIN_CSV)
df_h_raw  = pd.read_csv(HOLDOUT_CSV)

df_tr_raw = _clean_df(df_tr_raw)
df_h_raw  = _clean_df(df_h_raw)

df_tr_raw, order_col_tr = _order_df(df_tr_raw)
df_h_raw,  order_col_h  = _order_df(df_h_raw)

df_tr, feature_cols = _get_feature_cols(df_tr_raw)

# enforce SAME feature cols for holdout
for c in feature_cols:
    if c not in df_h_raw.columns:
        df_h_raw[c] = np.nan
df_h, _ = _get_feature_cols(df_h_raw)  # will parse to numeric too

F = len(feature_cols)

X_tr_all, y_tr_all = make_raw_sequence_dataset(df_tr, feature_cols, LABEL_COL, SEQ_LEN, MIN_NON_NAN_RATIO)
X_h_all,  y_h_all  = make_raw_sequence_dataset(df_h,  feature_cols, LABEL_COL, SEQ_LEN, MIN_NON_NAN_RATIO)

print("\n================= PHANTOMLIMB RAWSEQ TRANSFORMER =================")
print(f"[TRAIN]   rows={len(df_tr)} order_col={order_col_tr} sequences={X_tr_all.shape} ydist={dict(zip(*np.unique(y_tr_all, return_counts=True)))} F={F}")
print(f"[HOLDOUT] rows={len(df_h)}  order_col={order_col_h}  sequences={X_h_all.shape}  ydist={dict(zip(*np.unique(y_h_all,  return_counts=True)))} F={F}")

if len(y_tr_all) < 50:
    raise RuntimeError("Too few TRAIN sequences. Check ordering + NaNs + MIN_NON_NAN_RATIO.")
if len(y_h_all) < 20:
    print("[WARN] Very few HOLDOUT sequences. Metrics may be noisy.")

# inner train/val split ONLY from TRAIN (same as CV inner split)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_tr_all, y_tr_all,
    test_size=0.15,
    random_state=SEED,
    stratify=y_tr_all
)

# ---- impute+scale FIT ON TRAIN ONLY (flatten time) ----
Xtr_flat  = X_tr.reshape(-1, F)
Xval_flat = X_val.reshape(-1, F)
Xh_flat   = X_h_all.reshape(-1, F)

fill, mean, std = fit_impute_scale_keepdims(Xtr_flat)

Xtr_flat  = apply_impute_scale_keepdims(Xtr_flat,  fill, mean, std)
Xval_flat = apply_impute_scale_keepdims(Xval_flat, fill, mean, std)
Xh_flat   = apply_impute_scale_keepdims(Xh_flat,   fill, mean, std)

X_tr = Xtr_flat.reshape(-1, SEQ_LEN, F).astype(np.float32)
X_val = Xval_flat.reshape(-1, SEQ_LEN, F).astype(np.float32)
X_h  = Xh_flat.reshape(-1, SEQ_LEN, F).astype(np.float32)

# class_weight balanced on inner-train
class_weight = None
if len(np.unique(y_tr)) == 2:
    classes = np.array([0, 1])
    cw = compute_class_weight(class_weight="balanced", classes=classes, y=y_tr)
    class_weight = {0: float(cw[0]), 1: float(cw[1])}

model = build_rawseq_transformer(
    seq_len=SEQ_LEN, feat_dim=F,
    d_model=64, num_heads=2, key_dim=32,
    ff_dim=128, depth=2, dropout=0.15,
    lr=7e-5, weight_decay=1e-4,
    label_smoothing=0.0
)

early = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=0)
rlrop = ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, min_lr=1e-6, verbose=0)

train_ds = make_tf_dataset(X_tr,  y_tr,  batch_size=64, shuffle=True,  drop_remainder=True)
val_ds   = make_tf_dataset(X_val, y_val, batch_size=64, shuffle=False, drop_remainder=False)

model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=25,
    verbose=0,
    class_weight=class_weight,
    callbacks=[early, rlrop]
)

# threshold tuning on VAL (same as your CV)
p_val = model.predict(X_val, batch_size=64, verbose=0).reshape(-1)
thr, val_f1 = tune_threshold_on_val(y_val, p_val)

# evaluate on HOLDOUT once
p_h = model.predict(X_h, batch_size=64, verbose=0).reshape(-1)
yhat_h = (p_h >= thr).astype(int)
m_h = eval_binary(y_h_all, yhat_h)

print("\n================= TRAIN -> HOLDOUT (RAWSEQ TRANSFORMER) =================")
print(f"[VAL]     tuned_thr={thr:.2f} val_f1={val_f1:.3f}")
print(f"[HOLDOUT] acc={m_h['acc']:.3f} f1={m_h['f1']:.3f} prec={m_h['precision']:.3f} rec={m_h['recall']:.3f} | cm={m_h['cm']}")

# -------------------------
# 6) Save model + preprocessing params + meta
# -------------------------
model_path = os.path.join(OUT_DIR, "model.keras")
meta_path  = os.path.join(OUT_DIR, "meta.json")

model.save(model_path)

meta = {
    "app": "PhantomLimb",
    "task": "Temporal_rawseq_transformer",
    "trained_on": TRAIN_CSV,
    "evaluated_on_holdout": HOLDOUT_CSV,
    "seed": SEED,
    "label_col": LABEL_COL,
    "seq_len": SEQ_LEN,
    "min_non_nan_ratio": float(MIN_NON_NAN_RATIO),
    "order_col_train": order_col_tr,
    "order_col_holdout": order_col_h,
    "n_train_sequences_total": int(len(y_tr_all)),
    "n_train_sequences_inner_train": int(len(y_tr)),
    "n_train_sequences_inner_val": int(len(y_val)),
    "n_holdout_sequences": int(len(y_h_all)),
    "F": int(F),
    "feature_cols": feature_cols,
    "keepdims_impute_fill": fill.tolist(),
    "keepdims_scale_mean": mean.tolist(),
    "keepdims_scale_std": std.tolist(),
    "tuned_threshold": float(thr),
    "val_f1_at_tuned_threshold": float(val_f1),
    "holdout_metrics": m_h,
    "notes": "Same per-app raw-seq transformer pipeline; trained only on PhantomLimb train; threshold tuned on inner val from train; holdout evaluated once."
}
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print("\nSaved:")
print(" -", model_path)
print(" -", meta_path)


================= PHANTOMLIMB RAWSEQ TRANSFORMER =================
[TRAIN]   rows=4105 order_col=EntryID sequences=(4101, 5, 27) ydist={np.int64(0): np.int64(2660), np.int64(1): np.int64(1441)} F=27
[HOLDOUT] rows=391  order_col=EntryID  sequences=(387, 5, 27)  ydist={np.int64(0): np.int64(286), np.int64(1): np.int64(101)} F=27

================= TRAIN -> HOLDOUT (RAWSEQ TRANSFORMER) =================
[VAL]     tuned_thr=0.60 val_f1=0.914
[HOLDOUT] acc=0.946 f1=0.894 prec=0.908 rec=0.881 | cm=[[277, 9], [12, 89]]

Saved:
 - /content/phantomlimb_rawseq_transformer_holdout/model.keras
 - /content/phantomlimb_rawseq_transformer_holdout/meta.json


In [ ]:
# ===============================================================
# CONFERENCE-CORRECT:
# PHANTOMLIMB TRAIN -> inner (train/val) from TRAIN -> HOLDOUT test once
# Temporal Transformer on RAW SEQ (T=5, F=ALL METRIC COLS)
#
# - Feature columns defined ONLY from TRAIN and reused (same order) for HOLDOUT
# - Impute+scale fit ONLY on inner-train (flatten time)
# - Threshold tuned ONLY on inner-val (optional but allowed)
# - HOLDOUT used ONLY for final evaluation
# ===============================================================

import os, json
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.layers import Input, Dense, Dropout, LayerNormalization, MultiHeadAttention, GlobalAveragePooling1D
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"

SEQ_LEN = 5
MIN_NON_NAN_RATIO = 0.4
LABEL_COL = "Temporal"

TRAIN_CSV   = "/content/metrics_PhantomLimb.csv"
HOLDOUT_CSV = "/content/metrics_PhantomLimb_Holdout.csv"

OUT_DIR = "/content/phantomlimb_rawseq_transformer_holdout_CONFERENCE"
os.makedirs(OUT_DIR, exist_ok=True)

# -------------------------
# helpers
# -------------------------
def _clean_df(df):
    df = df.copy().replace("", np.nan)
    df[LABEL_COL] = pd.to_numeric(df[LABEL_COL], errors="coerce")
    df = df.dropna(subset=[LABEL_COL]).copy()
    df[LABEL_COL] = df[LABEL_COL].astype(int)
    return df

def _order_df(df):
    order_col = None
    for c in ["EntryID", "entryid", "Timestamp", "timestamp", "Frame", "frame"]:
        if c in df.columns:
            order_col = c
            break
    if order_col is None:
        return df.reset_index(drop=True), None

    tmp = pd.to_numeric(df[order_col], errors="coerce")
    if tmp.notna().any():
        df = df.assign(_order_tmp=tmp).sort_values("_order_tmp").drop(columns=["_order_tmp"]).reset_index(drop=True)
    else:
        df = df.sort_values(order_col).reset_index(drop=True)
    return df, order_col

def _get_feature_cols_from_train(df):
    id_cols = [c for c in ["GlobalID", "EntryID", "Sequence_ID", "Person_ID", "FPS"] if c in df.columns]
    drop_cols = ["App", "Spatial", LABEL_COL] + id_cols
    feature_cols = [c for c in df.columns if c not in drop_cols]
    return feature_cols

def _coerce_features(df, feature_cols):
    out = df.copy()
    for c in feature_cols:
        if c not in out.columns:
            out[c] = np.nan
        out[c] = pd.to_numeric(out[c], errors="coerce")
    return out

def window_is_valid(window, min_non_nan_ratio=0.4):
    total = window.size
    non_nan = np.isfinite(window).sum()
    return (non_nan / total) >= min_non_nan_ratio

def make_raw_sequence_dataset(app_df, feature_cols, label_col=LABEL_COL, seq_len=5, min_non_nan_ratio=0.4):
    X = app_df[feature_cols].values.astype(float)
    y = app_df[label_col].values.astype(int)

    if len(app_df) < seq_len:
        return np.empty((0, seq_len, len(feature_cols))), np.empty((0,), dtype=int)

    X_seq, y_seq = [], []
    center = seq_len // 2
    for i in range(len(app_df) - seq_len + 1):
        window = X[i:i+seq_len]
        if not window_is_valid(window, min_non_nan_ratio=min_non_nan_ratio):
            continue
        X_seq.append(window)
        y_seq.append(y[i + center])

    if len(X_seq) == 0:
        return np.empty((0, seq_len, len(feature_cols))), np.empty((0,), dtype=int)

    return np.stack(X_seq, axis=0), np.array(y_seq, dtype=int)

def eval_binary(y_true, y_pred):
    return {
        "acc": float(accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "cm": confusion_matrix(y_true, y_pred, labels=[0,1]).tolist(),
    }

def tune_threshold_on_val(y_val, p_val, grid=None):
    if grid is None:
        grid = np.linspace(0.05, 0.95, 19)
    best_thr, best_f1 = 0.5, -1.0
    for t in grid:
        yhat = (p_val >= t).astype(int)
        f1 = f1_score(y_val, yhat, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_thr = float(t)
    return best_thr, float(best_f1)

def transformer_encoder(inputs, num_heads, key_dim, ff_dim, dropout=0.15):
    attn = MultiHeadAttention(num_heads=num_heads, key_dim=key_dim)(inputs, inputs)
    attn = Dropout(dropout)(attn)
    x = LayerNormalization(epsilon=1e-6)(inputs + attn)

    ffn = Dense(ff_dim, activation="relu")(x)
    ffn = Dense(x.shape[-1])(ffn)
    ffn = Dropout(dropout)(ffn)
    return LayerNormalization(epsilon=1e-6)(x + ffn)

def build_rawseq_transformer(seq_len, feat_dim,
                            d_model=64, num_heads=2, key_dim=32,
                            ff_dim=128, depth=2, dropout=0.15,
                            lr=7e-5, weight_decay=1e-4,
                            label_smoothing=0.0):
    inp = Input(shape=(seq_len, feat_dim), name="raw_seq_in")
    x = Dense(d_model, activation="relu")(inp)
    for _ in range(depth):
        x = transformer_encoder(x, num_heads=num_heads, key_dim=key_dim, ff_dim=ff_dim, dropout=dropout)
    x = GlobalAveragePooling1D()(x)
    x = Dense(64, activation="relu")(x)
    x = Dropout(dropout)(x)
    out = Dense(1, activation="sigmoid")(x)

    model = Model(inp, out)
    opt = tf.keras.optimizers.AdamW(learning_rate=lr, weight_decay=weight_decay)
    loss = tf.keras.losses.BinaryCrossentropy(label_smoothing=label_smoothing)
    model.compile(optimizer=opt, loss=loss, metrics=["accuracy"])
    return model

def make_tf_dataset(X, y, batch_size, shuffle=True, drop_remainder=False):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if shuffle:
        ds = ds.shuffle(buffer_size=min(len(y), 5000), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(batch_size, drop_remainder=drop_remainder).prefetch(tf.data.AUTOTUNE)
    return ds

def fit_impute_scale_keepdims(X_train_flat):
    X = X_train_flat.astype(float, copy=True)
    fill = np.nanmedian(X, axis=0)
    fill = np.where(np.isfinite(fill), fill, 0.0)
    X = np.where(np.isfinite(X), X, fill)

    mean = X.mean(axis=0)
    std  = X.std(axis=0)
    std  = np.where(std > 0, std, 1.0)
    return fill, mean, std

def apply_impute_scale_keepdims(X_flat, fill, mean, std):
    X = X_flat.astype(float, copy=True)
    X = np.where(np.isfinite(X), X, fill)
    return (X - mean) / std

# -------------------------
# LOAD + PREP
# -------------------------
df_tr_raw = _order_df(_clean_df(pd.read_csv(TRAIN_CSV)))[0]
df_h_raw  = _order_df(_clean_df(pd.read_csv(HOLDOUT_CSV)))[0]
df_tr_raw, order_col_tr = _order_df(df_tr_raw)
df_h_raw,  order_col_h  = _order_df(df_h_raw)

feature_cols = _get_feature_cols_from_train(df_tr_raw)
df_tr = _coerce_features(df_tr_raw, feature_cols)
df_h  = _coerce_features(df_h_raw,  feature_cols)
F = len(feature_cols)

X_tr_all, y_tr_all = make_raw_sequence_dataset(df_tr, feature_cols, LABEL_COL, SEQ_LEN, MIN_NON_NAN_RATIO)
X_h_all,  y_h_all  = make_raw_sequence_dataset(df_h,  feature_cols, LABEL_COL, SEQ_LEN, MIN_NON_NAN_RATIO)

print("\n================= PHANTOMLIMB RAWSEQ TRANSFORMER (CONFERENCE) =================")
print(f"[TRAIN]   order_col={order_col_tr} sequences={X_tr_all.shape} ydist={dict(zip(*np.unique(y_tr_all, return_counts=True)))} F={F}")
print(f"[HOLDOUT] order_col={order_col_h}  sequences={X_h_all.shape}  ydist={dict(zip(*np.unique(y_h_all,  return_counts=True)))} F={F}")

# inner split ONLY from TRAIN
X_tr, X_val, y_tr, y_val = train_test_split(
    X_tr_all, y_tr_all, test_size=0.15, random_state=SEED, stratify=y_tr_all
)

# impute+scale fit ONLY on inner-train
Xtr_flat  = X_tr.reshape(-1, F)
Xval_flat = X_val.reshape(-1, F)
Xh_flat   = X_h_all.reshape(-1, F)

fill, mean, std = fit_impute_scale_keepdims(Xtr_flat)
Xtr_flat  = apply_impute_scale_keepdims(Xtr_flat,  fill, mean, std)
Xval_flat = apply_impute_scale_keepdims(Xval_flat, fill, mean, std)
Xh_flat   = apply_impute_scale_keepdims(Xh_flat,   fill, mean, std)

X_tr = Xtr_flat.reshape(-1, SEQ_LEN, F).astype(np.float32)
X_val = Xval_flat.reshape(-1, SEQ_LEN, F).astype(np.float32)
X_h  = Xh_flat.reshape(-1, SEQ_LEN, F).astype(np.float32)

# class_weight balanced on inner-train
class_weight = None
if len(np.unique(y_tr)) == 2:
    cw = compute_class_weight(class_weight="balanced", classes=np.array([0,1]), y=y_tr)
    class_weight = {0: float(cw[0]), 1: float(cw[1])}

model = build_rawseq_transformer(seq_len=SEQ_LEN, feat_dim=F)

early = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=0)
rlrop = ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, min_lr=1e-6, verbose=0)

train_ds = make_tf_dataset(X_tr,  y_tr,  batch_size=64, shuffle=True,  drop_remainder=True)
val_ds   = make_tf_dataset(X_val, y_val, batch_size=64, shuffle=False, drop_remainder=False)

model.fit(train_ds, validation_data=val_ds, epochs=25, verbose=0,
          class_weight=class_weight, callbacks=[early, rlrop])

# threshold tuned ONLY on val (train-derived)
p_val = model.predict(X_val, batch_size=64, verbose=0).reshape(-1)
thr, val_f1 = tune_threshold_on_val(y_val, p_val)

# evaluate ON HOLDOUT ONCE
p_h = model.predict(X_h, batch_size=64, verbose=0).reshape(-1)
yhat_h = (p_h >= thr).astype(int)
m_h = eval_binary(y_h_all, yhat_h)

print("\n================= TRAIN -> HOLDOUT (RAWSEQ TRANSFORMER, CONFERENCE) =================")
print(f"[VAL]     tuned_thr={thr:.2f} val_f1={val_f1:.3f}")
print(f"[HOLDOUT] acc={m_h['acc']:.3f} f1={m_h['f1']:.3f} prec={m_h['precision']:.3f} rec={m_h['recall']:.3f} | cm={m_h['cm']}")

# save
model_path = os.path.join(OUT_DIR, "model.keras")
meta_path  = os.path.join(OUT_DIR, "meta.json")
model.save(model_path)

meta = {
    "app": "PhantomLimb",
    "task": "Temporal_rawseq_transformer",
    "protocol": "Train on train split only; tune threshold on train-derived val; evaluate once on holdout.",
    "trained_on": TRAIN_CSV,
    "evaluated_on_holdout": HOLDOUT_CSV,
    "seed": SEED,
    "seq_len": SEQ_LEN,
    "min_non_nan_ratio": float(MIN_NON_NAN_RATIO),
    "F": int(F),
    "feature_cols": feature_cols,
    "tuned_threshold": float(thr),
    "val_f1_at_tuned_threshold": float(val_f1),
    "holdout_metrics": m_h
}
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print("\nSaved:")
print(" -", model_path)
print(" -", meta_path)


================= PHANTOMLIMB RAWSEQ TRANSFORMER (CONFERENCE) =================
[TRAIN]   order_col=EntryID sequences=(4101, 5, 27) ydist={np.int64(0): np.int64(2660), np.int64(1): np.int64(1441)} F=27
[HOLDOUT] order_col=EntryID  sequences=(387, 5, 27)  ydist={np.int64(0): np.int64(286), np.int64(1): np.int64(101)} F=27

================= TRAIN -> HOLDOUT (RAWSEQ TRANSFORMER, CONFERENCE) =================
[VAL]     tuned_thr=0.55 val_f1=0.928
[HOLDOUT] acc=0.747 f1=0.655 prec=0.508 rec=0.921 | cm=[[196, 90], [8, 93]]

Saved:
 - /content/phantomlimb_rawseq_transformer_holdout_CONFERENCE/model.keras
 - /content/phantomlimb_rawseq_transformer_holdout_CONFERENCE/meta.json


# -Spatial

## HGB/ XGB/ RF - metrics only

In [ ]:
# ============================================================
# PER-APP (within each app) SPATIAL (frame-level) study
# Runs TOP-3 spatial models: RF, HGB, XGBoost
# - Same style as your per-app temporal code:
#   * per-app 5-fold StratifiedKFold
#   * pick "best fold" by (accuracy, then f1)
#   * save best model + meta.json per app + per model
# - Uses Spatial label (frame-level) and feature columns from each app CSV
# - NO SMOTE / NO OVERSAMPLING
# ============================================================

import os, json
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

SEED = 42
np.random.seed(SEED)

LABEL_COL = "Spatial"

# --------------------------
# Feature selection policy (choose ONE)
# --------------------------
USE_ALL_FEATURES_EXCEPT_META = True   # matches your LOAO patch logic
USE_INVARIANT27_ONLY = False          # set True ONLY if your per-app CSVs contain these exact columns

INVARIANT27 = [
    "missing_joints_count","missing_joints_ratio","collapsed_joints_count",
    "center_of_mass_x","center_of_mass_y","center_of_mass_z","distance_from_origin",
    "bbox_width","bbox_height","bbox_depth","bbox_volume","max_joint_distance_from_com",
    "distance_from_floor","min_foot_height_above_floor","below_floor",
    "left_forearm_length","right_forearm_length","left_shin_length","right_shin_length",
    "arm_length_symmetry","leg_length_symmetry",
    "body_upright_x","body_upright_y","body_upright_z",
    "body_forward_x","body_forward_y","body_forward_z"
]

# --------------------------
# Optional: if PhantomLimb per-app CSV is huge and you want comparable N
# --------------------------
DOWNSAMPLE_PHANTOMLIMB = False
TARGET_N_PL = 400

# --------------------------
# Models (top 3)
# --------------------------
def make_rf():
    return Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", RandomForestClassifier(
            n_estimators=400,
            max_depth=None,
            min_samples_leaf=5,
            max_features="sqrt",
            class_weight="balanced",
            random_state=SEED,
            n_jobs=-1
        ))
    ])

def make_hgb():
    return Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", HistGradientBoostingClassifier(
            random_state=SEED,
            max_depth=6,
            learning_rate=0.05,
            max_iter=400
        ))
    ])

def make_xgb():
    try:
        import xgboost as xgb
    except Exception as e:
        raise RuntimeError(
            "XGBoost is not available. Install with: !pip -q install xgboost\n"
            f"Import error: {repr(e)}"
        )
    return Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", xgb.XGBClassifier(
            random_state=SEED,
            n_estimators=1200,
            learning_rate=0.03,
            max_depth=6,
            subsample=0.9,
            colsample_bytree=0.9,
            reg_lambda=1.0,
            reg_alpha=0.0,
            min_child_weight=1.0,
            gamma=0.0,
            tree_method="hist",
            objective="binary:logistic",
            eval_metric="logloss",
            n_jobs=-1
        ))
    ])

MODELS = {
    "RF": make_rf,
    "HGB": make_hgb,
    "XGBoost": make_xgb,
}

# --------------------------
# Metrics helper
# --------------------------
def eval_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true, y_pred, zero_division=0)
    f1   = f1_score(y_true, y_pred, zero_division=0)
    cm   = confusion_matrix(y_true, y_pred, labels=[0,1]).tolist()
    return acc, f1, prec, rec, cm

# --------------------------
# Per-app runner (frame-level)
# --------------------------
def run_app_spatial_from_metrics_csv(app_name, csv_path, save_root):
    df = pd.read_csv(csv_path).replace("", np.nan)

    if LABEL_COL not in df.columns:
        raise RuntimeError(f"[{app_name}] CSV missing label col '{LABEL_COL}'")

    # label
    df[LABEL_COL] = pd.to_numeric(df[LABEL_COL], errors="coerce")
    df = df.dropna(subset=[LABEL_COL]).copy()
    df[LABEL_COL] = df[LABEL_COL].astype(int)

    # optional downsample
    if DOWNSAMPLE_PHANTOMLIMB and app_name.lower() == "phantomlimb" and len(df) > TARGET_N_PL:
        df = df.sample(TARGET_N_PL, random_state=SEED).copy()
        df[LABEL_COL] = df[LABEL_COL].astype(int)

    # feature columns
    id_cols = [c for c in ["GlobalID", "EntryID", "entryid", "Timestamp", "timestamp", "Frame", "frame"] if c in df.columns]
    drop_cols = set(["App", "Temporal", LABEL_COL] + id_cols)

    if USE_INVARIANT27_ONLY:
        missing = [c for c in INVARIANT27 if c not in df.columns]
        if missing:
            raise RuntimeError(f"[{app_name}] missing INVARIANT27 cols: {missing[:10]}")
        feature_cols = INVARIANT27[:]
    else:
        feature_cols = [c for c in df.columns if c not in drop_cols]

    # coerce features numeric
    for c in feature_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    X_all = df[feature_cols].to_numpy(dtype=float)
    y_all = df[LABEL_COL].to_numpy(dtype=int)

    print(f"\n================= {app_name.upper()} (SPATIAL frame) =================")
    print("[INFO] rows:", len(df), "| label dist:", pd.Series(y_all).value_counts().to_dict())
    print("[INFO] features:", len(feature_cols), "| first 15:", feature_cols[:15])

    if len(y_all) < 50:
        raise RuntimeError(f"[{app_name}] too few rows for 5-fold CV after filtering.")

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

    app_out = []

    for model_name, factory in MODELS.items():
        model = factory()

        fold_rows = []
        best_key = (-1.0, -1.0)     # (acc, f1)
        best_fold = None
        best_model = None

        for fold, (tr, te) in enumerate(skf.split(X_all, y_all), 1):
            m = factory()
            m.fit(X_all[tr], y_all[tr])
            yhat = m.predict(X_all[te])

            acc, f1, prec, rec, cm = eval_metrics(y_all[te], yhat)
            fold_rows.append({
                "app": app_name, "label": LABEL_COL, "task": "frame",
                "model": model_name, "fold": fold,
                "accuracy": float(acc), "f1": float(f1),
                "precision": float(prec), "recall": float(rec),
                "cm": cm
            })
            print(f"[{app_name} | {model_name} | FOLD {fold}] "
                  f"acc={acc:.3f} f1={f1:.3f} prec={prec:.3f} rec={rec:.3f} | cm={cm}")

            key = (acc, f1)
            if key > best_key:
                best_key = key
                best_fold = fold
                best_model = m

        # save
        save_dir = os.path.join(save_root, f"saved_{app_name.lower()}_spatial_{model_name.lower()}_5fold")
        os.makedirs(save_dir, exist_ok=True)
        joblib.dump(best_model, os.path.join(save_dir, "model.joblib"))

        meta = {
            "app": app_name,
            "task": "Spatial_frame",
            "model": model_name,
            "seed": SEED,
            "label_col": LABEL_COL,
            "feature_policy": ("INVARIANT27_ONLY" if USE_INVARIANT27_ONLY else "ALL_FEATURES_EXCEPT_META"),
            "feature_cols": feature_cols,
            "frame_F": int(len(feature_cols)),
            "cv_n_splits": 5,
            "fold_metrics": fold_rows,
            "best_fold": int(best_fold),
            "best_key_(acc,f1)": [float(best_key[0]), float(best_key[1])],
            "notes": "Per-app Spatial (frame) study; choose best fold by (accuracy, f1). No SMOTE."
        }
        with open(os.path.join(save_dir, "meta.json"), "w") as f:
            json.dump(meta, f, indent=2)

        # summary row (for easy final table)
        app_out.append({
            "app": app_name,
            "model": model_name,
            "best_fold": int(best_fold),
            "best_acc": float(best_key[0]),
            "best_f1": float(best_key[1]),
            "cv_mean_acc": float(np.mean([r["accuracy"] for r in fold_rows])),
            "cv_mean_f1": float(np.mean([r["f1"] for r in fold_rows])),
            "cv_mean_precision": float(np.mean([r["precision"] for r in fold_rows])),
            "cv_mean_recall": float(np.mean([r["recall"] for r in fold_rows])),
            "saved_dir": save_dir
        })

        print(f"[SAVE] {app_name} | {model_name} -> {save_dir}")

    return pd.DataFrame(app_out)

# --------------------------
# RUN ALL 6 APPS (per-app CSVs)
# --------------------------
apps = {
    "Archery":      "/content/metrics_Archery.csv",
    "PhantomLimb":  "/content/metrics_PhantomLimb.csv",
    "PianoTiles":   "/content/metrics_PianoTiles.csv",
    "Puzzle":       "/content/metrics_Puzzle.csv",
    "Sea":          "/content/metrics_Sea.csv",
    "War":          "/content/metrics_War.csv",
}

SAVE_ROOT = "/content/per_app_spatial_top3_models"
os.makedirs(SAVE_ROOT, exist_ok=True)

all_summaries = []
for app, path in apps.items():
    df_sum = run_app_spatial_from_metrics_csv(app, path, SAVE_ROOT)
    all_summaries.append(df_sum)

summary_df = pd.concat(all_summaries, ignore_index=True)

print("\n================= SUMMARY (per app × model) =================")
print(summary_df.sort_values(["app","cv_mean_acc"], ascending=[True, False]).to_string(index=False))

print("\n================= BEST MODEL PER APP (by cv_mean_acc then cv_mean_f1) =================")
best_per_app = (summary_df
                .sort_values(["app","cv_mean_acc","cv_mean_f1"], ascending=[True, False, False])
                .groupby("app", as_index=False)
                .head(1))
print(best_per_app.to_string(index=False))

OUT_PATH = "/content/per_app_spatial_top3_summary.csv"
summary_df.to_csv(OUT_PATH, index=False)
print("\nSaved:", OUT_PATH)
print("Saved models/meta under:", SAVE_ROOT)


================= ARCHERY (SPATIAL frame) =================
[INFO] rows: 425 | label dist: {1: 342, 0: 83}
[INFO] features: 27 | first 15: ['missing_joints_count', 'missing_joints_ratio', 'collapsed_joints_count', 'center_of_mass_x', 'center_of_mass_y', 'center_of_mass_z', 'distance_from_origin', 'bbox_width', 'bbox_height', 'bbox_depth', 'bbox_volume', 'max_joint_distance_from_com', 'distance_from_floor', 'min_foot_height_above_floor', 'below_floor']
[Archery | RF | FOLD 1] acc=0.976 f1=0.985 prec=1.000 rec=0.971 | cm=[[17, 0], [2, 66]]
[Archery | RF | FOLD 2] acc=0.988 f1=0.993 prec=1.000 rec=0.985 | cm=[[17, 0], [1, 67]]
[Archery | RF | FOLD 3] acc=0.976 f1=0.985 prec=0.985 rec=0.985 | cm=[[16, 1], [1, 67]]
[Archery | RF | FOLD 4] acc=0.965 f1=0.978 prec=0.971 rec=0.986 | cm=[[14, 2], [1, 68]]
[Archery | RF | FOLD 5] acc=0.929 f1=0.957 prec=0.957 rec=0.957 | cm=[[13, 3], [3, 66]]
[SAVE] Archery | RF -> /content/per_app_spatial_top3_models/saved_archery_spatial_rf_5fold
[Archery | H

/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [12 13 14]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [12 13 14]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [12 13 14]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


[Puzzle | RF | FOLD 1] acc=0.983 f1=0.988 prec=1.000 rec=0.976 | cm=[[18, 0], [1, 41]]


/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [12 13 14]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [12 13 14]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


[Puzzle | RF | FOLD 2] acc=0.983 f1=0.989 prec=0.977 rec=1.000 | cm=[[16, 1], [0, 43]]


...

[Puzzle | XGBoost | FOLD 3] acc=0.933 f1=0.956 prec=0.915 rec=1.000 | cm=[[13, 4], [0, 43]]


/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [12 13 14]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [12 13 14]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


[Puzzle | XGBoost | FOLD 4] acc=0.950 f1=0.966 prec=0.935 rec=1.000 | cm=[[14, 3], [0, 43]]


/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [12 13 14]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


[Puzzle | XGBoost | FOLD 5] acc=0.967 f1=0.977 prec=0.956 rec=1.000 | cm=[[15, 2], [0, 43]]
[SAVE] Puzzle | XGBoost -> /content/per_app_spatial_top3_models/saved_puzzle_spatial_xgboost_5fold

================= SEA (SPATIAL frame) =================
[INFO] rows: 300 | label dist: {0: 168, 1: 132}
[INFO] features: 27 | first 15: ['missing_joints_count', 'missing_joints_ratio', 'collapsed_joints_count', 'center_of_mass_x', 'center_of_mass_y', 'center_of_mass_z', 'distance_from_origin', 'bbox_width', 'bbox_height', 'bbox_depth', 'bbox_volume', 'max_joint_distance_from_com', 'distance_from_floor', 'min_foot_height_above_floor', 'below_floor']
[Sea | RF | FOLD 1] acc=1.000 f1=1.000 prec=1.000 rec=1.000 | cm=[[34, 0], [0, 26]]
[Sea | RF | FOLD 2] acc=0.983 f1=0.980 prec=1.000 rec=0.962 | cm=[[34, 0], [1, 25]]
[Sea | RF | FOLD 3] acc=0.950 f1=0.945 prec=0.897 rec=1.000 | cm=[[31, 3], [0, 26]]
[Sea | RF | FOLD 4] acc=0.950 f1=0.947 prec=0.900 rec=1.000 | cm=[[30, 3], [0, 27]]
[Sea | RF | FOLD 5]

In [ ]:
# ============================================================
# PHANTOMLIMB SPATIAL — HOLDOUT EVALUATION
# Uses trained per-app models (RF / HGB / XGBoost)
# Evaluates ONCE on metrics_PhantomLimb_Holdout.csv
# ============================================================

import os
import numpy as np
import pandas as pd
import joblib

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

SEED = 42
LABEL_COL = "Spatial"

HOLDOUT_CSV = "/content/metrics_PhantomLimb_Holdout.csv"
MODEL_ROOT = "/content/per_app_spatial_top3_models"

MODELS = {
    "RF":      f"{MODEL_ROOT}/saved_phantomlimb_spatial_rf_5fold/model.joblib",
    "HGB":     f"{MODEL_ROOT}/saved_phantomlimb_spatial_hgb_5fold/model.joblib",
    "XGBoost": f"{MODEL_ROOT}/saved_phantomlimb_spatial_xgboost_5fold/model.joblib",
}

# --------------------------
# Load holdout
# --------------------------
df = pd.read_csv(HOLDOUT_CSV).replace("", np.nan)

if LABEL_COL not in df.columns:
    raise RuntimeError("Holdout CSV missing Spatial label")

df[LABEL_COL] = pd.to_numeric(df[LABEL_COL], errors="coerce")
df = df.dropna(subset=[LABEL_COL]).copy()
df[LABEL_COL] = df[LABEL_COL].astype(int)

# Feature selection (must match training!)
id_cols = [c for c in ["GlobalID", "EntryID", "entryid", "Timestamp", "timestamp", "Frame", "frame"] if c in df.columns]
drop_cols = set(["App", "Temporal", LABEL_COL] + id_cols)
feature_cols = [c for c in df.columns if c not in drop_cols]

for c in feature_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

X = df[feature_cols].values
y = df[LABEL_COL].values

print("\n================= PHANTOMLIMB HOLDOUT =================")
print("Rows:", len(df))
print("Label dist:", pd.Series(y).value_counts().to_dict())
print("Features:", len(feature_cols))

# --------------------------
# Eval helper
# --------------------------
def eval_metrics(y_true, y_pred):
    return {
        "acc": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "cm": confusion_matrix(y_true, y_pred).tolist(),
    }

# --------------------------
# Run evaluation
# --------------------------
rows = []
for name, path in MODELS.items():
    model = joblib.load(path)
    yhat = model.predict(X)
    m = eval_metrics(y, yhat)

    rows.append({
        "Model": name,
        "Accuracy": round(m["acc"] * 100, 1),
        "Precision": round(m["precision"] * 100, 1),
        "Recall": round(m["recall"] * 100, 1),
        "F1": round(m["f1"] * 100, 1),
        "CM": m["cm"]
    })

    print(f"[{name}] "
          f"acc={m['acc']:.3f} prec={m['precision']:.3f} "
          f"rec={m['recall']:.3f} f1={m['f1']:.3f} "
          f"cm={m['cm']}")

results_df = pd.DataFrame(rows)
print("\n================= HOLDOUT SUMMARY =================")
print(results_df)

OUT = "/content/phantomlimb_spatial_holdout_results.csv"
results_df.to_csv(OUT, index=False)
print("\nSaved:", OUT)


================= PHANTOMLIMB HOLDOUT =================
Rows: 391
Label dist: {1: 214, 0: 177}
Features: 27
[RF] acc=0.611 prec=0.626 rec=0.720 f1=0.670 cm=[[85, 92], [60, 154]]
[HGB] acc=0.639 prec=0.644 rec=0.762 f1=0.698 cm=[[87, 90], [51, 163]]
[XGBoost] acc=0.606 prec=0.625 rec=0.701 f1=0.661 cm=[[87, 90], [64, 150]]

================= HOLDOUT SUMMARY =================
     Model  Accuracy  Precision  Recall    F1                     CM
0       RF      61.1       62.6    72.0  67.0  [[85, 92], [60, 154]]
1      HGB      63.9       64.4    76.2  69.8  [[87, 90], [51, 163]]
2  XGBoost      60.6       62.5    70.1  66.1  [[87, 90], [64, 150]]

Saved: /content/phantomlimb_spatial_holdout_results.csv


## Image + Metrics(XGB)

In [ ]:
# ============================================================
# ✅ PER-APP SPATIAL HYBRID: Image (.npy stacks) + Invariant27 + HGB(teacher)
#   - Spatial label (frame-level)
#   - No raw coordinates
#   - Uses your existing per-app metrics CSVs + per-app .npy image stacks on Drive
#   - 5-fold Stratified CV per app
#   - Outputs table-ready mean accuracy/precision/recall/f1 per app
# ============================================================

import os, numpy as np, pandas as pd
import cv2 as cv

from google.colab import drive

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingClassifier

import tensorflow as tf
from tensorflow.keras.layers import (
    Input, Dense, Flatten, Conv2D, MaxPooling2D, LayerNormalization, Dropout,
    MultiHeadAttention, Concatenate, Lambda, GlobalAveragePooling1D
)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# -----------------------
# 0) Config
# -----------------------
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

LABEL_COL = "Spatial"
IMG_HW = (112, 112)
BATCH_SIZE = 16
EPOCHS = 10

INVARIANT27 = [
    "missing_joints_count","missing_joints_ratio","collapsed_joints_count",
    "center_of_mass_x","center_of_mass_y","center_of_mass_z","distance_from_origin",
    "bbox_width","bbox_height","bbox_depth","bbox_volume","max_joint_distance_from_com",
    "distance_from_floor","min_foot_height_above_floor","below_floor",
    "left_forearm_length","right_forearm_length","left_shin_length","right_shin_length",
    "arm_length_symmetry","leg_length_symmetry",
    "body_upright_x","body_upright_y","body_upright_z",
    "body_forward_x","body_forward_y","body_forward_z"
]

# -----------------------
# 1) Paths (metrics CSVs on /content) + image .npy stacks on Drive
# -----------------------
drive.mount("/content/drive", force_remount=False)

APPS = {
    "PianoTiles":   "/content/metrics_PianoTiles.csv",
    "Archery":      "/content/metrics_Archery.csv",
    "Puzzle":       "/content/metrics_Puzzle.csv",
    "Sea":          "/content/metrics_Sea.csv",
    "War":          "/content/metrics_War.csv",
    "PhantomLimb":  "/content/metrics_PhantomLimb.csv",
}

# Image stacks (exact ones you used before)
# NOTE: Archery is 3 chunks; PhantomLimb is BIG + many chunks; others single.
IMG_STACKS = {
    "PianoTiles": [
        "/content/drive/MyDrive/VR_AR_testing/Data_Logger_and_Scripts/processed_img/img_grey_PT_p1_5fps.npy",
    ],
    "Archery": [
        "/content/drive/MyDrive/VR_Archery/archery01/processed_img/img_grey_AR_p1_5fps_01.npy",
        "/content/drive/MyDrive/VR_Archery/archery02/processed_img/img_grey_AR_p1_5fps_02.npy",
        "/content/drive/MyDrive/VR_Archery/archery03/processed_img/img_grey_AR_p1_5fps_03.npy",
    ],
    "Puzzle": [
        "/content/drive/MyDrive/VR_Puzzle/puzzle02/processed_img/img_grey_PUZ_p1_5fps_02.npy",
    ],
    "Sea": [
        "/content/drive/MyDrive/VR_GameOver_Sea/gameover_sea01/processed_img/img_grey_GO_SEA_p1_5fps_01.npy",
    ],
    "War": [
        "/content/drive/MyDrive/VR_GameOver_War/gameover_war01/processed_img/img_grey_GO_WAR_p1_5fps_01.npy",
    ],
    "PhantomLimb": [
        # BIG (main)
        "/content/drive/MyDrive/VR_AR_testing/Data_Logger_and_Scripts/processed_img/preprocessed_images_grey.npy",
        # extras (train)
        "/content/drive/MyDrive/VR_AR_testing/Data_Logger_and_Scripts/processed_img/img_grey_PL_p1_5fps.npy",
        "/content/drive/MyDrive/VR_AR_testing/Data_Logger_and_Scripts/processed_img/img_grey_PL_p1_1fps.npy",
        "/content/drive/MyDrive/VR_AR_testing/Data_Logger_and_Scripts/processed_img/img_grey_PL_p1_3fps.npy",
        "/content/drive/MyDrive/VR_AR_testing/Data_Logger_and_Scripts/processed_img/img_grey_PL_p2_5fps.npy",
        "/content/drive/MyDrive/VR_AR_testing/Data_Logger_and_Scripts/processed_img/img_grey_PL_p2_1fps.npy",
        "/content/drive/MyDrive/VR_AR_testing/Data_Logger_and_Scripts/processed_img/img_grey_PL_p2_3fps.npy",
        "/content/drive/MyDrive/VR_AR_testing/Data_Logger_and_Scripts/processed_img/img_grey_PL_p3_5fps.npy",
        "/content/drive/MyDrive/VR_AR_testing/Data_Logger_and_Scripts/processed_img/img_grey_PL_p4_5fps.npy",
        "/content/drive/MyDrive/VR_AR_testing/Data_Logger_and_Scripts/processed_img/img_grey_PL_p4_3fps.npy",
        "/content/drive/MyDrive/VR_AR_testing/Data_Logger_and_Scripts/processed_img/img_grey_PL_p4_1fps.npy",
        "/content/drive/MyDrive/VR_AR_testing/Data_Logger_and_Scripts/processed_img/img_grey_PL_p5_5fps.npy",
        "/content/drive/MyDrive/VR_AR_testing/Data_Logger_and_Scripts/processed_img/img_grey_PL_p5_3fps.npy",
        "/content/drive/MyDrive/VR_AR_testing/Data_Logger_and_Scripts/processed_img/img_grey_PL_p5_1fps.npy",
        "/content/drive/MyDrive/VR_AR_testing/Data_Logger_and_Scripts/processed_img/img_grey_PL_p6_5fps.npy",
        "/content/drive/MyDrive/VR_AR_testing/Data_Logger_and_Scripts/processed_img/img_grey_PL_p6_3fps.npy",
        "/content/drive/MyDrive/VR_AR_testing/Data_Logger_and_Scripts/processed_img/img_grey_PL_p6_1fps.npy",
    ],
}

# Optional: PhantomLimb holdout (if you want later)
PHANTOM_HOLDOUT_CSV = "/content/metrics_PhantomLimb_Holdout.csv"
PHANTOM_HOLDOUT_STACKS = [
    "/content/drive/MyDrive/VR_AR_testing/Data_Logger_and_Scripts/processed_img/img_grey_PL_H_p1_5fps-black.npy",
    "/content/drive/MyDrive/VR_AR_testing/Data_Logger_and_Scripts/processed_img/img_grey_PL_H_p2_3fps-black.npy",
    "/content/drive/MyDrive/VR_AR_testing/Data_Logger_and_Scripts/processed_img/img_grey_PL_H_p2_5fps-black.npy",
    "/content/drive/MyDrive/VR_AR_testing/Data_Logger_and_Scripts/processed_img/img_grey_PL_H_p3_1fps-black.npy",
    "/content/drive/MyDrive/VR_AR_testing/Data_Logger_and_Scripts/processed_img/img_grey_PL_H_p3_1fps-white.npy",
    "/content/drive/MyDrive/VR_AR_testing/Data_Logger_and_Scripts/processed_img/img_grey_PL_H_p3_5fps-white.npy",
]

# -----------------------
# 2) Helpers
# -----------------------
def ensure_nhwc1(arr):
    # (N,H,W) or (N,H,W,1) -> (N,H,W,1)
    if arr.ndim == 3:
        arr = arr[..., None]
    if arr.ndim != 4 or arr.shape[-1] != 1:
        raise ValueError(f"Expected (N,H,W,1). Got {arr.shape}")
    return arr

def resize_to_112(arr_nhwc1, target_hw=(112,112)):
    arr_nhwc1 = ensure_nhwc1(arr_nhwc1)
    N, H, W, C = arr_nhwc1.shape
    if (H, W) == target_hw:
        out = arr_nhwc1.astype(np.float32, copy=False)
    else:
        out = np.empty((N, target_hw[0], target_hw[1], 1), dtype=np.float32)
        for i in range(N):
            img = arr_nhwc1[i,:,:,0]
            img_r = cv.resize(img, (target_hw[1], target_hw[0]), interpolation=cv.INTER_AREA)
            out[i,:,:,0] = img_r
    mx = out.max()
    if mx > 1.5:
        out = out / 255.0
    return out.astype(np.float32)

def load_image_stack_list(paths, app_name):
    arrays = []
    for p in paths:
        if not os.path.exists(p):
            raise FileNotFoundError(f"[{app_name}] Missing image stack: {p}")
        a = np.load(p)
        a = ensure_nhwc1(a)
        arrays.append(a)
        print(f"[{app_name}] loaded {os.path.basename(p)} shape={a.shape}")
    out = np.concatenate(arrays, axis=0)
    out = resize_to_112(out, IMG_HW)
    print(f"[{app_name}] stacked images -> {out.shape}")
    return out

def compute_metrics(y_true, y_pred01):
    acc = accuracy_score(y_true, y_pred01)
    prec = precision_score(y_true, y_pred01, zero_division=0)
    rec = recall_score(y_true, y_pred01, zero_division=0)
    f1 = f1_score(y_true, y_pred01, zero_division=0)
    cm = confusion_matrix(y_true, y_pred01, labels=[0,1]).tolist()
    return acc, prec, rec, f1, cm

# -----------------------
# 3) Load + pair (metrics rows <-> image rows)
#    Pairing rule: row order alignment
#    - If lengths mismatch, we truncate to min length and warn.
# -----------------------
def load_paired_spatial_from_stack(metrics_csv, image_stack_paths, app_name="APP"):
    df = pd.read_csv(metrics_csv).replace("", np.nan)

    if LABEL_COL not in df.columns:
        raise RuntimeError(f"[{app_name}] missing {LABEL_COL} column")

    missing_cols = [c for c in INVARIANT27 if c not in df.columns]
    if missing_cols:
        raise RuntimeError(f"[{app_name}] missing invariant cols (first 10): {missing_cols[:10]} | total={len(missing_cols)}")

    # numeric label
    df[LABEL_COL] = pd.to_numeric(df[LABEL_COL], errors="coerce")
    keep = df[LABEL_COL].notna()
    df = df.loc[keep].copy()
    df[LABEL_COL] = df[LABEL_COL].astype(int)

    # tabular
    Xtab = df[INVARIANT27].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=np.float32)
    Xtab[~np.isfinite(Xtab)] = 0.0
    y = df[LABEL_COL].to_numpy(dtype=int)

    # images
    Ximg = load_image_stack_list(image_stack_paths, app_name)

    # align lengths (row-order)
    n = min(len(y), len(Ximg))
    if len(y) != len(Ximg):
        print(f"[WARN] {app_name} length mismatch: metrics_rows={len(y)} vs images_rows={len(Ximg)} -> truncating to {n}")

    Xtab = Xtab[:n]
    y = y[:n]
    Ximg = Ximg[:n]

    print(f"[{app_name}] paired: Ximg={Ximg.shape} Xtab={Xtab.shape} y={y.shape} dist={pd.Series(y).value_counts().to_dict()}")

    if len(y) < 50:
        raise RuntimeError(f"[{app_name}] too few paired rows after filtering: {len(y)}")

    return Ximg, Xtab, y

# -----------------------
# 4) HGB teacher
# -----------------------
def make_hgb_teacher():
    return Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", HistGradientBoostingClassifier(
            random_state=SEED,
            max_depth=6,
            learning_rate=0.05,
            max_iter=400
        ))
    ])

# -----------------------
# 5) Keras hybrid model
# -----------------------
def transformer_encoder(inputs, num_heads=4, key_dim=64, ff_dim=128, dropout_rate=0.1):
    attn = MultiHeadAttention(num_heads=num_heads, key_dim=key_dim)(inputs, inputs)
    attn = Dropout(dropout_rate)(attn)
    x = LayerNormalization(epsilon=1e-6)(inputs + attn)

    ffn = Dense(ff_dim, activation="relu")(x)
    ffn = Dense(inputs.shape[-1])(ffn)
    ffn = Dropout(dropout_rate)(ffn)
    return LayerNormalization(epsilon=1e-6)(x + ffn)

def create_image_branch(image_input_shape=(112,112,1), token_dim=64):
    image_input = Input(shape=image_input_shape, name="image_input")
    x = Conv2D(16, (3,3), activation="relu", padding="same")(image_input)
    x = MaxPooling2D((2,2))(x)
    x = Conv2D(32, (3,3), activation="relu", padding="same")(x)
    x = MaxPooling2D((2,2))(x)
    x = Flatten()(x)
    token = Dense(token_dim, activation="relu")(x)
    token = Lambda(lambda z: tf.expand_dims(z, axis=1))(token)  # (B,1,64)
    return image_input, token

def create_metrics_hgb_token(tab_plus1_dim, token_dim=64):
    tab_in = Input(shape=(tab_plus1_dim,), name="metrics_plus_hgb_input")  # 27 metrics + 1 teacher prob
    x = Dense(64, activation="relu")(tab_in)
    x = Dropout(0.2)(x)
    x = Dense(token_dim, activation="relu")(x)
    token = Lambda(lambda z: tf.expand_dims(z, axis=1))(x)  # (B,1,64)
    return tab_in, token

def build_spatial_hybrid_model(image_input_shape=(112,112,1),
                               tab_plus1_dim=28,
                               token_dim=64, num_heads=4, key_dim=64, ff_dim=128, depth=2):
    image_input, image_token = create_image_branch(image_input_shape, token_dim=token_dim)
    tab_in, tab_token = create_metrics_hgb_token(tab_plus1_dim, token_dim=token_dim)

    x_img = image_token
    for _ in range(depth):
        x_img = transformer_encoder(x_img, num_heads=num_heads, key_dim=key_dim, ff_dim=ff_dim, dropout_rate=0.1)

    tokens = Concatenate(axis=1)([x_img, tab_token])  # (B,2,64)
    for _ in range(depth):
        tokens = transformer_encoder(tokens, num_heads=num_heads, key_dim=key_dim, ff_dim=ff_dim, dropout_rate=0.1)

    pooled = GlobalAveragePooling1D()(tokens)
    h = Dense(64, activation="relu")(pooled)
    out = Dense(1, activation="sigmoid", name="output")(h)

    model = Model(inputs=[image_input, tab_in], outputs=out)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss="binary_crossentropy", metrics=["accuracy"])
    return model

# -----------------------
# 6) 5-fold per app
# -----------------------
def run_spatial_5fold_hybrid(Ximg, Xtab27, y, app_name="APP", epochs=EPOCHS, batch_size=BATCH_SIZE):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    rows = []

    for fold, (tr, te) in enumerate(skf.split(np.zeros(len(y)), y), 1):
        print(f"\n[{app_name}] Fold {fold}/5")

        Ximg_tr, Ximg_te = Ximg[tr], Ximg[te]
        Xtab_tr, Xtab_te = Xtab27[tr], Xtab27[te]
        y_tr, y_te = y[tr], y[te]

        # Train HGB teacher on metrics only
        hgb = make_hgb_teacher()
        hgb.fit(Xtab_tr, y_tr)

        # Teacher prob
        p_tr = hgb.predict_proba(Xtab_tr)[:, 1].reshape(-1, 1).astype(np.float32)
        p_te = hgb.predict_proba(Xtab_te)[:, 1].reshape(-1, 1).astype(np.float32)

        # Scale metrics
        scaler = StandardScaler()
        Xtab_tr_s = scaler.fit_transform(Xtab_tr).astype(np.float32)
        Xtab_te_s = scaler.transform(Xtab_te).astype(np.float32)

        # Metrics + teacher prob => 28 dims
        Xtabh_tr = np.concatenate([Xtab_tr_s, p_tr], axis=1).astype(np.float32)
        Xtabh_te = np.concatenate([Xtab_te_s, p_te], axis=1).astype(np.float32)

        model = build_spatial_hybrid_model(image_input_shape=Ximg.shape[1:], tab_plus1_dim=Xtabh_tr.shape[1])

        es = EarlyStopping(monitor="val_accuracy", patience=3, restore_best_weights=True, verbose=0)
        rlrop = ReduceLROnPlateau(monitor="val_accuracy", factor=0.2, patience=1, min_lr=1e-7, verbose=0)

        model.fit(
            [Ximg_tr, Xtabh_tr], y_tr,
            validation_data=([Ximg_te, Xtabh_te], y_te),
            epochs=epochs,
            batch_size=batch_size,
            verbose=0,
            callbacks=[es, rlrop]
        )

        prob = model.predict([Ximg_te, Xtabh_te], batch_size=batch_size, verbose=0).reshape(-1)
        pred = (prob > 0.5).astype(int)

        acc, prec, rec, f1, cm = compute_metrics(y_te, pred)
        print(f"[{app_name} | HYBRID] acc={acc*100:.1f} prec={prec*100:.1f} rec={rec*100:.1f} f1={f1*100:.1f} cm={cm}")

        rows.append({"app": app_name, "fold": fold, "accuracy": acc, "precision": prec, "recall": rec, "f1": f1})

        tf.keras.backend.clear_session()

    return pd.DataFrame(rows)

# -----------------------
# 7) Run all apps + print table-ready means
# -----------------------
all_folds = []
for app, csv_path in APPS.items():
    print("\n" + "="*70)
    print(f"RUN APP: {app}")
    print("="*70)

    Ximg, Xtab, y = load_paired_spatial_from_stack(csv_path, IMG_STACKS[app], app_name=app)
    folds_df = run_spatial_5fold_hybrid(Ximg, Xtab, y, app_name=app)
    all_folds.append(folds_df)

results_df = pd.concat(all_folds, ignore_index=True)
summary = (results_df.groupby("app")[["accuracy","precision","recall","f1"]]
           .mean()
           .sort_index())

print("\n=========== HYBRID PER-APP 5-FOLD MEAN (READY FOR TABLE) ===========")
print((summary * 100).round(1))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

RUN APP: PianoTiles
[PianoTiles] loaded img_grey_PT_p1_5fps.npy shape=(440, 224, 224, 1)
[PianoTiles] stacked images -> (440, 112, 112, 1)
[PianoTiles] paired: Ximg=(440, 112, 112, 1) Xtab=(440, 27) y=(440,) dist={0: 228, 1: 212}

[PianoTiles] Fold 1/5
[PianoTiles | HYBRID] acc=87.5 prec=83.0 rec=92.9 f1=87.6 cm=[[38, 8], [3, 39]]

[PianoTiles] Fold 2/5
[PianoTiles | HYBRID] acc=84.1 prec=79.2 rec=90.5 f1=84.4 cm=[[36, 10], [4, 38]]

[PianoTiles] Fold 3/5


[PianoTiles | HYBRID] acc=85.2 prec=82.2 rec=88.1 f1=85.1 cm=[[38, 8], [5, 37]]

[PianoTiles] Fold 4/5


[PianoTiles | HYBRID] acc=86.4 prec=89.7 rec=81.4 f1=85.4 cm=[[41, 4], [8, 35]]

[PianoTiles] Fold 5/5
[PianoTiles | HYBRID] acc=89.8 prec=90.5 rec=88.4 f1=89.4 cm=[[41, 4], [5, 38]]

RUN APP: Archery
[Archery] loaded img_grey_AR_p1_5fps_01.npy shape=(100, 224, 224, 1)
[Archery] loaded img_grey_AR_p1_5fps_02.npy shape=(175, 224, 224, 1)
[Archery] loaded img_grey_AR_p1_5fps_03.npy shape=(150, 224, 224, 1)
[Archery] stacked images -> (425, 112, 112, 1)
[Archery] paired: Ximg=(425, 112, 112, 1) Xtab=(425, 27) y=(425,) dist={1: 342, 0: 83}

[Archery] Fold 1/5
[Archery | HYBRID] acc=94.1 prec=93.2 rec=100.0 f1=96.5 cm=[[12, 5], [0, 68]]

[Archery] Fold 2/5
[Archery | HYBRID] acc=98.8 prec=98.6 rec=100.0 f1=99.3 cm=[[16, 1], [0, 68]]

[Archery] Fold 3/5
[Archery | HYBRID] acc=91.8 prec=90.7 rec=100.0 f1=95.1 cm=[[10, 7], [0, 68]]

[Archery] Fold 4/5
[Archery | HYBRID] acc=92.9 prec=92.0 rec=100.0 f1=95.8 cm=[[10, 6], [0, 69]]

[Archery] Fold 5/5
[Archery | HYBRID] acc=92.9 prec=93.2 rec=98.6

In [ ]:
# ============================================================
# ✅ PHANTOMLIMB HOLDOUT EVAL (paper-style)
# Train on PhantomLimb train stacks + metrics
# Test on PhantomLimb holdout stacks + holdout metrics
# ============================================================

def load_paired_spatial_train_test(train_csv, train_img_paths, test_csv, test_img_paths, app_name="PhantomLimb"):
    # ---- train ----
    df_tr = pd.read_csv(train_csv).replace("", np.nan)
    df_tr[LABEL_COL] = pd.to_numeric(df_tr[LABEL_COL], errors="coerce")
    df_tr = df_tr[df_tr[LABEL_COL].notna()].copy()
    df_tr[LABEL_COL] = df_tr[LABEL_COL].astype(int)

    Xtab_tr = df_tr[INVARIANT27].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=np.float32)
    Xtab_tr[~np.isfinite(Xtab_tr)] = 0.0
    y_tr = df_tr[LABEL_COL].to_numpy(dtype=int)

    Ximg_tr = load_image_stack_list(train_img_paths, app_name + "_TRAIN")

    n_tr = min(len(y_tr), len(Ximg_tr))
    if len(y_tr) != len(Ximg_tr):
        print(f"[WARN] {app_name} TRAIN mismatch: metrics={len(y_tr)} imgs={len(Ximg_tr)} -> trunc {n_tr}")
    Xtab_tr, y_tr, Ximg_tr = Xtab_tr[:n_tr], y_tr[:n_tr], Ximg_tr[:n_tr]

    # ---- test ----
    df_te = pd.read_csv(test_csv).replace("", np.nan)
    df_te[LABEL_COL] = pd.to_numeric(df_te[LABEL_COL], errors="coerce")
    df_te = df_te[df_te[LABEL_COL].notna()].copy()
    df_te[LABEL_COL] = df_te[LABEL_COL].astype(int)

    Xtab_te = df_te[INVARIANT27].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=np.float32)
    Xtab_te[~np.isfinite(Xtab_te)] = 0.0
    y_te = df_te[LABEL_COL].to_numpy(dtype=int)

    Ximg_te = load_image_stack_list(test_img_paths, app_name + "_HOLDOUT")

    n_te = min(len(y_te), len(Ximg_te))
    if len(y_te) != len(Ximg_te):
        print(f"[WARN] {app_name} HOLDOUT mismatch: metrics={len(y_te)} imgs={len(Ximg_te)} -> trunc {n_te}")
    Xtab_te, y_te, Ximg_te = Xtab_te[:n_te], y_te[:n_te], Ximg_te[:n_te]

    print(f"[{app_name}] TRAIN: Ximg={Ximg_tr.shape} Xtab={Xtab_tr.shape} y={y_tr.shape} dist={pd.Series(y_tr).value_counts().to_dict()}")
    print(f"[{app_name}] HOLDOUT: Ximg={Ximg_te.shape} Xtab={Xtab_te.shape} y={y_te.shape} dist={pd.Series(y_te).value_counts().to_dict()}")

    return Ximg_tr, Xtab_tr, y_tr, Ximg_te, Xtab_te, y_te

def run_phantom_holdout_eval(train_csv, train_stacks, test_csv, test_stacks, epochs=EPOCHS, batch_size=BATCH_SIZE):
    Ximg_tr, Xtab_tr, y_tr, Ximg_te, Xtab_te, y_te = load_paired_spatial_train_test(
        train_csv, train_stacks, test_csv, test_stacks, app_name="PhantomLimb"
    )

    # teacher (train only)
    hgb = make_hgb_teacher()
    hgb.fit(Xtab_tr, y_tr)

    p_tr = hgb.predict_proba(Xtab_tr)[:, 1].reshape(-1, 1).astype(np.float32)
    p_te = hgb.predict_proba(Xtab_te)[:, 1].reshape(-1, 1).astype(np.float32)

    scaler = StandardScaler()
    Xtab_tr_s = scaler.fit_transform(Xtab_tr).astype(np.float32)
    Xtab_te_s = scaler.transform(Xtab_te).astype(np.float32)

    Xtabh_tr = np.concatenate([Xtab_tr_s, p_tr], axis=1).astype(np.float32)
    Xtabh_te = np.concatenate([Xtab_te_s, p_te], axis=1).astype(np.float32)

    model = build_spatial_hybrid_model(image_input_shape=Ximg_tr.shape[1:], tab_plus1_dim=Xtabh_tr.shape[1])

    es = EarlyStopping(monitor="val_accuracy", patience=3, restore_best_weights=True, verbose=0)
    rlrop = ReduceLROnPlateau(monitor="val_accuracy", factor=0.2, patience=1, min_lr=1e-7, verbose=0)

    model.fit(
        [Ximg_tr, Xtabh_tr], y_tr,
        validation_split=0.1,  # keep it simple; or split manually if you want
        epochs=epochs,
        batch_size=batch_size,
        verbose=0,
        callbacks=[es, rlrop]
    )

    prob = model.predict([Ximg_te, Xtabh_te], batch_size=batch_size, verbose=0).reshape(-1)
    pred = (prob > 0.5).astype(int)

    acc, prec, rec, f1, cm = compute_metrics(y_te, pred)
    print("\n================ PHANTOMLIMB HOLDOUT (HYBRID) ================")
    print(f"acc={acc*100:.1f} prec={prec*100:.1f} rec={rec*100:.1f} f1={f1*100:.1f} cm={cm}")

    tf.keras.backend.clear_session()
    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1, "cm": cm}

# ---- RUN IT ----
holdout_res = run_phantom_holdout_eval(
    train_csv=APPS["PhantomLimb"],
    train_stacks=IMG_STACKS["PhantomLimb"],
    test_csv=PHANTOM_HOLDOUT_CSV,
    test_stacks=PHANTOM_HOLDOUT_STACKS,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE
)

[PhantomLimb_TRAIN] loaded preprocessed_images_grey.npy shape=(1300, 224, 224, 1)
[PhantomLimb_TRAIN] loaded img_grey_PL_p1_5fps.npy shape=(150, 224, 224, 1)
[PhantomLimb_TRAIN] loaded img_grey_PL_p1_1fps.npy shape=(180, 224, 224, 1)
[PhantomLimb_TRAIN] loaded img_grey_PL_p1_3fps.npy shape=(168, 224, 224, 1)
[PhantomLimb_TRAIN] loaded img_grey_PL_p2_5fps.npy shape=(150, 224, 224, 1)
[PhantomLimb_TRAIN] loaded img_grey_PL_p2_1fps.npy shape=(100, 224, 224, 1)
[PhantomLimb_TRAIN] loaded img_grey_PL_p2_3fps.npy shape=(123, 224, 224, 1)
[PhantomLimb_TRAIN] loaded img_grey_PL_p3_5fps.npy shape=(175, 224, 224, 1)
[PhantomLimb_TRAIN] loaded img_grey_PL_p4_5fps.npy shape=(325, 224, 224, 1)
[PhantomLimb_TRAIN] loaded img_grey_PL_p4_3fps.npy shape=(229, 224, 224, 1)
[PhantomLimb_TRAIN] loaded img_grey_PL_p4_1fps.npy shape=(140, 224, 224, 1)
[PhantomLimb_TRAIN] loaded img_grey_PL_p5_5fps.npy shape=(275, 224, 224, 1)
[PhantomLimb_TRAIN] loaded img_grey_PL_p5_3fps.npy shape=(152, 224, 224, 1)
[Phant

# Metrics Study

# -Temporal

## LR - per app

In [ ]:
# ===============================================================
# Metric Influence Study (Conference-grade)
# - Baseline: your Temporal LR_balanced + Patch-B aggregation
# - Per-app 5-fold CV on sequences
# - Permutation importance by BASE METRIC (permute 6 agg columns together)
# - Group ablation (drop groups of metrics, rerun CV)
#
# Outputs:
#   /content/metric_influence_baseline_cv.csv
#   /content/metric_influence_perm_importance_by_metric.csv
#   /content/metric_influence_group_ablation.csv
#   /content/metric_influence_metric_ranking_across_apps.csv
# ===============================================================

import os, json
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# -------------------------
# Settings
# -------------------------
SEED = 42
np.random.seed(SEED)

SEQ_LEN = 5
AGG_USE = ("mean", "std", "delta", "range", "mean_abs_vel", "vel_std")
MIN_NON_NAN_RATIO = 0.4
LABEL_COL = "Temporal"

N_SPLITS = 5
PERM_REPEATS = 5          # lower to 3 for faster
DO_PERM_IMPORTANCE = True
DO_GROUP_ABLATION = True
DO_DROP_ONE_METRIC = False   # optional (very expensive): 6 apps * 27 metrics * 5 folds

# -------------------------
# Invariant metrics (27)
# -------------------------
INVARIANT27 = [
    "missing_joints_count","missing_joints_ratio","collapsed_joints_count",
    "center_of_mass_x","center_of_mass_y","center_of_mass_z","distance_from_origin",
    "bbox_width","bbox_height","bbox_depth","bbox_volume","max_joint_distance_from_com",
    "distance_from_floor","min_foot_height_above_floor","below_floor",
    "left_forearm_length","right_forearm_length","left_shin_length","right_shin_length",
    "arm_length_symmetry","leg_length_symmetry",
    "body_upright_x","body_upright_y","body_upright_z",
    "body_forward_x","body_forward_y","body_forward_z"
]

# -------------------------
# Metric groups (edit if you want)
# -------------------------
METRIC_GROUPS = {
    "validity_missingness": [
        "missing_joints_count","missing_joints_ratio","collapsed_joints_count"
    ],
    "global_position": [
        "center_of_mass_x","center_of_mass_y","center_of_mass_z","distance_from_origin"
    ],
    "bbox_body_extent": [
        "bbox_width","bbox_height","bbox_depth","bbox_volume","max_joint_distance_from_com"
    ],
    "floor_safety": [
        "distance_from_floor","min_foot_height_above_floor","below_floor"
    ],
    "anatomy_symmetry": [
        "left_forearm_length","right_forearm_length","left_shin_length","right_shin_length",
        "arm_length_symmetry","leg_length_symmetry"
    ],
    "orientation": [
        "body_upright_x","body_upright_y","body_upright_z",
        "body_forward_x","body_forward_y","body_forward_z"
    ],
}

# --------------------------
# Patch-B NaN-safe aggregation
# --------------------------
def nanmean_no_warn(x, axis=0):
    x = np.asarray(x, float)
    valid = np.isfinite(x)
    denom = valid.sum(axis=axis)
    num = np.where(valid, x, 0.0).sum(axis=axis)
    return np.divide(num, denom, out=np.full_like(num, np.nan), where=(denom > 0))

def nanstd_no_warn(x, axis=0):
    mu = nanmean_no_warn(x, axis=axis)
    diff2 = (x - np.expand_dims(mu, axis=axis)) ** 2
    valid = np.isfinite(diff2)
    denom = valid.sum(axis=axis)
    num = np.where(valid, diff2, 0.0).sum(axis=axis)
    return np.sqrt(np.divide(num, denom, out=np.full_like(num, np.nan), where=(denom > 0)))

def nanrange_no_warn(x, axis=0):
    valid = np.isfinite(x)
    mx = np.max(np.where(valid, x, -np.inf), axis=axis)
    mn = np.min(np.where(valid, x, +np.inf), axis=axis)
    return np.where(valid.any(axis=axis), mx - mn, np.nan)

def window_is_valid(window):
    return (np.isfinite(window).sum() / window.size) >= MIN_NON_NAN_RATIO

def aggregate_window(window):
    feats = [
        nanmean_no_warn(window, 0),
        nanstd_no_warn(window, 0),
        window[-1] - window[0],
        nanrange_no_warn(window, 0),
    ]
    diff = np.diff(window, axis=0)
    feats += [
        nanmean_no_warn(np.abs(diff), 0),
        nanstd_no_warn(diff, 0),
    ]
    return np.concatenate(feats)

def _order_df(df):
    ORDER_COL = None
    for c in ["EntryID", "entryid", "Timestamp", "timestamp", "Frame", "frame"]:
        if c in df.columns:
            ORDER_COL = c
            break
    if ORDER_COL is None:
        return df.reset_index(drop=True), None
    tmp = pd.to_numeric(df[ORDER_COL], errors="coerce")
    if tmp.notna().any():
        df = df.assign(_order_tmp=tmp).sort_values("_order_tmp").drop(columns="_order_tmp")
    else:
        df = df.sort_values(ORDER_COL)
    return df.reset_index(drop=True), ORDER_COL

def make_sequence_dataset(df_app, feature_cols):
    df = df_app.copy().replace("", np.nan)
    df[LABEL_COL] = pd.to_numeric(df[LABEL_COL], errors="coerce")
    df = df.dropna(subset=[LABEL_COL]).copy()
    df[LABEL_COL] = df[LABEL_COL].astype(int)

    df, ORDER_COL = _order_df(df)

    # numeric features
    df_num = df.apply(pd.to_numeric, errors="coerce")
    X = df_num[feature_cols].to_numpy(dtype=float)
    y = df[LABEL_COL].to_numpy(dtype=int)

    X_seq, y_seq = [], []
    mid = SEQ_LEN // 2
    for i in range(len(X) - SEQ_LEN + 1):
        w = X[i:i+SEQ_LEN]
        if not window_is_valid(w):
            continue
        X_seq.append(aggregate_window(w))
        y_seq.append(int(y[i + mid]))

    return np.asarray(X_seq), np.asarray(y_seq), ORDER_COL

def make_lr_balanced():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            class_weight="balanced",
            max_iter=3000,
            random_state=SEED,
            n_jobs=-1
        ))
    ])

def eval_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=[0,1]).tolist()
    return float(acc), float(f1), float(prec), float(rec), cm

# -------------------------
# Aggregated feature naming + index mapping
# -------------------------
AGG_NAMES = list(AGG_USE)

def get_agg_feature_names(feature_cols):
    names = []
    for agg in AGG_NAMES:
        for c in feature_cols:
            names.append(f"{agg}__{c}")
    return names

def metric_to_indices(feature_cols):
    # each base metric maps to its 6 agg-derived columns
    idx = {}
    F = len(feature_cols)
    for j, c in enumerate(feature_cols):
        idx[c] = [k*F + j for k in range(len(AGG_NAMES))]
    return idx

# -------------------------
# Permutation importance (by base metric)
# -------------------------
def perm_importance_by_metric(model, X_test, y_test, feature_cols, n_repeats=5):
    # baseline
    base_pred = model.predict(X_test)
    base_acc = accuracy_score(y_test, base_pred)
    base_f1  = f1_score(y_test, base_pred, zero_division=0)

    idx_map = metric_to_indices(feature_cols)
    rng = np.random.default_rng(SEED)

    drops = {}
    for c, inds in idx_map.items():
        acc_scores = []
        f1_scores  = []
        for _ in range(n_repeats):
            Xp = X_test.copy()
            perm = rng.permutation(len(Xp))
            Xp[:, inds] = Xp[perm][:, inds]
            p = model.predict(Xp)
            acc_scores.append(accuracy_score(y_test, p))
            f1_scores.append(f1_score(y_test, p, zero_division=0))

        drops[c] = {
            "acc_drop": float(base_acc - np.mean(acc_scores)),
            "f1_drop":  float(base_f1  - np.mean(f1_scores)),
        }

    return float(base_acc), float(base_f1), drops

# -------------------------
# Core runner: baseline CV + perm importance
# -------------------------
def run_baseline_and_perm(app_name, df_app, feature_cols):
    X_all, y_all, ORDER_COL = make_sequence_dataset(df_app, feature_cols)
    if len(y_all) < 50:
        raise RuntimeError(f"[{app_name}] Too few sequences after filtering; check ordering / NaNs / MIN_NON_NAN_RATIO.")

    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

    baseline_rows = []
    perm_rows = []

    for fold, (tr, te) in enumerate(skf.split(X_all, y_all), 1):
        m = make_lr_balanced()
        m.fit(X_all[tr], y_all[tr])
        yhat = m.predict(X_all[te])
        acc, f1, prec, rec, cm = eval_metrics(y_all[te], yhat)

        baseline_rows.append({
            "app": app_name, "fold": fold,
            "accuracy": acc, "f1": f1, "precision": prec, "recall": rec,
            "tn": cm[0][0], "fp": cm[0][1], "fn": cm[1][0], "tp": cm[1][1],
            "order_col": ORDER_COL,
            "n_test": int(len(te)),
        })

        if DO_PERM_IMPORTANCE:
            base_acc, base_f1, drops = perm_importance_by_metric(
                m, X_all[te], y_all[te], feature_cols, n_repeats=PERM_REPEATS
            )
            for metric, d in drops.items():
                perm_rows.append({
                    "app": app_name, "fold": fold,
                    "metric": metric,
                    "base_acc": base_acc,
                    "base_f1": base_f1,
                    "acc_drop": d["acc_drop"],
                    "f1_drop":  d["f1_drop"],
                })

        print(f"[{app_name} FOLD {fold}] acc={acc:.3f} f1={f1:.3f} prec={prec:.3f} rec={rec:.3f} | cm={cm}")

    baseline_df = pd.DataFrame(baseline_rows)
    perm_df = pd.DataFrame(perm_rows) if len(perm_rows) else pd.DataFrame(columns=["app","fold","metric","base_f1","f1_drop"])
    return baseline_df, perm_df

# -------------------------
# Group ablation runner
# -------------------------
def run_group_ablation(app_name, df_app, feature_cols, groups_dict):
    # baseline once (for delta reporting)
    X0, y0, _ = make_sequence_dataset(df_app, feature_cols)
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

    # baseline cv mean f1
    f1s0 = []
    for tr, te in skf.split(X0, y0):
        m0 = make_lr_balanced()
        m0.fit(X0[tr], y0[tr])
        f1s0.append(f1_score(y0[te], m0.predict(X0[te]), zero_division=0))
    base_mean_f1 = float(np.mean(f1s0))

    rows = []
    for gname, gmetrics in groups_dict.items():
        kept = [c for c in feature_cols if c not in set(gmetrics)]
        Xg, yg, _ = make_sequence_dataset(df_app, kept)

        # run cv
        f1s = []
        for tr, te in skf.split(Xg, yg):
            mg = make_lr_balanced()
            mg.fit(Xg[tr], yg[tr])
            f1s.append(f1_score(yg[te], mg.predict(Xg[te]), zero_division=0))

        mean_f1 = float(np.mean(f1s))
        rows.append({
            "app": app_name,
            "group": gname,
            "dropped_metrics": ",".join(gmetrics),
            "base_mean_f1": base_mean_f1,
            "ablated_mean_f1": mean_f1,
            "delta_f1": float(mean_f1 - base_mean_f1),  # negative means group helped
            "n_metrics_dropped": int(len(gmetrics)),
        })

        print(f"[{app_name} GROUP ABLATION] drop={gname:>18s}  base_f1={base_mean_f1:.3f}  ablated_f1={mean_f1:.3f}  delta={mean_f1-base_mean_f1:+.3f}")

    return pd.DataFrame(rows)

# -------------------------
# Optional: Drop-one-metric ablation (very expensive)
# -------------------------
def run_drop_one_metric(app_name, df_app, feature_cols):
    X0, y0, _ = make_sequence_dataset(df_app, feature_cols)
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

    f1s0 = []
    for tr, te in skf.split(X0, y0):
        m0 = make_lr_balanced()
        m0.fit(X0[tr], y0[tr])
        f1s0.append(f1_score(y0[te], m0.predict(X0[te]), zero_division=0))
    base_mean_f1 = float(np.mean(f1s0))

    rows = []
    for drop_metric in feature_cols:
        kept = [c for c in feature_cols if c != drop_metric]
        Xg, yg, _ = make_sequence_dataset(df_app, kept)
        f1s = []
        for tr, te in skf.split(Xg, yg):
            mg = make_lr_balanced()
            mg.fit(Xg[tr], yg[tr])
            f1s.append(f1_score(yg[te], mg.predict(Xg[te]), zero_division=0))
        mean_f1 = float(np.mean(f1s))
        rows.append({
            "app": app_name,
            "dropped_metric": drop_metric,
            "base_mean_f1": base_mean_f1,
            "ablated_mean_f1": mean_f1,
            "delta_f1": float(mean_f1 - base_mean_f1),
        })
        print(f"[{app_name} DROP-ONE] drop={drop_metric:>28s} base_f1={base_mean_f1:.3f} ablated_f1={mean_f1:.3f} delta={mean_f1-base_mean_f1:+.3f}")
    return pd.DataFrame(rows)

# ===============================================================
# RUN ALL 6 APPS (update paths if needed)
# ===============================================================
apps = {
    "Archery":      "/content/metrics_Archery.csv",
    "PhantomLimb":  "/content/metrics_PhantomLimb.csv",
    "PianoTiles":   "/content/metrics_PianoTiles.csv",
    "Puzzle":       "/content/metrics_Puzzle.csv",
    "Sea":          "/content/metrics_Sea.csv",
    "War":          "/content/metrics_War.csv",
}

feature_cols = INVARIANT27[:]  # exact 27

baseline_all = []
perm_all = []
group_ablation_all = []
drop_one_all = []

for app, path in apps.items():
    print("\n" + "="*70)
    print(f"APP: {app} | CSV: {path}")
    print("="*70)

    df_app = pd.read_csv(path)

    # sanity checks
    missing = [c for c in feature_cols if c not in df_app.columns]
    if missing:
        raise RuntimeError(f"[{app}] Missing invariant cols: {missing}")
    if LABEL_COL not in df_app.columns:
        raise RuntimeError(f"[{app}] Missing label col '{LABEL_COL}'")

    # baseline + permutation importance
    bdf, pdf = run_baseline_and_perm(app, df_app, feature_cols)
    baseline_all.append(bdf)
    perm_all.append(pdf)

    # group ablation
    if DO_GROUP_ABLATION:
        gadf = run_group_ablation(app, df_app, feature_cols, METRIC_GROUPS)
        group_ablation_all.append(gadf)

    # optional drop-one metric
    if DO_DROP_ONE_METRIC:
        do_df = run_drop_one_metric(app, df_app, feature_cols)
        drop_one_all.append(do_df)

# -------------------------
# Save outputs
# -------------------------
out_dir = "/content"
os.makedirs(out_dir, exist_ok=True)

baseline_df = pd.concat(baseline_all, ignore_index=True)
baseline_path = os.path.join(out_dir, "metric_influence_baseline_cv.csv")
baseline_df.to_csv(baseline_path, index=False)

perm_df = pd.concat(perm_all, ignore_index=True) if len(perm_all) else pd.DataFrame()
perm_path = os.path.join(out_dir, "metric_influence_perm_importance_by_metric.csv")
perm_df.to_csv(perm_path, index=False)

if DO_GROUP_ABLATION and len(group_ablation_all):
    group_ablation_df = pd.concat(group_ablation_all, ignore_index=True)
else:
    group_ablation_df = pd.DataFrame()
group_ablation_path = os.path.join(out_dir, "metric_influence_group_ablation.csv")
group_ablation_df.to_csv(group_ablation_path, index=False)

if DO_DROP_ONE_METRIC and len(drop_one_all):
    drop_one_df = pd.concat(drop_one_all, ignore_index=True)
else:
    drop_one_df = pd.DataFrame()
drop_one_path = os.path.join(out_dir, "metric_influence_drop_one_metric.csv")
drop_one_df.to_csv(drop_one_path, index=False)

# -------------------------
# Cross-app ranking summary (from permutation importance)
# -------------------------
# "General" metrics: high mean drop + supported across many apps
if len(perm_df):
    pivot = (perm_df
             .groupby(["app","metric"])["f1_drop"]
             .mean()
             .reset_index())

    # per-metric across apps
    metric_summary = (pivot
        .groupby("metric")["f1_drop"]
        .agg(
            mean_drop="mean",
            std_drop="std",
            support_apps=lambda x: int((x > 0.01).sum()),  # threshold for "meaningful"
            apps_total="count"
        )
        .reset_index()
    )
    metric_summary["score"] = metric_summary["mean_drop"] * metric_summary["support_apps"] / (1.0 + metric_summary["std_drop"].fillna(0.0))
    metric_summary = metric_summary.sort_values("score", ascending=False)

    rank_path = os.path.join(out_dir, "metric_influence_metric_ranking_across_apps.csv")
    metric_summary.to_csv(rank_path, index=False)
else:
    rank_path = None

print("\n================= SAVED FILES =================")
print("Baseline CV:", baseline_path)
print("Perm importance:", perm_path)
print("Group ablation:", group_ablation_path)
print("Drop-one metric:", drop_one_path)
if rank_path:
    print("Metric ranking:", rank_path)

print("\nDone.")


APP: Archery | CSV: /content/metrics_Archery.csv
[Archery FOLD 1] acc=0.929 f1=0.951 prec=0.951 rec=0.951 | cm=[[21, 3], [3, 58]]
[Archery FOLD 2] acc=0.952 f1=0.966 prec=0.983 rec=0.950 | cm=[[23, 1], [3, 57]]
[Archery FOLD 3] acc=0.905 f1=0.930 prec=0.981 rec=0.883 | cm=[[23, 1], [7, 53]]
[Archery FOLD 4] acc=0.952 f1=0.967 prec=0.952 rec=0.983 | cm=[[21, 3], [1, 59]]
[Archery FOLD 5] acc=0.988 f1=0.992 prec=1.000 rec=0.984 | cm=[[23, 0], [1, 60]]
[Archery GROUP ABLATION] drop=validity_missingness  base_f1=0.961  ablated_f1=0.961  delta=+0.000
[Archery GROUP ABLATION] drop=   global_position  base_f1=0.961  ablated_f1=0.961  delta=+0.000
[Archery GROUP ABLATION] drop=  bbox_body_extent  base_f1=0.961  ablated_f1=0.926  delta=-0.035
[Archery GROUP ABLATION] drop=      floor_safety  base_f1=0.961  ablated_f1=0.961  delta=+0.000
[Archery GROUP ABLATION] drop=  anatomy_symmetry  base_f1=0.961  ablated_f1=0.953  delta=-0.008
[Archery GROUP ABLATION] drop=       orientation  base_f1=0.961

/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [ 12  13  14  39  40  41  66  67  68  93  94  95 120 121 122 147 148 149]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [ 12  13  14  39  40  41  66  67  68  93  94  95 120 121 122 147 148 149]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [ 12  13  14  39  40  41  66  67  68  93  94  95 120 121 122 147 148 149]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping featur

[Puzzle FOLD 1] acc=0.946 f1=0.964 prec=1.000 rec=0.930 | cm=[[13, 0], [3, 40]]


/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [ 12  13  14  39  40  41  66  67  68  93  94  95 120 121 122 147 148 149]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [ 12  13  14  39  40  41  66  67  68  93  94  95 120 121 122 147 148 149]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [ 12  13  14  39  40  41  66  67  68  93  94  95 120 121 122 147 148 149]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping featur

[Puzzle FOLD 2] acc=0.945 f1=0.965 prec=0.953 rec=0.976 | cm=[[11, 2], [1, 41]]


...

[Puzzle GROUP ABLATION] drop=  bbox_body_extent  base_f1=0.972  ablated_f1=0.972  delta=+0.000
[Puzzle GROUP ABLATION] drop=      floor_safety  base_f1=0.972  ablated_f1=0.972  delta=+0.000


/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [ 12  13  14  33  34  35  54  55  56  75  76  77  96  97  98 117 118 119]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [ 12  13  14  33  34  35  54  55  56  75  76  77  96  97  98 117 118 119]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [ 12  13  14  33  34  35  54  55  56  75  76  77  96  97  98 117 118 119]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping featur

[Puzzle GROUP ABLATION] drop=  anatomy_symmetry  base_f1=0.972  ablated_f1=0.965  delta=-0.007
[Puzzle GROUP ABLATION] drop=       orientation  base_f1=0.972  ablated_f1=0.976  delta=+0.005

APP: Sea | CSV: /content/metrics_Sea.csv


/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [ 12  13  14  33  34  35  54  55  56  75  76  77  96  97  98 117 118 119]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [ 12  13  14  33  34  35  54  55  56  75  76  77  96  97  98 117 118 119]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


[Sea FOLD 1] acc=0.983 f1=0.981 prec=1.000 rec=0.963 | cm=[[33, 0], [1, 26]]
[Sea FOLD 2] acc=1.000 f1=1.000 prec=1.000 rec=1.000 | cm=[[33, 0], [0, 26]]
[Sea FOLD 3] acc=0.983 f1=0.982 prec=0.964 rec=1.000 | cm=[[31, 1], [0, 27]]
[Sea FOLD 4] acc=0.915 f1=0.915 prec=0.844 rec=1.000 | cm=[[27, 5], [0, 27]]
[Sea FOLD 5] acc=0.949 f1=0.945 prec=0.929 rec=0.963 | cm=[[30, 2], [1, 26]]
[Sea GROUP ABLATION] drop=validity_missingness  base_f1=0.965  ablated_f1=0.965  delta=+0.000
[Sea GROUP ABLATION] drop=   global_position  base_f1=0.965  ablated_f1=0.951  delta=-0.014
[Sea GROUP ABLATION] drop=  bbox_body_extent  base_f1=0.965  ablated_f1=0.961  delta=-0.004
[Sea GROUP ABLATION] drop=      floor_safety  base_f1=0.965  ablated_f1=0.968  delta=+0.003
[Sea GROUP ABLATION] drop=  anatomy_symmetry  base_f1=0.965  ablated_f1=0.965  delta=+0.001
[Sea GROUP ABLATION] drop=       orientation  base_f1=0.965  ablated_f1=0.953  delta=-0.011

APP: War | CSV: /content/metrics_War.csv
[War FOLD 1] acc=0.

In [ ]:
# ===============================================================
# Metric Influence Study (Conference-grade) — ACC-first, F1 guardrail
# - Baseline: Temporal LR_balanced + Patch-B aggregation
# - Per-app 5-fold CV on sequences
# - Permutation importance by BASE METRIC (permute 6 agg columns together)
# - Group ablation (drop groups of metrics, rerun CV)
#
# Outputs:
#   /content/metric_influence_baseline_cv.csv
#   /content/metric_influence_perm_importance_by_metric.csv
#   /content/metric_influence_group_ablation.csv
#   /content/metric_influence_metric_ranking_across_apps.csv
#   /content/metric_influence_group_ranking_across_apps.csv
# ===============================================================

import os, json
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# -------------------------
# Settings
# -------------------------
SEED = 42
np.random.seed(SEED)

SEQ_LEN = 5
AGG_USE = ("mean", "std", "delta", "range", "mean_abs_vel", "vel_std")
MIN_NON_NAN_RATIO = 0.4
LABEL_COL = "Temporal"

N_SPLITS = 5
PERM_REPEATS = 5          # set to 3 if you want faster
DO_PERM_IMPORTANCE = True
DO_GROUP_ABLATION = True
DO_DROP_ONE_METRIC = False   # optional (very expensive): 6 apps * 27 metrics * 5 folds

# NOTE on runtime:
# Perm importance cost ~ (6 apps * 5 folds * 27 metrics * PERM_REPEATS) predictions.
# With PERM_REPEATS=5 this is usually fine on Colab but not instant.

# -------------------------
# Invariant metrics (27)
# -------------------------
INVARIANT27 = [
    "missing_joints_count","missing_joints_ratio","collapsed_joints_count",
    "center_of_mass_x","center_of_mass_y","center_of_mass_z","distance_from_origin",
    "bbox_width","bbox_height","bbox_depth","bbox_volume","max_joint_distance_from_com",
    "distance_from_floor","min_foot_height_above_floor","below_floor",
    "left_forearm_length","right_forearm_length","left_shin_length","right_shin_length",
    "arm_length_symmetry","leg_length_symmetry",
    "body_upright_x","body_upright_y","body_upright_z",
    "body_forward_x","body_forward_y","body_forward_z"
]

# -------------------------
# Metric groups (edit if you want)
# -------------------------
METRIC_GROUPS = {
    "validity_missingness": [
        "missing_joints_count","missing_joints_ratio","collapsed_joints_count"
    ],
    "global_position": [
        "center_of_mass_x","center_of_mass_y","center_of_mass_z","distance_from_origin"
    ],
    "bbox_body_extent": [
        "bbox_width","bbox_height","bbox_depth","bbox_volume","max_joint_distance_from_com"
    ],
    "floor_safety": [
        "distance_from_floor","min_foot_height_above_floor","below_floor"
    ],
    "anatomy_symmetry": [
        "left_forearm_length","right_forearm_length","left_shin_length","right_shin_length",
        "arm_length_symmetry","leg_length_symmetry"
    ],
    "orientation": [
        "body_upright_x","body_upright_y","body_upright_z",
        "body_forward_x","body_forward_y","body_forward_z"
    ],
}

# --------------------------
# Patch-B NaN-safe aggregation
# --------------------------
def nanmean_no_warn(x, axis=0):
    x = np.asarray(x, float)
    valid = np.isfinite(x)
    denom = valid.sum(axis=axis)
    num = np.where(valid, x, 0.0).sum(axis=axis)
    return np.divide(num, denom, out=np.full_like(num, np.nan), where=(denom > 0))

def nanstd_no_warn(x, axis=0):
    mu = nanmean_no_warn(x, axis=axis)
    diff2 = (x - np.expand_dims(mu, axis=axis)) ** 2
    valid = np.isfinite(diff2)
    denom = valid.sum(axis=axis)
    num = np.where(valid, diff2, 0.0).sum(axis=axis)
    return np.sqrt(np.divide(num, denom, out=np.full_like(num, np.nan), where=(denom > 0)))

def nanrange_no_warn(x, axis=0):
    valid = np.isfinite(x)
    mx = np.max(np.where(valid, x, -np.inf), axis=axis)
    mn = np.min(np.where(valid, x, +np.inf), axis=axis)
    return np.where(valid.any(axis=axis), mx - mn, np.nan)

def window_is_valid(window):
    return (np.isfinite(window).sum() / window.size) >= MIN_NON_NAN_RATIO

def aggregate_window(window):
    feats = [
        nanmean_no_warn(window, 0),
        nanstd_no_warn(window, 0),
        window[-1] - window[0],
        nanrange_no_warn(window, 0),
    ]
    diff = np.diff(window, axis=0)
    feats += [
        nanmean_no_warn(np.abs(diff), 0),
        nanstd_no_warn(diff, 0),
    ]
    return np.concatenate(feats)

def _order_df(df):
    ORDER_COL = None
    for c in ["EntryID", "entryid", "Timestamp", "timestamp", "Frame", "frame"]:
        if c in df.columns:
            ORDER_COL = c
            break
    if ORDER_COL is None:
        return df.reset_index(drop=True), None
    tmp = pd.to_numeric(df[ORDER_COL], errors="coerce")
    if tmp.notna().any():
        df = df.assign(_order_tmp=tmp).sort_values("_order_tmp").drop(columns="_order_tmp")
    else:
        df = df.sort_values(ORDER_COL)
    return df.reset_index(drop=True), ORDER_COL

def make_sequence_dataset(df_app, feature_cols):
    df = df_app.copy().replace("", np.nan)
    df[LABEL_COL] = pd.to_numeric(df[LABEL_COL], errors="coerce")
    df = df.dropna(subset=[LABEL_COL]).copy()
    df[LABEL_COL] = df[LABEL_COL].astype(int)

    df, ORDER_COL = _order_df(df)

    df_num = df.apply(pd.to_numeric, errors="coerce")
    X = df_num[feature_cols].to_numpy(dtype=float)
    y = df[LABEL_COL].to_numpy(dtype=int)

    X_seq, y_seq = [], []
    mid = SEQ_LEN // 2
    for i in range(len(X) - SEQ_LEN + 1):
        w = X[i:i+SEQ_LEN]
        if not window_is_valid(w):
            continue
        X_seq.append(aggregate_window(w))
        y_seq.append(int(y[i + mid]))

    return np.asarray(X_seq), np.asarray(y_seq), ORDER_COL

def make_lr_balanced():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            class_weight="balanced",
            max_iter=3000,
            random_state=SEED,
            n_jobs=-1
        ))
    ])

def eval_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true, y_pred, zero_division=0)
    f1   = f1_score(y_true, y_pred, zero_division=0)
    cm   = confusion_matrix(y_true, y_pred, labels=[0,1]).tolist()
    return float(acc), float(f1), float(prec), float(rec), cm

# -------------------------
# Aggregated feature naming + index mapping
# -------------------------
AGG_NAMES = list(AGG_USE)

def metric_to_indices(feature_cols):
    # each base metric maps to its 6 agg-derived columns
    idx = {}
    F = len(feature_cols)
    for j, c in enumerate(feature_cols):
        idx[c] = [k*F + j for k in range(len(AGG_NAMES))]
    return idx

# -------------------------
# Permutation importance (by base metric) — ACC first, keep F1 too
# -------------------------
def perm_importance_by_metric(model, X_test, y_test, feature_cols, n_repeats=5):
    base_pred = model.predict(X_test)
    base_acc = accuracy_score(y_test, base_pred)
    base_f1  = f1_score(y_test, base_pred, zero_division=0)

    idx_map = metric_to_indices(feature_cols)
    rng = np.random.default_rng(SEED)

    drops = {}
    for c, inds in idx_map.items():
        acc_scores = []
        f1_scores  = []
        for _ in range(n_repeats):
            Xp = X_test.copy()
            perm = rng.permutation(len(Xp))
            # permute all agg columns jointly (keeps within-metric structure)
            Xp[:, inds] = Xp[perm][:, inds]
            p = model.predict(Xp)
            acc_scores.append(accuracy_score(y_test, p))
            f1_scores.append(f1_score(y_test, p, zero_division=0))

        drops[c] = {
            "acc_drop": float(base_acc - np.mean(acc_scores)),
            "f1_drop":  float(base_f1  - np.mean(f1_scores)),
        }

    return float(base_acc), float(base_f1), drops

# -------------------------
# Core runner: baseline CV + permutation importance
# -------------------------
def run_baseline_and_perm(app_name, df_app, feature_cols):
    X_all, y_all, ORDER_COL = make_sequence_dataset(df_app, feature_cols)
    if len(y_all) < 50:
        raise RuntimeError(f"[{app_name}] Too few sequences after filtering; check ordering / NaNs / MIN_NON_NAN_RATIO.")

    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

    baseline_rows = []
    perm_rows = []

    for fold, (tr, te) in enumerate(skf.split(X_all, y_all), 1):
        m = make_lr_balanced()
        m.fit(X_all[tr], y_all[tr])
        yhat = m.predict(X_all[te])
        acc, f1, prec, rec, cm = eval_metrics(y_all[te], yhat)

        baseline_rows.append({
            "app": app_name, "fold": fold,
            "accuracy": acc, "f1": f1, "precision": prec, "recall": rec,
            "tn": cm[0][0], "fp": cm[0][1], "fn": cm[1][0], "tp": cm[1][1],
            "order_col": ORDER_COL,
            "n_test": int(len(te)),
        })

        if DO_PERM_IMPORTANCE:
            base_acc, base_f1, drops = perm_importance_by_metric(
                m, X_all[te], y_all[te], feature_cols, n_repeats=PERM_REPEATS
            )
            for metric, d in drops.items():
                perm_rows.append({
                    "app": app_name, "fold": fold,
                    "metric": metric,
                    "base_acc": base_acc,
                    "base_f1": base_f1,
                    "acc_drop": d["acc_drop"],
                    "f1_drop":  d["f1_drop"],
                })

        print(f"[{app_name} FOLD {fold}] acc={acc:.3f} f1={f1:.3f} prec={prec:.3f} rec={rec:.3f} | cm={cm}")

    baseline_df = pd.DataFrame(baseline_rows)
    perm_df = pd.DataFrame(perm_rows) if len(perm_rows) else pd.DataFrame(
        columns=["app","fold","metric","base_acc","base_f1","acc_drop","f1_drop"]
    )
    return baseline_df, perm_df

# -------------------------
# Group ablation runner — ACC first, keep F1 too
# -------------------------
def run_group_ablation(app_name, df_app, feature_cols, groups_dict):
    X0, y0, _ = make_sequence_dataset(df_app, feature_cols)
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

    base_accs, base_f1s = [], []
    for tr, te in skf.split(X0, y0):
        m0 = make_lr_balanced()
        m0.fit(X0[tr], y0[tr])
        p0 = m0.predict(X0[te])
        base_accs.append(accuracy_score(y0[te], p0))
        base_f1s.append(f1_score(y0[te], p0, zero_division=0))

    base_mean_acc = float(np.mean(base_accs))
    base_mean_f1  = float(np.mean(base_f1s))

    rows = []
    for gname, gmetrics in groups_dict.items():
        kept = [c for c in feature_cols if c not in set(gmetrics)]
        Xg, yg, _ = make_sequence_dataset(df_app, kept)

        accs, f1s = [], []
        for tr, te in skf.split(Xg, yg):
            mg = make_lr_balanced()
            mg.fit(Xg[tr], yg[tr])
            pg = mg.predict(Xg[te])
            accs.append(accuracy_score(yg[te], pg))
            f1s.append(f1_score(yg[te], pg, zero_division=0))

        mean_acc = float(np.mean(accs))
        mean_f1  = float(np.mean(f1s))

        rows.append({
            "app": app_name,
            "group": gname,
            "dropped_metrics": ",".join(gmetrics),
            "base_mean_acc": base_mean_acc,
            "ablated_mean_acc": mean_acc,
            "delta_acc": float(mean_acc - base_mean_acc),  # negative => group helped
            "base_mean_f1": base_mean_f1,
            "ablated_mean_f1": mean_f1,
            "delta_f1": float(mean_f1 - base_mean_f1),
            "n_metrics_dropped": int(len(gmetrics)),
        })

        print(f"[{app_name} GROUP ABLATION] drop={gname:>18s}  "
              f"base_acc={base_mean_acc:.3f} ablated_acc={mean_acc:.3f} delta_acc={mean_acc-base_mean_acc:+.3f} | "
              f"base_f1={base_mean_f1:.3f} ablated_f1={mean_f1:.3f} delta_f1={mean_f1-base_mean_f1:+.3f}")

    return pd.DataFrame(rows)

# -------------------------
# Optional: Drop-one-metric ablation (very expensive) — ACC first, keep F1 too
# -------------------------
def run_drop_one_metric(app_name, df_app, feature_cols):
    X0, y0, _ = make_sequence_dataset(df_app, feature_cols)
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

    base_accs, base_f1s = [], []
    for tr, te in skf.split(X0, y0):
        m0 = make_lr_balanced()
        m0.fit(X0[tr], y0[tr])
        p0 = m0.predict(X0[te])
        base_accs.append(accuracy_score(y0[te], p0))
        base_f1s.append(f1_score(y0[te], p0, zero_division=0))
    base_mean_acc = float(np.mean(base_accs))
    base_mean_f1  = float(np.mean(base_f1s))

    rows = []
    for drop_metric in feature_cols:
        kept = [c for c in feature_cols if c != drop_metric]
        Xg, yg, _ = make_sequence_dataset(df_app, kept)

        accs, f1s = [], []
        for tr, te in skf.split(Xg, yg):
            mg = make_lr_balanced()
            mg.fit(Xg[tr], yg[tr])
            pg = mg.predict(Xg[te])
            accs.append(accuracy_score(yg[te], pg))
            f1s.append(f1_score(yg[te], pg, zero_division=0))

        mean_acc = float(np.mean(accs))
        mean_f1  = float(np.mean(f1s))

        rows.append({
            "app": app_name,
            "dropped_metric": drop_metric,
            "base_mean_acc": base_mean_acc,
            "ablated_mean_acc": mean_acc,
            "delta_acc": float(mean_acc - base_mean_acc),
            "base_mean_f1": base_mean_f1,
            "ablated_mean_f1": mean_f1,
            "delta_f1": float(mean_f1 - base_mean_f1),
        })

        print(f"[{app_name} DROP-ONE] drop={drop_metric:>28s} "
              f"base_acc={base_mean_acc:.3f} ablated_acc={mean_acc:.3f} delta_acc={mean_acc-base_mean_acc:+.3f} | "
              f"base_f1={base_mean_f1:.3f} ablated_f1={mean_f1:.3f} delta_f1={mean_f1-base_mean_f1:+.3f}")

    return pd.DataFrame(rows)

# ===============================================================
# RUN ALL 6 APPS
# ===============================================================
apps = {
    "Archery":      "/content/metrics_Archery.csv",
    "PhantomLimb":  "/content/metrics_PhantomLimb.csv",
    "PianoTiles":   "/content/metrics_PianoTiles.csv",
    "Puzzle":       "/content/metrics_Puzzle.csv",
    "Sea":          "/content/metrics_Sea.csv",
    "War":          "/content/metrics_War.csv",
}

feature_cols = INVARIANT27[:]  # exact 27

baseline_all = []
perm_all = []
group_ablation_all = []
drop_one_all = []

for app, path in apps.items():
    print("\n" + "="*70)
    print(f"APP: {app} | CSV: {path}")
    print("="*70)

    df_app = pd.read_csv(path)

    # sanity checks
    missing = [c for c in feature_cols if c not in df_app.columns]
    if missing:
        raise RuntimeError(f"[{app}] Missing invariant cols: {missing}")
    if LABEL_COL not in df_app.columns:
        raise RuntimeError(f"[{app}] Missing label col '{LABEL_COL}'")

    # baseline + permutation importance
    bdf, pdf = run_baseline_and_perm(app, df_app, feature_cols)
    baseline_all.append(bdf)
    perm_all.append(pdf)

    # group ablation
    if DO_GROUP_ABLATION:
        gadf = run_group_ablation(app, df_app, feature_cols, METRIC_GROUPS)
        group_ablation_all.append(gadf)

    # optional drop-one metric
    if DO_DROP_ONE_METRIC:
        do_df = run_drop_one_metric(app, df_app, feature_cols)
        drop_one_all.append(do_df)

# -------------------------
# Save outputs
# -------------------------
out_dir = "/content"
os.makedirs(out_dir, exist_ok=True)

baseline_df = pd.concat(baseline_all, ignore_index=True)
baseline_path = os.path.join(out_dir, "metric_influence_baseline_cv.csv")
baseline_df.to_csv(baseline_path, index=False)

perm_df = pd.concat(perm_all, ignore_index=True) if len(perm_all) else pd.DataFrame()
perm_path = os.path.join(out_dir, "metric_influence_perm_importance_by_metric.csv")
perm_df.to_csv(perm_path, index=False)

if DO_GROUP_ABLATION and len(group_ablation_all):
    group_ablation_df = pd.concat(group_ablation_all, ignore_index=True)
else:
    group_ablation_df = pd.DataFrame()
group_ablation_path = os.path.join(out_dir, "metric_influence_group_ablation.csv")
group_ablation_df.to_csv(group_ablation_path, index=False)

if DO_DROP_ONE_METRIC and len(drop_one_all):
    drop_one_df = pd.concat(drop_one_all, ignore_index=True)
else:
    drop_one_df = pd.DataFrame()
drop_one_path = os.path.join(out_dir, "metric_influence_drop_one_metric.csv")
drop_one_df.to_csv(drop_one_path, index=False)

# -------------------------
# Cross-app ranking summary (from permutation importance)
# - Rank by mean ACC drop (primary), then mean F1 drop (tie-break)
# -------------------------
if len(perm_df):
    pivot = (perm_df
             .groupby(["app","metric"])[["acc_drop","f1_drop"]]
             .mean()
             .reset_index())

    metric_summary = (pivot
        .groupby("metric")
        .agg(
            mean_acc_drop=("acc_drop","mean"),
            std_acc_drop=("acc_drop","std"),
            mean_f1_drop=("f1_drop","mean"),
            std_f1_drop=("f1_drop","std"),
            support_apps_acc=("acc_drop", lambda x: int((x > 0.01).sum())),
            support_apps_f1=("f1_drop",  lambda x: int((x > 0.01).sum())),
            apps_total=("acc_drop","count"),
        )
        .reset_index()
    )

    # ACC-first score; stability penalty via std; tie-break via mean_f1_drop later
    metric_summary["score"] = (
        metric_summary["mean_acc_drop"] * metric_summary["support_apps_acc"]
        / (1.0 + metric_summary["std_acc_drop"].fillna(0.0))
    )

    metric_summary = metric_summary.sort_values(
        ["score", "mean_f1_drop"],
        ascending=[False, False]
    )

    rank_path = os.path.join(out_dir, "metric_influence_metric_ranking_across_apps.csv")
    metric_summary.to_csv(rank_path, index=False)
else:
    rank_path = None

# -------------------------
# Optional: Cross-app GROUP ranking summary (from group ablation)
# - If dropping a group hurts ACC (delta_acc negative), that group is useful
# -------------------------
if len(group_ablation_df):
    group_summary = (group_ablation_df
        .groupby("group")
        .agg(
            mean_delta_acc=("delta_acc","mean"),
            std_delta_acc=("delta_acc","std"),
            mean_delta_f1=("delta_f1","mean"),
            std_delta_f1=("delta_f1","std"),
            apps_total=("delta_acc","count"),
        )
        .reset_index()
    )

    # More negative delta_acc => more important group
    group_summary["score"] = (
        (-group_summary["mean_delta_acc"])
        / (1.0 + group_summary["std_delta_acc"].fillna(0.0))
    )

    group_summary = group_summary.sort_values(
        ["score", "mean_delta_f1"],
        ascending=[False, True]   # delta_f1: more negative is worse, so True helps keep it aligned
    )

    group_rank_path = os.path.join(out_dir, "metric_influence_group_ranking_across_apps.csv")
    group_summary.to_csv(group_rank_path, index=False)
else:
    group_rank_path = None

print("\n================= SAVED FILES =================")
print("Baseline CV:", baseline_path)
print("Perm importance:", perm_path)
print("Group ablation:", group_ablation_path)
print("Drop-one metric:", drop_one_path)
if rank_path:
    print("Metric ranking:", rank_path)
if group_rank_path:
    print("Group ranking:", group_rank_path)

print("\nDone.")


APP: Archery | CSV: /content/metrics_Archery.csv
[Archery FOLD 1] acc=0.929 f1=0.951 prec=0.951 rec=0.951 | cm=[[21, 3], [3, 58]]
[Archery FOLD 2] acc=0.952 f1=0.966 prec=0.983 rec=0.950 | cm=[[23, 1], [3, 57]]
[Archery FOLD 3] acc=0.905 f1=0.930 prec=0.981 rec=0.883 | cm=[[23, 1], [7, 53]]
[Archery FOLD 4] acc=0.952 f1=0.967 prec=0.952 rec=0.983 | cm=[[21, 3], [1, 59]]
[Archery FOLD 5] acc=0.988 f1=0.992 prec=1.000 rec=0.984 | cm=[[23, 0], [1, 60]]
[Archery GROUP ABLATION] drop=validity_missingness  base_acc=0.945 ablated_acc=0.945 delta_acc=+0.000 | base_f1=0.961 ablated_f1=0.961 delta_f1=+0.000
[Archery GROUP ABLATION] drop=   global_position  base_acc=0.945 ablated_acc=0.945 delta_acc=-0.000 | base_f1=0.961 ablated_f1=0.961 delta_f1=+0.000
[Archery GROUP ABLATION] drop=  bbox_body_extent  base_acc=0.945 ablated_acc=0.898 delta_acc=-0.048 | base_f1=0.961 ablated_f1=0.926 delta_f1=-0.035
[Archery GROUP ABLATION] drop=      floor_safety  base_acc=0.945 ablated_acc=0.945 delta_acc=+0.

/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [ 12  13  14  39  40  41  66  67  68  93  94  95 120 121 122 147 148 149]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [ 12  13  14  39  40  41  66  67  68  93  94  95 120 121 122 147 148 149]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [ 12  13  14  39  40  41  66  67  68  93  94  95 120 121 122 147 148 149]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping featur

[Puzzle FOLD 1] acc=0.946 f1=0.964 prec=1.000 rec=0.930 | cm=[[13, 0], [3, 40]]


/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [ 12  13  14  39  40  41  66  67  68  93  94  95 120 121 122 147 148 149]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [ 12  13  14  39  40  41  66  67  68  93  94  95 120 121 122 147 148 149]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [ 12  13  14  39  40  41  66  67  68  93  94  95 120 121 122 147 148 149]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping featur

[Puzzle FOLD 2] acc=0.945 f1=0.965 prec=0.953 rec=0.976 | cm=[[11, 2], [1, 41]]


...

[Puzzle GROUP ABLATION] drop=  bbox_body_extent  base_acc=0.957 ablated_acc=0.957 delta_acc=+0.000 | base_f1=0.972 ablated_f1=0.972 delta_f1=+0.000
[Puzzle GROUP ABLATION] drop=      floor_safety  base_acc=0.957 ablated_acc=0.957 delta_acc=+0.000 | base_f1=0.972 ablated_f1=0.972 delta_f1=+0.000


/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [ 12  13  14  33  34  35  54  55  56  75  76  77  96  97  98 117 118 119]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [ 12  13  14  33  34  35  54  55  56  75  76  77  96  97  98 117 118 119]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [ 12  13  14  33  34  35  54  55  56  75  76  77  96  97  98 117 118 119]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping featur

[Puzzle GROUP ABLATION] drop=  anatomy_symmetry  base_acc=0.957 ablated_acc=0.946 delta_acc=-0.011 | base_f1=0.972 ablated_f1=0.965 delta_f1=-0.007
[Puzzle GROUP ABLATION] drop=       orientation  base_acc=0.957 ablated_acc=0.964 delta_acc=+0.007 | base_f1=0.972 ablated_f1=0.976 delta_f1=+0.005

APP: Sea | CSV: /content/metrics_Sea.csv


/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [ 12  13  14  33  34  35  54  55  56  75  76  77  96  97  98 117 118 119]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


[Sea FOLD 1] acc=0.983 f1=0.981 prec=1.000 rec=0.963 | cm=[[33, 0], [1, 26]]
[Sea FOLD 2] acc=1.000 f1=1.000 prec=1.000 rec=1.000 | cm=[[33, 0], [0, 26]]
[Sea FOLD 3] acc=0.983 f1=0.982 prec=0.964 rec=1.000 | cm=[[31, 1], [0, 27]]
[Sea FOLD 4] acc=0.915 f1=0.915 prec=0.844 rec=1.000 | cm=[[27, 5], [0, 27]]
[Sea FOLD 5] acc=0.949 f1=0.945 prec=0.929 rec=0.963 | cm=[[30, 2], [1, 26]]
[Sea GROUP ABLATION] drop=validity_missingness  base_acc=0.966 ablated_acc=0.966 delta_acc=+0.000 | base_f1=0.965 ablated_f1=0.965 delta_f1=+0.000
[Sea GROUP ABLATION] drop=   global_position  base_acc=0.966 ablated_acc=0.953 delta_acc=-0.014 | base_f1=0.965 ablated_f1=0.951 delta_f1=-0.014
[Sea GROUP ABLATION] drop=  bbox_body_extent  base_acc=0.966 ablated_acc=0.963 delta_acc=-0.003 | base_f1=0.965 ablated_f1=0.961 delta_f1=-0.004
[Sea GROUP ABLATION] drop=      floor_safety  base_acc=0.966 ablated_acc=0.970 delta_acc=+0.003 | base_f1=0.965 ablated_f1=0.968 delta_f1=+0.003
[Sea GROUP ABLATION] drop=  anato

In [ ]:
# ============================================================
# Comprehensive "metrics study" notebook block (paper-ready)
# For a DataFrame `df` with:
#   - label_col: binary label (0/1)
#   - metric_cols: your candidate invariant metrics (numeric)
# Optional:
#   - group_col: person/run/session id to prevent leakage
#
# What this produces (all fold-pure):
#  1) Single-metric performance (F1-focused) + CI
#  2) Ablations: all-metrics, drop-one, add-one-from-best
#  3) Correlation + redundancy groups
#  4) Permutation importance (fold-pure) for top model
#  5) Sanity checks: label shuffle, duplicate cross-split rate
#  6) Exports: CSV tables you can paste into the paper
# ============================================================

import os
import numpy as np
import pandas as pd

from dataclasses import dataclass
from typing import List, Optional, Dict, Tuple

from sklearn.model_selection import StratifiedKFold
try:
    from sklearn.model_selection import StratifiedGroupKFold
    HAS_SGK = True
except Exception:
    HAS_SGK = False

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    f1_score, accuracy_score, precision_score, recall_score, confusion_matrix
)
from sklearn.inspection import permutation_importance


# -----------------------------
# Utilities
# -----------------------------

def _pick_numeric_cols(df: pd.DataFrame, exclude: set) -> List[str]:
    return [c for c in df.columns if c not in exclude and pd.api.types.is_numeric_dtype(df[c])]

def _drop_train_all_nan_cols(X_tr: pd.DataFrame, X_te: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, List[str]]:
    all_nan_cols = X_tr.columns[X_tr.isna().all()].tolist()
    keep = [c for c in X_tr.columns if c not in all_nan_cols]
    return X_tr[keep], X_te[keep], all_nan_cols

def _dup_cross_split_rate(X_train: pd.DataFrame, X_test: pd.DataFrame) -> float:
    sentinel = "__NaN__SENTINEL__"
    tr = X_train.copy().astype("object").fillna(sentinel)
    te = X_test.copy().astype("object").fillna(sentinel)
    tr_hash = pd.util.hash_pandas_object(tr, index=False)
    te_hash = pd.util.hash_pandas_object(te, index=False)
    tr_set = set(tr_hash.values.tolist())
    dup = np.isin(te_hash.values, list(tr_set))
    return float(np.mean(dup)) if len(dup) else 0.0

def _bootstrap_ci(values: np.ndarray, n_boot: int = 2000, alpha: float = 0.05, seed: int = 0) -> Tuple[float, float]:
    rng = np.random.RandomState(seed)
    vals = np.asarray(values, dtype=float)
    if len(vals) == 0:
        return (np.nan, np.nan)
    boots = []
    for _ in range(n_boot):
        sample = rng.choice(vals, size=len(vals), replace=True)
        boots.append(np.mean(sample))
    lo = np.percentile(boots, 100 * (alpha / 2))
    hi = np.percentile(boots, 100 * (1 - alpha / 2))
    return float(lo), float(hi)

def _make_pipeline() -> Pipeline:
    # Strong and stable baseline for metrics tables
    return Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=5000, class_weight="balanced", solver="lbfgs")),
    ])

@dataclass
class FoldResult:
    fold: int
    f1: float
    acc: float
    prec: float
    rec: float
    n_features: int
    dropped_all_nan_cols: int
    dup_cross_split_rate: float
    cm_00: int
    cm_01: int
    cm_10: int
    cm_11: int

def _evaluate_one_feature_set_cv(
    df: pd.DataFrame,
    label_col: str,
    feature_cols: List[str],
    group_col: Optional[str] = None,
    n_splits: int = 5,
    random_state: int = 42,
    positive_label: int = 1,
    verbose: bool = False,
) -> Tuple[pd.DataFrame, Dict]:
    X = df[feature_cols].copy()
    y = df[label_col].astype(int).to_numpy()

    if group_col is not None:
        if not HAS_SGK:
            raise RuntimeError("StratifiedGroupKFold not available in this sklearn version.")
        groups = df[group_col].to_numpy()
        splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
        split_iter = splitter.split(X, y, groups)
    else:
        splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
        split_iter = splitter.split(X, y)

    rows: List[FoldResult] = []
    cms = []

    for fold, (tr_idx, te_idx) in enumerate(split_iter, start=1):
        X_tr = X.iloc[tr_idx].copy()
        X_te = X.iloc[te_idx].copy()
        y_tr = y[tr_idx]
        y_te = y[te_idx]

        X_tr, X_te, all_nan_cols = _drop_train_all_nan_cols(X_tr, X_te)
        dup_rate = _dup_cross_split_rate(X_tr, X_te)

        pipe = _make_pipeline()
        pipe.fit(X_tr, y_tr)
        y_pred = pipe.predict(X_te)

        f1 = f1_score(y_te, y_pred, pos_label=positive_label, zero_division=0)
        acc = accuracy_score(y_te, y_pred)
        prec = precision_score(y_te, y_pred, pos_label=positive_label, zero_division=0)
        rec = recall_score(y_te, y_pred, pos_label=positive_label, zero_division=0)
        cm = confusion_matrix(y_te, y_pred, labels=[0, 1])
        cms.append(cm)

        rows.append(FoldResult(
            fold=fold, f1=float(f1), acc=float(acc), prec=float(prec), rec=float(rec),
            n_features=int(X_tr.shape[1]),
            dropped_all_nan_cols=int(len(all_nan_cols)),
            dup_cross_split_rate=float(dup_rate),
            cm_00=int(cm[0,0]), cm_01=int(cm[0,1]), cm_10=int(cm[1,0]), cm_11=int(cm[1,1])
        ))

        if verbose:
            print(f"[Fold {fold}] f1={f1:.3f} acc={acc:.3f} prec={prec:.3f} rec={rec:.3f} "
                  f"| feats={X_tr.shape[1]} | dropNaN={len(all_nan_cols)} | dup={dup_rate:.3f}")

    per_fold = pd.DataFrame([r.__dict__ for r in rows])

    cm_sum = np.sum(np.stack(cms, axis=0), axis=0)
    tn, fp, fn, tp = cm_sum.ravel().tolist()
    micro_acc = (tp + tn) / max(1, (tp + tn + fp + fn))
    micro_prec = tp / max(1, (tp + fp))
    micro_rec = tp / max(1, (tp + fn))
    micro_f1 = 2 * micro_prec * micro_rec / max(1e-12, (micro_prec + micro_rec))

    f1_ci = _bootstrap_ci(per_fold["f1"].to_numpy(), seed=random_state)

    summary = {
        "mean_f1": float(per_fold["f1"].mean()),
        "std_f1": float(per_fold["f1"].std(ddof=1)) if len(per_fold) > 1 else 0.0,
        "ci95_f1_lo": f1_ci[0],
        "ci95_f1_hi": f1_ci[1],
        "mean_acc": float(per_fold["acc"].mean()),
        "mean_prec": float(per_fold["prec"].mean()),
        "mean_rec": float(per_fold["rec"].mean()),
        "mean_dup_cross_split_rate": float(per_fold["dup_cross_split_rate"].mean()),
        "cm_sum": cm_sum,
        "micro_f1_from_cm_sum": float(micro_f1),
        "micro_acc_from_cm_sum": float(micro_acc),
        "micro_prec_from_cm_sum": float(micro_prec),
        "micro_rec_from_cm_sum": float(micro_rec),
        "n_features_requested": int(len(feature_cols)),
        "n_features_used_mean": float(per_fold["n_features"].mean()),
    }

    return per_fold, summary


# -----------------------------
# Study: correlation + redundancy
# -----------------------------

def metrics_redundancy_report(df: pd.DataFrame, metric_cols: List[str], corr_abs_threshold: float = 0.90) -> pd.DataFrame:
    X = df[metric_cols].copy()
    # median-fill for corr only (NOT for training)
    X = X.fillna(X.median(numeric_only=True))
    corr = X.corr(method="spearman").abs()
    # upper triangle pairs
    pairs = []
    cols = corr.columns.tolist()
    for i in range(len(cols)):
        for j in range(i+1, len(cols)):
            v = corr.iloc[i, j]
            if np.isfinite(v) and v >= corr_abs_threshold:
                pairs.append({"metric_a": cols[i], "metric_b": cols[j], "abs_spearman_corr": float(v)})
    pairs_df = pd.DataFrame(pairs).sort_values("abs_spearman_corr", ascending=False)
    return pairs_df


# -----------------------------
# Study: single-metric table
# -----------------------------

def single_metric_study(
    df: pd.DataFrame,
    label_col: str,
    metric_cols: List[str],
    group_col: Optional[str] = None,
    n_splits: int = 5,
    random_state: int = 42,
    positive_label: int = 1,
) -> pd.DataFrame:
    rows = []
    for m in metric_cols:
        per_fold, summary = _evaluate_one_feature_set_cv(
            df, label_col, [m],
            group_col=group_col, n_splits=n_splits, random_state=random_state, positive_label=positive_label
        )
        rows.append({
            "metric": m,
            "mean_f1": summary["mean_f1"],
            "ci95_f1_lo": summary["ci95_f1_lo"],
            "ci95_f1_hi": summary["ci95_f1_hi"],
            "mean_acc": summary["mean_acc"],
            "mean_prec": summary["mean_prec"],
            "mean_rec": summary["mean_rec"],
            "mean_dup_cross_split_rate": summary["mean_dup_cross_split_rate"],
            "micro_f1": summary["micro_f1_from_cm_sum"],
        })
    out = pd.DataFrame(rows).sort_values("mean_f1", ascending=False).reset_index(drop=True)
    return out


# -----------------------------
# Study: ablations (all, drop-one, add-one)
# -----------------------------

def ablation_study(
    df: pd.DataFrame,
    label_col: str,
    metric_cols: List[str],
    group_col: Optional[str] = None,
    n_splits: int = 5,
    random_state: int = 42,
    positive_label: int = 1,
    top_k_single: int = 10,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    # 1) ALL metrics baseline
    _, all_sum = _evaluate_one_feature_set_cv(
        df, label_col, metric_cols,
        group_col=group_col, n_splits=n_splits, random_state=random_state, positive_label=positive_label
    )
    all_row = pd.DataFrame([{
        "setting": "ALL",
        "mean_f1": all_sum["mean_f1"],
        "ci95_f1_lo": all_sum["ci95_f1_lo"],
        "ci95_f1_hi": all_sum["ci95_f1_hi"],
        "mean_acc": all_sum["mean_acc"],
        "mean_prec": all_sum["mean_prec"],
        "mean_rec": all_sum["mean_rec"],
        "micro_f1": all_sum["micro_f1_from_cm_sum"],
        "mean_dup_cross_split_rate": all_sum["mean_dup_cross_split_rate"],
        "n_metrics": len(metric_cols),
    }])

    # 2) DROP-ONE: remove each metric from ALL
    drop_rows = []
    for m in metric_cols:
        cols = [c for c in metric_cols if c != m]
        _, s = _evaluate_one_feature_set_cv(
            df, label_col, cols,
            group_col=group_col, n_splits=n_splits, random_state=random_state, positive_label=positive_label
        )
        drop_rows.append({
            "dropped_metric": m,
            "mean_f1": s["mean_f1"],
            "delta_vs_all_f1": s["mean_f1"] - all_sum["mean_f1"],
            "mean_acc": s["mean_acc"],
            "micro_f1": s["micro_f1_from_cm_sum"],
        })
    drop_df = pd.DataFrame(drop_rows).sort_values("delta_vs_all_f1").reset_index(drop=True)  # most harmful drop first

    # 3) ADD-ONE: start from best single metric, add one metric at a time
    single_df = single_metric_study(
        df, label_col, metric_cols,
        group_col=group_col, n_splits=n_splits, random_state=random_state, positive_label=positive_label
    )
    seeds = single_df.head(top_k_single)["metric"].tolist()
    add_rows = []
    for seed in seeds:
        remaining = [m for m in metric_cols if m != seed]
        # Evaluate adding each remaining metric to the seed
        for m in remaining:
            cols = [seed, m]
            _, s = _evaluate_one_feature_set_cv(
                df, label_col, cols,
                group_col=group_col, n_splits=n_splits, random_state=random_state, positive_label=positive_label
            )
            add_rows.append({
                "seed_metric": seed,
                "added_metric": m,
                "mean_f1": s["mean_f1"],
                "mean_acc": s["mean_acc"],
            })
    add_df = pd.DataFrame(add_rows).sort_values("mean_f1", ascending=False).reset_index(drop=True)

    return all_row, drop_df, add_df


# -----------------------------
# Study: permutation importance (fold-pure)
# -----------------------------

def permutation_importance_cv(
    df: pd.DataFrame,
    label_col: str,
    metric_cols: List[str],
    group_col: Optional[str] = None,
    n_splits: int = 5,
    random_state: int = 42,
    positive_label: int = 1,
    n_repeats: int = 15,
) -> pd.DataFrame:
    X = df[metric_cols].copy()
    y = df[label_col].astype(int).to_numpy()

    if group_col is not None:
        if not HAS_SGK:
            raise RuntimeError("StratifiedGroupKFold not available in this sklearn version.")
        groups = df[group_col].to_numpy()
        splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
        split_iter = splitter.split(X, y, groups)
    else:
        splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
        split_iter = splitter.split(X, y)

    imp_accum = {m: [] for m in metric_cols}

    for fold, (tr_idx, te_idx) in enumerate(split_iter, start=1):
        X_tr = X.iloc[tr_idx].copy()
        X_te = X.iloc[te_idx].copy()
        y_tr = y[tr_idx]
        y_te = y[te_idx]

        X_tr, X_te, _ = _drop_train_all_nan_cols(X_tr, X_te)

        pipe = _make_pipeline()
        pipe.fit(X_tr, y_tr)

        # Permutation importance uses scoring; we want F1
        def f1_scorer(estimator, X_eval, y_eval):
            y_pred = estimator.predict(X_eval)
            return f1_score(y_eval, y_pred, pos_label=positive_label, zero_division=0)

        r = permutation_importance(
            pipe, X_te, y_te,
            scoring=f1_scorer,
            n_repeats=n_repeats,
            random_state=random_state,
            n_jobs=-1
        )

        # Align importances with the columns actually used (after NaN drop)
        used_cols = X_tr.columns.tolist()
        for i, col in enumerate(used_cols):
            imp_accum[col].append(float(r.importances_mean[i]))

    rows = []
    for m, vals in imp_accum.items():
        if len(vals) == 0:
            continue
        rows.append({
            "metric": m,
            "mean_perm_importance_f1": float(np.mean(vals)),
            "std_perm_importance_f1": float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0,
            "n_folds_used": int(len(vals)),
        })
    return pd.DataFrame(rows).sort_values("mean_perm_importance_f1", ascending=False).reset_index(drop=True)


# -----------------------------
# Sanity: label shuffle
# -----------------------------

def label_shuffle_sanity(
    df: pd.DataFrame,
    label_col: str,
    metric_cols: List[str],
    group_col: Optional[str] = None,
    n_splits: int = 5,
    random_state: int = 42,
    positive_label: int = 1,
) -> Dict:
    rng = np.random.RandomState(random_state)
    df2 = df.copy()
    df2[label_col] = rng.permutation(df2[label_col].to_numpy())
    _, s = _evaluate_one_feature_set_cv(
        df2, label_col, metric_cols,
        group_col=group_col, n_splits=n_splits, random_state=random_state, positive_label=positive_label
    )
    return s


# ============================================================
# ======================= RUN THIS PART ======================
# ============================================================

# ---- EDIT THESE 3 LINES ----
LABEL_COL = "Temporal"       # <-- change
GROUP_COL = None              # e.g., "PersonID" or "RunID" or "MovementID" (optional but recommended if you have it)
METRIC_COLS = None            # <-- set list of metric columns; if None, auto-select numeric (excluding label/group)

# df = ...  # ensure df is loaded

exclude = {LABEL_COL}
if GROUP_COL is not None:
    exclude.add(GROUP_COL)

if METRIC_COLS is None:
    METRIC_COLS = _pick_numeric_cols(df, exclude=exclude)

print(f"#metrics={len(METRIC_COLS)} | label={LABEL_COL} | group={GROUP_COL}")

# Output directory
OUT_DIR = "metrics_study_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

# 0) Redundancy / correlation report
redundant_pairs = metrics_redundancy_report(df, METRIC_COLS, corr_abs_threshold=0.90)
redundant_pairs.to_csv(os.path.join(OUT_DIR, "redundant_pairs_abs_spearman_ge_0.90.csv"), index=False)
print(f"Saved redundant pairs: {len(redundant_pairs)}")

# 1) Single-metric leaderboard
single_table = single_metric_study(
    df, LABEL_COL, METRIC_COLS,
    group_col=GROUP_COL, n_splits=5, random_state=42, positive_label=1
)
single_table.to_csv(os.path.join(OUT_DIR, "single_metric_leaderboard.csv"), index=False)
print("Top single metrics:")
print(single_table.head(10))

# 2) Ablations: ALL, DROP-ONE, ADD-ONE
all_row, drop_one_table, add_one_table = ablation_study(
    df, LABEL_COL, METRIC_COLS,
    group_col=GROUP_COL, n_splits=5, random_state=42, positive_label=1,
    top_k_single=10
)
all_row.to_csv(os.path.join(OUT_DIR, "all_metrics_baseline.csv"), index=False)
drop_one_table.to_csv(os.path.join(OUT_DIR, "drop_one_ablation.csv"), index=False)
add_one_table.to_csv(os.path.join(OUT_DIR, "add_one_pairs_from_top_singles.csv"), index=False)

print("\nALL-metrics baseline:")
print(all_row)

print("\nMost harmful drops (lower delta = more important metric):")
print(drop_one_table.head(15))

print("\nBest 2-metric pairs (seed from top singles):")
print(add_one_table.head(15))

# 3) Permutation importance on ALL metrics (fold-pure)
perm_imp = permutation_importance_cv(
    df, LABEL_COL, METRIC_COLS,
    group_col=GROUP_COL, n_splits=5, random_state=42, positive_label=1,
    n_repeats=15
)
perm_imp.to_csv(os.path.join(OUT_DIR, "permutation_importance_f1.csv"), index=False)
print("\nTop permutation importances:")
print(perm_imp.head(15))

# 4) Sanity check: label shuffle on ALL metrics (should collapse)
shuf = label_shuffle_sanity(df, LABEL_COL, METRIC_COLS, group_col=GROUP_COL, n_splits=5, random_state=42, positive_label=1)
print("\nLabel-shuffle sanity (expect low F1):")
print({k: shuf[k] for k in ["mean_f1","mean_acc","micro_f1_from_cm_sum","mean_dup_cross_split_rate"]})

print(f"\nAll outputs saved in: {OUT_DIR}")

#metrics=24 | label=Temporal | group=None
Saved redundant pairs: 2


/tmp/ipython-input-590476670.py:57: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  tr = X_train.copy().astype("object").fillna(sentinel)
/tmp/ipython-input-590476670.py:58: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  te = X_test.copy().astype("object").fillna(sentinel)
/tmp/ipython-input-590476670.py:57: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcastin

Top single metrics:
                        metric   mean_f1  ci95_f1_lo  ci95_f1_hi  mean_acc  \
0                      Spatial  0.747721    0.732728    0.763537  0.780374   
1             left_shin_length  0.668556    0.629016    0.697210  0.745987   
2            right_shin_length  0.653043    0.620106    0.680645  0.719033   
3               body_forward_y  0.641364    0.616808    0.670272  0.646211   
4          leg_length_symmetry  0.609513    0.592973    0.631493  0.627300   
5         missing_joints_count  0.602125    0.599928    0.604086  0.439224   
6         missing_joints_ratio  0.602125    0.599928    0.604086  0.439224   
7       collapsed_joints_count  0.595863    0.592526    0.598699  0.428327   
8                  bbox_height  0.581146    0.546653    0.606613  0.619843   
9  max_joint_distance_from_com  0.580928    0.545695    0.607562  0.618695   

   mean_prec  mean_rec  mean_dup_cross_split_rate  micro_f1  
0   0.730085  0.767568                   1.000000  0.747860

/tmp/ipython-input-590476670.py:57: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  tr = X_train.copy().astype("object").fillna(sentinel)
/tmp/ipython-input-590476670.py:58: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  te = X_test.copy().astype("object").fillna(sentinel)
/tmp/ipython-input-590476670.py:57: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcastin


ALL-metrics baseline:
  setting   mean_f1  ci95_f1_lo  ci95_f1_hi  mean_acc  mean_prec  mean_rec  \
0     ALL  0.892751    0.885998    0.899144  0.908258   0.886001       0.9   

   micro_f1  mean_dup_cross_split_rate  n_metrics  
0  0.892761                   0.005161         24  

Most harmful drops (lower delta = more important metric):
            dropped_metric   mean_f1  delta_vs_all_f1  mean_acc  micro_f1
0                  Spatial  0.862251        -0.030499  0.878444  0.862338
1           body_forward_z  0.872906        -0.019845  0.890493  0.872751
2           body_forward_x  0.884825        -0.007926  0.900802  0.884744
3     distance_from_origin  0.887131        -0.005619  0.903672  0.887097
4         center_of_mass_x  0.888464        -0.004287  0.904820  0.888441
5      leg_length_symmetry  0.891532        -0.001219  0.907112  0.891566
6              bbox_volume  0.891639        -0.001112  0.907107  0.891566
7      arm_length_symmetry  0.892128        -0.000623  0.907685  

/tmp/ipython-input-590476670.py:57: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  tr = X_train.copy().astype("object").fillna(sentinel)
/tmp/ipython-input-590476670.py:58: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  te = X_test.copy().astype("object").fillna(sentinel)
/tmp/ipython-input-590476670.py:57: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcastin


Label-shuffle sanity (expect low F1):
{'mean_f1': 0.4031758149434534, 'mean_acc': 0.46846820142937123, 'micro_f1_from_cm_sum': 0.4038585209003216, 'mean_dup_cross_split_rate': 0.005157593123209169}

All outputs saved in: metrics_study_outputs


## LR - cross app

In [ ]:
# ============================================================
# Cross-App "metrics study" (LOAO / cross-app, paper-ready)
# Same outputs as per-app CV, but evaluation unit = held-out APP
#
# Produces:
#  0) Redundancy / Spearman abs corr pairs
#  1) Single-metric LOAO leaderboard + CI over apps
#  2) Ablations: ALL, DROP-ONE, ADD-ONE (LOAO)
#  3) Permutation importance (LOAO, test-app permutation; train on other apps)
#  4) Sanity: label shuffle (LOAO)
#  5) Exports CSVs for paper tables
# ============================================================

import os
import numpy as np
import pandas as pd

from dataclasses import dataclass
from typing import List, Optional, Dict, Tuple

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    f1_score, accuracy_score, precision_score, recall_score, confusion_matrix
)
from sklearn.inspection import permutation_importance


# -----------------------------
# Utilities (reuse style)
# -----------------------------
def _pick_numeric_cols(df: pd.DataFrame, exclude: set) -> List[str]:
    return [c for c in df.columns if c not in exclude and pd.api.types.is_numeric_dtype(df[c])]

def _drop_train_all_nan_cols(X_tr: pd.DataFrame, X_te: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, List[str]]:
    all_nan_cols = X_tr.columns[X_tr.isna().all()].tolist()
    keep = [c for c in X_tr.columns if c not in all_nan_cols]
    return X_tr[keep], X_te[keep], all_nan_cols

def _dup_cross_split_rate(X_train: pd.DataFrame, X_test: pd.DataFrame) -> float:
    sentinel = "__NaN__SENTINEL__"
    tr = X_train.copy().astype("object").fillna(sentinel)
    te = X_test.copy().astype("object").fillna(sentinel)
    tr_hash = pd.util.hash_pandas_object(tr, index=False)
    te_hash = pd.util.hash_pandas_object(te, index=False)
    tr_set = set(tr_hash.values.tolist())
    dup = np.isin(te_hash.values, list(tr_set))
    return float(np.mean(dup)) if len(dup) else 0.0

def _bootstrap_ci(values: np.ndarray, n_boot: int = 5000, alpha: float = 0.05, seed: int = 0) -> Tuple[float, float]:
    rng = np.random.RandomState(seed)
    vals = np.asarray(values, dtype=float)
    vals = vals[np.isfinite(vals)]
    if len(vals) == 0:
        return (np.nan, np.nan)
    boots = []
    for _ in range(n_boot):
        sample = rng.choice(vals, size=len(vals), replace=True)
        boots.append(np.mean(sample))
    lo = np.percentile(boots, 100 * (alpha / 2))
    hi = np.percentile(boots, 100 * (1 - alpha / 2))
    return float(lo), float(hi)

def _make_pipeline() -> Pipeline:
    # match your best cross-app LR: stable, balanced
    return Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=5000, class_weight="balanced", solver="lbfgs")),
    ])


@dataclass
class AppResult:
    held_out_app: str
    f1: float
    acc: float
    prec: float
    rec: float
    n_train: int
    n_test: int
    n_features_used: int
    dropped_all_nan_cols: int
    dup_cross_split_rate: float
    cm_00: int
    cm_01: int
    cm_10: int
    cm_11: int


# -----------------------------
# Core LOAO evaluator (feature-set)
# -----------------------------
def _evaluate_one_feature_set_loao(
    df: pd.DataFrame,
    app_col: str,
    label_col: str,
    feature_cols: List[str],
    apps: Optional[List[str]] = None,
    positive_label: int = 1,
    random_state: int = 42,
    verbose: bool = False,
) -> Tuple[pd.DataFrame, Dict]:
    if apps is None:
        apps = sorted(df[app_col].dropna().unique().tolist())

    rows: List[AppResult] = []
    cms = []

    for test_app in apps:
        train_df = df[df[app_col] != test_app].copy()
        test_df  = df[df[app_col] == test_app].copy()

        # If an app has no samples, skip (shouldn't happen but safe)
        if len(train_df) == 0 or len(test_df) == 0:
            continue

        X_tr = train_df[feature_cols].copy()
        y_tr = train_df[label_col].astype(int).to_numpy()

        X_te = test_df[feature_cols].copy()
        y_te = test_df[label_col].astype(int).to_numpy()

        # Fold-pure NaN column drop: based ONLY on training (train apps)
        X_tr, X_te, all_nan_cols = _drop_train_all_nan_cols(X_tr, X_te)

        dup_rate = _dup_cross_split_rate(X_tr, X_te)

        pipe = _make_pipeline()
        pipe.fit(X_tr, y_tr)
        y_pred = pipe.predict(X_te)

        f1 = f1_score(y_te, y_pred, pos_label=positive_label, zero_division=0)
        acc = accuracy_score(y_te, y_pred)
        prec = precision_score(y_te, y_pred, pos_label=positive_label, zero_division=0)
        rec = recall_score(y_te, y_pred, pos_label=positive_label, zero_division=0)

        cm = confusion_matrix(y_te, y_pred, labels=[0, 1])
        cms.append(cm)

        rows.append(AppResult(
            held_out_app=str(test_app),
            f1=float(f1), acc=float(acc), prec=float(prec), rec=float(rec),
            n_train=int(len(y_tr)), n_test=int(len(y_te)),
            n_features_used=int(X_tr.shape[1]),
            dropped_all_nan_cols=int(len(all_nan_cols)),
            dup_cross_split_rate=float(dup_rate),
            cm_00=int(cm[0,0]), cm_01=int(cm[0,1]), cm_10=int(cm[1,0]), cm_11=int(cm[1,1]),
        ))

        if verbose:
            print(f"[LOAO] HELD-OUT={test_app:12s} f1={f1:.3f} acc={acc:.3f} "
                  f"prec={prec:.3f} rec={rec:.3f} | feats={X_tr.shape[1]} "
                  f"| dropNaN={len(all_nan_cols)} | dup={dup_rate:.3f} | n_test={len(y_te)}")

    per_app = pd.DataFrame([r.__dict__ for r in rows])

    # Aggregate confusion matrix across held-out apps (micro)
    if len(cms) > 0:
        cm_sum = np.sum(np.stack(cms, axis=0), axis=0)
        tn, fp, fn, tp = cm_sum.ravel().tolist()
        micro_acc = (tp + tn) / max(1, (tp + tn + fp + fn))
        micro_prec = tp / max(1, (tp + fp))
        micro_rec = tp / max(1, (tp + fn))
        micro_f1 = 2 * micro_prec * micro_rec / max(1e-12, (micro_prec + micro_rec))
    else:
        cm_sum = np.array([[0,0],[0,0]])
        micro_acc = micro_prec = micro_rec = micro_f1 = np.nan

    # CI over held-out apps (your cross-app unit)
    f1_ci = _bootstrap_ci(per_app["f1"].to_numpy(), seed=random_state)

    summary = {
        "mean_f1": float(per_app["f1"].mean()) if len(per_app) else np.nan,
        "std_f1": float(per_app["f1"].std(ddof=1)) if len(per_app) > 1 else 0.0,
        "ci95_f1_lo": f1_ci[0],
        "ci95_f1_hi": f1_ci[1],
        "mean_acc": float(per_app["acc"].mean()) if len(per_app) else np.nan,
        "mean_prec": float(per_app["prec"].mean()) if len(per_app) else np.nan,
        "mean_rec": float(per_app["rec"].mean()) if len(per_app) else np.nan,
        "mean_dup_cross_split_rate": float(per_app["dup_cross_split_rate"].mean()) if len(per_app) else np.nan,
        "cm_sum": cm_sum,
        "micro_f1_from_cm_sum": float(micro_f1),
        "micro_acc_from_cm_sum": float(micro_acc),
        "micro_prec_from_cm_sum": float(micro_prec),
        "micro_rec_from_cm_sum": float(micro_rec),
        "n_apps_used": int(len(per_app)),
        "n_features_requested": int(len(feature_cols)),
        "n_features_used_mean": float(per_app["n_features_used"].mean()) if len(per_app) else np.nan,
    }

    return per_app, summary


# -----------------------------
# Redundancy / correlation (same)
# -----------------------------
def metrics_redundancy_report(df: pd.DataFrame, metric_cols: List[str], corr_abs_threshold: float = 0.90) -> pd.DataFrame:
    X = df[metric_cols].copy()
    X = X.fillna(X.median(numeric_only=True))
    corr = X.corr(method="spearman").abs()
    pairs = []
    cols = corr.columns.tolist()
    for i in range(len(cols)):
        for j in range(i+1, len(cols)):
            v = corr.iloc[i, j]
            if np.isfinite(v) and v >= corr_abs_threshold:
                pairs.append({"metric_a": cols[i], "metric_b": cols[j], "abs_spearman_corr": float(v)})
    return pd.DataFrame(pairs).sort_values("abs_spearman_corr", ascending=False).reset_index(drop=True)


# -----------------------------
# Single-metric LOAO table
# -----------------------------
def single_metric_study_loao(
    df: pd.DataFrame,
    app_col: str,
    label_col: str,
    metric_cols: List[str],
    apps: Optional[List[str]] = None,
    random_state: int = 42,
    positive_label: int = 1,
) -> pd.DataFrame:
    rows = []
    for m in metric_cols:
        _, s = _evaluate_one_feature_set_loao(
            df, app_col, label_col, [m],
            apps=apps, positive_label=positive_label, random_state=random_state
        )
        rows.append({
            "metric": m,
            "mean_f1": s["mean_f1"],
            "ci95_f1_lo": s["ci95_f1_lo"],
            "ci95_f1_hi": s["ci95_f1_hi"],
            "mean_acc": s["mean_acc"],
            "mean_prec": s["mean_prec"],
            "mean_rec": s["mean_rec"],
            "micro_f1": s["micro_f1_from_cm_sum"],
            "mean_dup_cross_split_rate": s["mean_dup_cross_split_rate"],
            "n_apps_used": s["n_apps_used"],
        })
    return pd.DataFrame(rows).sort_values("mean_f1", ascending=False).reset_index(drop=True)


# -----------------------------
# Ablations (ALL, DROP-ONE, ADD-ONE) under LOAO
# -----------------------------
def ablation_study_loao(
    df: pd.DataFrame,
    app_col: str,
    label_col: str,
    metric_cols: List[str],
    apps: Optional[List[str]] = None,
    random_state: int = 42,
    positive_label: int = 1,
    top_k_single: int = 10,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:

    # ALL baseline
    _, all_sum = _evaluate_one_feature_set_loao(
        df, app_col, label_col, metric_cols,
        apps=apps, positive_label=positive_label, random_state=random_state
    )
    all_row = pd.DataFrame([{
        "setting": "ALL_LOAO",
        "mean_f1": all_sum["mean_f1"],
        "ci95_f1_lo": all_sum["ci95_f1_lo"],
        "ci95_f1_hi": all_sum["ci95_f1_hi"],
        "mean_acc": all_sum["mean_acc"],
        "mean_prec": all_sum["mean_prec"],
        "mean_rec": all_sum["mean_rec"],
        "micro_f1": all_sum["micro_f1_from_cm_sum"],
        "mean_dup_cross_split_rate": all_sum["mean_dup_cross_split_rate"],
        "n_metrics": len(metric_cols),
        "n_apps_used": all_sum["n_apps_used"],
    }])

    # DROP-ONE
    drop_rows = []
    for m in metric_cols:
        cols = [c for c in metric_cols if c != m]
        _, s = _evaluate_one_feature_set_loao(
            df, app_col, label_col, cols,
            apps=apps, positive_label=positive_label, random_state=random_state
        )
        drop_rows.append({
            "dropped_metric": m,
            "mean_f1": s["mean_f1"],
            "delta_vs_all_f1": s["mean_f1"] - all_sum["mean_f1"],
            "mean_acc": s["mean_acc"],
            "micro_f1": s["micro_f1_from_cm_sum"],
        })
    drop_df = pd.DataFrame(drop_rows).sort_values("delta_vs_all_f1").reset_index(drop=True)

    # ADD-ONE pairs from top singles
    single_df = single_metric_study_loao(
        df, app_col, label_col, metric_cols,
        apps=apps, random_state=random_state, positive_label=positive_label
    )
    seeds = single_df.head(top_k_single)["metric"].tolist()
    add_rows = []
    for seed in seeds:
        for m in metric_cols:
            if m == seed:
                continue
            cols = [seed, m]
            _, s = _evaluate_one_feature_set_loao(
                df, app_col, label_col, cols,
                apps=apps, positive_label=positive_label, random_state=random_state
            )
            add_rows.append({
                "seed_metric": seed,
                "added_metric": m,
                "mean_f1": s["mean_f1"],
                "mean_acc": s["mean_acc"],
                "micro_f1": s["micro_f1_from_cm_sum"],
            })
    add_df = pd.DataFrame(add_rows).sort_values("mean_f1", ascending=False).reset_index(drop=True)

    return all_row, drop_df, add_df


# -----------------------------
# Permutation importance under LOAO (test-app permutation; train on other apps)
# -----------------------------
def permutation_importance_loao(
    df: pd.DataFrame,
    app_col: str,
    label_col: str,
    metric_cols: List[str],
    apps: Optional[List[str]] = None,
    random_state: int = 42,
    positive_label: int = 1,
    n_repeats: int = 15,
) -> pd.DataFrame:
    if apps is None:
        apps = sorted(df[app_col].dropna().unique().tolist())

    imp_accum = {m: [] for m in metric_cols}

    for test_app in apps:
        train_df = df[df[app_col] != test_app].copy()
        test_df  = df[df[app_col] == test_app].copy()
        if len(train_df) == 0 or len(test_df) == 0:
            continue

        X_tr = train_df[metric_cols].copy()
        y_tr = train_df[label_col].astype(int).to_numpy()

        X_te = test_df[metric_cols].copy()
        y_te = test_df[label_col].astype(int).to_numpy()

        X_tr, X_te, _ = _drop_train_all_nan_cols(X_tr, X_te)

        pipe = _make_pipeline()
        pipe.fit(X_tr, y_tr)

        def f1_scorer(estimator, X_eval, y_eval):
            y_pred = estimator.predict(X_eval)
            return f1_score(y_eval, y_pred, pos_label=positive_label, zero_division=0)

        r = permutation_importance(
            pipe, X_te, y_te,
            scoring=f1_scorer,
            n_repeats=n_repeats,
            random_state=random_state,
            n_jobs=-1
        )

        used_cols = X_tr.columns.tolist()
        for i, col in enumerate(used_cols):
            imp_accum[col].append(float(r.importances_mean[i]))

    rows = []
    for m, vals in imp_accum.items():
        vals = [v for v in vals if np.isfinite(v)]
        if len(vals) == 0:
            continue
        rows.append({
            "metric": m,
            "mean_perm_importance_f1": float(np.mean(vals)),
            "std_perm_importance_f1": float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0,
            "n_apps_used": int(len(vals)),
        })

    return pd.DataFrame(rows).sort_values("mean_perm_importance_f1", ascending=False).reset_index(drop=True)


# -----------------------------
# Sanity: label shuffle under LOAO
# -----------------------------
def label_shuffle_sanity_loao(
    df: pd.DataFrame,
    app_col: str,
    label_col: str,
    metric_cols: List[str],
    apps: Optional[List[str]] = None,
    random_state: int = 42,
    positive_label: int = 1,
) -> Dict:
    rng = np.random.RandomState(random_state)
    df2 = df.copy()
    df2[label_col] = rng.permutation(df2[label_col].to_numpy())
    _, s = _evaluate_one_feature_set_loao(
        df2, app_col, label_col, metric_cols,
        apps=apps, positive_label=positive_label, random_state=random_state
    )
    return s


# ============================================================
# ======================= RUN THIS PART ======================
# ============================================================

# ---- EDIT THESE LINES ----
APP_COL   = "App"        # e.g., "App"
LABEL_COL = "Temporal"   # your label
METRIC_COLS = None       # set list explicitly, or auto-select numeric
APPS = None              # or list like ["Archery","PhantomLimb",...]
POS_LABEL = 1
SEED = 42

# df = ...  # ensure df is loaded

exclude = {APP_COL, LABEL_COL}
if METRIC_COLS is None:
    METRIC_COLS = _pick_numeric_cols(df, exclude=exclude)

if APPS is None:
    APPS = sorted(df[APP_COL].dropna().unique().tolist())

print(f"#apps={len(APPS)} | #metrics={len(METRIC_COLS)} | label={LABEL_COL} | app_col={APP_COL}")

OUT_DIR = "metrics_study_outputs_loao"
os.makedirs(OUT_DIR, exist_ok=True)

# 0) Redundancy report (global; same as before)
redundant_pairs = metrics_redundancy_report(df, METRIC_COLS, corr_abs_threshold=0.90)
redundant_pairs.to_csv(os.path.join(OUT_DIR, "redundant_pairs_abs_spearman_ge_0.90.csv"), index=False)
print(f"Saved redundant pairs: {len(redundant_pairs)}")

# 1) Single-metric LOAO leaderboard
single_table = single_metric_study_loao(
    df, APP_COL, LABEL_COL, METRIC_COLS,
    apps=APPS, random_state=SEED, positive_label=POS_LABEL
)
single_table.to_csv(os.path.join(OUT_DIR, "single_metric_leaderboard.csv"), index=False)
print("Top single metrics (LOAO):")
print(single_table.head(10))

# 2) Ablations LOAO: ALL, DROP-ONE, ADD-ONE
all_row, drop_one_table, add_one_table = ablation_study_loao(
    df, APP_COL, LABEL_COL, METRIC_COLS,
    apps=APPS, random_state=SEED, positive_label=POS_LABEL,
    top_k_single=10
)
all_row.to_csv(os.path.join(OUT_DIR, "all_metrics_baseline.csv"), index=False)
drop_one_table.to_csv(os.path.join(OUT_DIR, "drop_one_ablation.csv"), index=False)
add_one_table.to_csv(os.path.join(OUT_DIR, "add_one_pairs_from_top_singles.csv"), index=False)

print("\nALL-metrics baseline (LOAO):")
print(all_row)

print("\nMost harmful drops (more negative delta = more important metric):")
print(drop_one_table.head(15))

print("\nBest 2-metric pairs (seeded from top singles, LOAO):")
print(add_one_table.head(15))

# 3) Permutation importance LOAO on ALL metrics (train on other apps, permute on held-out app)
perm_imp = permutation_importance_loao(
    df, APP_COL, LABEL_COL, METRIC_COLS,
    apps=APPS, random_state=SEED, positive_label=POS_LABEL,
    n_repeats=15
)
perm_imp.to_csv(os.path.join(OUT_DIR, "permutation_importance_f1.csv"), index=False)
print("\nTop permutation importances (LOAO):")
print(perm_imp.head(15))

# 4) Sanity: label shuffle (should collapse)
shuf = label_shuffle_sanity_loao(df, APP_COL, LABEL_COL, METRIC_COLS, apps=APPS, random_state=SEED, positive_label=POS_LABEL)
print("\nLabel-shuffle sanity (LOAO; expect low F1):")
print({k: shuf[k] for k in ["mean_f1","mean_acc","micro_f1_from_cm_sum","mean_dup_cross_split_rate","n_apps_used"]})

print(f"\nAll LOAO outputs saved in: {OUT_DIR}")

#apps=6 | #metrics=24 | label=Temporal | app_col=App
Saved redundant pairs: 2


/tmp/ipython-input-473236407.py:46: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  te = X_test.copy().astype("object").fillna(sentinel)
/tmp/ipython-input-473236407.py:46: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  te = X_test.copy().astype("object").fillna(sentinel)
/tmp/ipython-input-473236407.py:46: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting

Top single metrics (LOAO):
                   metric   mean_f1  ci95_f1_lo  ci95_f1_hi  mean_acc  \
0                 Spatial  0.719707    0.410137    0.980561  0.835696   
1  collapsed_joints_count  0.627769    0.448659    0.794497  0.499438   
2          body_forward_x  0.570532    0.438303    0.702492  0.625843   
3          body_forward_z  0.559353    0.307588    0.773665  0.562537   
4     leg_length_symmetry  0.542107    0.267838    0.788985  0.676693   
5     distance_from_floor  0.524597    0.268241    0.776147  0.512502   
6     arm_length_symmetry  0.513261    0.258975    0.761495  0.605209   
7          body_forward_y  0.502207    0.201995    0.794388  0.666513   
8    missing_joints_count  0.490004    0.249591    0.711664  0.425559   
9    missing_joints_ratio  0.490004    0.249591    0.711664  0.425559   

   mean_prec  mean_rec  micro_f1  mean_dup_cross_split_rate  n_apps_used  
0   0.827455  0.698504  0.747860                   1.000000            6  
1   0.496469  0.991

/tmp/ipython-input-473236407.py:45: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  tr = X_train.copy().astype("object").fillna(sentinel)
/tmp/ipython-input-473236407.py:46: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  te = X_test.copy().astype("object").fillna(sentinel)
/tmp/ipython-input-473236407.py:45: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcastin


ALL-metrics baseline (LOAO):
    setting   mean_f1  ci95_f1_lo  ci95_f1_hi  mean_acc  mean_prec  mean_rec  \
0  ALL_LOAO  0.604893    0.399495    0.811652  0.610245    0.62186  0.844981   

   micro_f1  mean_dup_cross_split_rate  n_metrics  n_apps_used  
0  0.621622                        0.0         24            6  

Most harmful drops (more negative delta = more important metric):
                 dropped_metric   mean_f1  delta_vs_all_f1  mean_acc  micro_f1
0          distance_from_origin  0.446030        -0.158863  0.478726  0.559129
1                   bbox_volume  0.454239        -0.150654  0.516423  0.470790
2                    bbox_depth  0.458071        -0.146822  0.525297  0.475145
3                    bbox_width  0.502614        -0.102279  0.536563  0.501684
4                body_forward_z  0.541646        -0.063247  0.650881  0.610227
5                body_forward_x  0.549707        -0.055186  0.585762  0.600299
6              center_of_mass_x  0.563831        -0.041062 

/tmp/ipython-input-473236407.py:45: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  tr = X_train.copy().astype("object").fillna(sentinel)
/tmp/ipython-input-473236407.py:46: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  te = X_test.copy().astype("object").fillna(sentinel)
/tmp/ipython-input-473236407.py:45: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcastin


Label-shuffle sanity (LOAO; expect low F1):
{'mean_f1': 0.4185308935840906, 'mean_acc': 0.49567039240952293, 'micro_f1_from_cm_sum': 0.4879518072289157, 'mean_dup_cross_split_rate': 0.0, 'n_apps_used': 6}

All LOAO outputs saved in: metrics_study_outputs_loao


# -Spatial

## HGB - per app

In [ ]:
# ===============================================================
# Spatial Metric Influence Study (Conference-grade)
# - Baseline: Spatial HGB (metrics-only, 27 invariants)
# - Per-app 5-fold Stratified CV (frame-level)
# - Permutation importance by BASE METRIC (permute 1 metric column)
# - Group ablation (drop groups of metrics, rerun CV)
#
# Outputs (saved to /content):
#   spatial_metric_influence_baseline_cv.csv
#   spatial_metric_influence_perm_importance_by_metric.csv
#   spatial_metric_influence_group_ablation.csv
#   spatial_metric_influence_metric_ranking_across_apps.csv
# ===============================================================

import os
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.ensemble import HistGradientBoostingClassifier

# -------------------------
# Settings
# -------------------------
SEED = 42
np.random.seed(SEED)

LABEL_COL = "Spatial"
N_SPLITS = 5

PERM_REPEATS = 5
DO_PERM_IMPORTANCE = True
DO_GROUP_ABLATION = True
DO_DROP_ONE_METRIC = False   # optional (very expensive): 6 apps * 27 * 5 folds

# -------------------------
# Invariant metrics (27)
# -------------------------
INVARIANT27 = [
    "missing_joints_count","missing_joints_ratio","collapsed_joints_count",
    "center_of_mass_x","center_of_mass_y","center_of_mass_z","distance_from_origin",
    "bbox_width","bbox_height","bbox_depth","bbox_volume","max_joint_distance_from_com",
    "distance_from_floor","min_foot_height_above_floor","below_floor",
    "left_forearm_length","right_forearm_length","left_shin_length","right_shin_length",
    "arm_length_symmetry","leg_length_symmetry",
    "body_upright_x","body_upright_y","body_upright_z",
    "body_forward_x","body_forward_y","body_forward_z"
]

# -------------------------
# Metric groups (same spirit as your temporal study)
# -------------------------
METRIC_GROUPS = {
    "validity_missingness": [
        "missing_joints_count","missing_joints_ratio","collapsed_joints_count"
    ],
    "global_position": [
        "center_of_mass_x","center_of_mass_y","center_of_mass_z","distance_from_origin"
    ],
    "bbox_body_extent": [
        "bbox_width","bbox_height","bbox_depth","bbox_volume","max_joint_distance_from_com"
    ],
    "floor_safety": [
        "distance_from_floor","min_foot_height_above_floor","below_floor"
    ],
    "anatomy_symmetry": [
        "left_forearm_length","right_forearm_length","left_shin_length","right_shin_length",
        "arm_length_symmetry","leg_length_symmetry"
    ],
    "orientation": [
        "body_upright_x","body_upright_y","body_upright_z",
        "body_forward_x","body_forward_y","body_forward_z"
    ],
}

# -------------------------
# Apps -> per-app metrics CSV paths
# -------------------------
apps = {
    "Archery":      "/content/metrics_Archery.csv",
    "PhantomLimb":  "/content/metrics_PhantomLimb.csv",
    "PianoTiles":   "/content/metrics_PianoTiles.csv",
    "Puzzle":       "/content/metrics_Puzzle.csv",
    "Sea":          "/content/metrics_Sea.csv",
    "War":          "/content/metrics_War.csv",
}

# -------------------------
# Helpers
# -------------------------
def make_hgb():
    # Conference-grade stable default; tweak later if needed
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", HistGradientBoostingClassifier(
            random_state=SEED,
            max_depth=6,
            learning_rate=0.05,
            max_iter=400
        ))
    ])

def eval_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=[0,1]).tolist()
    return float(acc), float(f1), float(prec), float(rec), cm

def load_xy_spatial(df, feature_cols):
    df = df.copy().replace("", np.nan)

    # label
    df[LABEL_COL] = pd.to_numeric(df[LABEL_COL], errors="coerce")
    df = df.dropna(subset=[LABEL_COL]).copy()
    df[LABEL_COL] = df[LABEL_COL].astype(int)

    # features
    df_num = df[feature_cols].apply(pd.to_numeric, errors="coerce")
    X = df_num.to_numpy(dtype=float)
    y = df[LABEL_COL].to_numpy(dtype=int)

    return X, y

def perm_importance_by_metric(model, X_test, y_test, feature_cols, n_repeats=5):
    # baseline
    base_pred = model.predict(X_test)
    base_acc = accuracy_score(y_test, base_pred)
    base_f1  = f1_score(y_test, base_pred, zero_division=0)

    rng = np.random.default_rng(SEED)
    drops = {}

    # permute ONE column at a time (spatial = no agg columns)
    for j, c in enumerate(feature_cols):
        acc_scores, f1_scores = [], []
        for _ in range(n_repeats):
            Xp = X_test.copy()
            perm = rng.permutation(len(Xp))
            Xp[:, j] = Xp[perm, j]
            p = model.predict(Xp)
            acc_scores.append(accuracy_score(y_test, p))
            f1_scores.append(f1_score(y_test, p, zero_division=0))

        drops[c] = {
            "acc_drop": float(base_acc - np.mean(acc_scores)),
            "f1_drop":  float(base_f1  - np.mean(f1_scores)),
        }

    return float(base_acc), float(base_f1), drops

def run_baseline_and_perm(app_name, df_app, feature_cols):
    X_all, y_all = load_xy_spatial(df_app, feature_cols)
    if len(y_all) < 50:
        raise RuntimeError(f"[{app_name}] Too few rows after filtering (n={len(y_all)}).")

    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

    baseline_rows = []
    perm_rows = []

    for fold, (tr, te) in enumerate(skf.split(np.zeros(len(y_all)), y_all), 1):
        m = make_hgb()
        m.fit(X_all[tr], y_all[tr])
        yhat = m.predict(X_all[te])
        acc, f1, prec, rec, cm = eval_metrics(y_all[te], yhat)

        baseline_rows.append({
            "app": app_name, "fold": fold,
            "accuracy": acc, "f1": f1, "precision": prec, "recall": rec,
            "tn": cm[0][0], "fp": cm[0][1], "fn": cm[1][0], "tp": cm[1][1],
            "n_test": int(len(te)),
        })

        if DO_PERM_IMPORTANCE:
            base_acc, base_f1, drops = perm_importance_by_metric(
                m, X_all[te], y_all[te], feature_cols, n_repeats=PERM_REPEATS
            )
            for metric, d in drops.items():
                perm_rows.append({
                    "app": app_name, "fold": fold,
                    "metric": metric,
                    "base_acc": base_acc,
                    "base_f1": base_f1,
                    "acc_drop": d["acc_drop"],
                    "f1_drop":  d["f1_drop"],
                })

        print(f"[{app_name} FOLD {fold}] acc={acc:.3f} f1={f1:.3f} prec={prec:.3f} rec={rec:.3f} | cm={cm}")

    baseline_df = pd.DataFrame(baseline_rows)
    perm_df = pd.DataFrame(perm_rows) if len(perm_rows) else pd.DataFrame(columns=["app","fold","metric","base_f1","f1_drop"])
    return baseline_df, perm_df

def run_group_ablation(app_name, df_app, feature_cols, groups_dict):
    X0, y0 = load_xy_spatial(df_app, feature_cols)
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

    # baseline mean f1
    f1s0 = []
    for tr, te in skf.split(np.zeros(len(y0)), y0):
        m0 = make_hgb()
        m0.fit(X0[tr], y0[tr])
        f1s0.append(f1_score(y0[te], m0.predict(X0[te]), zero_division=0))
    base_mean_f1 = float(np.mean(f1s0))

    rows = []
    for gname, gmetrics in groups_dict.items():
        kept = [c for c in feature_cols if c not in set(gmetrics)]
        Xg, yg = load_xy_spatial(df_app, kept)

        f1s = []
        for tr, te in skf.split(np.zeros(len(yg)), yg):
            mg = make_hgb()
            mg.fit(Xg[tr], yg[tr])
            f1s.append(f1_score(yg[te], mg.predict(Xg[te]), zero_division=0))

        mean_f1 = float(np.mean(f1s))
        rows.append({
            "app": app_name,
            "group": gname,
            "dropped_metrics": ",".join(gmetrics),
            "base_mean_f1": base_mean_f1,
            "ablated_mean_f1": mean_f1,
            "delta_f1": float(mean_f1 - base_mean_f1),  # negative means group helped
            "n_metrics_dropped": int(len(gmetrics)),
        })

        print(f"[{app_name} GROUP ABLATION] drop={gname:>18s}  base_f1={base_mean_f1:.3f}  ablated_f1={mean_f1:.3f}  delta={mean_f1-base_mean_f1:+.3f}")

    return pd.DataFrame(rows)

def run_drop_one_metric(app_name, df_app, feature_cols):
    X0, y0 = load_xy_spatial(df_app, feature_cols)
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

    f1s0 = []
    for tr, te in skf.split(np.zeros(len(y0)), y0):
        m0 = make_hgb()
        m0.fit(X0[tr], y0[tr])
        f1s0.append(f1_score(y0[te], m0.predict(X0[te]), zero_division=0))
    base_mean_f1 = float(np.mean(f1s0))

    rows = []
    for drop_metric in feature_cols:
        kept = [c for c in feature_cols if c != drop_metric]
        Xg, yg = load_xy_spatial(df_app, kept)

        f1s = []
        for tr, te in skf.split(np.zeros(len(yg)), yg):
            mg = make_hgb()
            mg.fit(Xg[tr], yg[tr])
            f1s.append(f1_score(yg[te], mg.predict(Xg[te]), zero_division=0))

        mean_f1 = float(np.mean(f1s))
        rows.append({
            "app": app_name,
            "dropped_metric": drop_metric,
            "base_mean_f1": base_mean_f1,
            "ablated_mean_f1": mean_f1,
            "delta_f1": float(mean_f1 - base_mean_f1),
        })
        print(f"[{app_name} DROP-ONE] drop={drop_metric:>28s} base_f1={base_mean_f1:.3f} ablated_f1={mean_f1:.3f} delta={mean_f1-base_mean_f1:+.3f}")
    return pd.DataFrame(rows)

# ===============================================================
# RUN ALL APPS
# ===============================================================
feature_cols = INVARIANT27[:]

baseline_all = []
perm_all = []
group_ablation_all = []
drop_one_all = []

for app, path in apps.items():
    print("\n" + "="*70)
    print(f"APP: {app} | CSV: {path}")
    print("="*70)

    df_app = pd.read_csv(path)

    # sanity checks
    missing = [c for c in feature_cols if c not in df_app.columns]
    if missing:
        raise RuntimeError(f"[{app}] Missing invariant cols: {missing}")
    if LABEL_COL not in df_app.columns:
        raise RuntimeError(f"[{app}] Missing label col '{LABEL_COL}'")

    # baseline + permutation importance
    bdf, pdf = run_baseline_and_perm(app, df_app, feature_cols)
    baseline_all.append(bdf)
    perm_all.append(pdf)

    # group ablation
    if DO_GROUP_ABLATION:
        gadf = run_group_ablation(app, df_app, feature_cols, METRIC_GROUPS)
        group_ablation_all.append(gadf)

    # optional drop-one metric ablation
    if DO_DROP_ONE_METRIC:
        do_df = run_drop_one_metric(app, df_app, feature_cols)
        drop_one_all.append(do_df)

# -------------------------
# Save outputs
# -------------------------
out_dir = "/content"
os.makedirs(out_dir, exist_ok=True)

baseline_df = pd.concat(baseline_all, ignore_index=True)
baseline_path = os.path.join(out_dir, "spatial_metric_influence_baseline_cv.csv")
baseline_df.to_csv(baseline_path, index=False)

perm_df = pd.concat(perm_all, ignore_index=True) if len(perm_all) else pd.DataFrame()
perm_path = os.path.join(out_dir, "spatial_metric_influence_perm_importance_by_metric.csv")
perm_df.to_csv(perm_path, index=False)

if DO_GROUP_ABLATION and len(group_ablation_all):
    group_ablation_df = pd.concat(group_ablation_all, ignore_index=True)
else:
    group_ablation_df = pd.DataFrame()
group_ablation_path = os.path.join(out_dir, "spatial_metric_influence_group_ablation.csv")
group_ablation_df.to_csv(group_ablation_path, index=False)

if DO_DROP_ONE_METRIC and len(drop_one_all):
    drop_one_df = pd.concat(drop_one_all, ignore_index=True)
else:
    drop_one_df = pd.DataFrame()
drop_one_path = os.path.join(out_dir, "spatial_metric_influence_drop_one_metric.csv")
drop_one_df.to_csv(drop_one_path, index=False)

# -------------------------
# Cross-app ranking summary (from permutation importance)
# -------------------------
if len(perm_df):
    pivot = (perm_df
             .groupby(["app","metric"])["f1_drop"]
             .mean()
             .reset_index())

    metric_summary = (pivot
        .groupby("metric")["f1_drop"]
        .agg(
            mean_drop="mean",
            std_drop="std",
            support_apps=lambda x: int((x > 0.01).sum()),  # threshold for "meaningful"
            apps_total="count"
        )
        .reset_index()
    )
    metric_summary["score"] = metric_summary["mean_drop"] * metric_summary["support_apps"] / (1.0 + metric_summary["std_drop"].fillna(0.0))
    metric_summary = metric_summary.sort_values("score", ascending=False)

    rank_path = os.path.join(out_dir, "spatial_metric_influence_metric_ranking_across_apps.csv")
    metric_summary.to_csv(rank_path, index=False)
else:
    rank_path = None

print("\n================= SAVED FILES =================")
print("Baseline CV:", baseline_path)
print("Perm importance:", perm_path)
print("Group ablation:", group_ablation_path)
print("Drop-one metric:", drop_one_path)
if rank_path:
    print("Metric ranking:", rank_path)
print("\nDone.")


APP: Archery | CSV: /content/metrics_Archery.csv
[Archery FOLD 1] acc=0.988 f1=0.993 prec=1.000 rec=0.985 | cm=[[17, 0], [1, 67]]
[Archery FOLD 2] acc=0.976 f1=0.985 prec=0.985 rec=0.985 | cm=[[16, 1], [1, 67]]
[Archery FOLD 3] acc=0.953 f1=0.971 prec=0.957 rec=0.985 | cm=[[14, 3], [1, 67]]
[Archery FOLD 4] acc=0.976 f1=0.986 prec=0.986 rec=0.986 | cm=[[15, 1], [1, 68]]
[Archery FOLD 5] acc=0.929 f1=0.957 prec=0.944 rec=0.971 | cm=[[12, 4], [2, 67]]
[Archery GROUP ABLATION] drop=validity_missingness  base_f1=0.978  ablated_f1=0.978  delta=+0.000
[Archery GROUP ABLATION] drop=   global_position  base_f1=0.978  ablated_f1=0.978  delta=-0.000
[Archery GROUP ABLATION] drop=  bbox_body_extent  base_f1=0.978  ablated_f1=0.975  delta=-0.003
[Archery GROUP ABLATION] drop=      floor_safety  base_f1=0.978  ablated_f1=0.977  delta=-0.002
[Archery GROUP ABLATION] drop=  anatomy_symmetry  base_f1=0.978  ablated_f1=0.980  delta=+0.001
[Archery GROUP ABLATION] drop=       orientation  base_f1=0.978

/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [12 13 14]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [12 13 14]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [12 13 14]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [12 13 14]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/

[Puzzle FOLD 1] acc=1.000 f1=1.000 prec=1.000 rec=1.000 | cm=[[18, 0], [0, 42]]


/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [12 13 14]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [12 13 14]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [12 13 14]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [12 13 14]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/

[Puzzle FOLD 2] acc=0.983 f1=0.989 prec=0.977 rec=1.000 | cm=[[16, 1], [0, 43]]


...

[Puzzle GROUP ABLATION] drop=  bbox_body_extent  base_f1=0.975  ablated_f1=0.979  delta=+0.004
[Puzzle GROUP ABLATION] drop=      floor_safety  base_f1=0.975  ablated_f1=0.975  delta=+0.000


/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [12 13 14]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [12 13 14]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [12 13 14]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [12 13 14]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/

[Puzzle GROUP ABLATION] drop=  anatomy_symmetry  base_f1=0.975  ablated_f1=0.972  delta=-0.003


/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [12 13 14]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [12 13 14]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [12 13 14]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [12 13 14]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/

[Puzzle GROUP ABLATION] drop=       orientation  base_f1=0.975  ablated_f1=0.975  delta=+0.000

APP: Sea | CSV: /content/metrics_Sea.csv
[Sea FOLD 1] acc=0.983 f1=0.980 prec=1.000 rec=0.962 | cm=[[34, 0], [1, 25]]
[Sea FOLD 2] acc=1.000 f1=1.000 prec=1.000 rec=1.000 | cm=[[34, 0], [0, 26]]
[Sea FOLD 3] acc=0.967 f1=0.963 prec=0.929 rec=1.000 | cm=[[32, 2], [0, 26]]
[Sea FOLD 4] acc=0.967 f1=0.964 prec=0.931 rec=1.000 | cm=[[31, 2], [0, 27]]
[Sea FOLD 5] acc=0.933 f1=0.926 prec=0.926 rec=0.926 | cm=[[31, 2], [2, 25]]
[Sea GROUP ABLATION] drop=validity_missingness  base_f1=0.967  ablated_f1=0.967  delta=+0.000
[Sea GROUP ABLATION] drop=   global_position  base_f1=0.967  ablated_f1=0.967  delta=-0.000
[Sea GROUP ABLATION] drop=  bbox_body_extent  base_f1=0.967  ablated_f1=0.966  delta=-0.001
[Sea GROUP ABLATION] drop=      floor_safety  base_f1=0.967  ablated_f1=0.967  delta=+0.000
[Sea GROUP ABLATION] drop=  anatomy_symmetry  base_f1=0.967  ablated_f1=0.963  delta=-0.004
[Sea GROUP ABLAT

## HGB - cross App

In [ ]:
# ============================================================
# Cross-App Spatial "metrics study" (LOAO / paper-ready)
# Model: HGB (metrics-only, 27 invariants)
#
# Produces (folder: /content/spatial_metrics_study_outputs_loao):
#  0) redundant_pairs_abs_spearman_ge_0.90.csv
#  1) single_metric_leaderboard_loao.csv
#  2) all_metrics_baseline_loao.csv
#  3) drop_one_ablation_loao.csv
#  4) add_one_pairs_from_top_singles_loao.csv
#  5) permutation_importance_loao_f1.csv
#  6) label_shuffle_sanity_loao.json
# ============================================================

import os, json
import numpy as np
import pandas as pd

from dataclasses import dataclass
from typing import List, Optional, Dict, Tuple

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score, confusion_matrix
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance

# -----------------------------
# Settings
# -----------------------------
SEED = 42
np.random.seed(SEED)

APP_COL   = "App"       # MUST exist in combined df
LABEL_COL = "Spatial"   # spatial label
POS_LABEL = 1

# Use your invariant list
METRIC_COLS = [
    "missing_joints_count","missing_joints_ratio","collapsed_joints_count",
    "center_of_mass_x","center_of_mass_y","center_of_mass_z","distance_from_origin",
    "bbox_width","bbox_height","bbox_depth","bbox_volume","max_joint_distance_from_com",
    "distance_from_floor","min_foot_height_above_floor","below_floor",
    "left_forearm_length","right_forearm_length","left_shin_length","right_shin_length",
    "arm_length_symmetry","leg_length_symmetry",
    "body_upright_x","body_upright_y","body_upright_z",
    "body_forward_x","body_forward_y","body_forward_z"
]

CORR_ABS_THRESH = 0.90
BOOTSTRAP_N = 5000
PERM_REPEATS = 15
TOP_K_SINGLE_FOR_ADDONE = 10

OUT_DIR = "/content/spatial_metrics_study_outputs_loao"
os.makedirs(OUT_DIR, exist_ok=True)

# -----------------------------
# Utilities
# -----------------------------
def _drop_train_all_nan_cols(X_tr: pd.DataFrame, X_te: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, List[str]]:
    all_nan_cols = X_tr.columns[X_tr.isna().all()].tolist()
    keep = [c for c in X_tr.columns if c not in all_nan_cols]
    return X_tr[keep], X_te[keep], all_nan_cols

def _dup_cross_split_rate(X_train: pd.DataFrame, X_test: pd.DataFrame) -> float:
    sentinel = "__NaN__SENTINEL__"
    tr = X_train.copy().astype("object").fillna(sentinel)
    te = X_test.copy().astype("object").fillna(sentinel)
    tr_hash = pd.util.hash_pandas_object(tr, index=False)
    te_hash = pd.util.hash_pandas_object(te, index=False)
    tr_set = set(tr_hash.values.tolist())
    dup = np.isin(te_hash.values, list(tr_set))
    return float(np.mean(dup)) if len(dup) else 0.0

def _bootstrap_ci(values: np.ndarray, n_boot: int = 5000, alpha: float = 0.05, seed: int = 0) -> Tuple[float, float]:
    rng = np.random.RandomState(seed)
    vals = np.asarray(values, dtype=float)
    vals = vals[np.isfinite(vals)]
    if len(vals) == 0:
        return (np.nan, np.nan)
    boots = []
    for _ in range(n_boot):
        sample = rng.choice(vals, size=len(vals), replace=True)
        boots.append(np.mean(sample))
    lo = np.percentile(boots, 100 * (alpha / 2))
    hi = np.percentile(boots, 100 * (1 - alpha / 2))
    return float(lo), float(hi)

def _make_hgb_pipeline() -> Pipeline:
    return Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", HistGradientBoostingClassifier(
            random_state=SEED,
            max_depth=6,
            learning_rate=0.05,
            max_iter=400
        ))
    ])

@dataclass
class AppResult:
    held_out_app: str
    f1: float
    acc: float
    prec: float
    rec: float
    n_train: int
    n_test: int
    n_features_used: int
    dropped_all_nan_cols: int
    dup_cross_split_rate: float
    cm_00: int
    cm_01: int
    cm_10: int
    cm_11: int




# ============================================================
# FORCE BUILD the correct combined df for LOAO (overwrite df)
# ============================================================
paths = {
    "Archery":      "/content/metrics_Archery.csv",
    "PhantomLimb":  "/content/metrics_PhantomLimb.csv",
    "PianoTiles":   "/content/metrics_PianoTiles.csv",
    "Puzzle":       "/content/metrics_Puzzle.csv",
    "Sea":          "/content/metrics_Sea.csv",
    "War":          "/content/metrics_War.csv",
}

dfs = []
for app, p in paths.items():
    d = pd.read_csv(p).replace("", np.nan)
    d["App"] = app
    dfs.append(d)
    print(f"[LOAD] {app:<12s} rows={len(d)} from {p}")

df = pd.concat(dfs, ignore_index=True)
print("[OK] combined df:", df.shape, "| has App?", ("App" in df.columns))


# -----------------------------
# Core LOAO evaluator (feature-set)
# -----------------------------
def evaluate_one_feature_set_loao(
    df: pd.DataFrame,
    app_col: str,
    label_col: str,
    feature_cols: List[str],
    apps: Optional[List[str]] = None,
    positive_label: int = 1,
    random_state: int = 42,
    verbose: bool = True,
) -> Tuple[pd.DataFrame, Dict]:

    if apps is None:
        apps = sorted(df[app_col].dropna().unique().tolist())

    rows: List[AppResult] = []
    cms = []

    for test_app in apps:
        train_df = df[df[app_col] != test_app].copy()
        test_df  = df[df[app_col] == test_app].copy()

        if len(train_df) == 0 or len(test_df) == 0:
            continue

        X_tr = train_df[feature_cols].copy()
        y_tr = train_df[label_col].astype(int).to_numpy()

        X_te = test_df[feature_cols].copy()
        y_te = test_df[label_col].astype(int).to_numpy()

        # train-only all-NaN column drop
        X_tr, X_te, all_nan_cols = _drop_train_all_nan_cols(X_tr, X_te)

        dup_rate = _dup_cross_split_rate(X_tr, X_te)

        pipe = _make_hgb_pipeline()
        pipe.fit(X_tr, y_tr)
        y_pred = pipe.predict(X_te)

        f1 = f1_score(y_te, y_pred, pos_label=positive_label, zero_division=0)
        acc = accuracy_score(y_te, y_pred)
        prec = precision_score(y_te, y_pred, pos_label=positive_label, zero_division=0)
        rec = recall_score(y_te, y_pred, pos_label=positive_label, zero_division=0)

        cm = confusion_matrix(y_te, y_pred, labels=[0, 1])
        cms.append(cm)

        rows.append(AppResult(
            held_out_app=str(test_app),
            f1=float(f1), acc=float(acc), prec=float(prec), rec=float(rec),
            n_train=int(len(y_tr)), n_test=int(len(y_te)),
            n_features_used=int(X_tr.shape[1]),
            dropped_all_nan_cols=int(len(all_nan_cols)),
            dup_cross_split_rate=float(dup_rate),
            cm_00=int(cm[0,0]), cm_01=int(cm[0,1]), cm_10=int(cm[1,0]), cm_11=int(cm[1,1]),
        ))

        if verbose:
            print(f"[LOAO-HGB] HELD-OUT={test_app:12s} f1={f1:.3f} acc={acc:.3f} "
                  f"prec={prec:.3f} rec={rec:.3f} | feats={X_tr.shape[1]} "
                  f"| dropNaN={len(all_nan_cols)} | dup={dup_rate:.3f} | n_test={len(y_te)}")

    per_app = pd.DataFrame([r.__dict__ for r in rows])

    # micro from summed cm
    if len(cms) > 0:
        cm_sum = np.sum(np.stack(cms, axis=0), axis=0)
        tn, fp, fn, tp = cm_sum.ravel().tolist()
        micro_acc = (tp + tn) / max(1, (tp + tn + fp + fn))
        micro_prec = tp / max(1, (tp + fp))
        micro_rec = tp / max(1, (tp + fn))
        micro_f1 = 2 * micro_prec * micro_rec / max(1e-12, (micro_prec + micro_rec))
    else:
        cm_sum = np.array([[0,0],[0,0]])
        micro_acc = micro_prec = micro_rec = micro_f1 = np.nan

    f1_ci = _bootstrap_ci(per_app["f1"].to_numpy(), n_boot=BOOTSTRAP_N, seed=random_state)

    summary = {
        "mean_f1": float(per_app["f1"].mean()) if len(per_app) else np.nan,
        "std_f1": float(per_app["f1"].std(ddof=1)) if len(per_app) > 1 else 0.0,
        "ci95_f1_lo": f1_ci[0],
        "ci95_f1_hi": f1_ci[1],
        "mean_acc": float(per_app["acc"].mean()) if len(per_app) else np.nan,
        "mean_prec": float(per_app["prec"].mean()) if len(per_app) else np.nan,
        "mean_rec": float(per_app["rec"].mean()) if len(per_app) else np.nan,
        "mean_dup_cross_split_rate": float(per_app["dup_cross_split_rate"].mean()) if len(per_app) else np.nan,
        "cm_sum": cm_sum.tolist(),
        "micro_f1_from_cm_sum": float(micro_f1),
        "micro_acc_from_cm_sum": float(micro_acc),
        "micro_prec_from_cm_sum": float(micro_prec),
        "micro_rec_from_cm_sum": float(micro_rec),
        "n_apps_used": int(len(per_app)),
        "n_features_requested": int(len(feature_cols)),
        "n_features_used_mean": float(per_app["n_features_used"].mean()) if len(per_app) else np.nan,
    }

    return per_app, summary

# -----------------------------
# Redundancy / correlation
# -----------------------------
def metrics_redundancy_report(df: pd.DataFrame, metric_cols: List[str], corr_abs_threshold: float = 0.90) -> pd.DataFrame:
    X = df[metric_cols].copy()
    X = X.apply(pd.to_numeric, errors="coerce")
    X = X.fillna(X.median(numeric_only=True))
    corr = X.corr(method="spearman").abs()
    pairs = []
    cols = corr.columns.tolist()
    for i in range(len(cols)):
        for j in range(i+1, len(cols)):
            v = corr.iloc[i, j]
            if np.isfinite(v) and v >= corr_abs_threshold:
                pairs.append({"metric_a": cols[i], "metric_b": cols[j], "abs_spearman_corr": float(v)})
    return pd.DataFrame(pairs).sort_values("abs_spearman_corr", ascending=False).reset_index(drop=True)

# -----------------------------
# Single-metric LOAO leaderboard
# -----------------------------
def single_metric_study_loao(df, app_col, label_col, metric_cols, apps=None, random_state=42, positive_label=1):
    rows = []
    for m in metric_cols:
        _, s = evaluate_one_feature_set_loao(
            df, app_col, label_col, [m],
            apps=apps, positive_label=positive_label, random_state=random_state, verbose=False
        )
        rows.append({
            "metric": m,
            "mean_f1": s["mean_f1"],
            "ci95_f1_lo": s["ci95_f1_lo"],
            "ci95_f1_hi": s["ci95_f1_hi"],
            "mean_acc": s["mean_acc"],
            "mean_prec": s["mean_prec"],
            "mean_rec": s["mean_rec"],
            "micro_f1": s["micro_f1_from_cm_sum"],
            "mean_dup_cross_split_rate": s["mean_dup_cross_split_rate"],
            "n_apps_used": s["n_apps_used"],
        })
    return pd.DataFrame(rows).sort_values("mean_f1", ascending=False).reset_index(drop=True)

# -----------------------------
# Ablations (ALL, DROP-ONE, ADD-ONE) LOAO
# -----------------------------
def ablation_study_loao(df, app_col, label_col, metric_cols, apps=None, random_state=42, positive_label=1, top_k_single=10):
    # ALL
    _, all_sum = evaluate_one_feature_set_loao(
        df, app_col, label_col, metric_cols,
        apps=apps, positive_label=positive_label, random_state=random_state, verbose=False
    )
    all_row = pd.DataFrame([{
        "setting": "ALL_LOAO_HGB",
        "mean_f1": all_sum["mean_f1"],
        "ci95_f1_lo": all_sum["ci95_f1_lo"],
        "ci95_f1_hi": all_sum["ci95_f1_hi"],
        "mean_acc": all_sum["mean_acc"],
        "mean_prec": all_sum["mean_prec"],
        "mean_rec": all_sum["mean_rec"],
        "micro_f1": all_sum["micro_f1_from_cm_sum"],
        "mean_dup_cross_split_rate": all_sum["mean_dup_cross_split_rate"],
        "n_metrics": len(metric_cols),
        "n_apps_used": all_sum["n_apps_used"],
    }])

    # DROP-ONE
    drop_rows = []
    for m in metric_cols:
        cols = [c for c in metric_cols if c != m]
        _, s = evaluate_one_feature_set_loao(
            df, app_col, label_col, cols,
            apps=apps, positive_label=positive_label, random_state=random_state, verbose=False
        )
        drop_rows.append({
            "dropped_metric": m,
            "mean_f1": s["mean_f1"],
            "delta_vs_all_f1": s["mean_f1"] - all_sum["mean_f1"],
            "mean_acc": s["mean_acc"],
            "micro_f1": s["micro_f1_from_cm_sum"],
        })
    drop_df = pd.DataFrame(drop_rows).sort_values("delta_vs_all_f1").reset_index(drop=True)

    # ADD-ONE seeded from top singles
    single_df = single_metric_study_loao(df, app_col, label_col, metric_cols, apps=apps, random_state=random_state, positive_label=positive_label)
    seeds = single_df.head(top_k_single)["metric"].tolist()

    add_rows = []
    for seed in seeds:
        for m in metric_cols:
            if m == seed:
                continue
            cols = [seed, m]
            _, s = evaluate_one_feature_set_loao(
                df, app_col, label_col, cols,
                apps=apps, positive_label=positive_label, random_state=random_state, verbose=False
            )
            add_rows.append({
                "seed_metric": seed,
                "added_metric": m,
                "mean_f1": s["mean_f1"],
                "mean_acc": s["mean_acc"],
                "micro_f1": s["micro_f1_from_cm_sum"],
            })
    add_df = pd.DataFrame(add_rows).sort_values("mean_f1", ascending=False).reset_index(drop=True)

    return all_row, drop_df, add_df

# -----------------------------
# Permutation importance LOAO (permute on held-out app)
# -----------------------------
def permutation_importance_loao(df, app_col, label_col, metric_cols, apps=None, random_state=42, positive_label=1, n_repeats=15):
    if apps is None:
        apps = sorted(df[app_col].dropna().unique().tolist())

    imp_accum = {m: [] for m in metric_cols}

    for test_app in apps:
        train_df = df[df[app_col] != test_app].copy()
        test_df  = df[df[app_col] == test_app].copy()
        if len(train_df) == 0 or len(test_df) == 0:
            continue

        X_tr = train_df[metric_cols].copy()
        y_tr = train_df[label_col].astype(int).to_numpy()

        X_te = test_df[metric_cols].copy()
        y_te = test_df[label_col].astype(int).to_numpy()

        X_tr, X_te, _ = _drop_train_all_nan_cols(X_tr, X_te)

        pipe = _make_hgb_pipeline()
        pipe.fit(X_tr, y_tr)

        def f1_scorer(estimator, X_eval, y_eval):
            y_pred = estimator.predict(X_eval)
            return f1_score(y_eval, y_pred, pos_label=positive_label, zero_division=0)

        r = permutation_importance(
            pipe, X_te, y_te,
            scoring=f1_scorer,
            n_repeats=n_repeats,
            random_state=random_state,
            n_jobs=-1
        )

        used_cols = X_tr.columns.tolist()
        for i, col in enumerate(used_cols):
            imp_accum[col].append(float(r.importances_mean[i]))

        print(f"[PERM-LOAO] held-out={test_app:12s} | used_cols={len(used_cols)}")

    rows = []
    for m, vals in imp_accum.items():
        vals = [v for v in vals if np.isfinite(v)]
        if len(vals) == 0:
            continue
        rows.append({
            "metric": m,
            "mean_perm_importance_f1": float(np.mean(vals)),
            "std_perm_importance_f1": float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0,
            "n_apps_used": int(len(vals)),
        })
    return pd.DataFrame(rows).sort_values("mean_perm_importance_f1", ascending=False).reset_index(drop=True)

# -----------------------------
# Sanity: label shuffle LOAO
# -----------------------------
def label_shuffle_sanity_loao(df, app_col, label_col, metric_cols, apps=None, random_state=42, positive_label=1):
    rng = np.random.RandomState(random_state)
    df2 = df.copy()
    df2[label_col] = rng.permutation(df2[label_col].to_numpy())
    _, s = evaluate_one_feature_set_loao(df2, app_col, label_col, metric_cols, apps=apps, random_state=random_state, positive_label=positive_label, verbose=False)
    return s

# ============================================================
# LOAD + RUN
# ============================================================

# ---- YOU MUST HAVE A COMBINED DF CALLED df ----
# It must contain:
#   - df["App"]      : app name strings (6 apps)
#   - df["Spatial"]  : 0/1 label
#   - the 27 metrics columns listed above
#
# If you already have df_all from your earlier sheet concatenation, do:
#   df = df_all.copy()
#
# Otherwise: build df by concatenating your per-app metrics CSVs and adding App column.

# Clean label + keep only needed columns
df = df.copy().replace("", np.nan)
df[LABEL_COL] = pd.to_numeric(df[LABEL_COL], errors="coerce")


# ============================================================
# FIX: guarantee an App column exists (and is named APP_COL)
# ============================================================

APP_COL   = "App"
LABEL_COL = "Spatial"

if "df" in globals():
    df = df.copy()
else:
    paths = {
        "Archery":      "/content/metrics_Archery.csv",
        "PhantomLimb":  "/content/metrics_PhantomLimb.csv",
        "PianoTiles":   "/content/metrics_PianoTiles.csv",
        "Puzzle":       "/content/metrics_Puzzle.csv",
        "Sea":          "/content/metrics_Sea.csv",
        "War":          "/content/metrics_War.csv",
    }
    dfs = []
    for app, p in paths.items():
        d = pd.read_csv(p).replace("", np.nan)
        d[APP_COL] = app               # <-- force App column
        dfs.append(d)
        print(f"[LOAD] {app:<12s} rows={len(d)} from {p}")
    df = pd.concat(dfs, ignore_index=True)

# If user already had an app column but with a different name, normalize it
if APP_COL not in df.columns:
    for alt in ["app", "APP", "Application", "application", "Game", "game"]:
        if alt in df.columns:
            df[APP_COL] = df[alt].astype(str)
            break

if APP_COL not in df.columns:
    raise RuntimeError(f"Missing '{APP_COL}' column. Available columns: {list(df.columns)[:30]} ...")

# Clean label + keep only needed columns
df = df.copy().replace("", np.nan)
df[LABEL_COL] = pd.to_numeric(df[LABEL_COL], errors="coerce")
df = df.dropna(subset=[APP_COL, LABEL_COL]).copy()
df[LABEL_COL] = df[LABEL_COL].astype(int)

# Ensure metric cols numeric (HGB pipeline imputes NaNs)
for c in METRIC_COLS:
    if c not in df.columns:
        raise RuntimeError(f"Missing metric column: {c}")
    df[c] = pd.to_numeric(df[c], errors="coerce")

APPS = sorted(df[APP_COL].dropna().unique().tolist())
print(f"#apps={len(APPS)} | apps={APPS}")
print(f"#rows={len(df)} | #metrics={len(METRIC_COLS)} | label={LABEL_COL}")

# 0) Redundancy report
redundant_pairs = metrics_redundancy_report(df, METRIC_COLS, corr_abs_threshold=CORR_ABS_THRESH)
redundant_pairs_path = os.path.join(OUT_DIR, f"redundant_pairs_abs_spearman_ge_{CORR_ABS_THRESH:.2f}.csv")
redundant_pairs.to_csv(redundant_pairs_path, index=False)
print(f"[SAVE] redundant pairs: {len(redundant_pairs)} -> {redundant_pairs_path}")

# 1) Single-metric LOAO leaderboard
single_table = single_metric_study_loao(df, APP_COL, LABEL_COL, METRIC_COLS, apps=APPS, random_state=SEED, positive_label=POS_LABEL)
single_path = os.path.join(OUT_DIR, "single_metric_leaderboard_loao.csv")
single_table.to_csv(single_path, index=False)
print("\nTop single metrics (LOAO-HGB):")
print(single_table.head(10))
print(f"[SAVE] {single_path}")

# 2) Ablations LOAO: ALL, DROP-ONE, ADD-ONE
all_row, drop_one_table, add_one_table = ablation_study_loao(
    df, APP_COL, LABEL_COL, METRIC_COLS,
    apps=APPS, random_state=SEED, positive_label=POS_LABEL,
    top_k_single=TOP_K_SINGLE_FOR_ADDONE
)
all_path = os.path.join(OUT_DIR, "all_metrics_baseline_loao.csv")
drop_path = os.path.join(OUT_DIR, "drop_one_ablation_loao.csv")
add_path  = os.path.join(OUT_DIR, "add_one_pairs_from_top_singles_loao.csv")
all_row.to_csv(all_path, index=False)
drop_one_table.to_csv(drop_path, index=False)
add_one_table.to_csv(add_path, index=False)

print("\nALL-metrics baseline (LOAO-HGB):")
print(all_row)
print("\nMost harmful drops (more negative delta = more important):")
print(drop_one_table.head(15))
print("\nBest 2-metric pairs (seeded from top singles):")
print(add_one_table.head(15))
print(f"[SAVE] {all_path}")
print(f"[SAVE] {drop_path}")
print(f"[SAVE] {add_path}")

# 3) Permutation importance LOAO (train on other apps, permute on held-out app)
perm_imp = permutation_importance_loao(
    df, APP_COL, LABEL_COL, METRIC_COLS,
    apps=APPS, random_state=SEED, positive_label=POS_LABEL,
    n_repeats=PERM_REPEATS
)
perm_path = os.path.join(OUT_DIR, "permutation_importance_loao_f1.csv")
perm_imp.to_csv(perm_path, index=False)
print("\nTop permutation importances (LOAO-HGB):")
print(perm_imp.head(15))
print(f"[SAVE] {perm_path}")

# 4) Sanity: label shuffle (should collapse)
shuf = label_shuffle_sanity_loao(df, APP_COL, LABEL_COL, METRIC_COLS, apps=APPS, random_state=SEED, positive_label=POS_LABEL)
shuf_path = os.path.join(OUT_DIR, "label_shuffle_sanity_loao.json")
with open(shuf_path, "w") as f:
    json.dump(shuf, f, indent=2)
print("\nLabel-shuffle sanity (LOAO-HGB; expect low F1):")
print({k: shuf.get(k) for k in ["mean_f1","mean_acc","micro_f1_from_cm_sum","mean_dup_cross_split_rate","n_apps_used"]})
print(f"[SAVE] {shuf_path}")

print(f"\nAll LOAO outputs saved in: {OUT_DIR}")

[LOAD] Archery      rows=425 from /content/metrics_Archery.csv
[LOAD] PhantomLimb  rows=4105 from /content/metrics_PhantomLimb.csv
[LOAD] PianoTiles   rows=440 from /content/metrics_PianoTiles.csv
[LOAD] Puzzle       rows=300 from /content/metrics_Puzzle.csv
[LOAD] Sea          rows=300 from /content/metrics_Sea.csv
[LOAD] War          rows=281 from /content/metrics_War.csv
[OK] combined df: (5851, 32) | has App? True
#apps=6 | apps=['Archery', 'PhantomLimb', 'PianoTiles', 'Puzzle', 'Sea', 'War']
#rows=5851 | #metrics=27 | label=Spatial
[SAVE] redundant pairs: 2 -> /content/spatial_metrics_study_outputs_loao/redundant_pairs_abs_spearman_ge_0.90.csv


/tmp/ipython-input-2874821201.py:68: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  tr = X_train.copy().astype("object").fillna(sentinel)
/tmp/ipython-input-2874821201.py:69: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  te = X_test.copy().astype("object").fillna(sentinel)
/tmp/ipython-input-2874821201.py:68: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcas


Top single metrics (LOAO-HGB):
                 metric   mean_f1  ci95_f1_lo  ci95_f1_hi  mean_acc  \
0           below_floor  0.727676    0.581440    0.854734  0.702836   
1            bbox_depth  0.659827    0.573558    0.746264  0.622844   
2        body_upright_z  0.639025    0.531898    0.735726  0.632891   
3      center_of_mass_z  0.623569    0.356582    0.831990  0.674774   
4        body_upright_x  0.563515    0.513121    0.628234  0.552165   
5  distance_from_origin  0.533261    0.296508    0.747961  0.552776   
6            bbox_width  0.529294    0.407060    0.664761  0.529938   
7        body_forward_z  0.526360    0.439630    0.661786  0.558484   
8   leg_length_symmetry  0.523295    0.471361    0.575598  0.517433   
9        body_upright_y  0.512392    0.467577    0.562341  0.517293   

   mean_prec  mean_rec  micro_f1  mean_dup_cross_split_rate  n_apps_used  
0   0.715564  0.874794  0.820088                   0.000000            6  
1   0.601598  0.750078  0.606769    

/tmp/ipython-input-2874821201.py:68: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  tr = X_train.copy().astype("object").fillna(sentinel)
/tmp/ipython-input-2874821201.py:69: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  te = X_test.copy().astype("object").fillna(sentinel)
/tmp/ipython-input-2874821201.py:68: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcas


ALL-metrics baseline (LOAO-HGB):
        setting   mean_f1  ci95_f1_lo  ci95_f1_hi  mean_acc  mean_prec  \
0  ALL_LOAO_HGB  0.589371     0.32938    0.793902   0.64783   0.625136   

   mean_rec  micro_f1  mean_dup_cross_split_rate  n_metrics  n_apps_used  
0   0.65625   0.72734                        0.0         27            6  

Most harmful drops (more negative delta = more important):
                 dropped_metric   mean_f1  delta_vs_all_f1  mean_acc  micro_f1
0           distance_from_floor  0.467362        -0.122009  0.562311  0.624609
1              center_of_mass_z  0.551162        -0.038208  0.619089  0.734758
2          distance_from_origin  0.557499        -0.031872  0.602554  0.714626
3                body_upright_z  0.563354        -0.026017  0.618010  0.716104
4              center_of_mass_x  0.572381        -0.016989  0.629417  0.721711
5                    bbox_width  0.572833        -0.016537  0.621996  0.724837
6              center_of_mass_y  0.573859        -0.01

/tmp/ipython-input-2874821201.py:68: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  tr = X_train.copy().astype("object").fillna(sentinel)
/tmp/ipython-input-2874821201.py:69: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  te = X_test.copy().astype("object").fillna(sentinel)
/tmp/ipython-input-2874821201.py:68: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcas


Label-shuffle sanity (LOAO-HGB; expect low F1):
{'mean_f1': 0.2990048675131587, 'mean_acc': 0.5097664315793536, 'micro_f1_from_cm_sum': 0.4988954970263381, 'mean_dup_cross_split_rate': 0.0, 'n_apps_used': 6}
[SAVE] /content/spatial_metrics_study_outputs_loao/label_shuffle_sanity_loao.json

All LOAO outputs saved in: /content/spatial_metrics_study_outputs_loao
